# Exact inference: SmolLM2-135M

This notebook downloads the language model SmolLM2-135M and 64 prompts at pinned revisions and checks them by SHA-256. It runs every prompt through an integer version of the model, in which the only floating-point operation is a product of integer matrices small enough to be exact in double precision, and through a plain float32 version of the same weights. It prints how close the two are: top-1 agreement over every prompt position, the logit differences, and how many of the 64 greedy continuations of 48 tokens are the same. The last cell prints the SHA-256 of the integer logits and continuations with MATCH or DIFFERENT against the hashes in the repository, then the SHA-256 of the float32 ones on this machine.


In [ ]:
# The folder's code and tables, written here and checked by SHA-256.
import hashlib, os
FILES = {}
FILES['exact_llm.py'] = ('b0decd062c08c9cbdf6ac6f1a8630c38eec9eb222fa246d7ade7d13b7ff5e461', '"""Exact inference for a language model: HuggingFaceTB/SmolLM2-135M (and -360M, -1.7B) in integer arithmetic.\n\nThe method of proofs/exact-embeddings on a decoder. Every tensor is an integer array with a power-of-two scale;\nelement-wise work is NumPy int64; the only floating-point operation is matrix multiplication by the machine\'s\nfloat64 BLAS on integer-valued arrays whose partial sums stay below 2^53 (bounds.py). exp, SiLU and the rotary cos\nand sin are integer tables (tables/, make_tables.py). The output is the next-token logits at every position, as\nintegers at scale 2^-16, and greedy continuations computed from them.\n\nThe same file holds the pinned downloads, a byte-level BPE tokenizer in plain Python, a plain float32 forward pass\nof the same weights (the baseline), an onnxruntime baseline on the onnx-community export, and the evaluation.\n\nThe model is a block of pins.json, chosen by --model (or the environment variable EXACT_LLM_MODEL); its sizes come\nfrom its pinned config.json. SmolLM2-135M is the default and runs every engine; the larger two run exact-16, the\nfloat32 baseline and onnxruntime where an export is pinned.\n\nusage:\n  python exact_llm.py check [--model M] [--routes f64,f32,i8] [--no-float] [--no-onnx] [--peaks FILE]\n                                                  digests, the hashes against expected*.sha256, baselines, distance\n  python exact_llm.py hash --engine exact|exact16-f64|exact16-f32|exact16-i8|float32|onnx [--batch N] [--threads N]\n                           [--model M]\n"""\nimport gzip\nimport hashlib\nimport json\nimport math\nimport os\nimport struct\nimport sys\nimport time\nimport unicodedata\nimport urllib.request\nfrom fractions import Fraction\n\nimport numpy as np\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nPINS = json.load(open(os.path.join(HERE, "pins.json")))\nCACHE = os.environ.get("EXACT_LLM_CACHE", os.path.join(HERE, ".cache"))\nMODELS = tuple(PINS["models"])                  # the first is the default\nGEN = 48                                        # greedy continuation length\n\n\n# ---- downloads ------------------------------------------------------------------------------------------------\ndef sha256_file(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for chunk in iter(lambda: f.read(1 << 20), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef fetch(url, path, digest):\n    if not (os.path.exists(path) and sha256_file(path) == digest):\n        os.makedirs(os.path.dirname(path), exist_ok=True)\n        req = urllib.request.Request(url, headers={"User-Agent": "exact-llm"})\n        with urllib.request.urlopen(req) as r, open(path + ".part", "wb") as f:\n            while True:\n                chunk = r.read(1 << 20)\n                if not chunk:\n                    break\n                f.write(chunk)\n        os.replace(path + ".part", path)\n    got = sha256_file(path)\n    if got != digest:\n        raise SystemExit("digest mismatch for %s: %s, pinned %s" % (path, got, digest))\n    return path\n\n\ndef model_file(name):\n    return fetch("https://huggingface.co/%s/resolve/%s/%s" % (PIN["repo"], PIN["revision"], name),\n                 os.path.join(MCACHE, "model", name), PIN["files"][name])\n\n\ndef use_model(name):\n    """Select a model of pins.json and take its sizes from its pinned config.json. The default model\'s files stay at\n    the top of the cache; another model\'s go in a folder of its own name. Asserts what the code assumes: a Llama\n    decoder with SiLU, tied embedding, no biases, rotary positions on halves of 64-wide heads, no rope scaling."""\n    global MODEL, PIN, MCACHE, LAYERS, HIDDEN, HEADS, KV, HEAD, FF, VOCAB, THETA, EPS, NORM_BITS, RMS_Y, SUFFIX\n    if name not in PINS["models"]:\n        raise SystemExit("unknown model %s; pins.json has %s" % (name, ", ".join(MODELS)))\n    MODEL, PIN = name, PINS["models"][name]\n    MCACHE = CACHE if name == MODELS[0] else os.path.join(CACHE, name)\n    SUFFIX = "" if name == MODELS[0] else "-" + name                # expected-16<SUFFIX>.sha256, bounds-16<SUFFIX>.json\n    c = json.load(open(model_file("config.json"), encoding="utf-8"))\n    LAYERS, HIDDEN = c["num_hidden_layers"], c["hidden_size"]\n    HEADS, KV = c["num_attention_heads"], c["num_key_value_heads"]\n    HEAD = c.get("head_dim") or HIDDEN // HEADS\n    FF, VOCAB, THETA = c["intermediate_size"], c["vocab_size"], c["rope_theta"]\n    EPS = Fraction(repr(c["rms_norm_eps"]))                          # 1e-05 as the decimal it is written as\n    assert c["model_type"] == "llama" and c["hidden_act"] == "silu" and c["tie_word_embeddings"], name\n    assert not c.get("attention_bias") and not c.get("mlp_bias") and not c.get("rope_scaling"), name\n    assert not c.get("rope_interleaved", False) and HEAD == 64 and HEADS * HEAD == HIDDEN and HEADS % KV == 0, name\n    # RMS norm: x is shifted right until max |x| < 2^NORM_BITS, so the sum of HIDDEN squares stays below 2^62\n    NORM_BITS = (62 - (HIDDEN - 1).bit_length()) // 2\n    # every normalised value is at most sqrt(HIDDEN) times the root mean square; with the floors of the integer\n    # mean, square root and eps, at most sqrt(HIDDEN) (1 + 1/128): RMS_Y is the least integer at or above that\n    RMS_Y = math.isqrt(HIDDEN * 129 * 129 // (128 * 128))\n    while (128 * RMS_Y) ** 2 < HIDDEN * 129 * 129:\n        RMS_Y += 1\n\n\nuse_model(os.environ.get("EXACT_LLM_MODEL", MODELS[0]))\n\n\ndef prompts():\n    p = PINS["prompts"]\n    path = fetch("https://huggingface.co/datasets/%s/resolve/%s/%s" % (p["repo"], p["revision"], p["file"]),\n                 os.path.join(CACHE, "data", p["file"]), p["sha256"])\n    rows = [json.loads(line) for line in gzip.open(path, "rt", encoding="utf-8")]\n    return [r["sentence1"] for r in rows[:64]]\n\n\ndef text():\n    t = PINS["text"]\n    return open(fetch(t["url"], os.path.join(CACHE, "data", "input.txt"), t["sha256"]), encoding="utf-8").read()\n\n\n# ---- safetensors, parsed here; bfloat16 widened to float32 exactly (the 16 bits become the high half) -------------\ndef iter_safetensors(path, names=None, raw=False):\n    """Yield (name, float32 array) one tensor at a time, reading only that tensor\'s bytes. names: a collection of\n    names, or a function of (name, shape) that says which tensors to read. raw: the bfloat16 bits as uint16."""\n    with open(path, "rb") as f:\n        n = struct.unpack("<Q", f.read(8))[0]\n        header = json.loads(f.read(n).decode("utf-8"))\n        header.pop("__metadata__", None)\n        for name, t in header.items():\n            if names is not None and not (names(name, t["shape"]) if callable(names) else name in names):\n                continue\n            assert t["dtype"] == "BF16", (name, t["dtype"])\n            b, e = t["data_offsets"]\n            f.seek(8 + n + b)\n            bits = np.frombuffer(f.read(e - b), dtype="<u2").reshape(t["shape"])\n            yield name, bits if raw else widen(bits)\n\n\ndef widen(bits):\n    """bfloat16 bits (uint16) to float32, exactly: the 16 bits become the high half."""\n    return (bits.astype(np.uint32) << 16).view(np.float32)\n\n\ndef read_safetensors(path, names=None, raw=False):\n    return dict(iter_safetensors(path, names, raw))\n\n\n# ---- tokenizer: byte-level BPE, from the pinned tokenizer.json --------------------------------------------------\n# Pre-tokenizer: Digits (each digit alone), then ByteLevel with the GPT-2 pattern\n#   \'s|\'t|\'re|\'ve|\'m|\'ll|\'d| ?\\p{L}+| ?\\p{N}+| ?[^\\s\\p{L}\\p{N}]+|\\s+(?!\\S)|\\s+\n# written out below with unicodedata (Python\'s re has no \\p{L}). \\s is the Unicode White_Space set.\nWHITE = set("\\t\\n\\x0b\\x0c\\r \\x85\\xa0\u1680\u2028\u2029\u202f\u205f\u3000") | {chr(c) for c in range(0x2000, 0x200B)}\n\n\ndef _L(c):\n    return unicodedata.category(c)[0] == "L"\n\n\ndef _N(c):\n    return unicodedata.category(c)[0] == "N"\n\n\ndef _other(c):\n    return c not in WHITE and not _L(c) and not _N(c)\n\n\ndef gpt2_split(s):\n    out, i, n = [], 0, len(s)\n    while i < n:\n        c = s[i]\n        if c == "\'":\n            hit = next((suf for suf in ("s", "t", "re", "ve", "m", "ll", "d") if s.startswith(suf, i + 1)), None)\n            if hit:\n                out.append("\'" + hit)\n                i += 1 + len(hit)\n                continue\n        done = False\n        for cls in (_L, _N, _other):\n            j = i + 1 if (c == " " and i + 1 < n and cls(s[i + 1])) else i\n            if cls(s[j]):\n                k = j\n                while k < n and cls(s[k]):\n                    k += 1\n                out.append(s[i:k])\n                i, done = k, True\n                break\n        if done:\n            continue\n        k = i                                   # c is whitespace\n        while k < n and s[k] in WHITE:\n            k += 1\n        if k == n or k - i == 1:\n            out.append(s[i:k])\n            i = k\n        else:\n            out.append(s[i:k - 1])              # \\s+(?!\\S): leave the last one for the next word\n            i = k - 1\n    return out\n\n\ndef digits_split(s):\n    out, cur = [], ""\n    for c in s:\n        if _N(c):\n            if cur:\n                out.append(cur)\n                cur = ""\n            out.append(c)\n        else:\n            cur += c\n    if cur:\n        out.append(cur)\n    return out\n\n\ndef _byte_encoder():\n    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("\\xa1"), ord("\\xac") + 1)) + list(range(ord("\\xae"), 256))\n    cs, n = bs[:], 0\n    for b in range(256):\n        if b not in bs:\n            bs.append(b)\n            cs.append(256 + n)\n            n += 1\n    return {b: chr(c) for b, c in zip(bs, cs)}\n\n\nclass Tokenizer:\n    def __init__(self, path):\n        t = json.load(open(path, encoding="utf-8"))\n        assert t["normalizer"] is None and t["post_processor"] is None and t["model"]["type"] == "BPE"\n        self.vocab = t["model"]["vocab"]\n        self.rank = {tuple(m.split(" ")) if isinstance(m, str) else tuple(m): i for i, m in enumerate(t["model"]["merges"])}\n        self.added = sorted((a["content"] for a in t["added_tokens"]), key=len, reverse=True)\n        self.added_id = {a["content"]: a["id"] for a in t["added_tokens"]}\n        self.enc = _byte_encoder()\n        self.memo = {}\n\n    def bpe(self, word):\n        if word in self.memo:\n            return self.memo[word]\n        parts = list(word)\n        while len(parts) > 1:\n            best = min(((self.rank.get((parts[i], parts[i + 1]), 1 << 30), i) for i in range(len(parts) - 1)))\n            if best[0] == 1 << 30:\n                break\n            a, b = parts[best[1]], parts[best[1] + 1]\n            merged, i = [], 0\n            while i < len(parts):\n                if i < len(parts) - 1 and parts[i] == a and parts[i + 1] == b:\n                    merged.append(a + b)\n                    i += 2\n                else:\n                    merged.append(parts[i])\n                    i += 1\n            parts = merged\n        self.memo[word] = parts\n        return parts\n\n    def encode(self, s):\n        ids, i, start = [], 0, 0\n        chunks = []\n        while i < len(s):                       # added (special) tokens are cut out first, longest first\n            hit = next((a for a in self.added if s.startswith(a, i)), None)\n            if hit:\n                chunks.append((s[start:i], False))\n                chunks.append((hit, True))\n                i += len(hit)\n                start = i\n            else:\n                i += 1\n        chunks.append((s[start:], False))\n        for chunk, special in chunks:\n            if special:\n                ids.append(self.added_id[chunk])\n                continue\n            for piece in digits_split(chunk):\n                for word in gpt2_split(piece):\n                    for tok in self.bpe("".join(self.enc[b] for b in word.encode("utf-8"))):\n                        ids.append(self.vocab[tok])\n        return ids\n\n\ndef tokenizer():\n    return Tokenizer(model_file("tokenizer.json"))\n\n\n# ---- the float32 baseline: a plain forward pass of the same weights, with a key-value cache -----------------------\ndef n_params():\n    """Weights of the model, from its config: embedding (tied), per layer q, k, v, o, gate, up, down and two norms."""\n    attn = HIDDEN * (2 * HIDDEN + 2 * KV * HEAD)\n    return VOCAB * HIDDEN + LAYERS * (attn + 3 * HIDDEN * FF + 2 * HIDDEN) + HIDDEN\n\n\nLEAN_F32_BYTES = 1 << 32     # float32 weights above 4 GB are kept as bfloat16 and widened one matrix at a time\n\n\nclass _Layer(dict):\n    """One layer\'s float32 weights; a matrix kept as bfloat16 bits is widened to float32 [in, out] at each read."""\n    def __getitem__(self, k):\n        v = dict.__getitem__(self, k)\n        return np.ascontiguousarray(widen(v).T) if v.dtype == np.uint16 else v\n\n\nclass Float32:\n    def __init__(self, P=None, stats=None, lean=None):\n        """P: the weights as float32, or None to read them here. lean: keep the matrices as bfloat16 bits and widen\n        each at its use, giving BLAS the same float32 arrays (so the same results) in half the memory; by default\n        when the float32 weights would exceed LEAN_F32_BYTES."""\n        if P is None:\n            self.lean = 4 * n_params() > LEAN_F32_BYTES if lean is None else lean\n            P = read_safetensors(model_file("model.safetensors"), raw=self.lean)\n        else:\n            self.lean = False\n        f32 = lambda a: widen(a) if a.dtype == np.uint16 else a\n        self.E = f32(P["model.embed_tokens.weight"])\n        self.norm = f32(P["model.norm.weight"])\n        self.L = []\n        for i in range(LAYERS):\n            p = "model.layers.%d." % i\n            T = lambda n: P[p + n + ".weight"] if self.lean else np.ascontiguousarray(P[p + n + ".weight"].T)\n            self.L.append(_Layer({"q": T("self_attn.q_proj"), "k": T("self_attn.k_proj"),\n                                  "v": T("self_attn.v_proj"), "o": T("self_attn.o_proj"), "g": T("mlp.gate_proj"),\n                                  "u": T("mlp.up_proj"), "d": T("mlp.down_proj"),\n                                  "n1": f32(P[p + "input_layernorm.weight"]),\n                                  "n2": f32(P[p + "post_attention_layernorm.weight"])}))\n        del P\n        inv = (np.float32(1.0) / (np.float32(THETA) ** (np.arange(0, HEAD, 2, dtype=np.float32) / np.float32(HEAD))))\n        self.inv_freq = inv.astype(np.float32)\n        self.stats = stats\n\n    def site(self, name, i, x):\n        if self.stats is not None:\n            k = (name, i)\n            self.stats[k] = max(self.stats.get(k, 0.0), float(np.abs(x).max()))\n\n    def rms(self, x, w):\n        var = (x * x).mean(-1, keepdims=True)\n        return (x / np.sqrt(var + np.float32(float(EPS)))) * w\n\n    def rope(self, x, pos):\n        f = pos[..., None].astype(np.float32) * self.inv_freq                  # [B, L, 32]\n        cos = np.cos(np.concatenate([f, f], -1))[:, None]\n        sin = np.sin(np.concatenate([f, f], -1))[:, None]\n        rot = np.concatenate([-x[..., HEAD // 2:], x[..., :HEAD // 2]], -1)\n        return x * cos + rot * sin\n\n    def forward(self, ids, pos, cache, kvalid):\n        """ids, pos: [B, L]; cache: per layer (K, V) arrays [B, KV, T, 64] or None; kvalid: [B, L, Tk] bool."""\n        B, L = ids.shape\n        x = self.E[ids]\n        new = []\n        for i, w in enumerate(self.L):\n            self.site("residual", i, x)\n            h = self.rms(x, w["n1"])\n            self.site("attention input", i, h)\n            q = (h @ w["q"]).reshape(B, L, HEADS, HEAD).transpose(0, 2, 1, 3)\n            k = (h @ w["k"]).reshape(B, L, KV, HEAD).transpose(0, 2, 1, 3)\n            v = (h @ w["v"]).reshape(B, L, KV, HEAD).transpose(0, 2, 1, 3)\n            q, k = self.rope(q, pos), self.rope(k, pos)\n            self.site("q", i, q)\n            self.site("k", i, k)\n            self.site("v", i, v)\n            if cache is not None:\n                k = np.concatenate([cache[i][0], k], 2)\n                v = np.concatenate([cache[i][1], v], 2)\n            new.append((k, v))\n            kk = np.repeat(k, HEADS // KV, axis=1)\n            vv = np.repeat(v, HEADS // KV, axis=1)\n            s = (q @ kk.swapaxes(-1, -2)) / np.float32(8.0)\n            s = np.where(kvalid[:, None], s, np.float32(-np.inf))\n            s = np.exp(s - s.max(-1, keepdims=True))\n            ctx = ((s / s.sum(-1, keepdims=True)) @ vv).transpose(0, 2, 1, 3).reshape(B, L, HIDDEN)\n            self.site("o input", i, ctx)\n            x = x + ctx @ w["o"]\n            self.site("residual mid", i, x)\n            h = self.rms(x, w["n2"])\n            self.site("mlp input", i, h)\n            g, u = h @ w["g"], h @ w["u"]\n            self.site("gate", i, g)\n            self.site("up", i, u)\n            a = (g / (np.float32(1.0) + np.exp(-g))) * u\n            self.site("down input", i, a)\n            x = x + a @ w["d"]\n        self.site("residual", LAYERS, x)\n        h = self.rms(x, self.norm)\n        self.site("logits input", 0, h)\n        return h @ self.E.T, new\n\n\n# ---- the onnxruntime baseline: the onnx-community float32 export of the same model --------------------------------\ndef onnx_file(name):\n    o = PIN["onnx"]\n    return fetch("https://huggingface.co/%s/resolve/%s/%s" % (o["repo"], o["revision"], name),\n                 os.path.join(MCACHE, "onnx", name), o["files"][name])\n\n\nclass Onnx:\n    def __init__(self, threads=0):\n        import onnxruntime as ort\n        if PIN["onnx"] is None:\n            raise LookupError(PIN.get("onnx_note", "no onnx export pinned for " + MODEL))\n        for name in PIN["onnx"]["files"]:                                    # the graph and its external data, if any\n            onnx_file(name)\n        so = ort.SessionOptions()\n        if threads:\n            so.intra_op_num_threads = threads\n            so.inter_op_num_threads = 1\n        self.s = ort.InferenceSession(onnx_file("onnx/model.onnx"), so, providers=["CPUExecutionProvider"])\n        self.inputs = [i.name for i in self.s.get_inputs()]\n        self.outputs = [o.name for o in self.s.get_outputs()]\n\n    def forward(self, ids, pos, cache, kvalid):\n        """The model applies its own causal mask; it is given the keys each row may read (the last row of kvalid)."""\n        B = ids.shape[0]\n        feed = {"input_ids": ids.astype(np.int64), "attention_mask": kvalid[:, -1, :].astype(np.int64),\n                "position_ids": np.ascontiguousarray(pos, dtype=np.int64)}\n        for i in range(LAYERS):\n            k, v = cache[i] if cache is not None else (np.zeros((B, KV, 0, HEAD), np.float32),) * 2\n            feed["past_key_values.%d.key" % i] = k\n            feed["past_key_values.%d.value" % i] = v\n        res = dict(zip(self.outputs, self.s.run(None, {k: v for k, v in feed.items() if k in self.inputs})))\n        new = [(res["present.%d.key" % i], res["present.%d.value" % i]) for i in range(LAYERS)]\n        return res["logits"], new\n\n\n# ---- the exact model ----------------------------------------------------------------------------------------------\nA = 16                       # activations: integer x stands for x * 2^-16\nWB = 16                      # weights: integer w stands for w * 2^-WB (bfloat16 values rounded once)\nCLAMP = {                    # clamps in real units, per site; see bounds.py and the measured peaks\n    "residual": 2 ** 16, "attention input": 2 ** 12, "q": 2 ** 7, "k": 2 ** 7, "v": 2 ** 12, "o input": 2 ** 12,\n    "mlp input": 2 ** 11, "gate": 2 ** 12, "up": 2 ** 12, "down input": 2 ** 13, "logits input": 2 ** 14}\nSPLIT = {"down input": 256}  # the down product\'s 1,536 inputs in chunks of 256: each chunk exact in float64, summed in int64\n# Each clamp is the largest power of two the certificate allows (bounds.py): a product\'s input clamp times the largest\n# row sum of its integer weights below 2^53; q and k with 64 C_q C_k below 2^53. The measured peaks and the margin\n# clamp / peak are in the README; the number of clamped values is counted on every run.\nEXP_STEP, EXP_N = 8, 24 * 256\nSILU_STEP, SILU_LO, SILU_N = 8, 16, 32 * 256\nROPE = 20                    # cos and sin at 2^-20\nROPE_POS = 512\n# NORM_BITS and RMS_Y come from the width, in use_model: 26 and 25 at 576 (135M), 26 and 32 at 960, 25 and 46 at 2,048\n\n\ndef rshift_round(v, k):\n    """floor(v / 2^k + 1/2), round to nearest with halves up; k = 0 leaves v."""\n    return (v + (1 << (k - 1))) >> k if k > 0 else v\n\n\ndef isqrt(n):\n    """floor(sqrt(n)) for 0 <= n < 2^62, asserted by its bracket s^2 <= n < (s + 1)^2."""\n    assert (n >= 0).all() and (n < (1 << 62)).all()\n    s = np.sqrt(n.astype(np.float64)).astype(np.int64)\n    for _ in range(4):\n        s = np.where(s * s > n, s - 1, s)\n        s = np.where((s + 1) * (s + 1) <= n, s + 1, s)\n    assert ((s * s <= n) & ((s + 1) * (s + 1) > n)).all()\n    return s\n\n\ndef fdiv(num, den):\n    """floor(num / den) for den > 0, asserted by its bracket q den <= num < (q + 1) den."""\n    den = np.asarray(den, dtype=np.int64)\n    assert (den > 0).all()\n    q = num // den\n    assert ((q * den <= num) & (num < (q + 1) * den)).all()\n    return q\n\n\ndef load_table(name, shape=None):\n    with open(os.path.join(HERE, "tables", name)) as f:\n        t = np.array([int(line) for line in f if not line.startswith("#")], dtype=np.int64)\n    return t.reshape(shape) if shape else t\n\n\ndef int_weights(wb=None, small=False):\n    """The integer weights: round(w * 2^WB) of each bfloat16 value, ties to even (exact arithmetic plus one rounding;\n    most values need no rounding). Computed once from model.safetensors and kept as int32 .npy files in the cache,\n    transposed to [in, out]; opened memory-mapped. small: only the embedding and the norm weights, which is all\n    exact-16 reads at this scale (its products use int16_weights)."""\n    wb = WB if wb is None else wb\n    d = os.path.join(MCACHE, "int32-wb%d%s" % (wb, "-small" if small else ""))\n    done = os.path.join(d, "DONE")\n    want = (lambda name, shape: len(shape) == 1 or name == "model.embed_tokens.weight") if small else None\n    if not os.path.exists(done):\n        os.makedirs(d, exist_ok=True)\n        for name, a in iter_safetensors(model_file("model.safetensors"), want):\n            if a.ndim == 2:\n                a = a.T if name != "model.embed_tokens.weight" else a          # [in, out]; the embedding stays [V, 576]\n                out = np.empty(a.shape, dtype=np.int32)\n                for s in range(0, a.shape[0], 4096):\n                    v = np.rint(a[s:s + 4096].astype(np.float64) * 2.0 ** wb)\n                    assert np.abs(v).max() < 2 ** 31\n                    out[s:s + 4096] = v.astype(np.int32)\n            else:\n                out = np.rint(a.astype(np.float64) * 2.0 ** A).astype(np.int64)  # norm weights at 2^-16\n            np.save(os.path.join(d, name + ".npy"), np.ascontiguousarray(out))\n        open(done, "w").write("from %s\\n" % PIN["files"]["model.safetensors"])\n    return {f[:-4]: np.load(os.path.join(d, f), mmap_mode="r") for f in os.listdir(d) if f.endswith(".npy")}\n\n\ndef rope_table_names():\n    """cos and sin tables of the model\'s rotary base (make_tables.py): the 135M\'s 100,000 keeps its first names."""\n    return ("cos.txt", "sin.txt") if THETA == 100000 else ("cos-%d.txt" % THETA, "sin-%d.txt" % THETA)\n\n\nclass Exact:\n    def __init__(self, W=None, keep_f64=False, clamps=None):\n        self.W = W if W is not None else int_weights()\n        self.E = self.W["model.embed_tokens.weight"]                          # [V, HIDDEN] at 2^-WB\n        self.Et = load_table("exp.txt")\n        self.St = load_table("silu.txt")\n        cname, sname = rope_table_names()\n        self.cos = load_table(cname, (ROPE_POS, HEAD // 2))\n        self.sin = load_table(sname, (ROPE_POS, HEAD // 2))\n        self.clamps = CLAMP if clamps is None else clamps                    # real units, per site\n        self.C = {k: v << A for k, v in self.clamps.items()}\n        self.clamped, self.peak, self.layer_peak = {}, {}, {}\n        self.keep = keep_f64\n        self.f64 = {}\n        self.eps = [int(round(EPS * 2 ** (2 * A - 2 * s))) for s in range(A + 1)]   # eps at 2^-(32 - 2s)\n\n    def clamp(self, x, site, layer=None):\n        c = self.C[site]\n        a = np.abs(x)\n        n = int((a > c).sum())\n        if n:\n            self.clamped[site] = self.clamped.get(site, 0) + n\n        top = int(a.max()) if a.size else 0\n        self.peak[site] = max(self.peak.get(site, 0), top)\n        if layer is not None:\n            self.layer_peak[(site, layer)] = max(self.layer_peak.get((site, layer), 0), top)\n        return np.clip(x, -c, c)\n\n    def matrix(self, name):\n        """A weight matrix as float64 [in, out], converted from int32 (exactly). Kept for the run when keep_f64,\n        else converted at each use, one matrix at a time."""\n        w = self.f64.get(name)\n        if w is None:\n            w = np.asarray(self.W[name], dtype=np.float64)\n            if self.keep:\n                self.f64[name] = w\n        return w\n\n    def linear(self, x, name, site):\n        xc = self.clamp(x, site).astype(np.float64)\n        w = self.matrix(name)\n        n = SPLIT.get(site, w.shape[0])\n        y = sum((xc[:, s:s + n] @ w[s:s + n]).astype(np.int64) for s in range(0, w.shape[0], n))  # exact: bounds.py\n        return rshift_round(y, WB)\n\n    def rms(self, x, wname):\n        """RMS normalisation of each token (last axis, HIDDEN values), x and result at 2^-16."""\n        top = np.abs(x).max(-1, keepdims=True)\n        sh = np.maximum(0, np.frexp(top.astype(np.float64))[1] - NORM_BITS).astype(np.int64)  # bit length - NORM_BITS\n        xs = x >> sh\n        ss = (xs * xs).sum(-1, keepdims=True)                                 # at 2^-(32 - 2 sh), below 2^62\n        ms = fdiv(ss, HIDDEN)\n        assert (sh <= A).all()\n        eps = np.array(self.eps, dtype=np.int64)[sh]\n        r = isqrt(ms + np.maximum(eps, 1))                                    # sqrt(mean x^2 + eps) at 2^-(16 - sh)\n        y = fdiv(x << (A - sh), r)                                            # x / r at 2^-16\n        assert (np.abs(y) <= RMS_Y << A).all()                                # |x_i| <= sqrt(HIDDEN) rms: bounds.py\n        return rshift_round(y * np.asarray(self.W[wname]), A)\n\n    def rope_tables(self, pos):\n        """cos and sin of each position, read once per forward pass: [B, 1, L, 32] at 2^-20."""\n        return self.cos[pos][:, None], self.sin[pos][:, None]\n\n    def rope(self, x, cs):\n        c, s = cs\n        x1, x2 = x[..., :HEAD // 2], x[..., HEAD // 2:]\n        return np.concatenate([rshift_round(x1 * c - x2 * s, ROPE), rshift_round(x2 * c + x1 * s, ROPE)], -1)\n\n    def exp(self, t):\n        k = np.minimum(t >> EXP_STEP, EXP_N)\n        r = t & ((1 << EXP_STEP) - 1)\n        return (self.Et[k] * ((1 << EXP_STEP) - r) + self.Et[np.minimum(k + 1, EXP_N)] * r) >> EXP_STEP\n\n    def silu(self, x):\n        top = SILU_N << SILU_STEP\n        u = x + (SILU_LO << A)\n        uc = np.clip(u, 0, top - 1)\n        k, r = uc >> SILU_STEP, uc & ((1 << SILU_STEP) - 1)\n        g = (self.St[k] * ((1 << SILU_STEP) - r) + self.St[k + 1] * r) >> SILU_STEP\n        return np.where((u >= 0) & (u < top), g, np.where(u < 0, 0, x))\n\n    def cache_entry(self, k, v):\n        """What the key-value cache keeps for each new position: k and v clamped. Done once per position; the\n        cache is read at every later step."""\n        return self.clamp(k, "k"), self.clamp(v, "v")\n\n    def attention(self, q, k, v, kvalid):\n        """q [B, 9, L, 64]; k, v [B, 3, T, 64], clamped (cache_entry); head h reads key-value head h // 3. Scores at\n        2^-32; the scale 1/sqrt(64) = 2^-3 is the 3 in the shift by 19."""\n        B, _, L, _ = q.shape\n        qc = self.clamp(q, "q").astype(np.float64).reshape(B, KV, HEADS // KV, L, HEAD)\n        kc = k.astype(np.float64)[:, :, None]\n        vc = v.astype(np.float64)[:, :, None]\n        s = (qc @ kc.swapaxes(-1, -2)).astype(np.int64)                     # [B, 3, 3, L, T]\n        key = kvalid[:, None, None]\n        m = np.where(key, s, -(1 << 62)).max(-1, keepdims=True)\n        t = np.maximum(m - s, 0) >> (A + 3)\n        e = np.where(key, self.exp(t), 0)\n        p = fdiv(e << A, e.sum(-1, keepdims=True))\n        ctx = (p.astype(np.float64) @ vc).astype(np.int64)                   # at 2^-32\n        return rshift_round(ctx, A).reshape(B, HEADS, L, HEAD)\n\n    def forward(self, ids, pos, cache, kvalid):\n        B, L = ids.shape\n        x = rshift_round(np.asarray(self.E[ids], dtype=np.int64), WB - A)\n        cs = self.rope_tables(pos)\n        new = []\n        for i in range(LAYERS):\n            p = "model.layers.%d." % i\n            x = self.clamp(x, "residual", i)\n            h = self.rms(x, p + "input_layernorm.weight").reshape(B * L, HIDDEN)\n            q = self.linear(h, p + "self_attn.q_proj.weight", "attention input").reshape(B, L, HEADS, HEAD)\n            k = self.linear(h, p + "self_attn.k_proj.weight", "attention input").reshape(B, L, KV, HEAD)\n            v = self.linear(h, p + "self_attn.v_proj.weight", "attention input").reshape(B, L, KV, HEAD)\n            q = self.rope(q.transpose(0, 2, 1, 3), cs)\n            k = self.rope(k.transpose(0, 2, 1, 3), cs)\n            kv = self.cache_entry(k, v.transpose(0, 2, 1, 3))                 # each array [B, KV, L, ...]\n            if cache is not None:\n                kv = tuple(np.concatenate([c, e], 2) for c, e in zip(cache[i], kv))\n            new.append(kv)\n            ctx = self.attention(q, *kv, kvalid).transpose(0, 2, 1, 3).reshape(B * L, HIDDEN)\n            x = x + self.linear(ctx, p + "self_attn.o_proj.weight", "o input").reshape(B, L, HIDDEN)\n            x = self.clamp(x, "residual", i)\n            h = self.rms(x, p + "post_attention_layernorm.weight").reshape(B * L, HIDDEN)\n            g = self.clamp(self.linear(h, p + "mlp.gate_proj.weight", "mlp input"), "gate")\n            u = self.clamp(self.linear(h, p + "mlp.up_proj.weight", "mlp input"), "up")\n            a = rshift_round(self.silu(g) * u, A)\n            x = x + self.linear(a, p + "mlp.down_proj.weight", "down input").reshape(B, L, HIDDEN)\n        x = self.clamp(x, "residual", LAYERS)\n        h = self.rms(x, "model.norm.weight").reshape(B * L, HIDDEN)\n        return self.logits(h).reshape(B, L, VOCAB), new\n\n    def logits(self, h):\n        """h @ E^T at 2^-16, the embedding converted in blocks of 8,192 rows."""\n        hc = self.clamp(h, "logits input").astype(np.float64)\n        out = np.empty((h.shape[0], VOCAB), dtype=np.int32)\n        for s in range(0, VOCAB, 8192):\n            e = np.asarray(self.E[s:s + 8192], dtype=np.float64)\n            blk = rshift_round((hc @ e.T).astype(np.int64), WB)\n            assert np.abs(blk).max() < (1 << 31)\n            out[:, s:s + 8192] = blk\n        return out\n\n\n# ---- exact-16: every product a 16-bit by 16-bit integer product -------------------------------------------------\n# Every product operand is a signed 16-bit integer with its own power-of-two scale; everything between products is\n# Exact\'s. Product inputs: one shift per token vector (block16). Weights: one scale 2^g per output row, from the\n# bfloat16 values (int16_weights). q, k: one shift per head vector. v: whole, as two 16-bit halves 2^15 vh + vl\n# (two products, no shift, so nothing depends on other keys or later positions). p: at 2^-15, at most 2^15 - 1.\n# The PEEL16 largest |x| of each product input are kept whole and multiply the same 16-bit weights; each output is\n# rounded once. The integer products are exact on three routes (mm16), and the routes give the same bits.\nB16 = 16\nLIM16 = (1 << (B16 - 1)) - 1      # 32,767\nPEEL16 = 2\nCHUNK32 = 1024                    # f32 route: inputs per chunk, 1,024 x 2^14 = 2^24\nN_I8 = 33025                      # i8 route: n 255^2 < 2^31 for n <= 33,025\nROUTES = ("f64", "f32", "i8")\n\n\ndef shift_round(v, k):\n    """round(v 2^-k), halves up, for integers k >= 0 (an array broadcast against v); k = 0 leaves v."""\n    k = np.asarray(k, dtype=np.int64)\n    assert (k >= 0).all()\n    return (v + (np.left_shift(1, k) >> 1)) >> k\n\n\ndef bitlen(a):\n    """Bit length of each integer 0 <= a < 2^53 (frexp is exact there)."""\n    return np.frexp(a.astype(np.float64))[1].astype(np.int64)\n\n\ndef block16(x, axes=-1):\n    """x int64 -> (x16, sh, n): x16 = round(x 2^-sh) with |x16| <= 2^15 - 1, sh >= 0 the least shift that fits, one\n    shift per vector along `axes`; n counts the vectors whose rounding needed one more shift."""\n    top = np.abs(x).max(axis=axes, keepdims=True)\n    sh = np.maximum(0, bitlen(top) - (B16 - 1))\n    xb = shift_round(x, sh)\n    over = (np.abs(xb) > LIM16).any(axis=axes, keepdims=True)\n    sh = sh + over\n    xb = np.where(over, shift_round(x, sh), xb)\n    assert (np.abs(xb) <= LIM16).all()\n    return xb, sh, int(over.sum())\n\n\ndef int16_weights():\n    """Each weight matrix as int16 [in, out] = round(w 2^g), g per output row the largest with every entry inside\n    +-(2^15 - 1) (the embedding as [576, V], one g per token), from the bfloat16 values: the scaling is a power of\n    two, so it is exact in float32 and the one rounding is rint\'s (ties to even). Kept as .npy files in the cache."""\n    d = os.path.join(MCACHE, "int16-rows")\n    if not os.path.exists(os.path.join(d, "DONE")):\n        os.makedirs(d, exist_ok=True)\n        for name, a in iter_safetensors(model_file("model.safetensors")):\n            if a.ndim != 2:\n                continue\n            top = np.abs(a).max(1).astype(np.float64)                         # a: [out, in]\n            g = (B16 - 1) - np.frexp(top)[1]\n            g = np.where(top * np.exp2(g) > LIM16, g - 1, g)\n            wq = np.rint(a * np.exp2(g).astype(np.float32)[:, None])\n            assert np.abs(wq).max() <= LIM16\n            np.save(os.path.join(d, name + ".npy"), np.ascontiguousarray(wq.T.astype(np.int16)))\n            np.save(os.path.join(d, name + ".g.npy"), g.astype(np.int64))\n        open(os.path.join(d, "DONE"), "w").write("from %s\\n" % PIN["files"]["model.safetensors"])\n    return {f[:-4]: np.load(os.path.join(d, f), mmap_mode="r") for f in os.listdir(d) if f.endswith(".npy")}\n\n\ndef _pb(num, payload):\n    """One length-delimited protobuf field."""\n    def varint(n):\n        out = bytearray()\n        while True:\n            out.append((n & 0x7F) | (0x80 if n >> 7 else 0))\n            n >>= 7\n            if not n:\n                return bytes(out)\n    return varint(num << 3 | 2) + varint(len(payload)) + payload if isinstance(payload, bytes) else \\\n        varint(num << 3) + varint(payload)\n\n\ndef _mmi_model(ta, tb):\n    """ONNX model bytes: Y (int32) = MatMulInteger(A, B), element types ta, tb (2 uint8, 3 int8), opset 13."""\n    vi = lambda name, t: _pb(1, name) + _pb(2, _pb(1, _pb(1, t)))           # ValueInfo: name, tensor type, any shape\n    node = _pb(1, b"A") + _pb(1, b"B") + _pb(2, b"Y") + _pb(3, b"m") + _pb(4, b"MatMulInteger")\n    graph = _pb(1, node) + _pb(2, b"g") + _pb(11, vi(b"A", ta)) + _pb(11, vi(b"B", tb)) + _pb(12, vi(b"Y", 6))\n    return _pb(1, 8) + _pb(8, _pb(1, b"") + _pb(2, 13)) + _pb(7, graph)\n\n\nclass MatMulInteger:\n    """int32-accumulating products of unsigned bytes: onnxruntime\'s MatMulInteger (uint8 by uint8, the pair every\n    onnxruntime version implements, and one whose kernels add the byte products in 32 bits) where onnxruntime is\n    installed, else NumPy int32 (a stand-in, named "i8-numpy")."""\n\n    def __init__(self):\n        try:\n            import onnxruntime as ort\n        except ImportError:\n            self.name, self.s = "i8-numpy", None\n            return\n        self.name = "i8-onnxruntime-%s" % ort.__version__\n        self.s = ort.InferenceSession(_mmi_model(2, 2), providers=["CPUExecutionProvider"])\n\n    def __call__(self, a, b):\n        assert a.dtype == np.uint8 and b.dtype == np.uint8\n        if self.s is None:\n            return (a.astype(np.int32) @ b.astype(np.int32)).astype(np.int64)\n        return self.s.run(None, {"A": a, "B": b})[0].astype(np.int64)\n\n\ndef mm16(a, b, route, mmi=None):\n    """a @ b exactly, as int64, for integer arrays with every |entry| <= 2^15 - 1 (a [..., N, n], b [..., n, M]).\n    f64: one float64 product, |partial sums| <= n 2^30 < 2^53.\n    f32: a = 256 a1 + a0 with a0 in [-128, 127], a1 in [-128, 128], the same for b; four float32 products in chunks\n         of 1,024 inputs, |partial sums| <= 1024 x 2^14 = 2^24; the chunks added in int64.\n    i8:  unsigned: u = a + 2^15 in [1, 2^16 - 1], u = 256 u1 + u0 with u1, u0 in [0, 255] (uint8), the same for b;\n         four int32 MatMulInteger products of uint8 by uint8, partial sums <= n 255^2 < 2^31; then, in int64,\n         a b = u_a u_b - 2^15 (row sums of a) - 2^15 (column sums of b) - n 2^30.\n    Then a b (u_a u_b on the i8 route) = 2^16 G11 + 2^8 (G10 + G01) + G00, G_ij = a_i b_j, in int64."""\n    n = a.shape[-1]\n    if route == "f64":\n        return (a.astype(np.float64) @ b.astype(np.float64)).astype(np.int64)\n    # the digits in the operand\'s own type when it is int16 (the weights: no widening; every step below stays within\n    # [-2^15, 2^15)), else in int64\n    a = np.asarray(a, dtype=np.int64)\n    b = np.asarray(b) if np.asarray(b).dtype == np.int16 else np.asarray(b, dtype=np.int64)\n    if route == "f32":\n        def bal(x):                                   # x = 256 x1 + x0: x0 = (x & 255) - 256 [x & 255 >= 128]\n            x0 = ((x & 255) ^ 128) - 128\n            return (x >> 8) + (x0 < 0), x0\n        (a1, a0), (b1, b0) = bal(a), bal(b)\n        f = lambda x: x.astype(np.float32)\n        G = lambda x, y: sum((f(x[..., s:s + CHUNK32]) @ f(y[..., s:s + CHUNK32, :])).astype(np.int64)\n                             for s in range(0, n, CHUNK32))\n    elif route == "i8":\n        assert n <= N_I8\n        ua = a + (1 << 15)                                                    # int64, in [1, 2^16 - 1]\n        ub = (b.view(np.uint16) ^ 0x8000) if b.dtype == np.int16 else b + (1 << 15)   # int16: b + 2^15, mod 2^16\n        a1, a0, b1, b0 = (ua >> 8).astype(np.uint8), (ua & 255).astype(np.uint8), \\\n            (ub >> 8).astype(np.uint8), (ub & 255).astype(np.uint8)\n        U = (mmi(a1, b1) << 16) + ((mmi(a1, b0) + mmi(a0, b1)) << 8) + mmi(a0, b0)   # sum u_a u_b, exact\n        ra = a.sum(axis=-1, keepdims=True)                                    # [..., N, 1]\n        cb = b.sum(axis=-2, dtype=np.int64, keepdims=True)                    # [..., 1, M]\n        return U - ((ra + cb) << 15) - (n << 30)\n    else:\n        raise SystemExit("unknown route " + route)\n    return (G(a1, b1) << 16) + ((G(a1, b0) + G(a0, b1)) << 8) + G(a0, b0)\n\n\ndef peel_bits(n):\n    """Index bits of the peel key for rows of n values: 11 up to 2,048 (the 135M\'s keys), more for wider rows."""\n    return max(11, (n - 1).bit_length())\n\n\ndef peel_index(x, k):\n    """The k largest |x| of each row, ties to the lower index: unique keys |x| 2^b + (2^b - 1 - i), b = peel_bits(n),\n    so the set does not depend on the selection algorithm (|x| 2^b below 2^63: bounds16.py)."""\n    n = x.shape[-1]\n    b = peel_bits(n)\n    key = (np.abs(x) << b) + ((1 << b) - 1 - np.arange(n))\n    return np.argpartition(-key, k - 1, axis=-1)[:, :k]\n\n\ndef shmax(c):\n    """Largest block shift for integers within [-c, c]: bitlen(c) - 15, plus one for the rounding."""\n    return max(0, c.bit_length() - (B16 - 1)) + 1\n\n\nSITE16 = {"q_proj": "attention input", "k_proj": "attention input", "v_proj": "attention input",\n          "o_proj": "o input", "gate_proj": "mlp input", "up_proj": "mlp input", "down_proj": "down input",\n          "embed_tokens": "logits input"}\n\n\ndef weight_stats16():\n    """Per matrix kind, from the 16-bit weights and their row scales g: inputs n, the largest sum of |w16| over one\n    output\'s inputs (col), the g range, and lin = max over outputs of col 2^(g_max - g), so that an output is at\n    most (input bound) lin 2^-g_max. Computed once and kept beside the weights (stats16.json); each matrix is\n    opened on its own and let go, so at most one is in memory."""\n    d = os.path.join(MCACHE, "int16-rows")\n    path = os.path.join(d, "stats16.json")\n    if os.path.exists(path):\n        return json.load(open(path))\n    int16_weights()\n    st = {}\n    for kind in SITE16:\n        names = sorted(f[:-4] for f in os.listdir(d) if f.endswith("." + kind + ".weight.npy"))\n        gs = [np.load(os.path.join(d, n + ".g.npy")) for n in names]\n        gmin, gmax = min(int(g.min()) for g in gs), max(int(g.max()) for g in gs)\n        col, lin, n_in = 0, 0, None\n        for name, g in zip(names, gs):\n            w = np.load(os.path.join(d, name + ".npy"), mmap_mode="r")\n            n_in = w.shape[0]\n            for s in range(0, w.shape[1], 8192):\n                c = np.abs(np.asarray(w[:, s:s + 8192], dtype=np.int64)).sum(axis=0)\n                col = max(col, int(c.max()))\n                lin = max(lin, int((c << (gmax - g[s:s + 8192].astype(np.int64))).max()))   # below 2^41\n            del w\n        st[kind] = {"site": SITE16[kind], "inputs": int(n_in), "matrices": len(names),\n                    "col": col, "g_min": gmin, "g_max": gmax, "lin": lin}\n    json.dump(st, open(path, "w"), indent=1)\n    return st\n\n\ndef bounds16_rows(C, st):\n    """The int64 operations of exact-16 that depend on the clamps C (integers at 2^-16, per site), each with its\n    largest magnitude and its limit, from the weight statistics st. The clamp rule and bounds16.py use these."""\n    rows = []\n    lin_out = {}\n    for kind, k in st.items():\n        c = C[k["site"]]\n        sh = shmax(c)\n        T = k["col"] * (c + (1 << (sh - 1))) + (1 << (k["g_max"] - 1))     # 2^sh x16 . w16 + peeled + rounding\n        lin_out[kind] = ((k["lin"] * (c + (1 << (sh - 1)))) >> k["g_max"]) + 1\n        rows.append(("%s: 2^sh x16 . w16 plus the peeled terms (sh <= %d), plus rounding constant" % (kind, sh),\n                     T, 1 << 62, k["site"]))\n        rows.append(("%s: largest output" % kind, lin_out[kind], 1 << 62, k["site"]))\n        b = peel_bits(k["inputs"])\n        rows.append(("%s: peel key |x| 2^%d + index" % (kind, b), (c << b) + (1 << b), 1 << 63, k["site"]))\n    for kind in ("q_proj", "k_proj"):\n        rows.append(("rope: %s output times cos, two terms" % kind, 2 * lin_out[kind] << ROPE, 1 << 63,\n                     st[kind]["site"]))\n    qk = HEAD * LIM16 * LIM16 << (shmax(C["q"]) + shmax(C["k"]))\n    rows.append(("scores: q16 . k16 shifted by sh_q + sh_k, at 2^-32; max minus score", 2 * qk, 1 << 63, "q, k"))\n    rows.append(("v: the high half |v >> 15| <= 2^15 - 1", (C["v"] >> (B16 - 1)) + 1, 1 << (B16 - 1), "v"))\n    sp = ((1 << A) + ROPE_POS) // 2                                          # sum of p at 2^-15 over one row\n    rows.append(("context: 2^15 p16 . vh + p16 . vl", sp * (((C["v"] >> (B16 - 1)) + 1) << (B16 - 1)) + sp * LIM16,\n                 1 << 63, "v"))\n    rows.append(("gate times up: silu(g) u, both clamped", C["gate"] * C["up"], 1 << 63, "gate, up"))\n    rows.append(("rms: max |x| below 2^(NORM_BITS + 16), so the shift is at most 16", C["residual"],\n                 1 << (NORM_BITS + A), "residual"))\n    rows.append(("residual: clamp plus one linear output", C["residual"] + max(lin_out[k] for k in ("o_proj", "down_proj")),\n                 1 << 63, "residual"))\n    return rows, lin_out\n\n\ndef clamps16(st):\n    """The exact-16 clamps, in real units, by rule: for each site the largest power of two at which every row of\n    bounds16_rows below its limit still holds, the others held at 2^0; q and k together, gate and up together."""\n    groups = [("residual",), ("attention input",), ("o input",), ("mlp input",), ("down input",),\n              ("logits input",), ("q", "k"), ("v",), ("gate", "up")]\n    sites = [s for g in groups for s in g]\n    e = {s: 0 for s in sites}\n\n    def ok(group, x):\n        C = {s: 1 << (e[s] + A) for s in sites}\n        for s in group:\n            C[s] = 1 << (x + A)\n        rows, _ = bounds16_rows(C, st)\n        return all(v < lim for _, v, lim, _ in rows)\n    for g in groups:\n        assert ok(g, 0), g\n        x = 0\n        while x < 46 and ok(g, x + 1):\n            x += 1\n        for s in g:\n            e[s] = x\n    return {s: 1 << e[s] for s in sites}\n\n\nclass Exact16(Exact):\n    def __init__(self, route="f64"):\n        self.stats16 = weight_stats16()\n        self.W16 = int16_weights()\n        super().__init__(W=int_weights(small=True), clamps=clamps16(self.stats16))\n        self.route = route\n        self.mmi = MatMulInteger() if route == "i8" else None\n        self.extra = 0                                   # vectors whose rounding needed one more shift\n        self.p_held = 0                                  # p values held at 2^15 - 1\n\n    def product(self, x, name, dtype=np.int64):\n        """x int64 [N, in] at 2^-16, the matrix `name`; y at 2^-16:\n        y = round(2^-g (2^sh x16 . w16 + sum of the peeled x w16))."""\n        w, g = self.W16[name], self.W16[name + ".g"]\n        if PEEL16:\n            idx = peel_index(x, PEEL16)\n            xp = np.take_along_axis(x, idx, -1)\n            x = x.copy()\n            np.put_along_axis(x, idx, 0, -1)\n        xb, sh, n = block16(x)\n        self.extra += n\n        y = np.empty((x.shape[0], w.shape[1]), dtype=dtype)\n        for c in range(0, w.shape[1], 8192):                                  # output columns in blocks of 8,192\n            wc = w[:, c:c + 8192]                                             # int16, memory-mapped\n            T = np.left_shift(mm16(xb, wc, self.route, self.mmi), sh)\n            for j in range(PEEL16):\n                T += xp[:, j:j + 1] * wc[idx[:, j]]                           # int64 times int16: int64\n            blk = shift_round(T, g[None, c:c + 8192])\n            assert dtype == np.int64 or np.abs(blk).max() < (1 << 31)\n            y[:, c:c + 8192] = blk\n        return y\n\n    def linear(self, x, name, site):\n        return self.product(self.clamp(x, site), name)\n\n    def logits(self, h):\n        y = self.product(self.clamp(h, "logits input"), "model.embed_tokens.weight", np.int32)\n        return y\n\n    def cache_entry(self, k, v):\n        """What the cache keeps for each new position, made once: k clamped and block-shifted per head vector (k16\n        and its shift), v clamped and split in two 16-bit halves, v = 2^15 vh + vl, vl in [0, 2^15 - 1], |vh| <= 2^13\n        within the clamp."""\n        kb, shk, n = block16(self.clamp(k, "k"))\n        self.extra += n\n        v = self.clamp(v, "v")\n        vh, vl = v >> (B16 - 1), v & LIM16\n        assert (np.abs(vh) <= LIM16).all()\n        return kb, shk, vh, vl\n\n    def attention(self, q, kb, shk, vh, vl, kvalid):\n        """As Exact\'s, with q k^T and p v as 16-bit products (k16, its shifts and the halves of v from the cache).\n        Scores at 2^-32, p at 2^-15."""\n        B, _, L, _ = q.shape\n        T, G = kb.shape[2], HEADS // KV\n        qb, shq, n1 = block16(self.clamp(q, "q"))                            # per head vector\n        qb = qb.reshape(B, KV, G * L, HEAD)\n        S = mm16(qb, kb.swapaxes(-1, -2), self.route, self.mmi).reshape(B, KV, G, L, T)\n        s = np.left_shift(S, shq.reshape(B, KV, G, L, 1) + shk[:, :, None].swapaxes(-1, -2))\n        key = kvalid[:, None, None]\n        m = np.where(key, s, -(1 << 62)).max(-1, keepdims=True)\n        t = np.maximum(m - s, 0) >> (A + 3)\n        e = np.where(key, self.exp(t), 0)\n        p = shift_round(fdiv(e << A, e.sum(-1, keepdims=True)), 1)            # at 2^-15\n        self.p_held += int((p > LIM16).sum())\n        pb = np.minimum(p, LIM16).reshape(B, KV, G * L, T)\n        self.extra += n1\n        c = np.left_shift(mm16(pb, vh, self.route, self.mmi), B16 - 1) + mm16(pb, vl, self.route, self.mmi)\n        ctx = shift_round(c, B16 - 1)                                         # 2^-15 x 2^-16 -> 2^-16\n        return ctx.reshape(B, KV, G, L, HEAD).reshape(B, HEADS, L, HEAD)\n\n\n# ---- drivers ------------------------------------------------------------------------------------------------------\ndef run_prompts(model, tok, texts, batch, on_logits=None, gen=GEN):\n    """Prefill each batch (right-padded), hand the logits of every prompt position to on_logits(row, logits), then\n    continue greedily for `gen` tokens with the key-value cache. A row attends only to its own prompt and its own\n    generated tokens; padded keys are masked. Returns the continuations, one list of ids per prompt."""\n    out = []\n    for s in range(0, len(texts), batch):\n        enc = [tok.encode(t) for t in texts[s:s + batch]]\n        B, Lmax = len(enc), max(len(e) for e in enc)\n        lens = np.array([len(e) for e in enc])\n        ids = np.zeros((B, Lmax), dtype=np.int64)\n        for b, e in enumerate(enc):\n            ids[b, :len(e)] = e\n        pos = np.broadcast_to(np.arange(Lmax), (B, Lmax))\n        idx = np.arange(Lmax)\n        kvalid = (idx[None, None, :] <= idx[None, :, None]) & (idx[None, None, :] < lens[:, None, None])\n        logits, cache = model.forward(ids, pos, None, kvalid)\n        for b in range(B):\n            if on_logits is not None:\n                on_logits(s + b, logits[b, :lens[b]])\n        nxt = np.array([int(np.argmax(logits[b, lens[b] - 1])) for b in range(B)])\n        cont = [[int(t)] for t in nxt]\n        for t in range(gen - 1):\n            T = Lmax + t + 1\n            idx = np.arange(T)\n            kv = (idx[None] < lens[:, None]) | ((idx[None] >= Lmax) & (idx[None] <= Lmax + t))\n            logits, cache = model.forward(nxt[:, None], (lens + t)[:, None], cache, kv[:, None, :])\n            nxt = np.array([int(np.argmax(logits[b, 0])) for b in range(B)])\n            for b in range(B):\n                cont[b].append(int(nxt[b]))\n        out += cont\n    return out\n\n\ndef text_windows(tok, n=16, size=512):\n    ids = tok.encode(text())[: n * size]\n    assert len(ids) == n * size\n    return [np.array(ids[i * size:(i + 1) * size], dtype=np.int64)[None] for i in range(n)]\n\n\ndef run_window(model, ids):\n    L = ids.shape[1]\n    pos = np.arange(L)[None]\n    kvalid = np.tril(np.ones((L, L), dtype=bool))[None]\n    logits, _ = model.forward(ids, pos, None, kvalid)\n    return logits[0]\n\n\n# ---- hashes, the text, the command line ---------------------------------------------------------------------------\ndef engine(name, threads=0):\n    if name == "exact":\n        return Exact()\n    if name in ("exact16-" + r for r in ROUTES):\n        return Exact16(name.split("-")[1])\n    if name == "float32":\n        return Float32()\n    if name == "onnx":\n        return Onnx(threads)\n    raise SystemExit("unknown engine " + name)\n\n\ndef as_bytes(logits):\n    """Exact logits as little-endian int32 (scale 2^-16); float logits as little-endian float32."""\n    return np.ascontiguousarray(logits).astype("<i4" if logits.dtype == np.int32 else "<f4").tobytes()\n\n\ndef prompt_run(eng, batch, keep=False):\n    """The 64 prompts: SHA-256 over the logits of every prompt position (prompt by prompt, position by position,\n    49,152 values each) and SHA-256 over the 64 continuations of 48 token ids (int32, little-endian)."""\n    h = hashlib.sha256()\n    kept = []\n\n    def on_logits(row, lg):\n        h.update(as_bytes(lg))\n        if keep:\n            kept.append(np.array(lg))\n    cont = run_prompts(eng, tokenizer(), prompts(), batch, on_logits)\n    hc = hashlib.sha256(np.array(cont, dtype="<i4").tobytes()).hexdigest()\n    return h.hexdigest(), hc, cont, kept\n\n\ndef text_run(eng):\n    """The 16 windows of 512 tokens: negative log-likelihood of each next token (in float64 from the logits; exact\n    logits are divided by 2^16 first) and the top-1 token at each of the 8,176 predicted positions."""\n    nll, top = [], []\n    for w in text_windows(tokenizer()):\n        lg = run_window(eng, w)\n        top.append(np.argmax(lg[:-1], -1))\n        scale = 2.0 ** A if lg.dtype == np.int32 else 1.0\n        for s in range(0, lg.shape[0] - 1, 128):                    # rows s .. predict tokens s+1 .., in blocks\n            e = min(s + 128, lg.shape[0] - 1)\n            z = lg[s:e].astype(np.float64) / scale\n            mx = z.max(-1, keepdims=True)\n            lse = np.log(np.exp(z - mx).sum(-1)) + mx[:, 0]\n            nll.append(lse - z[np.arange(e - s), w[0, s + 1:e + 1]])\n    return np.concatenate(nll), np.concatenate(top)\n\n\ndef expected(bits=None):\n    """The pinned pair: expected.sha256 (exact) and expected-16.sha256 (exact-16) for the default model;\n    expected-16-<model>.sha256 for another."""\n    path = os.path.join(HERE, "expected%s%s.sha256" % ("" if bits is None else "-%d" % bits, SUFFIX))\n    if not os.path.exists(path):\n        return None\n    lines = open(path).read().split("\\n")\n    return lines[0].split()[0], lines[1].split()[0]\n\n\ndef peaks_line(eng):\n    return "; ".join("%s %.3f / %s" % (k, v / 2.0 ** A, eng.clamps[k]) for k, v in sorted(eng.peak.items()))\n\n\ndef peaks_record(eng):\n    """The measured peaks, clamps and clamped counts of a run, per site, in real units (for bounds16.py)."""\n    return {"model": MODEL, "sites": {k: {"peak": v / 2.0 ** A, "clamp": eng.clamps[k],\n                                          "margin_log2": round(math.log2(eng.clamps[k] * 2.0 ** A / v), 3) if v else None,\n                                          "clamped": eng.clamped.get(k, 0)} for k, v in sorted(eng.peak.items())}}\n\n\ndef main(argv):\n    arg = lambda k, d: argv[argv.index(k) + 1] if k in argv else d\n    if "--model" in argv:\n        use_model(arg("--model", MODELS[0]))\n    if len(argv) > 1 and argv[1] == "hash":\n        name, batch, threads = arg("--engine", "exact"), int(arg("--batch", "8")), int(arg("--threads", "0"))\n        save = arg("--save", None)\n        eng = Exact(keep_f64=True) if name == "exact" and "--keep" in argv else engine(name, threads)\n        hl, hc, cont, kept = prompt_run(eng, batch, keep=save is not None)\n        print("%s %s %s batch=%d" % (hl, hc, name, batch))\n        if save:\n            np.save(save + ".logits.npy", np.concatenate(kept))\n            np.save(save + ".cont.npy", np.array(cont, dtype=np.int32))\n        if name.startswith("exact"):\n            print("clamped", sum(eng.clamped.values()), eng.clamped)\n            print("peaks (real) and clamps:", peaks_line(eng))\n            if arg("--peaks", None):\n                json.dump(peaks_record(eng), open(arg("--peaks", None), "w"), indent=1)\n        if name.startswith("exact16"):\n            print("route %s; vectors with one more shift %d; p held at 2^15 - 1: %d" % (\n                eng.mmi.name if eng.mmi else eng.route, eng.extra, eng.p_held))\n        return 0\n    if len(argv) > 1 and argv[1] == "text":\n        name, save = arg("--engine", "exact"), arg("--save", None)\n        nll, top = text_run(engine(name))\n        print("%s perplexity %.6f over %d tokens" % (name, math.exp(nll.mean()), nll.size))\n        if save:\n            np.save(save + ".nll.npy", nll)\n            np.save(save + ".top.npy", top)\n        return 0\n    if len(argv) > 1 and argv[1] == "check":\n        try:\n            import onnxruntime\n            ort_version = onnxruntime.__version__\n        except ImportError:\n            ort_version = "not installed (the i8 route uses NumPy int32)"\n        print("versions: Python %s, NumPy %s, onnxruntime %s; %s %s" % (\n            sys.version.split()[0], np.__version__, ort_version, sys.platform, os.uname().machine\n            if hasattr(os, "uname") else os.environ.get("PROCESSOR_ARCHITECTURE", "")))\n        print("model %s @ %s: %d layers, width %d, %d/%d heads, feed-forward %d, rotary base %d, %d weights" % (\n            MODEL, PIN["revision"][:7], LAYERS, HIDDEN, HEADS, KV, FF, THETA, n_params()))\n        t_all = time.time()\n        ok, runs = True, []                                    # runs: (name, logits, continuations) for the distance\n        if MODEL == MODELS[0]:                                 # the exact engine runs on the default model only\n            ex = Exact()\n            hl, hc, cont_e, lg_e = prompt_run(ex, 8, keep=True)\n            el, ec = expected()\n            ok = hl == el and hc == ec\n            print("exact    logits %s  continuations %s  %s" % (hl, hc, "MATCH" if ok else "DIFFERENT"))\n            print("         clamped values: %d" % sum(ex.clamped.values()))\n            runs.append(("exact", lg_e, cont_e))\n            del ex\n        t0 = time.time()\n        weight_stats16()\n        int_weights(small=True)\n        print("exact16  16-bit weights, int32 embedding and norms, weight statistics: ready in %.0f s" % (time.time() - t0))\n        e16 = expected(16)\n        if e16 is None:\n            print("exact16  no expected-16%s.sha256: the hashes below are not compared" % SUFFIX)\n        kept16 = False\n        for r in arg("--routes", ",".join(ROUTES)).split(","):\n            m16 = Exact16(r)\n            t0 = time.time()\n            h16l, h16c, cont, lg = prompt_run(m16, 8, keep=not kept16)\n            ok16 = (h16l, h16c) == e16\n            ok = ok and ok16\n            print("exact16  logits %s  continuations %s  %s  route %s, %.0f s, clamped %d" % (\n                h16l, h16c, "MATCH" if ok16 else "DIFFERENT", m16.mmi.name if m16.mmi else r, time.time() - t0,\n                sum(m16.clamped.values())))\n            if not kept16:\n                print("         peaks (real) / clamps: %s" % peaks_line(m16))\n                if arg("--peaks", None):\n                    json.dump(peaks_record(m16), open(arg("--peaks", None), "w"), indent=1)\n                runs.append(("exact16", lg, cont))\n                kept16 = True\n            del m16\n        if "--no-float" not in argv:\n            t0 = time.time()\n            f = Float32()\n            fl, fc, cont_f, lg_f = prompt_run(f, 8, keep=True)\n            print("float32  logits %s  continuations %s  %.0f s%s" % (\n                fl, fc, time.time() - t0, ", matrices widened from bfloat16 at each use" if f.lean else ""))\n            del f\n            g = np.concatenate(lg_f).astype(np.float64)\n            for nm, lg, cont in runs:\n                e = np.concatenate(lg).astype(np.float64) / 2.0 ** A\n                print("%s against float32 over %d prompt positions: top-1 agreement %d, largest |logit difference| "\n                      "%.5f, mean %.6f; continuations equal %d of %d" % (\n                          nm, e.shape[0], int((e.argmax(-1) == g.argmax(-1)).sum()), np.abs(e - g).max(),\n                          np.abs(e - g).mean(), sum(a == b for a, b in zip(cont, cont_f)), len(cont)))\n                del e\n            del g, lg_f\n        if "--no-float" not in argv and "--no-onnx" not in argv:\n            t0 = time.time()\n            try:\n                o = Onnx()\n            except ImportError:\n                print("onnx     onnxruntime is not installed here")\n            except LookupError as why:\n                print("onnx     skipped: %s" % why)\n            else:\n                ol, oc, _, _ = prompt_run(o, 8)\n                print("onnx     logits %s  continuations %s  %.0f s" % (ol, oc, time.time() - t0))\n                del o\n        print("check    %s, %.0f s" % ("every pair MATCH" if ok else "a pair DIFFERENT or not pinned", time.time() - t_all))\n        return 0 if ok else 1\n    print(__doc__)\n    return 2\n\n\nif __name__ == "__main__":\n    sys.exit(main(sys.argv))\n')
FILES['pins.json'] = ('c2c30a03da5a9d5e317659360c16025e78e4d78567f80f467c2e2aaba4bb0619', '{\n  "models": {\n    "SmolLM2-135M": {\n      "repo": "HuggingFaceTB/SmolLM2-135M",\n      "revision": "93efa2f097d58c2a74874c7e644dbc9b0cee75a2",\n      "license": "apache-2.0",\n      "files": {\n        "config.json": "1d556eab73b69c7f11f64c557a2f9c6f440bd4c6b89bb2584a6b498c92603843",\n        "tokenizer.json": "9ca9acddb6525a194ec8ac7a87f24fbba7232a9a15ffa1af0c1224fcd888e47c",\n        "model.safetensors": "80521b40281d6ce74e35c9282c22539e75aa0ac8578892b2a59955ef78d55da1"\n      },\n      "onnx": {\n        "repo": "onnx-community/SmolLM2-135M-ONNX",\n        "revision": "d0ae6834f1df45e0e95b5fdae95e536f9ca7cd3f",\n        "files": {\n          "onnx/model.onnx": "de4f7dbed6420825147b1fd97d7ea000042ad667ce30316bedd87a3ed49c48b1",\n          "onnx/model.onnx_data": "3ffe0420181424138de643144c94bd4c30126221cc636590f3ccb2fa352ae602"\n        }\n      }\n    },\n    "SmolLM2-360M": {\n      "repo": "HuggingFaceTB/SmolLM2-360M",\n      "revision": "f8027fd0eaeea54caa13c31d31b9fdc459c38b49",\n      "license": "apache-2.0",\n      "files": {\n        "config.json": "34f7801487078de7e434e19162c497e5cc6ff397080e40e8586627cb68a5168a",\n        "tokenizer.json": "9ca9acddb6525a194ec8ac7a87f24fbba7232a9a15ffa1af0c1224fcd888e47c",\n        "model.safetensors": "7aaff6661428bed033abba9522bec81938678642cca3181fe752b6ca9e1e540f"\n      },\n      "onnx": {\n        "repo": "onnx-community/SmolLM2-360M-ONNX",\n        "revision": "539c1088be2dcd673c39059d25a6bc4aec625504",\n        "files": {\n          "onnx/model.onnx": "09c211719c8c8c752b3aec0afc53bcb0f95402e575b2526b7a58f52da6bad620"\n        }\n      }\n    },\n    "SmolLM2-1.7B": {\n      "repo": "HuggingFaceTB/SmolLM2-1.7B",\n      "revision": "effd688a12921b4cc83e3312b6feb579f70f9c71",\n      "license": "apache-2.0",\n      "files": {\n        "config.json": "33397f6af8090f7290f27ce6fb1cd23a95fd0915a5c648dfd10b6002aaf7fc48",\n        "tokenizer.json": "9ca9acddb6525a194ec8ac7a87f24fbba7232a9a15ffa1af0c1224fcd888e47c",\n        "model.safetensors": "1193528982f4ac0c0b707ce36fd7dc03a0ef6f3e1a432deb886dce2e90c300c0"\n      },\n      "onnx": null,\n      "onnx_note": "onnx-community publishes no export of this model; check skips onnxruntime for it"\n    }\n  },\n  "prompts": {\n    "repo": "mteb/stsbenchmark-sts",\n    "revision": "96943a16ea6a35129e253c659081cb59daf81b30",\n    "file": "test.jsonl.gz",\n    "sha256": "524a75137d413dbfc40c3e84905f733714232647c3848521acb6bc375b834e1c",\n    "take": "sentence1 of the first 64 lines, in file order"\n  },\n  "text": {\n    "url": "https://raw.githubusercontent.com/karpathy/char-rnn/370cbcd448eb7daf32f21a6be560b70e0b33c4e3/data/tinyshakespeare/input.txt",\n    "sha256": "86c4e6aa9db7c042ec79f339dcb96d42b0075e16b8fc2e86bf0ca57e2dc565ed",\n    "take": "the first 8,192 tokens as sixteen windows of 512; the first 2,000 lines for the tokenizer comparison"\n  }\n}\n')
FILES['expected.sha256'] = ('90569fb33ea487a058c88018ca2c1dfb165373329fcc102e4d6b3a0f3ab39f40', '30ed613b6036d08046fbdbf273a0fb7b8cc04fb284ca49e0296980eed9cee773  logits: int32 little-endian (scale 2^-16) of every position of the 64 prompts, prompt by prompt, 49,152 each\na800ebadc848514e02bebbb8077d531cf0493acd2a4c173403ba1db43d7613cc  continuations: 64 x 48 greedy token ids, int32 little-endian\n')
FILES['tables/exp.txt'] = ('b77c98176ce6bab030710e68f747553319c954cf93afd0855fbedbf14179fdd0', '# exp.txt\n# E[k] = round(2^24 exp(-k/256)), k = 0 .. 6144\n# 6145 values, one per line, generated by make_tables.py\n16777216\n16711808\n16646655\n16581756\n16517109\n16452715\n16388572\n16324679\n16261035\n16197640\n16134491\n16071589\n16008931\n15946518\n15884349\n15822422\n15760736\n15699291\n15638085\n15577118\n15516388\n15455896\n15395639\n15335617\n15275829\n15216274\n15156952\n15097860\n15038999\n14980368\n14921965\n14863790\n14805841\n14748119\n14690621\n14633348\n14576298\n14519470\n14462864\n14406479\n14350313\n14294367\n14238638\n14183127\n14127832\n14072753\n14017889\n13963238\n13908801\n13854575\n13800561\n13746758\n13693165\n13639780\n13586604\n13533634\n13480872\n13428315\n13375963\n13323815\n13271870\n13220128\n13168588\n13117248\n13066109\n13015169\n12964428\n12913884\n12863538\n12813388\n12763433\n12713673\n12664107\n12614734\n12565554\n12516566\n12467768\n12419161\n12370743\n12322514\n12274473\n12226620\n12178953\n12131471\n12084175\n12037064\n11990136\n11943390\n11896828\n11850446\n11804246\n11758225\n11712384\n11666722\n11621238\n11575931\n11530801\n11485846\n11441067\n11396463\n11352032\n11307775\n11263690\n11219777\n11176035\n11132464\n11089063\n11045830\n11002767\n10959871\n10917143\n10874581\n10832185\n10789954\n10747888\n10705986\n10664247\n10622671\n10581257\n10540005\n10498913\n10457982\n10417210\n10376597\n10336143\n10295846\n10255706\n10215723\n10175896\n10136224\n10096706\n10057343\n10018133\n9979076\n9940172\n9901418\n9862816\n9824365\n9786063\n9747911\n9709908\n9672052\n9634345\n9596784\n9559370\n9522101\n9484978\n9447999\n9411165\n9374475\n9337927\n9301522\n9265259\n9229137\n9193156\n9157315\n9121614\n9086052\n9050629\n9015344\n8980197\n8945186\n8910312\n8875574\n8840972\n8806504\n8772171\n8737971\n8703905\n8669972\n8636171\n8602501\n8568963\n8535556\n8502279\n8469132\n8436114\n8403225\n8370464\n8337830\n8305324\n8272945\n8240692\n8208564\n8176562\n8144685\n8112932\n8081302\n8049796\n8018413\n7987152\n7956013\n7924996\n7894099\n7863323\n7832667\n7802130\n7771712\n7741413\n7711232\n7681169\n7651223\n7621394\n7591681\n7562084\n7532602\n7503235\n7473983\n7444844\n7415820\n7386908\n7358109\n7329423\n7300848\n7272385\n7244032\n7215791\n7187659\n7159637\n7131724\n7103920\n7076225\n7048637\n7021157\n6993784\n6966518\n6939358\n6912304\n6885356\n6858512\n6831773\n6805139\n6778608\n6752181\n6725856\n6699635\n6673515\n6647498\n6621582\n6595767\n6570052\n6544438\n6518924\n6493509\n6468193\n6442976\n6417857\n6392836\n6367913\n6343087\n6318357\n6293724\n6269188\n6244746\n6220400\n6196149\n6171993\n6147931\n6123962\n6100087\n6076305\n6052616\n6029019\n6005514\n5982101\n5958779\n5935548\n5912407\n5889357\n5866396\n5843525\n5820744\n5798051\n5775446\n5752930\n5730501\n5708160\n5685906\n5663739\n5641658\n5619663\n5597754\n5575931\n5554192\n5532539\n5510969\n5489484\n5468083\n5446765\n5425530\n5404378\n5383308\n5362320\n5341415\n5320590\n5299847\n5279185\n5258604\n5238102\n5217681\n5197339\n5177077\n5156893\n5136788\n5116762\n5096813\n5076943\n5057150\n5037434\n5017795\n4998232\n4978746\n4959336\n4940001\n4920742\n4901558\n4882448\n4863413\n4844453\n4825566\n4806753\n4788013\n4769346\n4750752\n4732231\n4713782\n4695405\n4677099\n4658865\n4640701\n4622609\n4604587\n4586636\n4568754\n4550942\n4533200\n4515526\n4497922\n4480386\n4462919\n4445520\n4428188\n4410924\n4393728\n4376598\n4359536\n4342539\n4325609\n4308745\n4291947\n4275214\n4258547\n4241945\n4225407\n4208933\n4192524\n4176179\n4159898\n4143680\n4127525\n4111434\n4095405\n4079438\n4063534\n4047692\n4031911\n4016192\n4000535\n3984938\n3969402\n3953927\n3938512\n3923157\n3907862\n3892627\n3877451\n3862334\n3847277\n3832278\n3817337\n3802454\n3787630\n3772864\n3758155\n3743503\n3728908\n3714371\n3699890\n3685465\n3671097\n3656785\n3642528\n3628327\n3614182\n3600092\n3586056\n3572075\n3558149\n3544277\n3530459\n3516696\n3502985\n3489328\n3475725\n3462174\n3448676\n3435231\n3421839\n3408498\n3395210\n3381973\n3368788\n3355654\n3342572\n3329540\n3316560\n3303630\n3290750\n3277921\n3265141\n3252412\n3239732\n3227101\n3214520\n3201988\n3189504\n3177070\n3164683\n3152345\n3140056\n3127814\n3115620\n3103473\n3091374\n3079321\n3067316\n3055358\n3043446\n3031581\n3019762\n3007989\n2996262\n2984581\n2972945\n2961355\n2949809\n2938309\n2926854\n2915443\n2904077\n2892755\n2881477\n2870243\n2859053\n2847907\n2836804\n2825744\n2814728\n2803754\n2792823\n2781935\n2771089\n2760286\n2749525\n2738805\n2728128\n2717492\n2706897\n2696344\n2685832\n2675361\n2664931\n2654541\n2644192\n2633883\n2623615\n2613386\n2603198\n2593049\n2582939\n2572869\n2562839\n2552847\n2542895\n2532981\n2523106\n2513269\n2503471\n2493711\n2483988\n2474304\n2464658\n2455049\n2445478\n2435944\n2426447\n2416987\n2407564\n2398178\n2388828\n2379515\n2370238\n2360998\n2351793\n2342624\n2333491\n2324394\n2315332\n2306305\n2297314\n2288357\n2279436\n2270549\n2261697\n2252880\n2244097\n2235348\n2226633\n2217952\n2209305\n2200692\n2192112\n2183566\n2175053\n2166573\n2158127\n2149713\n2141332\n2132984\n2124668\n2116385\n2108134\n2099915\n2091728\n2083573\n2075450\n2067359\n2059299\n2051270\n2043273\n2035307\n2027372\n2019468\n2011595\n2003753\n1995941\n1988159\n1980408\n1972687\n1964997\n1957336\n1949705\n1942104\n1934532\n1926990\n1919478\n1911994\n1904540\n1897115\n1889719\n1882351\n1875013\n1867703\n1860421\n1853168\n1845943\n1838747\n1831578\n1824438\n1817325\n1810240\n1803182\n1796152\n1789150\n1782175\n1775227\n1768306\n1761412\n1754544\n1747704\n1740891\n1734103\n1727343\n1720609\n1713901\n1707219\n1700563\n1693933\n1687329\n1680751\n1674198\n1667671\n1661169\n1654693\n1648242\n1641816\n1635415\n1629039\n1622688\n1616362\n1610061\n1603784\n1597531\n1591303\n1585099\n1578919\n1572764\n1566632\n1560524\n1554440\n1548380\n1542344\n1536330\n1530341\n1524375\n1518432\n1512512\n1506615\n1500741\n1494891\n1489063\n1483257\n1477475\n1471714\n1465977\n1460261\n1454568\n1448898\n1443249\n1437622\n1432017\n1426435\n1420873\n1415334\n1409816\n1404320\n1398845\n1393391\n1387959\n1382548\n1377158\n1371789\n1366441\n1361113\n1355807\n1350521\n1345256\n1340011\n1334787\n1329583\n1324400\n1319236\n1314093\n1308970\n1303867\n1298783\n1293720\n1288676\n1283652\n1278648\n1273663\n1268697\n1263751\n1258824\n1253916\n1249028\n1244158\n1239308\n1234476\n1229663\n1224869\n1220094\n1215337\n1210599\n1205880\n1201178\n1196495\n1191831\n1187184\n1182556\n1177945\n1173353\n1168779\n1164222\n1159683\n1155162\n1150658\n1146172\n1141704\n1137253\n1132819\n1128403\n1124003\n1119621\n1115256\n1110908\n1106577\n1102263\n1097966\n1093685\n1089421\n1085174\n1080944\n1076729\n1072532\n1068350\n1064185\n1060036\n1055903\n1051787\n1047686\n1043602\n1039533\n1035480\n1031443\n1027422\n1023417\n1019427\n1015452\n1011494\n1007550\n1003622\n999709\n995812\n991930\n988062\n984210\n980373\n976551\n972744\n968951\n965174\n961411\n957663\n953929\n950210\n946506\n942816\n939140\n935479\n931832\n928199\n924580\n920975\n917385\n913808\n910246\n906697\n903162\n899641\n896134\n892640\n889160\n885693\n882240\n878801\n875375\n871962\n868563\n865176\n861803\n858443\n855097\n851763\n848442\n845135\n841840\n838558\n835288\n832032\n828788\n825557\n822338\n819132\n815939\n812758\n809589\n806433\n803289\n800157\n797038\n793930\n790835\n787752\n784681\n781622\n778574\n775539\n772515\n769504\n766504\n763515\n760539\n757574\n754620\n751678\n748748\n745829\n742921\n740025\n737139\n734266\n731403\n728551\n725711\n722882\n720064\n717256\n714460\n711675\n708900\n706136\n703383\n700641\n697910\n695189\n692478\n689779\n687089\n684411\n681743\n679085\n676437\n673800\n671173\n668556\n665950\n663354\n660768\n658191\n655625\n653069\n650523\n647987\n645461\n642944\n640438\n637941\n635454\n632977\n630509\n628051\n625602\n623163\n620734\n618314\n615903\n613502\n611110\n608728\n606354\n603990\n601636\n599290\n596954\n594626\n592308\n589999\n587699\n585408\n583125\n580852\n578587\n576332\n574085\n571847\n569617\n567396\n565184\n562981\n560786\n558600\n556422\n554253\n552092\n549940\n547796\n545660\n543533\n541413\n539303\n537200\n535106\n533020\n530942\n528872\n526810\n524756\n522710\n520672\n518642\n516620\n514606\n512600\n510602\n508611\n506628\n504653\n502685\n500726\n498773\n496829\n494892\n492963\n491041\n489126\n487219\n485320\n483428\n481543\n479666\n477796\n475933\n474077\n472229\n470388\n468554\n466728\n464908\n463096\n461290\n459492\n457700\n455916\n454138\n452368\n450604\n448848\n447098\n445355\n443618\n441889\n440166\n438450\n436741\n435038\n433342\n431652\n429970\n428293\n426624\n424960\n423304\n421653\n420009\n418372\n416741\n415116\n413498\n411886\n410280\n408680\n407087\n405500\n403919\n402344\n400776\n399213\n397657\n396107\n394562\n393024\n391492\n389966\n388445\n386931\n385422\n383920\n382423\n380932\n379447\n377968\n376494\n375026\n373564\n372108\n370657\n369212\n367773\n366339\n364910\n363488\n362071\n360659\n359253\n357852\n356457\n355068\n353683\n352304\n350931\n349563\n348200\n346843\n345490\n344143\n342802\n341465\n340134\n338808\n337487\n336171\n334861\n333555\n332255\n330959\n329669\n328384\n327104\n325828\n324558\n323293\n322032\n320777\n319526\n318281\n317040\n315804\n314572\n313346\n312124\n310908\n309695\n308488\n307285\n306087\n304894\n303705\n302521\n301342\n300167\n298997\n297831\n296670\n295514\n294361\n293214\n292071\n290932\n289798\n288668\n287543\n286422\n285305\n284193\n283085\n281981\n280882\n279787\n278696\n277609\n276527\n275449\n274375\n273305\n272240\n271178\n270121\n269068\n268019\n266974\n265933\n264897\n263864\n262835\n261810\n260790\n259773\n258760\n257751\n256747\n255746\n254749\n253755\n252766\n251781\n250799\n249821\n248847\n247877\n246911\n245948\n244989\n244034\n243083\n242135\n241191\n240251\n239314\n238381\n237452\n236526\n235604\n234685\n233770\n232859\n231951\n231047\n230146\n229249\n228355\n227465\n226578\n225695\n224815\n223938\n223065\n222196\n221329\n220467\n219607\n218751\n217898\n217048\n216202\n215359\n214520\n213683\n212850\n212021\n211194\n210371\n209550\n208733\n207920\n207109\n206302\n205497\n204696\n203898\n203103\n202311\n201523\n200737\n199954\n199175\n198398\n197625\n196854\n196087\n195323\n194561\n193802\n193047\n192294\n191545\n190798\n190054\n189313\n188575\n187840\n187107\n186378\n185651\n184928\n184207\n183489\n182773\n182061\n181351\n180644\n179940\n179238\n178539\n177843\n177150\n176459\n175771\n175086\n174403\n173723\n173046\n172372\n171699\n171030\n170363\n169699\n169038\n168379\n167722\n167068\n166417\n165768\n165122\n164478\n163837\n163198\n162562\n161928\n161297\n160668\n160042\n159418\n158796\n158177\n157560\n156946\n156334\n155725\n155118\n154513\n153910\n153310\n152713\n152117\n151524\n150934\n150345\n149759\n149175\n148594\n148014\n147437\n146862\n146290\n145719\n145151\n144585\n144022\n143460\n142901\n142344\n141789\n141236\n140686\n140137\n139591\n139046\n138504\n137964\n137427\n136891\n136357\n135825\n135296\n134768\n134243\n133720\n133198\n132679\n132162\n131647\n131133\n130622\n130113\n129606\n129100\n128597\n128096\n127596\n127099\n126603\n126110\n125618\n125128\n124640\n124155\n123671\n123188\n122708\n122230\n121753\n121279\n120806\n120335\n119866\n119398\n118933\n118469\n118007\n117547\n117089\n116632\n116178\n115725\n115274\n114824\n114377\n113931\n113486\n113044\n112603\n112164\n111727\n111291\n110858\n110425\n109995\n109566\n109139\n108713\n108290\n107867\n107447\n107028\n106611\n106195\n105781\n105369\n104958\n104549\n104141\n103735\n103331\n102928\n102526\n102127\n101729\n101332\n100937\n100543\n100151\n99761\n99372\n98985\n98599\n98214\n97831\n97450\n97070\n96692\n96315\n95939\n95565\n95193\n94821\n94452\n94084\n93717\n93351\n92987\n92625\n92264\n91904\n91546\n91189\n90833\n90479\n90127\n89775\n89425\n89077\n88729\n88383\n88039\n87696\n87354\n87013\n86674\n86336\n85999\n85664\n85330\n84997\n84666\n84336\n84007\n83680\n83353\n83028\n82705\n82382\n82061\n81741\n81423\n81105\n80789\n80474\n80160\n79848\n79536\n79226\n78917\n78610\n78303\n77998\n77694\n77391\n77089\n76789\n76489\n76191\n75894\n75598\n75304\n75010\n74718\n74426\n74136\n73847\n73559\n73272\n72987\n72702\n72419\n72136\n71855\n71575\n71296\n71018\n70741\n70465\n70191\n69917\n69644\n69373\n69102\n68833\n68565\n68297\n68031\n67766\n67502\n67238\n66976\n66715\n66455\n66196\n65938\n65681\n65425\n65170\n64916\n64663\n64411\n64159\n63909\n63660\n63412\n63165\n62918\n62673\n62429\n62185\n61943\n61702\n61461\n61221\n60983\n60745\n60508\n60272\n60037\n59803\n59570\n59338\n59106\n58876\n58646\n58418\n58190\n57963\n57737\n57512\n57288\n57065\n56842\n56620\n56400\n56180\n55961\n55743\n55525\n55309\n55093\n54878\n54665\n54451\n54239\n54028\n53817\n53607\n53398\n53190\n52983\n52776\n52570\n52365\n52161\n51958\n51755\n51554\n51353\n51152\n50953\n50754\n50556\n50359\n50163\n49967\n49773\n49579\n49385\n49193\n49001\n48810\n48620\n48430\n48241\n48053\n47866\n47679\n47493\n47308\n47124\n46940\n46757\n46575\n46393\n46212\n46032\n45853\n45674\n45496\n45318\n45142\n44966\n44790\n44616\n44442\n44269\n44096\n43924\n43753\n43582\n43412\n43243\n43075\n42907\n42739\n42573\n42407\n42241\n42077\n41913\n41749\n41587\n41424\n41263\n41102\n40942\n40782\n40623\n40465\n40307\n40150\n39993\n39837\n39682\n39527\n39373\n39220\n39067\n38915\n38763\n38612\n38461\n38311\n38162\n38013\n37865\n37717\n37570\n37424\n37278\n37133\n36988\n36844\n36700\n36557\n36414\n36272\n36131\n35990\n35850\n35710\n35571\n35432\n35294\n35156\n35019\n34883\n34747\n34611\n34476\n34342\n34208\n34075\n33942\n33810\n33678\n33547\n33416\n33285\n33156\n33026\n32898\n32769\n32642\n32514\n32388\n32261\n32136\n32010\n31886\n31761\n31637\n31514\n31391\n31269\n31147\n31025\n30905\n30784\n30664\n30544\n30425\n30307\n30189\n30071\n29954\n29837\n29721\n29605\n29489\n29374\n29260\n29146\n29032\n28919\n28806\n28694\n28582\n28471\n28360\n28249\n28139\n28029\n27920\n27811\n27703\n27595\n27487\n27380\n27273\n27167\n27061\n26955\n26850\n26746\n26641\n26537\n26434\n26331\n26228\n26126\n26024\n25923\n25822\n25721\n25621\n25521\n25421\n25322\n25224\n25125\n25027\n24930\n24832\n24736\n24639\n24543\n24447\n24352\n24257\n24163\n24068\n23975\n23881\n23788\n23695\n23603\n23511\n23419\n23328\n23237\n23146\n23056\n22966\n22877\n22788\n22699\n22610\n22522\n22434\n22347\n22260\n22173\n22086\n22000\n21915\n21829\n21744\n21659\n21575\n21491\n21407\n21323\n21240\n21158\n21075\n20993\n20911\n20830\n20748\n20667\n20587\n20507\n20427\n20347\n20268\n20189\n20110\n20032\n19953\n19876\n19798\n19721\n19644\n19568\n19491\n19415\n19340\n19264\n19189\n19114\n19040\n18965\n18892\n18818\n18745\n18671\n18599\n18526\n18454\n18382\n18310\n18239\n18168\n18097\n18026\n17956\n17886\n17816\n17747\n17678\n17609\n17540\n17472\n17404\n17336\n17268\n17201\n17134\n17067\n17001\n16934\n16868\n16802\n16737\n16672\n16607\n16542\n16477\n16413\n16349\n16286\n16222\n16159\n16096\n16033\n15971\n15908\n15846\n15784\n15723\n15662\n15601\n15540\n15479\n15419\n15359\n15299\n15239\n15180\n15121\n15062\n15003\n14944\n14886\n14828\n14770\n14713\n14655\n14598\n14541\n14485\n14428\n14372\n14316\n14260\n14204\n14149\n14094\n14039\n13984\n13930\n13875\n13821\n13767\n13714\n13660\n13607\n13554\n13501\n13449\n13396\n13344\n13292\n13240\n13188\n13137\n13086\n13035\n12984\n12933\n12883\n12833\n12783\n12733\n12683\n12634\n12584\n12535\n12487\n12438\n12389\n12341\n12293\n12245\n12197\n12150\n12102\n12055\n12008\n11961\n11915\n11868\n11822\n11776\n11730\n11684\n11639\n11593\n11548\n11503\n11458\n11414\n11369\n11325\n11281\n11237\n11193\n11149\n11106\n11062\n11019\n10976\n10934\n10891\n10849\n10806\n10764\n10722\n10680\n10639\n10597\n10556\n10515\n10474\n10433\n10392\n10352\n10311\n10271\n10231\n10191\n10151\n10112\n10072\n10033\n9994\n9955\n9916\n9878\n9839\n9801\n9763\n9725\n9687\n9649\n9611\n9574\n9536\n9499\n9462\n9425\n9389\n9352\n9316\n9279\n9243\n9207\n9171\n9135\n9100\n9064\n9029\n8994\n8959\n8924\n8889\n8854\n8820\n8785\n8751\n8717\n8683\n8649\n8615\n8582\n8548\n8515\n8482\n8449\n8416\n8383\n8350\n8318\n8285\n8253\n8221\n8189\n8157\n8125\n8093\n8062\n8030\n7999\n7968\n7937\n7906\n7875\n7844\n7814\n7783\n7753\n7723\n7693\n7663\n7633\n7603\n7573\n7544\n7515\n7485\n7456\n7427\n7398\n7369\n7340\n7312\n7283\n7255\n7227\n7198\n7170\n7142\n7115\n7087\n7059\n7032\n7004\n6977\n6950\n6923\n6896\n6869\n6842\n6815\n6789\n6762\n6736\n6710\n6684\n6658\n6632\n6606\n6580\n6554\n6529\n6503\n6478\n6453\n6428\n6402\n6378\n6353\n6328\n6303\n6279\n6254\n6230\n6205\n6181\n6157\n6133\n6109\n6085\n6062\n6038\n6015\n5991\n5968\n5944\n5921\n5898\n5875\n5852\n5830\n5807\n5784\n5762\n5739\n5717\n5694\n5672\n5650\n5628\n5606\n5584\n5563\n5541\n5519\n5498\n5476\n5455\n5434\n5413\n5391\n5370\n5349\n5329\n5308\n5287\n5267\n5246\n5226\n5205\n5185\n5165\n5145\n5124\n5104\n5085\n5065\n5045\n5025\n5006\n4986\n4967\n4947\n4928\n4909\n4890\n4871\n4852\n4833\n4814\n4795\n4777\n4758\n4739\n4721\n4702\n4684\n4666\n4648\n4630\n4612\n4594\n4576\n4558\n4540\n4522\n4505\n4487\n4470\n4452\n4435\n4418\n4400\n4383\n4366\n4349\n4332\n4315\n4298\n4282\n4265\n4248\n4232\n4215\n4199\n4182\n4166\n4150\n4134\n4118\n4102\n4086\n4070\n4054\n4038\n4022\n4007\n3991\n3975\n3960\n3944\n3929\n3914\n3898\n3883\n3868\n3853\n3838\n3823\n3808\n3793\n3779\n3764\n3749\n3735\n3720\n3705\n3691\n3677\n3662\n3648\n3634\n3620\n3606\n3591\n3577\n3564\n3550\n3536\n3522\n3508\n3495\n3481\n3467\n3454\n3440\n3427\n3414\n3400\n3387\n3374\n3361\n3348\n3335\n3322\n3309\n3296\n3283\n3270\n3257\n3245\n3232\n3219\n3207\n3194\n3182\n3169\n3157\n3145\n3133\n3120\n3108\n3096\n3084\n3072\n3060\n3048\n3036\n3024\n3013\n3001\n2989\n2977\n2966\n2954\n2943\n2931\n2920\n2908\n2897\n2886\n2875\n2863\n2852\n2841\n2830\n2819\n2808\n2797\n2786\n2775\n2764\n2754\n2743\n2732\n2722\n2711\n2700\n2690\n2679\n2669\n2659\n2648\n2638\n2628\n2617\n2607\n2597\n2587\n2577\n2567\n2557\n2547\n2537\n2527\n2517\n2507\n2497\n2488\n2478\n2468\n2459\n2449\n2440\n2430\n2421\n2411\n2402\n2392\n2383\n2374\n2365\n2355\n2346\n2337\n2328\n2319\n2310\n2301\n2292\n2283\n2274\n2265\n2256\n2247\n2239\n2230\n2221\n2213\n2204\n2195\n2187\n2178\n2170\n2161\n2153\n2145\n2136\n2128\n2120\n2111\n2103\n2095\n2087\n2079\n2070\n2062\n2054\n2046\n2038\n2030\n2023\n2015\n2007\n1999\n1991\n1983\n1976\n1968\n1960\n1953\n1945\n1937\n1930\n1922\n1915\n1907\n1900\n1893\n1885\n1878\n1871\n1863\n1856\n1849\n1842\n1834\n1827\n1820\n1813\n1806\n1799\n1792\n1785\n1778\n1771\n1764\n1757\n1750\n1744\n1737\n1730\n1723\n1716\n1710\n1703\n1696\n1690\n1683\n1677\n1670\n1664\n1657\n1651\n1644\n1638\n1631\n1625\n1619\n1612\n1606\n1600\n1594\n1587\n1581\n1575\n1569\n1563\n1557\n1551\n1545\n1539\n1533\n1527\n1521\n1515\n1509\n1503\n1497\n1491\n1485\n1480\n1474\n1468\n1462\n1457\n1451\n1445\n1440\n1434\n1429\n1423\n1417\n1412\n1406\n1401\n1395\n1390\n1385\n1379\n1374\n1368\n1363\n1358\n1353\n1347\n1342\n1337\n1332\n1326\n1321\n1316\n1311\n1306\n1301\n1296\n1291\n1286\n1281\n1276\n1271\n1266\n1261\n1256\n1251\n1246\n1241\n1236\n1232\n1227\n1222\n1217\n1212\n1208\n1203\n1198\n1194\n1189\n1184\n1180\n1175\n1171\n1166\n1161\n1157\n1152\n1148\n1143\n1139\n1135\n1130\n1126\n1121\n1117\n1113\n1108\n1104\n1100\n1095\n1091\n1087\n1083\n1078\n1074\n1070\n1066\n1062\n1057\n1053\n1049\n1045\n1041\n1037\n1033\n1029\n1025\n1021\n1017\n1013\n1009\n1005\n1001\n997\n993\n990\n986\n982\n978\n974\n970\n967\n963\n959\n955\n952\n948\n944\n941\n937\n933\n930\n926\n922\n919\n915\n912\n908\n905\n901\n897\n894\n890\n887\n884\n880\n877\n873\n870\n866\n863\n860\n856\n853\n850\n846\n843\n840\n837\n833\n830\n827\n824\n820\n817\n814\n811\n808\n804\n801\n798\n795\n792\n789\n786\n783\n780\n777\n774\n771\n768\n765\n762\n759\n756\n753\n750\n747\n744\n741\n738\n735\n733\n730\n727\n724\n721\n718\n716\n713\n710\n707\n704\n702\n699\n696\n694\n691\n688\n685\n683\n680\n677\n675\n672\n670\n667\n664\n662\n659\n657\n654\n652\n649\n646\n644\n641\n639\n636\n634\n631\n629\n627\n624\n622\n619\n617\n614\n612\n610\n607\n605\n603\n600\n598\n596\n593\n591\n589\n586\n584\n582\n579\n577\n575\n573\n570\n568\n566\n564\n562\n559\n557\n555\n553\n551\n549\n546\n544\n542\n540\n538\n536\n534\n532\n530\n528\n526\n523\n521\n519\n517\n515\n513\n511\n509\n507\n505\n503\n501\n500\n498\n496\n494\n492\n490\n488\n486\n484\n482\n480\n479\n477\n475\n473\n471\n469\n467\n466\n464\n462\n460\n458\n457\n455\n453\n451\n450\n448\n446\n444\n443\n441\n439\n437\n436\n434\n432\n431\n429\n427\n426\n424\n422\n421\n419\n417\n416\n414\n413\n411\n409\n408\n406\n405\n403\n401\n400\n398\n397\n395\n394\n392\n391\n389\n388\n386\n384\n383\n382\n380\n379\n377\n376\n374\n373\n371\n370\n368\n367\n365\n364\n363\n361\n360\n358\n357\n356\n354\n353\n351\n350\n349\n347\n346\n345\n343\n342\n341\n339\n338\n337\n335\n334\n333\n331\n330\n329\n328\n326\n325\n324\n323\n321\n320\n319\n318\n316\n315\n314\n313\n311\n310\n309\n308\n307\n305\n304\n303\n302\n301\n299\n298\n297\n296\n295\n294\n293\n291\n290\n289\n288\n287\n286\n285\n284\n282\n281\n280\n279\n278\n277\n276\n275\n274\n273\n272\n271\n269\n268\n267\n266\n265\n264\n263\n262\n261\n260\n259\n258\n257\n256\n255\n254\n253\n252\n251\n250\n249\n248\n247\n246\n245\n244\n243\n242\n242\n241\n240\n239\n238\n237\n236\n235\n234\n233\n232\n231\n230\n230\n229\n228\n227\n226\n225\n224\n223\n223\n222\n221\n220\n219\n218\n217\n217\n216\n215\n214\n213\n212\n212\n211\n210\n209\n208\n207\n207\n206\n205\n204\n203\n203\n202\n201\n200\n199\n199\n198\n197\n196\n196\n195\n194\n193\n193\n192\n191\n190\n190\n189\n188\n187\n187\n186\n185\n184\n184\n183\n182\n182\n181\n180\n180\n179\n178\n177\n177\n176\n175\n175\n174\n173\n173\n172\n171\n171\n170\n169\n169\n168\n167\n167\n166\n165\n165\n164\n163\n163\n162\n162\n161\n160\n160\n159\n158\n158\n157\n157\n156\n155\n155\n154\n154\n153\n152\n152\n151\n151\n150\n149\n149\n148\n148\n147\n147\n146\n145\n145\n144\n144\n143\n143\n142\n141\n141\n140\n140\n139\n139\n138\n138\n137\n137\n136\n135\n135\n134\n134\n133\n133\n132\n132\n131\n131\n130\n130\n129\n129\n128\n128\n127\n127\n126\n126\n125\n125\n124\n124\n123\n123\n122\n122\n121\n121\n121\n120\n120\n119\n119\n118\n118\n117\n117\n116\n116\n115\n115\n115\n114\n114\n113\n113\n112\n112\n111\n111\n111\n110\n110\n109\n109\n108\n108\n108\n107\n107\n106\n106\n106\n105\n105\n104\n104\n103\n103\n103\n102\n102\n101\n101\n101\n100\n100\n100\n99\n99\n98\n98\n98\n97\n97\n96\n96\n96\n95\n95\n95\n94\n94\n93\n93\n93\n92\n92\n92\n91\n91\n91\n90\n90\n90\n89\n89\n89\n88\n88\n87\n87\n87\n86\n86\n86\n85\n85\n85\n84\n84\n84\n83\n83\n83\n83\n82\n82\n82\n81\n81\n81\n80\n80\n80\n79\n79\n79\n78\n78\n78\n78\n77\n77\n77\n76\n76\n76\n75\n75\n75\n75\n74\n74\n74\n73\n73\n73\n73\n72\n72\n72\n71\n71\n71\n71\n70\n70\n70\n69\n69\n69\n69\n68\n68\n68\n68\n67\n67\n67\n67\n66\n66\n66\n66\n65\n65\n65\n65\n64\n64\n64\n64\n63\n63\n63\n63\n62\n62\n62\n62\n61\n61\n61\n61\n60\n60\n60\n60\n59\n59\n59\n59\n59\n58\n58\n58\n58\n57\n57\n57\n57\n56\n56\n56\n56\n56\n55\n55\n55\n55\n55\n54\n54\n54\n54\n53\n53\n53\n53\n53\n52\n52\n52\n52\n52\n51\n51\n51\n51\n51\n50\n50\n50\n50\n50\n49\n49\n49\n49\n49\n49\n48\n48\n48\n48\n48\n47\n47\n47\n47\n47\n46\n46\n46\n46\n46\n46\n45\n45\n45\n45\n45\n45\n44\n44\n44\n44\n44\n43\n43\n43\n43\n43\n43\n42\n42\n42\n42\n42\n42\n41\n41\n41\n41\n41\n41\n41\n40\n40\n40\n40\n40\n40\n39\n39\n39\n39\n39\n39\n39\n38\n38\n38\n38\n38\n38\n37\n37\n37\n37\n37\n37\n37\n36\n36\n36\n36\n36\n36\n36\n35\n35\n35\n35\n35\n35\n35\n35\n34\n34\n34\n34\n34\n34\n34\n33\n33\n33\n33\n33\n33\n33\n33\n32\n32\n32\n32\n32\n32\n32\n32\n31\n31\n31\n31\n31\n31\n31\n31\n30\n30\n30\n30\n30\n30\n30\n30\n30\n29\n29\n29\n29\n29\n29\n29\n29\n29\n28\n28\n28\n28\n28\n28\n28\n28\n28\n27\n27\n27\n27\n27\n27\n27\n27\n27\n26\n26\n26\n26\n26\n26\n26\n26\n26\n26\n25\n25\n25\n25\n25\n25\n25\n25\n25\n25\n24\n24\n24\n24\n24\n24\n24\n24\n24\n24\n24\n23\n23\n23\n23\n23\n23\n23\n23\n23\n23\n23\n22\n22\n22\n22\n22\n22\n22\n22\n22\n22\n22\n22\n21\n21\n21\n21\n21\n21\n21\n21\n21\n21\n21\n21\n20\n20\n20\n20\n20\n20\n20\n20\n20\n20\n20\n20\n20\n19\n19\n19\n19\n19\n19\n19\n19\n19\n19\n19\n19\n19\n18\n18\n18\n18\n18\n18\n18\n18\n18\n18\n18\n18\n18\n18\n17\n17\n17\n17\n17\n17\n17\n17\n17\n17\n17\n17\n17\n17\n17\n17\n16\n16\n16\n16\n16\n16\n16\n16\n16\n16\n16\n16\n16\n16\n16\n16\n15\n15\n15\n15\n15\n15\n15\n15\n15\n15\n15\n15\n15\n15\n15\n15\n15\n14\n14\n14\n14\n14\n14\n14\n14\n14\n14\n14\n14\n14\n14\n14\n14\n14\n14\n13\n13\n13\n13\n13\n13\n13\n13\n13\n13\n13\n13\n13\n13\n13\n13\n13\n13\n13\n13\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n12\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n11\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n10\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n9\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n8\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n7\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n6\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n5\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n4\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n2\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n1\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n')
FILES['tables/silu.txt'] = ('f4fe2a2c80ed355c4b2b3841c90de0c9a4f4f22a5ae13cc79f7b8d01d3010edb', '# silu.txt\n# S[k] = round(2^16 x / (1 + exp(-x))), x = -16 + k/256, k = 0 .. 8192\n# 8193 values, one per line, generated by make_tables.py\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-1\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-2\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-3\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-4\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-5\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-6\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-7\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-8\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-9\n-10\n-10\n-10\n-10\n-10\n-10\n-10\n-10\n-10\n-10\n-10\n-10\n-10\n-10\n-10\n-10\n-10\n-10\n-10\n-10\n-10\n-10\n-10\n-10\n-10\n-10\n-10\n-10\n-11\n-11\n-11\n-11\n-11\n-11\n-11\n-11\n-11\n-11\n-11\n-11\n-11\n-11\n-11\n-11\n-11\n-11\n-11\n-11\n-11\n-11\n-11\n-11\n-11\n-11\n-12\n-12\n-12\n-12\n-12\n-12\n-12\n-12\n-12\n-12\n-12\n-12\n-12\n-12\n-12\n-12\n-12\n-12\n-12\n-12\n-12\n-12\n-12\n-13\n-13\n-13\n-13\n-13\n-13\n-13\n-13\n-13\n-13\n-13\n-13\n-13\n-13\n-13\n-13\n-13\n-13\n-13\n-13\n-13\n-13\n-14\n-14\n-14\n-14\n-14\n-14\n-14\n-14\n-14\n-14\n-14\n-14\n-14\n-14\n-14\n-14\n-14\n-14\n-14\n-14\n-15\n-15\n-15\n-15\n-15\n-15\n-15\n-15\n-15\n-15\n-15\n-15\n-15\n-15\n-15\n-15\n-15\n-15\n-15\n-16\n-16\n-16\n-16\n-16\n-16\n-16\n-16\n-16\n-16\n-16\n-16\n-16\n-16\n-16\n-16\n-16\n-17\n-17\n-17\n-17\n-17\n-17\n-17\n-17\n-17\n-17\n-17\n-17\n-17\n-17\n-17\n-17\n-17\n-18\n-18\n-18\n-18\n-18\n-18\n-18\n-18\n-18\n-18\n-18\n-18\n-18\n-18\n-18\n-18\n-19\n-19\n-19\n-19\n-19\n-19\n-19\n-19\n-19\n-19\n-19\n-19\n-19\n-19\n-19\n-20\n-20\n-20\n-20\n-20\n-20\n-20\n-20\n-20\n-20\n-20\n-20\n-20\n-20\n-21\n-21\n-21\n-21\n-21\n-21\n-21\n-21\n-21\n-21\n-21\n-21\n-21\n-22\n-22\n-22\n-22\n-22\n-22\n-22\n-22\n-22\n-22\n-22\n-22\n-22\n-23\n-23\n-23\n-23\n-23\n-23\n-23\n-23\n-23\n-23\n-23\n-23\n-24\n-24\n-24\n-24\n-24\n-24\n-24\n-24\n-24\n-24\n-24\n-24\n-25\n-25\n-25\n-25\n-25\n-25\n-25\n-25\n-25\n-25\n-25\n-25\n-26\n-26\n-26\n-26\n-26\n-26\n-26\n-26\n-26\n-26\n-26\n-27\n-27\n-27\n-27\n-27\n-27\n-27\n-27\n-27\n-27\n-28\n-28\n-28\n-28\n-28\n-28\n-28\n-28\n-28\n-28\n-29\n-29\n-29\n-29\n-29\n-29\n-29\n-29\n-29\n-29\n-30\n-30\n-30\n-30\n-30\n-30\n-30\n-30\n-30\n-30\n-31\n-31\n-31\n-31\n-31\n-31\n-31\n-31\n-31\n-32\n-32\n-32\n-32\n-32\n-32\n-32\n-32\n-32\n-33\n-33\n-33\n-33\n-33\n-33\n-33\n-33\n-34\n-34\n-34\n-34\n-34\n-34\n-34\n-34\n-34\n-35\n-35\n-35\n-35\n-35\n-35\n-35\n-35\n-36\n-36\n-36\n-36\n-36\n-36\n-36\n-36\n-37\n-37\n-37\n-37\n-37\n-37\n-37\n-38\n-38\n-38\n-38\n-38\n-38\n-38\n-38\n-39\n-39\n-39\n-39\n-39\n-39\n-39\n-40\n-40\n-40\n-40\n-40\n-40\n-40\n-41\n-41\n-41\n-41\n-41\n-41\n-41\n-42\n-42\n-42\n-42\n-42\n-42\n-42\n-43\n-43\n-43\n-43\n-43\n-43\n-43\n-44\n-44\n-44\n-44\n-44\n-44\n-45\n-45\n-45\n-45\n-45\n-45\n-45\n-46\n-46\n-46\n-46\n-46\n-46\n-47\n-47\n-47\n-47\n-47\n-47\n-48\n-48\n-48\n-48\n-48\n-48\n-49\n-49\n-49\n-49\n-49\n-49\n-50\n-50\n-50\n-50\n-50\n-50\n-51\n-51\n-51\n-51\n-51\n-52\n-52\n-52\n-52\n-52\n-52\n-53\n-53\n-53\n-53\n-53\n-54\n-54\n-54\n-54\n-54\n-55\n-55\n-55\n-55\n-55\n-55\n-56\n-56\n-56\n-56\n-56\n-57\n-57\n-57\n-57\n-57\n-58\n-58\n-58\n-58\n-58\n-59\n-59\n-59\n-59\n-59\n-60\n-60\n-60\n-60\n-61\n-61\n-61\n-61\n-61\n-62\n-62\n-62\n-62\n-62\n-63\n-63\n-63\n-63\n-64\n-64\n-64\n-64\n-64\n-65\n-65\n-65\n-65\n-66\n-66\n-66\n-66\n-66\n-67\n-67\n-67\n-67\n-68\n-68\n-68\n-68\n-69\n-69\n-69\n-69\n-70\n-70\n-70\n-70\n-71\n-71\n-71\n-71\n-72\n-72\n-72\n-72\n-73\n-73\n-73\n-73\n-74\n-74\n-74\n-74\n-75\n-75\n-75\n-75\n-76\n-76\n-76\n-76\n-77\n-77\n-77\n-77\n-78\n-78\n-78\n-79\n-79\n-79\n-79\n-80\n-80\n-80\n-80\n-81\n-81\n-81\n-82\n-82\n-82\n-82\n-83\n-83\n-83\n-84\n-84\n-84\n-84\n-85\n-85\n-85\n-86\n-86\n-86\n-87\n-87\n-87\n-87\n-88\n-88\n-88\n-89\n-89\n-89\n-90\n-90\n-90\n-91\n-91\n-91\n-91\n-92\n-92\n-92\n-93\n-93\n-93\n-94\n-94\n-94\n-95\n-95\n-95\n-96\n-96\n-96\n-97\n-97\n-97\n-98\n-98\n-98\n-99\n-99\n-99\n-100\n-100\n-100\n-101\n-101\n-101\n-102\n-102\n-103\n-103\n-103\n-104\n-104\n-104\n-105\n-105\n-105\n-106\n-106\n-107\n-107\n-107\n-108\n-108\n-108\n-109\n-109\n-109\n-110\n-110\n-111\n-111\n-111\n-112\n-112\n-113\n-113\n-113\n-114\n-114\n-114\n-115\n-115\n-116\n-116\n-116\n-117\n-117\n-118\n-118\n-119\n-119\n-119\n-120\n-120\n-121\n-121\n-121\n-122\n-122\n-123\n-123\n-124\n-124\n-124\n-125\n-125\n-126\n-126\n-127\n-127\n-127\n-128\n-128\n-129\n-129\n-130\n-130\n-130\n-131\n-131\n-132\n-132\n-133\n-133\n-134\n-134\n-135\n-135\n-136\n-136\n-136\n-137\n-137\n-138\n-138\n-139\n-139\n-140\n-140\n-141\n-141\n-142\n-142\n-143\n-143\n-144\n-144\n-145\n-145\n-146\n-146\n-147\n-147\n-148\n-148\n-149\n-149\n-150\n-150\n-151\n-151\n-152\n-152\n-153\n-153\n-154\n-154\n-155\n-155\n-156\n-157\n-157\n-158\n-158\n-159\n-159\n-160\n-160\n-161\n-161\n-162\n-163\n-163\n-164\n-164\n-165\n-165\n-166\n-166\n-167\n-168\n-168\n-169\n-169\n-170\n-170\n-171\n-172\n-172\n-173\n-173\n-174\n-175\n-175\n-176\n-176\n-177\n-178\n-178\n-179\n-179\n-180\n-181\n-181\n-182\n-183\n-183\n-184\n-184\n-185\n-186\n-186\n-187\n-188\n-188\n-189\n-190\n-190\n-191\n-191\n-192\n-193\n-193\n-194\n-195\n-195\n-196\n-197\n-197\n-198\n-199\n-199\n-200\n-201\n-202\n-202\n-203\n-204\n-204\n-205\n-206\n-206\n-207\n-208\n-209\n-209\n-210\n-211\n-211\n-212\n-213\n-214\n-214\n-215\n-216\n-216\n-217\n-218\n-219\n-219\n-220\n-221\n-222\n-222\n-223\n-224\n-225\n-225\n-226\n-227\n-228\n-229\n-229\n-230\n-231\n-232\n-232\n-233\n-234\n-235\n-236\n-236\n-237\n-238\n-239\n-240\n-241\n-241\n-242\n-243\n-244\n-245\n-245\n-246\n-247\n-248\n-249\n-250\n-250\n-251\n-252\n-253\n-254\n-255\n-256\n-257\n-257\n-258\n-259\n-260\n-261\n-262\n-263\n-264\n-264\n-265\n-266\n-267\n-268\n-269\n-270\n-271\n-272\n-273\n-274\n-274\n-275\n-276\n-277\n-278\n-279\n-280\n-281\n-282\n-283\n-284\n-285\n-286\n-287\n-288\n-289\n-290\n-291\n-292\n-293\n-294\n-295\n-296\n-297\n-298\n-299\n-300\n-301\n-302\n-303\n-304\n-305\n-306\n-307\n-308\n-309\n-310\n-311\n-312\n-313\n-314\n-315\n-316\n-317\n-318\n-320\n-321\n-322\n-323\n-324\n-325\n-326\n-327\n-328\n-329\n-330\n-332\n-333\n-334\n-335\n-336\n-337\n-338\n-339\n-341\n-342\n-343\n-344\n-345\n-346\n-348\n-349\n-350\n-351\n-352\n-353\n-355\n-356\n-357\n-358\n-359\n-361\n-362\n-363\n-364\n-366\n-367\n-368\n-369\n-370\n-372\n-373\n-374\n-375\n-377\n-378\n-379\n-381\n-382\n-383\n-384\n-386\n-387\n-388\n-390\n-391\n-392\n-394\n-395\n-396\n-397\n-399\n-400\n-401\n-403\n-404\n-406\n-407\n-408\n-410\n-411\n-412\n-414\n-415\n-417\n-418\n-419\n-421\n-422\n-424\n-425\n-426\n-428\n-429\n-431\n-432\n-434\n-435\n-437\n-438\n-439\n-441\n-442\n-444\n-445\n-447\n-448\n-450\n-451\n-453\n-454\n-456\n-457\n-459\n-460\n-462\n-464\n-465\n-467\n-468\n-470\n-471\n-473\n-474\n-476\n-478\n-479\n-481\n-482\n-484\n-486\n-487\n-489\n-491\n-492\n-494\n-495\n-497\n-499\n-500\n-502\n-504\n-505\n-507\n-509\n-511\n-512\n-514\n-516\n-517\n-519\n-521\n-523\n-524\n-526\n-528\n-530\n-531\n-533\n-535\n-537\n-538\n-540\n-542\n-544\n-546\n-547\n-549\n-551\n-553\n-555\n-557\n-558\n-560\n-562\n-564\n-566\n-568\n-570\n-571\n-573\n-575\n-577\n-579\n-581\n-583\n-585\n-587\n-589\n-591\n-593\n-595\n-597\n-599\n-601\n-603\n-605\n-607\n-609\n-611\n-613\n-615\n-617\n-619\n-621\n-623\n-625\n-627\n-629\n-631\n-633\n-635\n-637\n-639\n-642\n-644\n-646\n-648\n-650\n-652\n-654\n-657\n-659\n-661\n-663\n-665\n-667\n-670\n-672\n-674\n-676\n-679\n-681\n-683\n-685\n-688\n-690\n-692\n-694\n-697\n-699\n-701\n-704\n-706\n-708\n-711\n-713\n-715\n-718\n-720\n-722\n-725\n-727\n-729\n-732\n-734\n-737\n-739\n-742\n-744\n-746\n-749\n-751\n-754\n-756\n-759\n-761\n-764\n-766\n-769\n-771\n-774\n-776\n-779\n-781\n-784\n-787\n-789\n-792\n-794\n-797\n-800\n-802\n-805\n-807\n-810\n-813\n-815\n-818\n-821\n-823\n-826\n-829\n-832\n-834\n-837\n-840\n-843\n-845\n-848\n-851\n-854\n-856\n-859\n-862\n-865\n-868\n-870\n-873\n-876\n-879\n-882\n-885\n-888\n-891\n-893\n-896\n-899\n-902\n-905\n-908\n-911\n-914\n-917\n-920\n-923\n-926\n-929\n-932\n-935\n-938\n-941\n-944\n-947\n-950\n-954\n-957\n-960\n-963\n-966\n-969\n-972\n-975\n-979\n-982\n-985\n-988\n-991\n-995\n-998\n-1001\n-1004\n-1008\n-1011\n-1014\n-1017\n-1021\n-1024\n-1027\n-1031\n-1034\n-1037\n-1041\n-1044\n-1048\n-1051\n-1054\n-1058\n-1061\n-1065\n-1068\n-1071\n-1075\n-1078\n-1082\n-1085\n-1089\n-1092\n-1096\n-1100\n-1103\n-1107\n-1110\n-1114\n-1117\n-1121\n-1125\n-1128\n-1132\n-1136\n-1139\n-1143\n-1147\n-1150\n-1154\n-1158\n-1161\n-1165\n-1169\n-1173\n-1176\n-1180\n-1184\n-1188\n-1192\n-1196\n-1199\n-1203\n-1207\n-1211\n-1215\n-1219\n-1223\n-1227\n-1231\n-1235\n-1239\n-1243\n-1247\n-1251\n-1255\n-1259\n-1263\n-1267\n-1271\n-1275\n-1279\n-1283\n-1287\n-1291\n-1295\n-1300\n-1304\n-1308\n-1312\n-1316\n-1321\n-1325\n-1329\n-1333\n-1338\n-1342\n-1346\n-1350\n-1355\n-1359\n-1363\n-1368\n-1372\n-1377\n-1381\n-1385\n-1390\n-1394\n-1399\n-1403\n-1408\n-1412\n-1417\n-1421\n-1426\n-1430\n-1435\n-1439\n-1444\n-1449\n-1453\n-1458\n-1462\n-1467\n-1472\n-1476\n-1481\n-1486\n-1491\n-1495\n-1500\n-1505\n-1510\n-1514\n-1519\n-1524\n-1529\n-1534\n-1539\n-1544\n-1548\n-1553\n-1558\n-1563\n-1568\n-1573\n-1578\n-1583\n-1588\n-1593\n-1598\n-1603\n-1608\n-1614\n-1619\n-1624\n-1629\n-1634\n-1639\n-1644\n-1650\n-1655\n-1660\n-1665\n-1671\n-1676\n-1681\n-1686\n-1692\n-1697\n-1702\n-1708\n-1713\n-1719\n-1724\n-1730\n-1735\n-1740\n-1746\n-1751\n-1757\n-1762\n-1768\n-1774\n-1779\n-1785\n-1790\n-1796\n-1802\n-1807\n-1813\n-1819\n-1824\n-1830\n-1836\n-1842\n-1848\n-1853\n-1859\n-1865\n-1871\n-1877\n-1883\n-1888\n-1894\n-1900\n-1906\n-1912\n-1918\n-1924\n-1930\n-1936\n-1942\n-1949\n-1955\n-1961\n-1967\n-1973\n-1979\n-1985\n-1992\n-1998\n-2004\n-2010\n-2017\n-2023\n-2029\n-2036\n-2042\n-2048\n-2055\n-2061\n-2067\n-2074\n-2080\n-2087\n-2093\n-2100\n-2106\n-2113\n-2120\n-2126\n-2133\n-2139\n-2146\n-2153\n-2159\n-2166\n-2173\n-2180\n-2186\n-2193\n-2200\n-2207\n-2214\n-2220\n-2227\n-2234\n-2241\n-2248\n-2255\n-2262\n-2269\n-2276\n-2283\n-2290\n-2297\n-2304\n-2312\n-2319\n-2326\n-2333\n-2340\n-2347\n-2355\n-2362\n-2369\n-2377\n-2384\n-2391\n-2399\n-2406\n-2413\n-2421\n-2428\n-2436\n-2443\n-2451\n-2458\n-2466\n-2473\n-2481\n-2489\n-2496\n-2504\n-2512\n-2519\n-2527\n-2535\n-2543\n-2550\n-2558\n-2566\n-2574\n-2582\n-2590\n-2598\n-2606\n-2614\n-2622\n-2630\n-2638\n-2646\n-2654\n-2662\n-2670\n-2678\n-2686\n-2695\n-2703\n-2711\n-2719\n-2728\n-2736\n-2744\n-2753\n-2761\n-2770\n-2778\n-2786\n-2795\n-2803\n-2812\n-2820\n-2829\n-2838\n-2846\n-2855\n-2864\n-2872\n-2881\n-2890\n-2898\n-2907\n-2916\n-2925\n-2934\n-2943\n-2952\n-2961\n-2969\n-2978\n-2987\n-2997\n-3006\n-3015\n-3024\n-3033\n-3042\n-3051\n-3060\n-3070\n-3079\n-3088\n-3098\n-3107\n-3116\n-3126\n-3135\n-3144\n-3154\n-3163\n-3173\n-3182\n-3192\n-3202\n-3211\n-3221\n-3230\n-3240\n-3250\n-3260\n-3269\n-3279\n-3289\n-3299\n-3309\n-3319\n-3329\n-3339\n-3349\n-3359\n-3369\n-3379\n-3389\n-3399\n-3409\n-3419\n-3429\n-3440\n-3450\n-3460\n-3470\n-3481\n-3491\n-3501\n-3512\n-3522\n-3533\n-3543\n-3554\n-3564\n-3575\n-3586\n-3596\n-3607\n-3618\n-3628\n-3639\n-3650\n-3661\n-3671\n-3682\n-3693\n-3704\n-3715\n-3726\n-3737\n-3748\n-3759\n-3770\n-3781\n-3793\n-3804\n-3815\n-3826\n-3837\n-3849\n-3860\n-3871\n-3883\n-3894\n-3906\n-3917\n-3929\n-3940\n-3952\n-3963\n-3975\n-3987\n-3998\n-4010\n-4022\n-4033\n-4045\n-4057\n-4069\n-4081\n-4093\n-4105\n-4117\n-4129\n-4141\n-4153\n-4165\n-4177\n-4189\n-4201\n-4214\n-4226\n-4238\n-4250\n-4263\n-4275\n-4288\n-4300\n-4313\n-4325\n-4338\n-4350\n-4363\n-4375\n-4388\n-4401\n-4413\n-4426\n-4439\n-4452\n-4465\n-4478\n-4490\n-4503\n-4516\n-4529\n-4542\n-4556\n-4569\n-4582\n-4595\n-4608\n-4621\n-4635\n-4648\n-4661\n-4675\n-4688\n-4702\n-4715\n-4728\n-4742\n-4756\n-4769\n-4783\n-4796\n-4810\n-4824\n-4838\n-4851\n-4865\n-4879\n-4893\n-4907\n-4921\n-4935\n-4949\n-4963\n-4977\n-4991\n-5005\n-5020\n-5034\n-5048\n-5062\n-5077\n-5091\n-5105\n-5120\n-5134\n-5149\n-5163\n-5178\n-5193\n-5207\n-5222\n-5237\n-5251\n-5266\n-5281\n-5296\n-5311\n-5326\n-5341\n-5356\n-5371\n-5386\n-5401\n-5416\n-5431\n-5446\n-5461\n-5477\n-5492\n-5507\n-5523\n-5538\n-5554\n-5569\n-5585\n-5600\n-5616\n-5631\n-5647\n-5663\n-5678\n-5694\n-5710\n-5726\n-5742\n-5757\n-5773\n-5789\n-5805\n-5821\n-5838\n-5854\n-5870\n-5886\n-5902\n-5918\n-5935\n-5951\n-5967\n-5984\n-6000\n-6017\n-6033\n-6050\n-6066\n-6083\n-6100\n-6116\n-6133\n-6150\n-6167\n-6183\n-6200\n-6217\n-6234\n-6251\n-6268\n-6285\n-6302\n-6319\n-6337\n-6354\n-6371\n-6388\n-6406\n-6423\n-6440\n-6458\n-6475\n-6493\n-6510\n-6528\n-6545\n-6563\n-6581\n-6598\n-6616\n-6634\n-6652\n-6670\n-6688\n-6706\n-6724\n-6742\n-6760\n-6778\n-6796\n-6814\n-6832\n-6850\n-6869\n-6887\n-6905\n-6924\n-6942\n-6961\n-6979\n-6998\n-7016\n-7035\n-7053\n-7072\n-7091\n-7109\n-7128\n-7147\n-7166\n-7185\n-7204\n-7223\n-7242\n-7261\n-7280\n-7299\n-7318\n-7337\n-7357\n-7376\n-7395\n-7414\n-7434\n-7453\n-7473\n-7492\n-7512\n-7531\n-7551\n-7570\n-7590\n-7610\n-7630\n-7649\n-7669\n-7689\n-7709\n-7729\n-7749\n-7769\n-7789\n-7809\n-7829\n-7849\n-7869\n-7889\n-7910\n-7930\n-7950\n-7971\n-7991\n-8012\n-8032\n-8052\n-8073\n-8094\n-8114\n-8135\n-8156\n-8176\n-8197\n-8218\n-8239\n-8259\n-8280\n-8301\n-8322\n-8343\n-8364\n-8385\n-8407\n-8428\n-8449\n-8470\n-8491\n-8513\n-8534\n-8555\n-8577\n-8598\n-8620\n-8641\n-8663\n-8684\n-8706\n-8727\n-8749\n-8771\n-8793\n-8814\n-8836\n-8858\n-8880\n-8902\n-8924\n-8946\n-8968\n-8990\n-9012\n-9034\n-9056\n-9078\n-9100\n-9123\n-9145\n-9167\n-9190\n-9212\n-9234\n-9257\n-9279\n-9302\n-9324\n-9347\n-9369\n-9392\n-9415\n-9437\n-9460\n-9483\n-9506\n-9529\n-9551\n-9574\n-9597\n-9620\n-9643\n-9666\n-9689\n-9712\n-9735\n-9759\n-9782\n-9805\n-9828\n-9851\n-9875\n-9898\n-9921\n-9945\n-9968\n-9992\n-10015\n-10039\n-10062\n-10086\n-10109\n-10133\n-10156\n-10180\n-10204\n-10227\n-10251\n-10275\n-10299\n-10323\n-10346\n-10370\n-10394\n-10418\n-10442\n-10466\n-10490\n-10514\n-10538\n-10562\n-10586\n-10611\n-10635\n-10659\n-10683\n-10707\n-10732\n-10756\n-10780\n-10805\n-10829\n-10853\n-10878\n-10902\n-10927\n-10951\n-10976\n-11000\n-11025\n-11049\n-11074\n-11099\n-11123\n-11148\n-11173\n-11197\n-11222\n-11247\n-11271\n-11296\n-11321\n-11346\n-11371\n-11396\n-11420\n-11445\n-11470\n-11495\n-11520\n-11545\n-11570\n-11595\n-11620\n-11645\n-11670\n-11695\n-11720\n-11746\n-11771\n-11796\n-11821\n-11846\n-11871\n-11897\n-11922\n-11947\n-11972\n-11997\n-12023\n-12048\n-12073\n-12099\n-12124\n-12149\n-12175\n-12200\n-12225\n-12251\n-12276\n-12301\n-12327\n-12352\n-12378\n-12403\n-12429\n-12454\n-12480\n-12505\n-12530\n-12556\n-12581\n-12607\n-12632\n-12658\n-12683\n-12709\n-12734\n-12760\n-12786\n-12811\n-12837\n-12862\n-12888\n-12913\n-12939\n-12964\n-12990\n-13015\n-13041\n-13067\n-13092\n-13118\n-13143\n-13169\n-13194\n-13220\n-13245\n-13271\n-13297\n-13322\n-13348\n-13373\n-13399\n-13424\n-13450\n-13475\n-13501\n-13526\n-13552\n-13577\n-13603\n-13628\n-13654\n-13679\n-13705\n-13730\n-13756\n-13781\n-13806\n-13832\n-13857\n-13883\n-13908\n-13933\n-13959\n-13984\n-14009\n-14035\n-14060\n-14085\n-14110\n-14136\n-14161\n-14186\n-14211\n-14236\n-14262\n-14287\n-14312\n-14337\n-14362\n-14387\n-14412\n-14437\n-14462\n-14487\n-14512\n-14537\n-14562\n-14587\n-14611\n-14636\n-14661\n-14686\n-14710\n-14735\n-14760\n-14784\n-14809\n-14834\n-14858\n-14883\n-14907\n-14932\n-14956\n-14980\n-15005\n-15029\n-15053\n-15078\n-15102\n-15126\n-15150\n-15174\n-15198\n-15222\n-15246\n-15270\n-15294\n-15318\n-15342\n-15366\n-15389\n-15413\n-15437\n-15460\n-15484\n-15507\n-15531\n-15554\n-15578\n-15601\n-15624\n-15647\n-15671\n-15694\n-15717\n-15740\n-15763\n-15786\n-15808\n-15831\n-15854\n-15877\n-15899\n-15922\n-15944\n-15967\n-15989\n-16011\n-16034\n-16056\n-16078\n-16100\n-16122\n-16144\n-16166\n-16188\n-16210\n-16231\n-16253\n-16274\n-16296\n-16317\n-16339\n-16360\n-16381\n-16402\n-16423\n-16444\n-16465\n-16486\n-16507\n-16527\n-16548\n-16568\n-16589\n-16609\n-16629\n-16650\n-16670\n-16690\n-16710\n-16729\n-16749\n-16769\n-16788\n-16808\n-16827\n-16847\n-16866\n-16885\n-16904\n-16923\n-16942\n-16961\n-16979\n-16998\n-17016\n-17035\n-17053\n-17071\n-17089\n-17107\n-17125\n-17143\n-17160\n-17178\n-17195\n-17213\n-17230\n-17247\n-17264\n-17281\n-17298\n-17314\n-17331\n-17347\n-17364\n-17380\n-17396\n-17412\n-17428\n-17443\n-17459\n-17475\n-17490\n-17505\n-17520\n-17535\n-17550\n-17565\n-17580\n-17594\n-17608\n-17623\n-17637\n-17651\n-17665\n-17678\n-17692\n-17705\n-17719\n-17732\n-17745\n-17758\n-17770\n-17783\n-17795\n-17808\n-17820\n-17832\n-17844\n-17855\n-17867\n-17878\n-17890\n-17901\n-17912\n-17923\n-17933\n-17944\n-17954\n-17964\n-17974\n-17984\n-17994\n-18003\n-18013\n-18022\n-18031\n-18040\n-18048\n-18057\n-18065\n-18074\n-18082\n-18089\n-18097\n-18105\n-18112\n-18119\n-18126\n-18133\n-18140\n-18146\n-18152\n-18158\n-18164\n-18170\n-18175\n-18181\n-18186\n-18191\n-18195\n-18200\n-18204\n-18209\n-18213\n-18216\n-18220\n-18223\n-18226\n-18229\n-18232\n-18235\n-18237\n-18239\n-18241\n-18243\n-18245\n-18246\n-18247\n-18248\n-18249\n-18249\n-18249\n-18249\n-18249\n-18249\n-18248\n-18247\n-18246\n-18245\n-18244\n-18242\n-18240\n-18238\n-18235\n-18233\n-18230\n-18227\n-18223\n-18220\n-18216\n-18212\n-18208\n-18203\n-18198\n-18193\n-18188\n-18182\n-18177\n-18171\n-18164\n-18158\n-18151\n-18144\n-18137\n-18129\n-18122\n-18114\n-18105\n-18097\n-18088\n-18079\n-18070\n-18060\n-18050\n-18040\n-18030\n-18019\n-18008\n-17997\n-17985\n-17974\n-17962\n-17949\n-17937\n-17924\n-17911\n-17897\n-17884\n-17870\n-17855\n-17841\n-17826\n-17811\n-17796\n-17780\n-17764\n-17748\n-17731\n-17714\n-17697\n-17680\n-17662\n-17644\n-17625\n-17607\n-17588\n-17568\n-17549\n-17529\n-17509\n-17488\n-17467\n-17446\n-17425\n-17403\n-17381\n-17359\n-17336\n-17313\n-17290\n-17266\n-17242\n-17218\n-17193\n-17168\n-17143\n-17117\n-17091\n-17065\n-17038\n-17011\n-16984\n-16956\n-16928\n-16900\n-16871\n-16842\n-16813\n-16784\n-16754\n-16723\n-16692\n-16661\n-16630\n-16598\n-16566\n-16534\n-16501\n-16468\n-16434\n-16400\n-16366\n-16332\n-16297\n-16261\n-16226\n-16190\n-16153\n-16117\n-16079\n-16042\n-16004\n-15966\n-15927\n-15888\n-15849\n-15809\n-15769\n-15729\n-15688\n-15646\n-15605\n-15563\n-15520\n-15478\n-15434\n-15391\n-15347\n-15303\n-15258\n-15213\n-15167\n-15122\n-15075\n-15029\n-14982\n-14934\n-14886\n-14838\n-14789\n-14740\n-14691\n-14641\n-14591\n-14540\n-14489\n-14437\n-14386\n-14333\n-14281\n-14227\n-14174\n-14120\n-14066\n-14011\n-13956\n-13900\n-13844\n-13787\n-13731\n-13673\n-13616\n-13557\n-13499\n-13440\n-13380\n-13321\n-13260\n-13200\n-13139\n-13077\n-13015\n-12953\n-12890\n-12826\n-12763\n-12699\n-12634\n-12569\n-12504\n-12438\n-12371\n-12304\n-12237\n-12170\n-12101\n-12033\n-11964\n-11894\n-11825\n-11754\n-11683\n-11612\n-11540\n-11468\n-11396\n-11323\n-11249\n-11175\n-11101\n-11026\n-10950\n-10875\n-10798\n-10722\n-10645\n-10567\n-10489\n-10410\n-10331\n-10252\n-10172\n-10091\n-10011\n-9929\n-9847\n-9765\n-9682\n-9599\n-9516\n-9431\n-9347\n-9262\n-9176\n-9090\n-9004\n-8917\n-8829\n-8741\n-8653\n-8564\n-8475\n-8385\n-8295\n-8204\n-8112\n-8021\n-7928\n-7836\n-7743\n-7649\n-7555\n-7460\n-7365\n-7269\n-7173\n-7077\n-6980\n-6882\n-6784\n-6686\n-6587\n-6487\n-6387\n-6287\n-6186\n-6084\n-5982\n-5880\n-5777\n-5674\n-5570\n-5465\n-5360\n-5255\n-5149\n-5043\n-4936\n-4829\n-4721\n-4612\n-4504\n-4394\n-4285\n-4174\n-4063\n-3952\n-3840\n-3728\n-3615\n-3502\n-3388\n-3274\n-3159\n-3044\n-2928\n-2812\n-2695\n-2578\n-2460\n-2342\n-2223\n-2104\n-1984\n-1864\n-1743\n-1622\n-1500\n-1378\n-1255\n-1132\n-1008\n-884\n-759\n-634\n-508\n-382\n-255\n-128\n0\n128\n257\n386\n516\n646\n777\n908\n1040\n1172\n1305\n1438\n1572\n1706\n1841\n1976\n2112\n2248\n2385\n2522\n2660\n2798\n2937\n3076\n3216\n3356\n3497\n3638\n3780\n3922\n4065\n4208\n4352\n4496\n4641\n4786\n4931\n5078\n5224\n5372\n5519\n5667\n5816\n5965\n6115\n6265\n6416\n6567\n6718\n6870\n7023\n7176\n7330\n7484\n7638\n7793\n7949\n8105\n8261\n8418\n8576\n8734\n8892\n9051\n9211\n9371\n9531\n9692\n9853\n10015\n10177\n10340\n10504\n10667\n10832\n10996\n11161\n11327\n11493\n11660\n11827\n11995\n12163\n12331\n12500\n12670\n12840\n13010\n13181\n13353\n13524\n13697\n13870\n14043\n14217\n14391\n14565\n14741\n14916\n15092\n15269\n15446\n15623\n15801\n15979\n16158\n16338\n16517\n16698\n16878\n17059\n17241\n17423\n17605\n17788\n17972\n18156\n18340\n18525\n18710\n18895\n19082\n19268\n19455\n19643\n19830\n20019\n20208\n20397\n20586\n20776\n20967\n21158\n21349\n21541\n21734\n21926\n22119\n22313\n22507\n22701\n22896\n23092\n23287\n23484\n23680\n23877\n24075\n24272\n24471\n24669\n24869\n25068\n25268\n25468\n25669\n25870\n26072\n26274\n26477\n26679\n26883\n27086\n27291\n27495\n27700\n27905\n28111\n28317\n28524\n28731\n28938\n29146\n29354\n29562\n29771\n29981\n30190\n30401\n30611\n30822\n31033\n31245\n31457\n31670\n31882\n32096\n32309\n32523\n32738\n32952\n33167\n33383\n33599\n33815\n34032\n34249\n34466\n34684\n34902\n35121\n35339\n35559\n35778\n35998\n36219\n36439\n36660\n36882\n37104\n37326\n37548\n37771\n37994\n38218\n38442\n38666\n38891\n39116\n39341\n39566\n39792\n40019\n40246\n40473\n40700\n40928\n41156\n41384\n41613\n41842\n42071\n42301\n42531\n42761\n42992\n43223\n43454\n43686\n43918\n44150\n44383\n44616\n44849\n45083\n45317\n45551\n45786\n46021\n46256\n46491\n46727\n46963\n47200\n47436\n47673\n47911\n48148\n48386\n48624\n48863\n49102\n49341\n49580\n49820\n50060\n50300\n50541\n50782\n51023\n51265\n51506\n51748\n51991\n52233\n52476\n52719\n52963\n53206\n53450\n53695\n53939\n54184\n54429\n54674\n54920\n55166\n55412\n55658\n55905\n56152\n56399\n56647\n56894\n57142\n57391\n57639\n57888\n58137\n58386\n58636\n58885\n59135\n59386\n59636\n59887\n60138\n60389\n60640\n60892\n61144\n61396\n61649\n61901\n62154\n62407\n62661\n62914\n63168\n63422\n63676\n63931\n64186\n64441\n64696\n64951\n65207\n65463\n65719\n65975\n66231\n66488\n66745\n67002\n67259\n67517\n67775\n68033\n68291\n68549\n68808\n69067\n69326\n69585\n69844\n70104\n70363\n70623\n70884\n71144\n71405\n71665\n71926\n72187\n72449\n72710\n72972\n73234\n73496\n73758\n74020\n74283\n74546\n74809\n75072\n75335\n75599\n75863\n76126\n76390\n76655\n76919\n77184\n77448\n77713\n77978\n78243\n78509\n78774\n79040\n79306\n79572\n79838\n80104\n80371\n80637\n80904\n81171\n81438\n81706\n81973\n82241\n82508\n82776\n83044\n83312\n83581\n83849\n84118\n84386\n84655\n84924\n85193\n85463\n85732\n86002\n86271\n86541\n86811\n87081\n87352\n87622\n87892\n88163\n88434\n88705\n88976\n89247\n89518\n89789\n90061\n90333\n90604\n90876\n91148\n91420\n91692\n91965\n92237\n92510\n92782\n93055\n93328\n93601\n93874\n94147\n94421\n94694\n94968\n95241\n95515\n95789\n96063\n96337\n96611\n96885\n97160\n97434\n97709\n97983\n98258\n98533\n98808\n99083\n99358\n99633\n99909\n100184\n100460\n100735\n101011\n101287\n101562\n101838\n102114\n102390\n102667\n102943\n103219\n103496\n103772\n104049\n104325\n104602\n104879\n105156\n105433\n105710\n105987\n106264\n106541\n106819\n107096\n107374\n107651\n107929\n108206\n108484\n108762\n109040\n109318\n109596\n109874\n110152\n110430\n110709\n110987\n111265\n111544\n111822\n112101\n112379\n112658\n112937\n113216\n113494\n113773\n114052\n114331\n114610\n114889\n115169\n115448\n115727\n116006\n116286\n116565\n116845\n117124\n117404\n117683\n117963\n118243\n118522\n118802\n119082\n119362\n119642\n119922\n120202\n120482\n120762\n121042\n121322\n121602\n121882\n122163\n122443\n122723\n123004\n123284\n123564\n123845\n124125\n124406\n124686\n124967\n125248\n125528\n125809\n126090\n126370\n126651\n126932\n127213\n127493\n127774\n128055\n128336\n128617\n128898\n129179\n129460\n129741\n130022\n130303\n130584\n130865\n131146\n131428\n131709\n131990\n132271\n132552\n132834\n133115\n133396\n133677\n133959\n134240\n134521\n134803\n135084\n135365\n135647\n135928\n136210\n136491\n136772\n137054\n137335\n137617\n137898\n138180\n138461\n138743\n139024\n139306\n139587\n139869\n140150\n140432\n140713\n140995\n141276\n141558\n141839\n142121\n142403\n142684\n142966\n143247\n143529\n143810\n144092\n144373\n144655\n144937\n145218\n145500\n145781\n146063\n146344\n146626\n146907\n147189\n147470\n147752\n148034\n148315\n148597\n148878\n149160\n149441\n149723\n150004\n150286\n150567\n150848\n151130\n151411\n151693\n151974\n152256\n152537\n152819\n153100\n153381\n153663\n153944\n154225\n154507\n154788\n155069\n155351\n155632\n155913\n156195\n156476\n156757\n157038\n157319\n157601\n157882\n158163\n158444\n158725\n159006\n159288\n159569\n159850\n160131\n160412\n160693\n160974\n161255\n161536\n161817\n162098\n162379\n162660\n162940\n163221\n163502\n163783\n164064\n164345\n164625\n164906\n165187\n165467\n165748\n166029\n166309\n166590\n166871\n167151\n167432\n167712\n167993\n168273\n168554\n168834\n169115\n169395\n169675\n169956\n170236\n170516\n170797\n171077\n171357\n171637\n171917\n172198\n172478\n172758\n173038\n173318\n173598\n173878\n174158\n174438\n174718\n174998\n175277\n175557\n175837\n176117\n176397\n176676\n176956\n177236\n177515\n177795\n178074\n178354\n178633\n178913\n179192\n179472\n179751\n180031\n180310\n180589\n180869\n181148\n181427\n181706\n181985\n182265\n182544\n182823\n183102\n183381\n183660\n183939\n184218\n184497\n184775\n185054\n185333\n185612\n185891\n186169\n186448\n186727\n187005\n187284\n187562\n187841\n188119\n188398\n188676\n188954\n189233\n189511\n189789\n190068\n190346\n190624\n190902\n191180\n191458\n191736\n192014\n192292\n192570\n192848\n193126\n193404\n193682\n193959\n194237\n194515\n194793\n195070\n195348\n195625\n195903\n196180\n196458\n196735\n197013\n197290\n197567\n197845\n198122\n198399\n198676\n198953\n199231\n199508\n199785\n200062\n200339\n200616\n200893\n201169\n201446\n201723\n202000\n202276\n202553\n202830\n203106\n203383\n203660\n203936\n204212\n204489\n204765\n205042\n205318\n205594\n205871\n206147\n206423\n206699\n206975\n207251\n207527\n207803\n208079\n208355\n208631\n208907\n209183\n209458\n209734\n210010\n210286\n210561\n210837\n211112\n211388\n211663\n211939\n212214\n212490\n212765\n213040\n213315\n213591\n213866\n214141\n214416\n214691\n214966\n215241\n215516\n215791\n216066\n216341\n216616\n216891\n217165\n217440\n217715\n217989\n218264\n218538\n218813\n219087\n219362\n219636\n219911\n220185\n220459\n220734\n221008\n221282\n221556\n221830\n222104\n222378\n222652\n222926\n223200\n223474\n223748\n224022\n224296\n224570\n224843\n225117\n225391\n225664\n225938\n226211\n226485\n226758\n227032\n227305\n227578\n227852\n228125\n228398\n228671\n228945\n229218\n229491\n229764\n230037\n230310\n230583\n230856\n231129\n231401\n231674\n231947\n232220\n232492\n232765\n233038\n233310\n233583\n233855\n234128\n234400\n234673\n234945\n235217\n235490\n235762\n236034\n236306\n236578\n236850\n237123\n237395\n237667\n237939\n238211\n238482\n238754\n239026\n239298\n239570\n239841\n240113\n240385\n240656\n240928\n241199\n241471\n241742\n242014\n242285\n242557\n242828\n243099\n243371\n243642\n243913\n244184\n244455\n244726\n244997\n245268\n245539\n245810\n246081\n246352\n246623\n246894\n247165\n247435\n247706\n247977\n248247\n248518\n248789\n249059\n249330\n249600\n249871\n250141\n250411\n250682\n250952\n251222\n251492\n251763\n252033\n252303\n252573\n252843\n253113\n253383\n253653\n253923\n254193\n254463\n254733\n255002\n255272\n255542\n255812\n256081\n256351\n256620\n256890\n257160\n257429\n257698\n257968\n258237\n258507\n258776\n259045\n259315\n259584\n259853\n260122\n260391\n260660\n260930\n261199\n261468\n261737\n262006\n262274\n262543\n262812\n263081\n263350\n263619\n263887\n264156\n264425\n264693\n264962\n265230\n265499\n265767\n266036\n266304\n266573\n266841\n267110\n267378\n267646\n267914\n268183\n268451\n268719\n268987\n269255\n269523\n269791\n270059\n270327\n270595\n270863\n271131\n271399\n271667\n271935\n272202\n272470\n272738\n273005\n273273\n273541\n273808\n274076\n274343\n274611\n274878\n275146\n275413\n275681\n275948\n276215\n276483\n276750\n277017\n277284\n277551\n277819\n278086\n278353\n278620\n278887\n279154\n279421\n279688\n279955\n280222\n280489\n280755\n281022\n281289\n281556\n281822\n282089\n282356\n282622\n282889\n283156\n283422\n283689\n283955\n284222\n284488\n284755\n285021\n285287\n285554\n285820\n286086\n286352\n286619\n286885\n287151\n287417\n287683\n287949\n288215\n288481\n288747\n289013\n289279\n289545\n289811\n290077\n290343\n290609\n290875\n291140\n291406\n291672\n291938\n292203\n292469\n292734\n293000\n293266\n293531\n293797\n294062\n294328\n294593\n294858\n295124\n295389\n295654\n295920\n296185\n296450\n296716\n296981\n297246\n297511\n297776\n298041\n298306\n298571\n298837\n299102\n299367\n299631\n299896\n300161\n300426\n300691\n300956\n301221\n301486\n301750\n302015\n302280\n302544\n302809\n303074\n303338\n303603\n303868\n304132\n304397\n304661\n304926\n305190\n305454\n305719\n305983\n306248\n306512\n306776\n307041\n307305\n307569\n307833\n308098\n308362\n308626\n308890\n309154\n309418\n309682\n309946\n310210\n310474\n310738\n311002\n311266\n311530\n311794\n312058\n312322\n312586\n312849\n313113\n313377\n313641\n313904\n314168\n314432\n314695\n314959\n315223\n315486\n315750\n316013\n316277\n316540\n316804\n317067\n317331\n317594\n317857\n318121\n318384\n318647\n318911\n319174\n319437\n319701\n319964\n320227\n320490\n320753\n321016\n321280\n321543\n321806\n322069\n322332\n322595\n322858\n323121\n323384\n323647\n323910\n324173\n324436\n324698\n324961\n325224\n325487\n325750\n326012\n326275\n326538\n326801\n327063\n327326\n327589\n327851\n328114\n328376\n328639\n328902\n329164\n329427\n329689\n329952\n330214\n330477\n330739\n331001\n331264\n331526\n331788\n332051\n332313\n332575\n332838\n333100\n333362\n333624\n333887\n334149\n334411\n334673\n334935\n335197\n335459\n335722\n335984\n336246\n336508\n336770\n337032\n337294\n337556\n337818\n338080\n338341\n338603\n338865\n339127\n339389\n339651\n339912\n340174\n340436\n340698\n340960\n341221\n341483\n341745\n342006\n342268\n342530\n342791\n343053\n343314\n343576\n343838\n344099\n344361\n344622\n344884\n345145\n345406\n345668\n345929\n346191\n346452\n346714\n346975\n347236\n347498\n347759\n348020\n348281\n348543\n348804\n349065\n349326\n349588\n349849\n350110\n350371\n350632\n350893\n351154\n351416\n351677\n351938\n352199\n352460\n352721\n352982\n353243\n353504\n353765\n354026\n354287\n354548\n354808\n355069\n355330\n355591\n355852\n356113\n356374\n356634\n356895\n357156\n357417\n357677\n357938\n358199\n358460\n358720\n358981\n359242\n359502\n359763\n360023\n360284\n360545\n360805\n361066\n361326\n361587\n361847\n362108\n362368\n362629\n362889\n363150\n363410\n363671\n363931\n364191\n364452\n364712\n364973\n365233\n365493\n365754\n366014\n366274\n366534\n366795\n367055\n367315\n367575\n367836\n368096\n368356\n368616\n368876\n369137\n369397\n369657\n369917\n370177\n370437\n370697\n370957\n371217\n371477\n371737\n371997\n372257\n372517\n372777\n373037\n373297\n373557\n373817\n374077\n374337\n374597\n374857\n375117\n375377\n375636\n375896\n376156\n376416\n376676\n376936\n377195\n377455\n377715\n377975\n378234\n378494\n378754\n379013\n379273\n379533\n379792\n380052\n380312\n380571\n380831\n381091\n381350\n381610\n381869\n382129\n382388\n382648\n382908\n383167\n383427\n383686\n383946\n384205\n384465\n384724\n384983\n385243\n385502\n385762\n386021\n386280\n386540\n386799\n387059\n387318\n387577\n387837\n388096\n388355\n388615\n388874\n389133\n389392\n389652\n389911\n390170\n390429\n390689\n390948\n391207\n391466\n391725\n391985\n392244\n392503\n392762\n393021\n393280\n393539\n393798\n394058\n394317\n394576\n394835\n395094\n395353\n395612\n395871\n396130\n396389\n396648\n396907\n397166\n397425\n397684\n397943\n398202\n398461\n398720\n398979\n399237\n399496\n399755\n400014\n400273\n400532\n400791\n401050\n401308\n401567\n401826\n402085\n402344\n402602\n402861\n403120\n403379\n403637\n403896\n404155\n404414\n404672\n404931\n405190\n405449\n405707\n405966\n406225\n406483\n406742\n407001\n407259\n407518\n407776\n408035\n408294\n408552\n408811\n409069\n409328\n409587\n409845\n410104\n410362\n410621\n410879\n411138\n411396\n411655\n411913\n412172\n412430\n412689\n412947\n413206\n413464\n413722\n413981\n414239\n414498\n414756\n415015\n415273\n415531\n415790\n416048\n416306\n416565\n416823\n417081\n417340\n417598\n417856\n418115\n418373\n418631\n418890\n419148\n419406\n419664\n419923\n420181\n420439\n420697\n420956\n421214\n421472\n421730\n421989\n422247\n422505\n422763\n423021\n423279\n423538\n423796\n424054\n424312\n424570\n424828\n425086\n425345\n425603\n425861\n426119\n426377\n426635\n426893\n427151\n427409\n427667\n427925\n428183\n428441\n428699\n428957\n429215\n429473\n429731\n429989\n430247\n430505\n430763\n431021\n431279\n431537\n431795\n432053\n432311\n432569\n432827\n433085\n433343\n433601\n433859\n434117\n434374\n434632\n434890\n435148\n435406\n435664\n435922\n436179\n436437\n436695\n436953\n437211\n437469\n437726\n437984\n438242\n438500\n438758\n439015\n439273\n439531\n439789\n440046\n440304\n440562\n440820\n441077\n441335\n441593\n441851\n442108\n442366\n442624\n442881\n443139\n443397\n443655\n443912\n444170\n444428\n444685\n444943\n445201\n445458\n445716\n445973\n446231\n446489\n446746\n447004\n447262\n447519\n447777\n448034\n448292\n448550\n448807\n449065\n449322\n449580\n449837\n450095\n450352\n450610\n450868\n451125\n451383\n451640\n451898\n452155\n452413\n452670\n452928\n453185\n453443\n453700\n453958\n454215\n454473\n454730\n454987\n455245\n455502\n455760\n456017\n456275\n456532\n456790\n457047\n457304\n457562\n457819\n458077\n458334\n458591\n458849\n459106\n459364\n459621\n459878\n460136\n460393\n460650\n460908\n461165\n461423\n461680\n461937\n462195\n462452\n462709\n462966\n463224\n463481\n463738\n463996\n464253\n464510\n464768\n465025\n465282\n465539\n465797\n466054\n466311\n466569\n466826\n467083\n467340\n467598\n467855\n468112\n468369\n468626\n468884\n469141\n469398\n469655\n469913\n470170\n470427\n470684\n470941\n471199\n471456\n471713\n471970\n472227\n472484\n472742\n472999\n473256\n473513\n473770\n474027\n474285\n474542\n474799\n475056\n475313\n475570\n475827\n476084\n476342\n476599\n476856\n477113\n477370\n477627\n477884\n478141\n478398\n478655\n478912\n479170\n479427\n479684\n479941\n480198\n480455\n480712\n480969\n481226\n481483\n481740\n481997\n482254\n482511\n482768\n483025\n483282\n483539\n483796\n484053\n484310\n484567\n484824\n485081\n485338\n485595\n485852\n486109\n486366\n486623\n486880\n487137\n487394\n487651\n487908\n488165\n488422\n488679\n488936\n489193\n489450\n489707\n489964\n490221\n490478\n490734\n490991\n491248\n491505\n491762\n492019\n492276\n492533\n492790\n493047\n493304\n493560\n493817\n494074\n494331\n494588\n494845\n495102\n495359\n495615\n495872\n496129\n496386\n496643\n496900\n497157\n497414\n497670\n497927\n498184\n498441\n498698\n498955\n499211\n499468\n499725\n499982\n500239\n500495\n500752\n501009\n501266\n501523\n501780\n502036\n502293\n502550\n502807\n503064\n503320\n503577\n503834\n504091\n504347\n504604\n504861\n505118\n505375\n505631\n505888\n506145\n506402\n506658\n506915\n507172\n507429\n507685\n507942\n508199\n508456\n508712\n508969\n509226\n509482\n509739\n509996\n510253\n510509\n510766\n511023\n511279\n511536\n511793\n512050\n512306\n512563\n512820\n513076\n513333\n513590\n513846\n514103\n514360\n514617\n514873\n515130\n515387\n515643\n515900\n516157\n516413\n516670\n516927\n517183\n517440\n517697\n517953\n518210\n518466\n518723\n518980\n519236\n519493\n519750\n520006\n520263\n520520\n520776\n521033\n521289\n521546\n521803\n522059\n522316\n522573\n522829\n523086\n523342\n523599\n523856\n524112\n524369\n524625\n524882\n525139\n525395\n525652\n525908\n526165\n526422\n526678\n526935\n527191\n527448\n527704\n527961\n528218\n528474\n528731\n528987\n529244\n529500\n529757\n530013\n530270\n530527\n530783\n531040\n531296\n531553\n531809\n532066\n532322\n532579\n532835\n533092\n533349\n533605\n533862\n534118\n534375\n534631\n534888\n535144\n535401\n535657\n535914\n536170\n536427\n536683\n536940\n537196\n537453\n537709\n537966\n538222\n538479\n538735\n538992\n539248\n539505\n539761\n540018\n540274\n540531\n540787\n541044\n541300\n541557\n541813\n542070\n542326\n542583\n542839\n543096\n543352\n543608\n543865\n544121\n544378\n544634\n544891\n545147\n545404\n545660\n545917\n546173\n546430\n546686\n546942\n547199\n547455\n547712\n547968\n548225\n548481\n548737\n548994\n549250\n549507\n549763\n550020\n550276\n550532\n550789\n551045\n551302\n551558\n551815\n552071\n552327\n552584\n552840\n553097\n553353\n553609\n553866\n554122\n554379\n554635\n554892\n555148\n555404\n555661\n555917\n556174\n556430\n556686\n556943\n557199\n557455\n557712\n557968\n558225\n558481\n558737\n558994\n559250\n559507\n559763\n560019\n560276\n560532\n560788\n561045\n561301\n561557\n561814\n562070\n562327\n562583\n562839\n563096\n563352\n563608\n563865\n564121\n564377\n564634\n564890\n565147\n565403\n565659\n565916\n566172\n566428\n566685\n566941\n567197\n567454\n567710\n567966\n568223\n568479\n568735\n568992\n569248\n569504\n569761\n570017\n570273\n570530\n570786\n571042\n571299\n571555\n571811\n572068\n572324\n572580\n572837\n573093\n573349\n573605\n573862\n574118\n574374\n574631\n574887\n575143\n575400\n575656\n575912\n576169\n576425\n576681\n576937\n577194\n577450\n577706\n577963\n578219\n578475\n578732\n578988\n579244\n579500\n579757\n580013\n580269\n580526\n580782\n581038\n581294\n581551\n581807\n582063\n582320\n582576\n582832\n583088\n583345\n583601\n583857\n584113\n584370\n584626\n584882\n585139\n585395\n585651\n585907\n586164\n586420\n586676\n586932\n587189\n587445\n587701\n587957\n588214\n588470\n588726\n588982\n589239\n589495\n589751\n590007\n590264\n590520\n590776\n591032\n591289\n591545\n591801\n592057\n592314\n592570\n592826\n593082\n593339\n593595\n593851\n594107\n594364\n594620\n594876\n595132\n595389\n595645\n595901\n596157\n596414\n596670\n596926\n597182\n597438\n597695\n597951\n598207\n598463\n598720\n598976\n599232\n599488\n599744\n600001\n600257\n600513\n600769\n601026\n601282\n601538\n601794\n602050\n602307\n602563\n602819\n603075\n603331\n603588\n603844\n604100\n604356\n604613\n604869\n605125\n605381\n605637\n605894\n606150\n606406\n606662\n606918\n607175\n607431\n607687\n607943\n608199\n608456\n608712\n608968\n609224\n609480\n609737\n609993\n610249\n610505\n610761\n611017\n611274\n611530\n611786\n612042\n612298\n612555\n612811\n613067\n613323\n613579\n613836\n614092\n614348\n614604\n614860\n615116\n615373\n615629\n615885\n616141\n616397\n616654\n616910\n617166\n617422\n617678\n617934\n618191\n618447\n618703\n618959\n619215\n619471\n619728\n619984\n620240\n620496\n620752\n621008\n621265\n621521\n621777\n622033\n622289\n622545\n622802\n623058\n623314\n623570\n623826\n624082\n624339\n624595\n624851\n625107\n625363\n625619\n625875\n626132\n626388\n626644\n626900\n627156\n627412\n627669\n627925\n628181\n628437\n628693\n628949\n629205\n629462\n629718\n629974\n630230\n630486\n630742\n630998\n631255\n631511\n631767\n632023\n632279\n632535\n632791\n633048\n633304\n633560\n633816\n634072\n634328\n634584\n634841\n635097\n635353\n635609\n635865\n636121\n636377\n636634\n636890\n637146\n637402\n637658\n637914\n638170\n638426\n638683\n638939\n639195\n639451\n639707\n639963\n640219\n640476\n640732\n640988\n641244\n641500\n641756\n642012\n642268\n642525\n642781\n643037\n643293\n643549\n643805\n644061\n644317\n644574\n644830\n645086\n645342\n645598\n645854\n646110\n646366\n646622\n646879\n647135\n647391\n647647\n647903\n648159\n648415\n648671\n648928\n649184\n649440\n649696\n649952\n650208\n650464\n650720\n650976\n651233\n651489\n651745\n652001\n652257\n652513\n652769\n653025\n653281\n653538\n653794\n654050\n654306\n654562\n654818\n655074\n655330\n655586\n655842\n656099\n656355\n656611\n656867\n657123\n657379\n657635\n657891\n658147\n658403\n658660\n658916\n659172\n659428\n659684\n659940\n660196\n660452\n660708\n660964\n661221\n661477\n661733\n661989\n662245\n662501\n662757\n663013\n663269\n663525\n663782\n664038\n664294\n664550\n664806\n665062\n665318\n665574\n665830\n666086\n666342\n666599\n666855\n667111\n667367\n667623\n667879\n668135\n668391\n668647\n668903\n669159\n669415\n669672\n669928\n670184\n670440\n670696\n670952\n671208\n671464\n671720\n671976\n672232\n672488\n672745\n673001\n673257\n673513\n673769\n674025\n674281\n674537\n674793\n675049\n675305\n675561\n675818\n676074\n676330\n676586\n676842\n677098\n677354\n677610\n677866\n678122\n678378\n678634\n678890\n679147\n679403\n679659\n679915\n680171\n680427\n680683\n680939\n681195\n681451\n681707\n681963\n682219\n682476\n682732\n682988\n683244\n683500\n683756\n684012\n684268\n684524\n684780\n685036\n685292\n685548\n685804\n686061\n686317\n686573\n686829\n687085\n687341\n687597\n687853\n688109\n688365\n688621\n688877\n689133\n689389\n689645\n689902\n690158\n690414\n690670\n690926\n691182\n691438\n691694\n691950\n692206\n692462\n692718\n692974\n693230\n693486\n693742\n693999\n694255\n694511\n694767\n695023\n695279\n695535\n695791\n696047\n696303\n696559\n696815\n697071\n697327\n697583\n697839\n698095\n698352\n698608\n698864\n699120\n699376\n699632\n699888\n700144\n700400\n700656\n700912\n701168\n701424\n701680\n701936\n702192\n702448\n702705\n702961\n703217\n703473\n703729\n703985\n704241\n704497\n704753\n705009\n705265\n705521\n705777\n706033\n706289\n706545\n706801\n707057\n707313\n707570\n707826\n708082\n708338\n708594\n708850\n709106\n709362\n709618\n709874\n710130\n710386\n710642\n710898\n711154\n711410\n711666\n711922\n712178\n712434\n712691\n712947\n713203\n713459\n713715\n713971\n714227\n714483\n714739\n714995\n715251\n715507\n715763\n716019\n716275\n716531\n716787\n717043\n717299\n717555\n717811\n718067\n718324\n718580\n718836\n719092\n719348\n719604\n719860\n720116\n720372\n720628\n720884\n721140\n721396\n721652\n721908\n722164\n722420\n722676\n722932\n723188\n723444\n723700\n723956\n724213\n724469\n724725\n724981\n725237\n725493\n725749\n726005\n726261\n726517\n726773\n727029\n727285\n727541\n727797\n728053\n728309\n728565\n728821\n729077\n729333\n729589\n729845\n730101\n730357\n730613\n730870\n731126\n731382\n731638\n731894\n732150\n732406\n732662\n732918\n733174\n733430\n733686\n733942\n734198\n734454\n734710\n734966\n735222\n735478\n735734\n735990\n736246\n736502\n736758\n737014\n737270\n737526\n737782\n738039\n738295\n738551\n738807\n739063\n739319\n739575\n739831\n740087\n740343\n740599\n740855\n741111\n741367\n741623\n741879\n742135\n742391\n742647\n742903\n743159\n743415\n743671\n743927\n744183\n744439\n744695\n744951\n745207\n745463\n745719\n745976\n746232\n746488\n746744\n747000\n747256\n747512\n747768\n748024\n748280\n748536\n748792\n749048\n749304\n749560\n749816\n750072\n750328\n750584\n750840\n751096\n751352\n751608\n751864\n752120\n752376\n752632\n752888\n753144\n753400\n753656\n753912\n754168\n754424\n754680\n754937\n755193\n755449\n755705\n755961\n756217\n756473\n756729\n756985\n757241\n757497\n757753\n758009\n758265\n758521\n758777\n759033\n759289\n759545\n759801\n760057\n760313\n760569\n760825\n761081\n761337\n761593\n761849\n762105\n762361\n762617\n762873\n763129\n763385\n763641\n763897\n764153\n764409\n764665\n764921\n765177\n765434\n765690\n765946\n766202\n766458\n766714\n766970\n767226\n767482\n767738\n767994\n768250\n768506\n768762\n769018\n769274\n769530\n769786\n770042\n770298\n770554\n770810\n771066\n771322\n771578\n771834\n772090\n772346\n772602\n772858\n773114\n773370\n773626\n773882\n774138\n774394\n774650\n774906\n775162\n775418\n775674\n775930\n776186\n776442\n776698\n776954\n777211\n777467\n777723\n777979\n778235\n778491\n778747\n779003\n779259\n779515\n779771\n780027\n780283\n780539\n780795\n781051\n781307\n781563\n781819\n782075\n782331\n782587\n782843\n783099\n783355\n783611\n783867\n784123\n784379\n784635\n784891\n785147\n785403\n785659\n785915\n786171\n786427\n786683\n786939\n787195\n787451\n787707\n787963\n788219\n788475\n788731\n788987\n789243\n789499\n789755\n790011\n790267\n790523\n790779\n791035\n791291\n791548\n791804\n792060\n792316\n792572\n792828\n793084\n793340\n793596\n793852\n794108\n794364\n794620\n794876\n795132\n795388\n795644\n795900\n796156\n796412\n796668\n796924\n797180\n797436\n797692\n797948\n798204\n798460\n798716\n798972\n799228\n799484\n799740\n799996\n800252\n800508\n800764\n801020\n801276\n801532\n801788\n802044\n802300\n802556\n802812\n803068\n803324\n803580\n803836\n804092\n804348\n804604\n804860\n805116\n805372\n805628\n805884\n806140\n806396\n806652\n806908\n807164\n807420\n807676\n807932\n808188\n808444\n808700\n808956\n809212\n809469\n809725\n809981\n810237\n810493\n810749\n811005\n811261\n811517\n811773\n812029\n812285\n812541\n812797\n813053\n813309\n813565\n813821\n814077\n814333\n814589\n814845\n815101\n815357\n815613\n815869\n816125\n816381\n816637\n816893\n817149\n817405\n817661\n817917\n818173\n818429\n818685\n818941\n819197\n819453\n819709\n819965\n820221\n820477\n820733\n820989\n821245\n821501\n821757\n822013\n822269\n822525\n822781\n823037\n823293\n823549\n823805\n824061\n824317\n824573\n824829\n825085\n825341\n825597\n825853\n826109\n826365\n826621\n826877\n827133\n827389\n827645\n827901\n828157\n828413\n828669\n828925\n829181\n829437\n829693\n829949\n830205\n830461\n830717\n830973\n831229\n831485\n831741\n831997\n832253\n832509\n832765\n833021\n833277\n833534\n833790\n834046\n834302\n834558\n834814\n835070\n835326\n835582\n835838\n836094\n836350\n836606\n836862\n837118\n837374\n837630\n837886\n838142\n838398\n838654\n838910\n839166\n839422\n839678\n839934\n840190\n840446\n840702\n840958\n841214\n841470\n841726\n841982\n842238\n842494\n842750\n843006\n843262\n843518\n843774\n844030\n844286\n844542\n844798\n845054\n845310\n845566\n845822\n846078\n846334\n846590\n846846\n847102\n847358\n847614\n847870\n848126\n848382\n848638\n848894\n849150\n849406\n849662\n849918\n850174\n850430\n850686\n850942\n851198\n851454\n851710\n851966\n852222\n852478\n852734\n852990\n853246\n853502\n853758\n854014\n854270\n854526\n854782\n855038\n855294\n855550\n855806\n856062\n856318\n856574\n856830\n857086\n857342\n857598\n857854\n858110\n858366\n858622\n858878\n859134\n859390\n859646\n859902\n860158\n860414\n860670\n860926\n861182\n861438\n861694\n861950\n862206\n862462\n862718\n862974\n863230\n863486\n863742\n863998\n864254\n864510\n864766\n865022\n865278\n865534\n865790\n866046\n866302\n866558\n866814\n867070\n867326\n867582\n867838\n868094\n868350\n868606\n868862\n869118\n869374\n869630\n869887\n870143\n870399\n870655\n870911\n871167\n871423\n871679\n871935\n872191\n872447\n872703\n872959\n873215\n873471\n873727\n873983\n874239\n874495\n874751\n875007\n875263\n875519\n875775\n876031\n876287\n876543\n876799\n877055\n877311\n877567\n877823\n878079\n878335\n878591\n878847\n879103\n879359\n879615\n879871\n880127\n880383\n880639\n880895\n881151\n881407\n881663\n881919\n882175\n882431\n882687\n882943\n883199\n883455\n883711\n883967\n884223\n884479\n884735\n884991\n885247\n885503\n885759\n886015\n886271\n886527\n886783\n887039\n887295\n887551\n887807\n888063\n888319\n888575\n888831\n889087\n889343\n889599\n889855\n890111\n890367\n890623\n890879\n891135\n891391\n891647\n891903\n892159\n892415\n892671\n892927\n893183\n893439\n893695\n893951\n894207\n894463\n894719\n894975\n895231\n895487\n895743\n895999\n896255\n896511\n896767\n897023\n897279\n897535\n897791\n898047\n898303\n898559\n898815\n899071\n899327\n899583\n899839\n900095\n900351\n900607\n900863\n901119\n901375\n901631\n901887\n902143\n902399\n902655\n902911\n903167\n903423\n903679\n903935\n904191\n904447\n904703\n904959\n905215\n905471\n905727\n905983\n906239\n906495\n906751\n907007\n907263\n907519\n907775\n908031\n908287\n908543\n908799\n909055\n909311\n909567\n909823\n910079\n910335\n910591\n910847\n911103\n911359\n911615\n911871\n912127\n912383\n912639\n912895\n913151\n913407\n913663\n913919\n914175\n914431\n914687\n914943\n915199\n915455\n915711\n915967\n916223\n916479\n916735\n916991\n917247\n917503\n917759\n918015\n918271\n918527\n918783\n919039\n919295\n919551\n919807\n920063\n920319\n920575\n920831\n921087\n921343\n921599\n921855\n922111\n922367\n922623\n922879\n923135\n923391\n923647\n923903\n924159\n924415\n924671\n924927\n925183\n925439\n925695\n925951\n926207\n926463\n926719\n926975\n927231\n927487\n927743\n927999\n928255\n928511\n928767\n929023\n929279\n929535\n929791\n930047\n930303\n930559\n930815\n931071\n931327\n931583\n931839\n932095\n932351\n932607\n932863\n933119\n933375\n933631\n933887\n934143\n934399\n934655\n934911\n935167\n935423\n935679\n935935\n936191\n936447\n936703\n936959\n937215\n937471\n937727\n937983\n938239\n938495\n938751\n939007\n939263\n939519\n939775\n940031\n940287\n940543\n940799\n941055\n941311\n941567\n941823\n942079\n942335\n942591\n942847\n943103\n943359\n943615\n943871\n944127\n944383\n944639\n944895\n945151\n945407\n945663\n945919\n946175\n946431\n946687\n946943\n947199\n947456\n947712\n947968\n948224\n948480\n948736\n948992\n949248\n949504\n949760\n950016\n950272\n950528\n950784\n951040\n951296\n951552\n951808\n952064\n952320\n952576\n952832\n953088\n953344\n953600\n953856\n954112\n954368\n954624\n954880\n955136\n955392\n955648\n955904\n956160\n956416\n956672\n956928\n957184\n957440\n957696\n957952\n958208\n958464\n958720\n958976\n959232\n959488\n959744\n960000\n960256\n960512\n960768\n961024\n961280\n961536\n961792\n962048\n962304\n962560\n962816\n963072\n963328\n963584\n963840\n964096\n964352\n964608\n964864\n965120\n965376\n965632\n965888\n966144\n966400\n966656\n966912\n967168\n967424\n967680\n967936\n968192\n968448\n968704\n968960\n969216\n969472\n969728\n969984\n970240\n970496\n970752\n971008\n971264\n971520\n971776\n972032\n972288\n972544\n972800\n973056\n973312\n973568\n973824\n974080\n974336\n974592\n974848\n975104\n975360\n975616\n975872\n976128\n976384\n976640\n976896\n977152\n977408\n977664\n977920\n978176\n978432\n978688\n978944\n979200\n979456\n979712\n979968\n980224\n980480\n980736\n980992\n981248\n981504\n981760\n982016\n982272\n982528\n982784\n983040\n983296\n983552\n983808\n984064\n984320\n984576\n984832\n985088\n985344\n985600\n985856\n986112\n986368\n986624\n986880\n987136\n987392\n987648\n987904\n988160\n988416\n988672\n988928\n989184\n989440\n989696\n989952\n990208\n990464\n990720\n990976\n991232\n991488\n991744\n992000\n992256\n992512\n992768\n993024\n993280\n993536\n993792\n994048\n994304\n994560\n994816\n995072\n995328\n995584\n995840\n996096\n996352\n996608\n996864\n997120\n997376\n997632\n997888\n998144\n998400\n998656\n998912\n999168\n999424\n999680\n999936\n1000192\n1000448\n1000704\n1000960\n1001216\n1001472\n1001728\n1001984\n1002240\n1002496\n1002752\n1003008\n1003264\n1003520\n1003776\n1004032\n1004288\n1004544\n1004800\n1005056\n1005312\n1005568\n1005824\n1006080\n1006336\n1006592\n1006848\n1007104\n1007360\n1007616\n1007872\n1008128\n1008384\n1008640\n1008896\n1009152\n1009408\n1009664\n1009920\n1010176\n1010432\n1010688\n1010944\n1011200\n1011456\n1011712\n1011968\n1012224\n1012480\n1012736\n1012992\n1013248\n1013504\n1013760\n1014016\n1014272\n1014528\n1014784\n1015040\n1015296\n1015552\n1015808\n1016064\n1016320\n1016576\n1016832\n1017088\n1017344\n1017600\n1017856\n1018112\n1018368\n1018624\n1018880\n1019136\n1019392\n1019648\n1019904\n1020160\n1020416\n1020672\n1020928\n1021184\n1021440\n1021696\n1021952\n1022208\n1022464\n1022720\n1022976\n1023232\n1023488\n1023744\n1024000\n1024256\n1024512\n1024768\n1025024\n1025280\n1025536\n1025792\n1026048\n1026304\n1026560\n1026816\n1027072\n1027328\n1027584\n1027840\n1028096\n1028352\n1028608\n1028864\n1029120\n1029376\n1029632\n1029888\n1030144\n1030400\n1030656\n1030912\n1031168\n1031424\n1031680\n1031936\n1032192\n1032448\n1032704\n1032960\n1033216\n1033472\n1033728\n1033984\n1034240\n1034496\n1034752\n1035008\n1035264\n1035520\n1035776\n1036032\n1036288\n1036544\n1036800\n1037056\n1037312\n1037568\n1037824\n1038080\n1038336\n1038592\n1038848\n1039104\n1039360\n1039616\n1039872\n1040128\n1040384\n1040640\n1040896\n1041152\n1041408\n1041664\n1041920\n1042176\n1042432\n1042688\n1042944\n1043200\n1043456\n1043712\n1043968\n1044224\n1044480\n1044736\n1044992\n1045248\n1045504\n1045760\n1046016\n1046272\n1046528\n1046784\n1047040\n1047296\n1047552\n1047808\n1048064\n1048320\n1048576\n')
FILES['tables/cos.txt'] = ('29e6fcb7aa21f9f621a683dcd835ea6710735baff05caf9860f290166906c7b5', '# cos.txt\n# C[p, i] = round(2^20 cos(p 100000^(-i/32))), p = 0 .. 511, i = 0 .. 31, row p then i\n# 16384 values, one per line, generated by make_tables.py\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n566548\n803459\n926685\n988613\n1019231\n1034252\n1041592\n1045173\n1046918\n1047769\n1048183\n1048385\n1048483\n1048531\n1048554\n1048565\n1048571\n1048573\n1048575\n1048575\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n-436362\n182705\n589351\n815580\n932839\n991669\n1020734\n1034987\n1041951\n1045348\n1047004\n1047810\n1048203\n1048394\n1048488\n1048533\n1048555\n1048566\n1048571\n1048574\n1048575\n1048575\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n-1038082\n-523467\n115000\n549269\n794234\n921993\n986279\n1018083\n1033690\n1041318\n1045040\n1046853\n1047737\n1048167\n1048377\n1048479\n1048529\n1048553\n1048565\n1048571\n1048573\n1048575\n1048575\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n-685395\n-984906\n-386087\n220137\n611175\n827127\n938687\n994572\n1022161\n1035685\n1042292\n1045514\n1047085\n1047850\n1048222\n1048404\n1048492\n1048535\n1048556\n1048566\n1048571\n1048574\n1048575\n1048575\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n297441\n-985878\n-797414\n-134172\n393909\n709661\n878591\n964606\n1007400\n1028457\n1038762\n1043793\n1046246\n1047441\n1048023\n1048307\n1048445\n1048512\n1048545\n1048561\n1048569\n1048572\n1048574\n1048575\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1006812\n-525928\n-1023351\n-473135\n154594\n572807\n806792\n928379\n989454\n1019645\n1034454\n1041691\n1045221\n1046942\n1047780\n1048188\n1048387\n1048484\n1048531\n1048554\n1048565\n1048571\n1048573\n1048575\n1048575\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n790524\n179906\n-1011372\n-757986\n-93373\n420302\n724246\n886126\n968380\n1009263\n1029370\n1039209\n1044011\n1046352\n1047493\n1048048\n1048319\n1048451\n1048515\n1048546\n1048562\n1048569\n1048573\n1048574\n1048575\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n-152568\n801630\n-764260\n-956145\n-336114\n256314\n632052\n838122\n944245\n997328\n1023514\n1036347\n1042615\n1045672\n1047161\n1047887\n1048240\n1048413\n1048496\n1048537\n1048557\n1048567\n1048572\n1048574\n1048575\n1048575\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n-955389\n1048572\n-339468\n-1044948\n-560042\n85324\n531440\n784679\n917125\n983856\n1016891\n1033106\n1041033\n1044901\n1046786\n1047704\n1048151\n1048369\n1048475\n1048527\n1048552\n1048564\n1048570\n1048573\n1048575\n1048575\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n-879830\n805282\n164247\n-1014240\n-752624\n-87999\n423748\n726143\n887105\n968870\n1009505\n1029489\n1039267\n1044039\n1046366\n1047500\n1048052\n1048321\n1048452\n1048515\n1048547\n1048562\n1048569\n1048573\n1048574\n1048575\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n4641\n185503\n629777\n-867531\n-903081\n-258916\n310412\n662894\n854280\n952392\n1001362\n1025495\n1037315\n1043087\n1045902\n1047274\n1047942\n1048267\n1048426\n1048503\n1048540\n1048559\n1048568\n1048572\n1048574\n1048575\n1048576\n1048576\n1048576\n1048576\n1048576\n1048576\n884845\n-521003\n948891\n-621602\n-1002991\n-422760\n192942\n595343\n818755\n934448\n992469\n1021127\n1035179\n1042045\n1045394\n1047026\n1047821\n1048208\n1048397\n1048489\n1048534\n1048555\n1048566\n1048571\n1048574\n1048575\n1048575\n1048576\n1048576\n1048576\n1048576\n1048576\n951527\n-983928\n1047399\n-304580\n-1046763\n-575053\n72901\n523927\n780642\n915065\n982831\n1016386\n1032859\n1040913\n1044842\n1046757\n1047690\n1048145\n1048366\n1048474\n1048526\n1048552\n1048564\n1048570\n1048573\n1048575\n1048575\n1048576\n1048576\n1048576\n1048576\n1048576\n143379\n-986842\n902399\n47278\n-1031947\n-711635\n-48111\n449111\n740060\n894273\n972456\n1011274\n1030355\n1039690\n1044246\n1046466\n1047549\n1048076\n1048332\n1048457\n1048518\n1048548\n1048562\n1048569\n1048573\n1048574\n1048575\n1048576\n1048576\n1048576\n1048576\n1048576\n-796591\n-528384\n547602\n393728\n-959371\n-828774\n-168482\n371381\n697139\n872104\n961352\n1005792\n1027668\n1038377\n1043605\n1046154\n1047397\n1048002\n1048296\n1048440\n1048510\n1048544\n1048560\n1048568\n1048572\n1048574\n1048575\n1048576\n1048576\n1048576\n1048576\n1048576\n-1004179\n177106\n65495\n695147\n-833098\n-923269\n-286609\n291240\n652014\n848592\n949527\n999944\n1024799\n1036975\n1042921\n1045821\n1047234\n1047922\n1048258\n1048421\n1048501\n1048539\n1048558\n1048567\n1048572\n1048574\n1048575\n1048576\n1048576\n1048576\n1048576\n1048576\n-288530\n799795\n-431840\n917062\n-660196\n-992538\n-400918\n209208\n604827\n823774\n936990\n993730\n1021747\n1035482\n1042193\n1045466\n1047061\n1047838\n1048217\n1048401\n1048491\n1048535\n1048556\n1048566\n1048571\n1048574\n1048575\n1048575\n1048576\n1048576\n1048576\n1048576\n692392\n1048561\n-828777\n1034091\n-450342\n-1034690\n-509887\n125819\n555729\n797687\n923750\n987153\n1018513\n1033900\n1041421\n1045090\n1046878\n1047749\n1048173\n1048380\n1048480\n1048529\n1048553\n1048565\n1048571\n1048573\n1048575\n1048575\n1048576\n1048576\n1048576\n1048576\n1036732\n807099\n-1033033\n1032849\n-215282\n-1048573\n-612063\n41614\n504873\n770372\n909818\n980216\n1015099\n1032229\n1040605\n1044692\n1046684\n1047654\n1048127\n1048357\n1048470\n1048524\n1048551\n1048564\n1048570\n1048573\n1048575\n1048575\n1048576\n1048576\n1048576\n1048576\n427905\n188299\n-997121\n913480\n31828\n-1033806\n-706087\n-42862\n452422\n741870\n895204\n972921\n1011503\n1030468\n1039745\n1044272\n1046480\n1047555\n1048079\n1048334\n1048458\n1048519\n1048548\n1048562\n1048569\n1048573\n1048574\n1048575\n1048576\n1048576\n1048576\n1048576\n-574336\n-518535\n-729391\n689635\n277157\n-990794\n-790706\n-127060\n398540\n712227\n879918\n965271\n1007728\n1028618\n1038841\n1043832\n1046265\n1047450\n1048028\n1048309\n1048446\n1048513\n1048545\n1048561\n1048569\n1048572\n1048574\n1048575\n1048576\n1048576\n1048576\n1048576\n-1048535\n-982942\n-292086\n386915\n506972\n-920712\n-864792\n-210433\n343398\n681487\n863973\n957268\n1003774\n1026678\n1037893\n1043369\n1046039\n1047341\n1047974\n1048283\n1048433\n1048507\n1048542\n1048560\n1048568\n1048572\n1048574\n1048575\n1048576\n1048576\n1048576\n1048576\n-558716\n-987799\n213126\n39944\n708412\n-825474\n-927358\n-292440\n287170\n649697\n847379\n948916\n999641\n1024650\n1036902\n1042886\n1045804\n1047226\n1047918\n1048256\n1048420\n1048500\n1048539\n1048558\n1048567\n1048572\n1048574\n1048575\n1048576\n1048576\n1048576\n1048576\n444784\n-530837\n668788\n-311596\n870201\n-707683\n-977572\n-372549\n230035\n616908\n830150\n940217\n995331\n1022533\n1035867\n1042381\n1045558\n1047106\n1047860\n1048227\n1048406\n1048493\n1048536\n1048556\n1048566\n1048571\n1048574\n1048575\n1048575\n1048576\n1048576\n1048576\n1039351\n174305\n968965\n-627499\n983284\n-570557\n-1014764\n-450240\n172172\n583168\n812299\n931174\n990843\n1020328\n1034788\n1041854\n1045301\n1046981\n1047799\n1048198\n1048392\n1048486\n1048532\n1048555\n1048566\n1048571\n1048574\n1048575\n1048575\n1048576\n1048576\n1048576\n678344\n797954\n1043869\n-871633\n1041332\n-417842\n-1038439\n-525009\n113765\n548530\n793839\n921792\n986179\n1018034\n1033666\n1041306\n1045034\n1046851\n1047736\n1048167\n1048377\n1048479\n1048529\n1048553\n1048565\n1048571\n1048573\n1048575\n1048575\n1048576\n1048576\n1048576\n-306330\n1048541\n876086\n-1016078\n1041095\n-253711\n-1048282\n-596371\n54999\n513048\n774783\n912073\n981340\n1015652\n1032500\n1040737\n1044756\n1046715\n1047670\n1048135\n1048361\n1048471\n1048525\n1048551\n1048564\n1048570\n1048573\n1048575\n1048575\n1048576\n1048576\n1048576\n-1009365\n808910\n504624\n-1044313\n982587\n-82648\n-1044161\n-663862\n-3942\n476776\n755147\n902021\n976327\n1013182\n1031290\n1040146\n1044468\n1046575\n1047601\n1048101\n1048345\n1048463\n1048521\n1048549\n1048563\n1048570\n1048573\n1048574\n1048575\n1048576\n1048576\n1048576\n-784395\n191094\n15842\n-953108\n869082\n90673\n-1026132\n-727044\n-62870\n439770\n734944\n891640\n971139\n1010625\n1030037\n1039534\n1044170\n1046430\n1047531\n1048067\n1048328\n1048455\n1048517\n1048547\n1048562\n1048569\n1048573\n1048574\n1048575\n1048576\n1048576\n1048576\n161744\n-516063\n-476623\n-752896\n706934\n261517\n-994434\n-785508\n-121599\n402087\n714190\n880933\n965779\n1007979\n1028741\n1038901\n1043861\n1046279\n1047457\n1048031\n1048311\n1048447\n1048513\n1048545\n1048561\n1048569\n1048572\n1048574\n1048575\n1048576\n1048576\n1048576\n959177\n-981949\n-858278\n-466573\n505218\n425215\n-949489\n-838873\n-179944\n363784\n692901\n869904\n960248\n1005247\n1027401\n1038247\n1043542\n1046123\n1047381\n1047994\n1048293\n1048438\n1048509\n1048543\n1048560\n1048568\n1048572\n1048574\n1048575\n1048576\n1048576\n1048576\n874747\n-988749\n-1040394\n-126889\n275225\n577296\n-891897\n-886794\n-237720\n324922\n671092\n858558\n954545\n1002427\n1026018\n1037571\n1043212\n1045963\n1047303\n1047956\n1048274\n1048429\n1048504\n1048541\n1048559\n1048568\n1048572\n1048574\n1048575\n1048576\n1048576\n1048576\n-13922\n-533286\n-980631\n227309\n29826\n713604\n-822425\n-928959\n-294745\n285559\n648779\n846898\n948673\n999521\n1024591\n1036873\n1042872\n1045797\n1047222\n1047917\n1048255\n1048420\n1048500\n1048539\n1048558\n1048567\n1048572\n1048574\n1048575\n1048576\n1048576\n1048576\n-889790\n171502\n-692883\n555508\n-217241\n830415\n-741998\n-965095\n-350837\n245756\n625981\n834929\n942632\n996528\n1023122\n1036155\n1042521\n1045626\n1047139\n1047876\n1048235\n1048410\n1048495\n1048537\n1048557\n1048567\n1048571\n1048574\n1048575\n1048575\n1048576\n1048576\n-947590\n796108\n-244048\n820174\n-452150\n924538\n-651687\n-994967\n-405821\n205575\n602713\n822656\n936424\n993449\n1021609\n1035415\n1042160\n1045450\n1047053\n1047834\n1048215\n1048400\n1048490\n1048534\n1048556\n1048566\n1048571\n1048574\n1048575\n1048575\n1048576\n1048576\n-134180\n1048514\n261526\n991035\n-661751\n993401\n-552695\n-1018382\n-459521\n165078\n578993\n810081\n930049\n990284\n1020053\n1034654\n1041789\n1045269\n1046965\n1047791\n1048194\n1048390\n1048485\n1048532\n1048555\n1048566\n1048571\n1048574\n1048575\n1048575\n1048576\n1048576\n802595\n810715\n706298\n1048550\n-834313\n1035122\n-446341\n-1035187\n-511769\n124326\n554838\n797211\n923508\n987033\n1018454\n1033871\n1041407\n1045083\n1046874\n1047747\n1048172\n1048379\n1048480\n1048529\n1048553\n1048565\n1048571\n1048573\n1048575\n1048575\n1048576\n1048576\n1001467\n193887\n986864\n986142\n-960178\n1048562\n-334042\n-1045273\n-562399\n83383\n530268\n784050\n916804\n983697\n1016812\n1033068\n1041014\n1044891\n1046781\n1047702\n1048150\n1048369\n1048475\n1048527\n1048552\n1048564\n1048570\n1048573\n1048575\n1048575\n1048576\n1048576\n279595\n-513588\n1037996\n810947\n-1032300\n1033354\n-217293\n-1048575\n-611250\n42311\n505300\n770602\n909936\n980275\n1015128\n1032243\n1040612\n1044695\n1046686\n1047655\n1048128\n1048358\n1048470\n1048524\n1048551\n1048564\n1048570\n1048573\n1048575\n1048575\n1048576\n1048576\n-699335\n-980948\n847806\n543003\n-1046644\n989912\n-97650\n-1045072\n-658170\n1175\n479954\n756873\n902907\n976769\n1013400\n1031397\n1040199\n1044494\n1046587\n1047607\n1048104\n1048346\n1048464\n1048522\n1048549\n1048563\n1048570\n1048573\n1048575\n1048575\n1048576\n1048576\n-1035300\n-989692\n460512\n212955\n-1002406\n919424\n23294\n-1034786\n-703008\n-39964\n454247\n742868\n895717\n973177\n1011630\n1030530\n1039775\n1044287\n1046487\n1047558\n1048080\n1048335\n1048458\n1048519\n1048548\n1048562\n1048569\n1048573\n1048574\n1048575\n1048576\n1048576\n-419415\n-535730\n-33846\n-141448\n-902062\n823816\n143928\n-1017783\n-745625\n-81040\n428200\n728592\n888367\n969502\n1009817\n1029641\n1039341\n1044076\n1046384\n1047508\n1048056\n1048323\n1048453\n1048516\n1048547\n1048562\n1048569\n1048573\n1048574\n1048575\n1048576\n1048576\n582078\n168698\n-520335\n-479674\n-751228\n705700\n262645\n-994175\n-785883\n-121993\n401831\n714049\n880860\n965743\n1007961\n1028732\n1038897\n1043859\n1046278\n1047457\n1048031\n1048311\n1048447\n1048513\n1048545\n1048561\n1048569\n1048572\n1048574\n1048575\n1048576\n1048576\n1048412\n794256\n-885852\n-763039\n-558348\n568303\n377863\n-964115\n-823658\n-162757\n375162\n699246\n873196\n961900\n1006063\n1027801\n1038442\n1043637\n1046170\n1047404\n1048005\n1048298\n1048441\n1048510\n1048544\n1048560\n1048568\n1048572\n1048574\n1048575\n1048576\n1048576\n550840\n1048480\n-1045420\n-959135\n-334216\n415379\n488048\n-927797\n-858828\n-203271\n348211\n684187\n865377\n957974\n1004123\n1026850\n1037977\n1043410\n1046059\n1047350\n1047979\n1048285\n1048434\n1048507\n1048542\n1048560\n1048568\n1048572\n1048574\n1048575\n1048576\n1048576\n-453171\n812514\n-961939\n-1045533\n-91378\n251106\n591731\n-885457\n-891283\n-243471\n320999\n668878\n857404\n953965\n1002140\n1025877\n1037502\n1043178\n1045946\n1047295\n1047952\n1048272\n1048428\n1048504\n1048541\n1048559\n1048568\n1048572\n1048574\n1048575\n1048576\n1048576\n-1040539\n196679\n-654819\n-1012352\n156575\n79972\n687533\n-837371\n-920921\n-283297\n293546\n653325\n849278\n949873\n1000115\n1024883\n1037016\n1042941\n1045831\n1047239\n1047925\n1048259\n1048422\n1048501\n1048539\n1048558\n1048567\n1048572\n1048574\n1048575\n1048576\n1048576\n-671240\n-511109\n-195462\n-863387\n395764\n-93347\n774177\n-783850\n-947647\n-322687\n265873\n637534\n841002\n945699\n998048\n1023868\n1036520\n1042699\n1045713\n1047181\n1047897\n1048245\n1048415\n1048498\n1048538\n1048557\n1048567\n1048572\n1048574\n1048575\n1048575\n1048576\n315194\n-979941\n309338\n-615676\n612802\n-264115\n850508\n-725241\n-971377\n-361579\n238001\n621510\n832576\n941443\n995939\n1022832\n1036013\n1042452\n1045592\n1047123\n1047868\n1048231\n1048408\n1048494\n1048536\n1048557\n1048567\n1048571\n1048574\n1048575\n1048575\n1048576\n1011840\n-990627\n742221\n-297549\n795540\n-427667\n915510\n-661925\n-992036\n-399915\n209950\n605259\n824002\n937105\n993787\n1021775\n1035496\n1042200\n1045469\n1047063\n1047839\n1048217\n1048401\n1048491\n1048535\n1048556\n1048566\n1048571\n1048574\n1048575\n1048575\n1048576\n778205\n-538171\n1002546\n54609\n933752\n-579535\n968317\n-594313\n-1009559\n-437636\n181742\n588787\n815281\n932687\n991594\n1020697\n1034969\n1041942\n1045344\n1047002\n1047809\n1048203\n1048394\n1048487\n1048533\n1048555\n1048566\n1048571\n1048574\n1048575\n1048575\n1048576\n-170908\n165892\n1029791\n400521\n1019700\n-715568\n1008225\n-522844\n-1023890\n-474682\n153397\n572099\n806415\n928187\n989359\n1019598\n1034431\n1041680\n1045216\n1046939\n1047779\n1048188\n1048387\n1048484\n1048531\n1048554\n1048565\n1048571\n1048573\n1048575\n1048575\n1048576\n-962889\n792397\n817622\n700626\n1048574\n-832052\n1034704\n-447982\n-1034984\n-510998\n124938\n555203\n797406\n923608\n987082\n1018478\n1033883\n1041412\n1045086\n1046876\n1047748\n1048173\n1048380\n1048480\n1048529\n1048553\n1048565\n1048571\n1048573\n1048575\n1048575\n1048576\n-869594\n1048437\n415366\n920598\n1018759\n-925802\n1047400\n-370212\n-1042807\n-546527\n96385\n538105\n788255\n918948\n984764\n1017338\n1033325\n1041140\n1044953\n1046811\n1047716\n1048157\n1048372\n1048477\n1048528\n1048552\n1048565\n1048570\n1048573\n1048575\n1048575\n1048576\n23202\n814308\n-83458\n1035281\n931922\n-994257\n1046144\n-290039\n-1047332\n-581214\n67759\n520809\n778964\n914209\n982404\n1016176\n1032756\n1040862\n1044817\n1046745\n1047684\n1048142\n1048365\n1048473\n1048526\n1048552\n1048564\n1048570\n1048573\n1048575\n1048575\n1048576\n894666\n199470\n-562879\n1031558\n792925\n-1035547\n1030954\n-207984\n-1048546\n-615006\n39083\n503324\n769535\n909390\n980003\n1014993\n1032177\n1040580\n1044679\n1046678\n1047652\n1048126\n1048357\n1048469\n1048524\n1048551\n1048564\n1048570\n1048573\n1048575\n1048575\n1048576\n943579\n-508625\n-911437\n909854\n609547\n-1048545\n1002030\n-124578\n-1046446\n-647851\n10378\n485655\n759968\n904493\n977560\n1013790\n1031588\n1040292\n1044539\n1046610\n1047618\n1048110\n1048349\n1048465\n1048522\n1048550\n1048563\n1048570\n1048573\n1048575\n1048575\n1048576\n124969\n-978926\n-1048097\n684089\n392052\n-1032895\n959759\n-40365\n-1041037\n-679699\n-18336\n467808\n750267\n899517\n975076\n1012566\n1030988\n1039999\n1044396\n1046540\n1047584\n1048093\n1048341\n1048461\n1048520\n1048549\n1048563\n1048570\n1048573\n1048574\n1048575\n1048576\n-808536\n-991555\n-941087\n380084\n152613\n-989024\n904704\n44111\n-1032337\n-710501\n-47035\n449790\n740432\n894464\n972551\n1011321\n1030378\n1039701\n1044251\n1046469\n1047550\n1048076\n1048333\n1048457\n1048518\n1048548\n1048562\n1048569\n1048573\n1048574\n1048575\n1048576\n-998677\n-540608\n-615285\n32608\n-95367\n-918131\n837597\n128300\n-1020373\n-740208\n-75699\n431609\n730465\n889333\n969985\n1010055\n1029758\n1039398\n1044103\n1046397\n1047515\n1048059\n1048324\n1048453\n1048516\n1048547\n1048562\n1048569\n1048573\n1048574\n1048575\n1048576\n-270639\n163086\n-146437\n-318597\n-338010\n-822153\n759334\n211657\n-1005184\n-768776\n-104307\n413269\n720369\n884125\n967378\n1008769\n1029128\n1039090\n1043953\n1046324\n1047479\n1048042\n1048316\n1048449\n1048514\n1048546\n1048561\n1048569\n1048573\n1048574\n1048575\n1048576\n706223\n790533\n356456\n-633364\n-561734\n-703712\n670956\n293640\n-986816\n-796160\n-132836\n394779\n710144\n878841\n964731\n1007462\n1028487\n1038777\n1043800\n1046250\n1047443\n1048024\n1048307\n1048445\n1048512\n1048545\n1048561\n1048569\n1048572\n1048574\n1048575\n1048576\n1033787\n1048387\n776477\n-875692\n-754017\n-566045\n573640\n373717\n-965329\n-822318\n-161266\n376145\n699793\n873480\n962042\n1006134\n1027836\n1038459\n1043645\n1046174\n1047406\n1048006\n1048299\n1048441\n1048510\n1048544\n1048560\n1048568\n1048572\n1048574\n1048575\n1048576\n410892\n816095\n1015977\n-1017866\n-904097\n-412913\n468684\n451369\n-940790\n-847210\n-189575\n357373\n689318\n868044\n959313\n1004785\n1027174\n1038136\n1043488\n1046097\n1047369\n1047988\n1048290\n1048437\n1048508\n1048543\n1048560\n1048568\n1048572\n1048574\n1048575\n1048576\n-589776\n202259\n1019274\n-1043626\n-1003574\n-248499\n357484\n526091\n-913277\n-870797\n-217741\n338471\n678720\n862532\n956544\n1003416\n1026503\n1037808\n1043328\n1046019\n1047331\n1047969\n1048281\n1048432\n1048506\n1048542\n1048559\n1048568\n1048572\n1048574\n1048575\n1048576\n-1048206\n-506139\n785602\n-950025\n-1046879\n-77295\n241522\n597398\n-882876\n-893044\n-245745\n319445\n668001\n856946\n953734\n1002026\n1025821\n1037474\n1043165\n1045940\n1047292\n1047951\n1048271\n1048428\n1048504\n1048541\n1048559\n1048568\n1048572\n1048574\n1048575\n1048576\n-542921\n-977904\n369287\n-747768\n-1031590\n96020\n122344\n664829\n-849685\n-913915\n-273564\n300303\n657164\n851286\n950884\n1000616\n1025129\n1037136\n1043000\n1045859\n1047253\n1047932\n1048262\n1048423\n1048502\n1048540\n1048558\n1048567\n1048572\n1048574\n1048575\n1048576\n461523\n-992475\n-132883\n-459988\n-958561\n266712\n1535\n727944\n-813807\n-933380\n-301178\n281051\n646209\n845552\n947994\n999185\n1024426\n1036793\n1042832\n1045778\n1047213\n1047912\n1048253\n1048419\n1048499\n1048539\n1048558\n1048567\n1048572\n1048574\n1048575\n1048576\n1041645\n-543041\n-604159\n-119599\n-831881\n430117\n-119294\n786335\n-775356\n-951407\n-328566\n261696\n635140\n839745\n945064\n997734\n1023714\n1036444\n1042662\n1045695\n1047173\n1047893\n1048243\n1048414\n1048497\n1048538\n1048557\n1048567\n1048572\n1048574\n1048575\n1048575\n664083\n160278\n-934976\n234469\n-658639\n581770\n-238533\n839622\n-734454\n-967969\n-355708\n242245\n623958\n833865\n942094\n996262\n1022991\n1036091\n1042490\n1045611\n1047132\n1047873\n1048233\n1048409\n1048495\n1048536\n1048557\n1048567\n1048571\n1048574\n1048575\n1048575\n-324034\n788663\n-1048421\n561720\n-448532\n717528\n-354596\n887460\n-691230\n-983041\n-382583\n222707\n612665\n827913\n939085\n994769\n1022257\n1035732\n1042315\n1045525\n1047090\n1047852\n1048224\n1048404\n1048492\n1048535\n1048556\n1048566\n1048571\n1048574\n1048575\n1048575\n-1014236\n1048330\n-918122\n824727\n-213321\n833682\n-465935\n929538\n-645821\n-996600\n-409171\n203087\n601263\n821889\n936036\n993256\n1021514\n1035368\n1042137\n1045439\n1047048\n1047832\n1048214\n1048400\n1048490\n1048534\n1048556\n1048566\n1048571\n1048574\n1048575\n1048575\n-771954\n817876\n-574369\n993409\n33830\n927059\n-571067\n965583\n-598370\n-1008623\n-435452\n183392\n589754\n815794\n932947\n991723\n1020761\n1035000\n1041958\n1045351\n1047005\n1047811\n1048203\n1048395\n1048488\n1048533\n1048555\n1048566\n1048571\n1048574\n1048575\n1048575\n180059\n205046\n-97083\n1048473\n279088\n995106\n-668593\n995361\n-549027\n-1019094\n-461407\n163631\n578140\n809628\n929819\n990170\n1019997\n1034626\n1041775\n1045262\n1046962\n1047790\n1048193\n1048390\n1048485\n1048532\n1048554\n1048566\n1048571\n1048574\n1048575\n1048575\n966526\n-503648\n402774\n983623\n508724\n1035966\n-757213\n1018679\n-497949\n-1027996\n-487016\n143810\n566424\n803392\n926652\n988596\n1019223\n1034247\n1041590\n1045172\n1046918\n1047769\n1048183\n1048384\n1048483\n1048531\n1048554\n1048565\n1048571\n1048573\n1048575\n1048575\n864374\n-976875\n808990\n806274\n709887\n1048521\n-835747\n1035385\n-445296\n-1035315\n-512259\n123937\n554606\n797087\n923445\n987002\n1018439\n1033864\n1041403\n1045081\n1046874\n1047747\n1048172\n1048379\n1048480\n1048529\n1048553\n1048565\n1048571\n1048573\n1048575\n1048575\n-32480\n-993389\n1027127\n536711\n871317\n1032429\n-903148\n1045371\n-391236\n-1041040\n-537119\n104018\n542690\n790713\n920200\n985387\n1017644\n1033475\n1041213\n1044988\n1046828\n1047725\n1048162\n1048374\n1048478\n1048528\n1048553\n1048565\n1048570\n1048573\n1048575\n1048575\n-899472\n-545470\n1006468\n205763\n983978\n988129\n-958518\n1048573\n-335939\n-1045161\n-561576\n84061\n530678\n784270\n916916\n983752\n1016840\n1033081\n1041021\n1044895\n1046783\n1047703\n1048151\n1048369\n1048475\n1048527\n1048552\n1048564\n1048570\n1048573\n1048575\n1048575\n-939494\n157469\n751818\n-148718\n1041565\n916831\n-1001121\n1044969\n-279580\n-1047674\n-585611\n64074\n518571\n777759\n913593\n982098\n1016025\n1032682\n1040826\n1044800\n1046737\n1047680\n1048140\n1048364\n1048473\n1048526\n1048551\n1048564\n1048570\n1048573\n1048575\n1048575\n-115749\n786788\n322379\n-486190\n1040854\n820484\n-1030389\n1034583\n-222336\n-1048573\n-609208\n44063\n506372\n771181\n910232\n980422\n1015200\n1032278\n1040629\n1044704\n1046690\n1047657\n1048129\n1048358\n1048470\n1048524\n1048551\n1048564\n1048570\n1048573\n1048575\n1048575\n814414\n1048264\n-182009\n-768055\n981886\n701720\n-1045931\n1017482\n-164390\n-1047858\n-632348\n24036\n494083\n764536\n906832\n978727\n1014365\n1031870\n1040429\n1044606\n1046642\n1047634\n1048117\n1048353\n1048467\n1048523\n1048550\n1048563\n1048570\n1048573\n1048575\n1048575\n995809\n819652\n-644083\n-962078\n867960\n563784\n-1047541\n993777\n-105925\n-1045530\n-655013\n4000\n481706\n757825\n903394\n977012\n1013520\n1031456\n1040227\n1044508\n1046594\n1047611\n1048106\n1048347\n1048465\n1048522\n1048550\n1048563\n1048570\n1048573\n1048575\n1048575\n261662\n207832\n-956414\n-1046066\n705454\n410444\n-1035197\n963623\n-47124\n-1041591\n-677187\n-16037\n469243\n751048\n899919\n975277\n1012665\n1031037\n1040023\n1044408\n1046546\n1047587\n1048094\n1048341\n1048462\n1048520\n1048549\n1048563\n1048570\n1048573\n1048574\n1048575\n-713056\n-501154\n-1046391\n-1010415\n503462\n245890\n-1009065\n927214\n11826\n-1036049\n-698854\n-36068\n456697\n744207\n896405\n973521\n1011799\n1030613\n1039816\n1044307\n1046496\n1047563\n1048083\n1048336\n1048459\n1048519\n1048548\n1048562\n1048569\n1048573\n1048574\n1048575\n-1032194\n-975838\n-893094\n-859201\n273291\n74618\n-969491\n884787\n70738\n-1028912\n-719997\n-56086\n444070\n737300\n892853\n971746\n1010924\n1030184\n1039606\n1044205\n1046447\n1047539\n1048071\n1048330\n1048456\n1048518\n1048548\n1048562\n1048569\n1048573\n1048574\n1048575\n-402337\n-994295\n-532164\n-609719\n27824\n-98693\n-917003\n836618\n129427\n-1020190\n-740599\n-76084\n431363\n730330\n889264\n969951\n1010038\n1029750\n1039394\n1044101\n1046396\n1047514\n1048059\n1048324\n1048453\n1048516\n1048547\n1048562\n1048569\n1048573\n1048574\n1048575\n597427\n-547895\n-47512\n-290503\n-219200\n-269307\n-852300\n783019\n187706\n-1009898\n-760647\n-96054\n418581\n723297\n885637\n968135\n1009142\n1029311\n1039180\n1043997\n1046345\n1047489\n1048047\n1048318\n1048451\n1048515\n1048546\n1048562\n1048569\n1048573\n1048574\n1048575\n1047919\n154659\n448186\n61938\n-453956\n-432563\n-776244\n724338\n245392\n-998051\n-780124\n-115989\n405723\n716201\n881972\n966300\n1008236\n1028867\n1038963\n1043891\n1046294\n1047464\n1048035\n1048312\n1048448\n1048513\n1048546\n1048561\n1048569\n1048572\n1048574\n1048575\n534959\n784906\n839687\n407295\n-663303\n-584001\n-689848\n660955\n302302\n-984667\n-799016\n-135881\n392794\n709043\n878271\n964445\n1007321\n1028418\n1038743\n1043784\n1046242\n1047439\n1048022\n1048306\n1048445\n1048512\n1048545\n1048561\n1048569\n1048572\n1048574\n1048575\n-469839\n1048191\n1035970\n706070\n-835525\n-719483\n-594264\n593283\n358257\n-969766\n-817309\n-155724\n379794\n701824\n874533\n962570\n1006395\n1027964\n1038522\n1043676\n1046189\n1047413\n1048010\n1048300\n1048442\n1048511\n1048544\n1048560\n1048568\n1048572\n1048574\n1048575\n-1042670\n821421\n991402\n924090\n-960981\n-835308\n-490763\n521761\n413079\n-953373\n-834989\n-175510\n366727\n694544\n870757\n960676\n1005458\n1027505\n1038297\n1043566\n1046135\n1047387\n1047997\n1048294\n1048439\n1048509\n1048543\n1048560\n1048568\n1048572\n1048574\n1048575\n-656875\n210617\n716345\n1036421\n-1032650\n-928310\n-380725\n446851\n466595\n-935512\n-852043\n-195232\n353595\n687203\n866945\n958762\n1004512\n1027041\n1038070\n1043456\n1046082\n1047361\n1047984\n1048288\n1048436\n1048508\n1048543\n1048560\n1048568\n1048572\n1048574\n1048575\n332848\n-498656\n274747\n1030215\n-1046520\n-995949\n-265616\n369042\n518636\n-916210\n-868458\n-214882\n340400\n679804\n863097\n956828\n1003556\n1026572\n1037841\n1043344\n1046027\n1047334\n1047971\n1048282\n1048433\n1048506\n1048542\n1048559\n1048568\n1048572\n1048574\n1048575\n1016552\n-974795\n-230727\n906183\n-1001816\n-1036378\n-146969\n288838\n569037\n-895497\n-884221\n-234454\n327145\n672345\n859212\n954874\n1002590\n1026097\n1037610\n1043231\n1045972\n1047308\n1047958\n1048275\n1048430\n1048505\n1048541\n1048559\n1048568\n1048572\n1048574\n1048575\n765642\n-995194\n-682560\n678509\n-901039\n-1048490\n-26364\n206758\n617639\n-873406\n-899322\n-253941\n313831\n664828\n855290\n952901\n1001614\n1025618\n1037375\n1043117\n1045916\n1047281\n1047945\n1048269\n1048426\n1048503\n1048541\n1048559\n1048568\n1048572\n1048574\n1048575\n-189195\n-550316\n-975705\n373233\n-749829\n-1031956\n94592\n123337\n664289\n-849970\n-913748\n-273335\n300461\n657254\n851333\n950908\n1000628\n1025134\n1037139\n1043001\n1045860\n1047253\n1047932\n1048262\n1048423\n1048502\n1048540\n1048558\n1048567\n1048572\n1048574\n1048575\n-970088\n151848\n-1042011\n25270\n-556651\n-987227\n214288\n39116\n708838\n-825225\n-927490\n-292629\n287038\n649622\n847340\n948896\n999631\n1024645\n1036900\n1042885\n1045803\n1047225\n1047918\n1048256\n1048420\n1048500\n1048539\n1048558\n1048567\n1048572\n1048574\n1048575\n-859086\n783019\n-866062\n-325583\n-332317\n-915526\n331130\n-45360\n751147\n-799210\n-940535\n-311816\n273564\n641935\n843311\n946864\n998625\n1024151\n1036658\n1042767\n1045746\n1047197\n1047905\n1048249\n1048417\n1048498\n1048538\n1048558\n1048567\n1048572\n1048574\n1048575\n41755\n1048110\n-488763\n-639198\n-89382\n-818810\n443561\n-129541\n791081\n-771964\n-952876\n-330889\n260042\n634192\n839246\n944813\n997609\n1023652\n1036414\n1042648\n1045688\n1047169\n1047891\n1048242\n1048414\n1048497\n1048537\n1048557\n1048567\n1048572\n1048574\n1048575\n904207\n823184\n2167\n-879708\n158555\n-699723\n550083\n-212881\n828513\n-743529\n-964502\n-349841\n246473\n626393\n835147\n942742\n996583\n1023148\n1036168\n1042527\n1045629\n1047141\n1047877\n1048236\n1048410\n1048495\n1048537\n1048557\n1048567\n1048571\n1048574\n1048575\n935335\n213400\n492593\n-1019605\n397618\n-561519\n649278\n-294840\n863327\n-713950\n-975405\n-368666\n232860\n618541\n831012\n940652\n995546\n1022639\n1035919\n1042406\n1045570\n1047112\n1047863\n1048229\n1048407\n1048494\n1048536\n1048556\n1048566\n1048571\n1048574\n1048575\n106520\n-496154\n868497\n-1042888\n614426\n-407973\n739825\n-374885\n895411\n-683271\n-985576\n-387356\n219206\n610635\n826841\n938543\n994500\n1022125\n1035667\n1042283\n1045510\n1047083\n1047849\n1048222\n1048404\n1048492\n1048535\n1048556\n1048566\n1048571\n1048574\n1048575\n-820229\n-973744\n1042486\n-946895\n796844\n-243280\n820517\n-452497\n924665\n-651540\n-995008\n-405905\n205513\n602676\n822636\n936414\n993444\n1021606\n1035414\n1042159\n1045450\n1047053\n1047834\n1048215\n1048400\n1048490\n1048534\n1048556\n1048566\n1048571\n1048574\n1048575\n-992863\n-996085\n974110\n-742604\n934661\n-71941\n890279\n-527172\n950995\n-618806\n-1003695\n-424305\n191783\n594665\n818397\n934266\n992378\n1021083\n1035157\n1042034\n1045389\n1047024\n1047820\n1048208\n1048397\n1048489\n1048533\n1048555\n1048566\n1048571\n1048574\n1048575\n-252664\n-552733\n679264\n-453381\n1020165\n101365\n948182\n-598425\n974318\n-585119\n-1011628\n-442550\n178019\n586603\n814122\n932099\n991302\n1020554\n1034899\n1041908\n1045327\n1046994\n1047805\n1048201\n1048393\n1048487\n1048533\n1048555\n1048566\n1048571\n1048574\n1048575\n719833\n149036\n226498\n-112304\n1048568\n271900\n993455\n-665795\n994562\n-550532\n-1018803\n-460634\n164224\n578490\n809814\n929913\n990217\n1020020\n1034637\n1041781\n1045265\n1046963\n1047791\n1048193\n1048390\n1048485\n1048532\n1048554\n1048566\n1048571\n1048574\n1048575\n1030519\n781126\n-278927\n241617\n1018283\n435007\n1025495\n-728843\n1011661\n-515096\n-1025214\n-478550\n150399\n570326\n805471\n927708\n989121\n1019481\n1034374\n1041652\n1045202\n1046933\n1047776\n1048186\n1048386\n1048484\n1048531\n1048554\n1048565\n1048571\n1048573\n1048575\n393750\n1048022\n-719504\n567905\n931002\n586229\n1043875\n-787161\n1025562\n-478868\n-1030857\n-496291\n136548\n562113\n801095\n925484\n988015\n1018937\n1034108\n1041522\n1045139\n1046902\n1047761\n1048179\n1048383\n1048482\n1048530\n1048554\n1048565\n1048571\n1048573\n1048575\n-605031\n824941\n-992805\n829240\n791613\n721433\n1048350\n-840370\n1036220\n-441902\n-1035726\n-513850\n122672\n553852\n796684\n923240\n986900\n1018388\n1033839\n1041391\n1045075\n1046871\n1047745\n1048172\n1048379\n1048480\n1048529\n1048553\n1048565\n1048571\n1048573\n1048575\n-1047549\n216181\n-1035291\n995734\n607916\n836927\n1038861\n-888125\n1043603\n-404256\n-1039819\n-531222\n108775\n545543\n792240\n920978\n985774\n1017835\n1033568\n1041259\n1045011\n1046839\n1047730\n1048164\n1048375\n1048478\n1048528\n1048553\n1048565\n1048571\n1048573\n1048575\n-526956\n-493649\n-837084\n1048345\n390194\n929555\n1015533\n-930116\n1047686\n-365987\n-1043132\n-548400\n94858\n537186\n787763\n918697\n984639\n1017276\n1033295\n1041125\n1044946\n1046808\n1047715\n1048157\n1048372\n1048477\n1048528\n1048552\n1048565\n1048570\n1048573\n1048575\n478118\n-972686\n-444265\n981055\n150631\n996786\n978679\n-966070\n1048457\n-327155\n-1045663\n-565378\n80924\n528783\n783252\n916397\n983494\n1016712\n1033019\n1040991\n1044880\n1046776\n1047699\n1048149\n1048368\n1048475\n1048527\n1048552\n1048564\n1048570\n1048573\n1048575\n1043613\n-996969\n51840\n801561\n-97362\n1036782\n928787\n-995753\n1045914\n-287819\n-1047409\n-582149\n66976\n520334\n778708\n914078\n982339\n1016144\n1032741\n1040855\n1044814\n1046743\n1047683\n1048141\n1048364\n1048473\n1048526\n1048552\n1048564\n1048570\n1048573\n1048575\n649614\n-555145\n535893\n530392\n-339905\n1048452\n866525\n-1018974\n1040064\n-248040\n-1048371\n-598708\n53016\n511840\n774132\n911740\n981174\n1015570\n1032460\n1040718\n1044747\n1046711\n1047668\n1048134\n1048361\n1048471\n1048525\n1048551\n1048564\n1048570\n1048573\n1048575\n-341636\n146222\n895357\n198560\n-563424\n1031477\n792719\n-1035582\n1030926\n-207879\n-1048546\n-615048\n39047\n503302\n769523\n909384\n980000\n1014992\n1032177\n1040579\n1044679\n1046678\n1047652\n1048126\n1048357\n1048469\n1048524\n1048551\n1048564\n1048570\n1048573\n1048575\n-1018788\n779228\n1046662\n-155980\n-755408\n986319\n708354\n-1045468\n1018528\n-167398\n-1047935\n-631164\n25071\n494720\n764881\n907009\n978815\n1014409\n1031891\n1040440\n1044611\n1046645\n1047635\n1048118\n1048353\n1048467\n1048523\n1048550\n1048563\n1048570\n1048573\n1048575\n-759271\n1047925\n954630\n-492681\n-905110\n914214\n614554\n-1048569\n1002911\n-126659\n-1046538\n-647049\n11090\n486095\n760207\n904615\n977621\n1013820\n1031603\n1040299\n1044543\n1046611\n1047619\n1048110\n1048349\n1048465\n1048522\n1048550\n1048563\n1048570\n1048573\n1048575\n198317\n826693\n640657\n-773034\n-1004152\n817131\n512568\n-1044865\n984123\n-85725\n-1044356\n-662698\n-2893\n477428\n755501\n902203\n976417\n1013227\n1031312\n1040157\n1044473\n1046578\n1047603\n1048102\n1048345\n1048464\n1048521\n1048549\n1048563\n1048570\n1048573\n1048575\n973573\n218961\n177740\n-964973\n-1046991\n697722\n403754\n-1034379\n962223\n-44659\n-1041392\n-678104\n-16875\n468720\n750763\n899772\n975204\n1012629\n1031019\n1040014\n1044404\n1046544\n1047586\n1048094\n1048341\n1048462\n1048520\n1048549\n1048563\n1048570\n1048573\n1048574\n853731\n-491140\n-326500\n-1046548\n-1031229\n559250\n289561\n-1017179\n937282\n-3525\n-1037646\n-693263\n-30854\n459971\n745994\n897323\n973980\n1012026\n1030724\n1039870\n1044333\n1046509\n1047569\n1048086\n1048337\n1048460\n1048519\n1048548\n1048563\n1048569\n1048573\n1048574\n-51027\n-971621\n-754832\n-1008428\n-957747\n405499\n171512\n-993378\n909377\n37615\n-1033122\n-708169\n-44828\n451182\n741193\n894855\n972747\n1011417\n1030426\n1039724\n1044262\n1046475\n1047553\n1048078\n1048333\n1048458\n1048518\n1048548\n1048562\n1048569\n1048573\n1048574\n-908871\n-997846\n-1007675\n-854972\n-830660\n240668\n51178\n-963129\n878598\n78698\n-1027824\n-722817\n-58794\n442355\n736361\n892369\n971504\n1010804\n1030125\n1039578\n1044191\n1046440\n1047536\n1048069\n1048329\n1048456\n1048517\n1048548\n1048562\n1048569\n1048573\n1048574\n-931103\n-557554\n-1026246\n-603732\n-657079\n69262\n-69837\n-926630\n845041\n119659\n-1021755\n-737200\n-72749\n433489\n731498\n889865\n970251\n1010187\n1029823\n1039430\n1044119\n1046405\n1047518\n1048061\n1048325\n1048454\n1048517\n1048547\n1048562\n1048569\n1048573\n1048574\n-97283\n143408\n-806226\n-283443\n-446721\n-104036\n-189923\n-884116\n808812\n160435\n-1014920\n-751314\n-86692\n424585\n726604\n887342\n968989\n1009564\n1029517\n1039281\n1044046\n1046369\n1047501\n1048053\n1048321\n1048452\n1048516\n1048547\n1048562\n1048569\n1048573\n1048574\n825979\n777323\n-398769\n69264\n-211360\n-274492\n-307478\n-835864\n770026\n200965\n-1007324\n-765154\n-100619\n415645\n721679\n884801\n967717\n1008936\n1029210\n1039130\n1043973\n1046333\n1047484\n1048044\n1048317\n1048450\n1048515\n1048546\n1048561\n1048569\n1048573\n1048574\n989839\n1047821\n101397\n414049\n35832\n-437448\n-420938\n-782187\n728806\n241185\n-998972\n-778715\n-114528\n406669\n716724\n882242\n966435\n1008303\n1028899\n1038979\n1043899\n1046297\n1047466\n1048036\n1048313\n1048448\n1048514\n1048546\n1048561\n1048569\n1048572\n1048574\n243646\n828438\n577990\n711479\n281018\n-588452\n-528791\n-723434\n685282\n281034\n-989871\n-791991\n-128417\n397657\n711738\n879665\n965144\n1007666\n1028587\n1038826\n1043824\n1046261\n1047449\n1048027\n1048309\n1048446\n1048513\n1048545\n1048561\n1048569\n1048572\n1048574\n-726554\n221739\n920207\n927536\n510475\n-723379\n-629600\n-659985\n639592\n320450\n-980029\n-804978\n-142283\n388611\n706723\n877070\n963843\n1007023\n1028272\n1038672\n1043749\n1046225\n1047431\n1048018\n1048304\n1048444\n1048512\n1048545\n1048561\n1048569\n1048572\n1048574\n-1028764\n-488627\n1048487\n1037510\n711360\n-838542\n-722022\n-592252\n591879\n359373\n-969451\n-817671\n-156123\n379532\n701678\n874457\n962532\n1006376\n1027954\n1038517\n1043674\n1046188\n1047413\n1048009\n1048300\n1048442\n1048511\n1048544\n1048560\n1048568\n1048572\n1048574\n-385133\n-970549\n933006\n1028823\n872430\n-930794\n-804827\n-520676\n542295\n397742\n-958146\n-830065\n-169936\n370419\n696603\n871826\n961212\n1005724\n1027635\n1038361\n1043597\n1046151\n1047395\n1048001\n1048296\n1048440\n1048510\n1048544\n1048560\n1048568\n1048572\n1048574\n612587\n-998716\n600612\n902467\n984669\n-997616\n-876911\n-445720\n490997\n435499\n-946123\n-842157\n-183719\n361275\n691499\n869177\n959882\n1005066\n1027312\n1038203\n1043521\n1046113\n1047376\n1047992\n1048291\n1048437\n1048509\n1048543\n1048560\n1048568\n1048572\n1048574\n1047098\n-559959\n128583\n672896\n1041794\n-1037180\n-937315\n-367872\n438146\n472586\n-933390\n-853940\n-197469\n352099\n686366\n866510\n958543\n1004404\n1026988\n1038045\n1043443\n1046075\n1047358\n1047983\n1048287\n1048435\n1048507\n1048543\n1048560\n1048568\n1048572\n1048574\n518911\n140593\n-373340\n366365\n1040610\n-1048408\n-985233\n-287636\n383910\n508945\n-919958\n-865412\n-211183\n342892\n681203\n863825\n957194\n1003737\n1026660\n1037885\n1043365\n1046037\n1047340\n1047974\n1048283\n1048433\n1048506\n1048542\n1048560\n1048568\n1048572\n1048574\n-486360\n775413\n-788466\n17932\n981181\n-1030991\n-1020027\n-205533\n328461\n544520\n-905835\n-876568\n-224861\n333656\n676012\n861123\n955835\n1003065\n1026331\n1037724\n1043287\n1045999\n1047321\n1047965\n1048278\n1048431\n1048505\n1048542\n1048559\n1048568\n1048572\n1048574\n-1044474\n1047710\n-1020283\n-332552\n866835\n-985405\n-1041234\n-122096\n271973\n579257\n-891034\n-887404\n-238498\n324392\n670793\n858402\n954467\n1002389\n1025999\n1037561\n1043207\n1045960\n1047302\n1047956\n1048274\n1048429\n1048504\n1048541\n1048559\n1048568\n1048572\n1048574\n-642303\n830177\n-1014896\n-645001\n703971\n-912896\n-1048572\n-37867\n214625\n613101\n-875564\n-897916\n-252093\n315099\n665545\n855665\n953089\n1001707\n1025664\n1037398\n1043128\n1045922\n1047283\n1047946\n1048269\n1048427\n1048503\n1048541\n1048559\n1048568\n1048572\n1048574\n350398\n224516\n-773558\n-883681\n501704\n-815446\n-1041942\n46608\n156599\n646002\n-859438\n-908100\n-265643\n305778\n660269\n852909\n951702\n1001021\n1025327\n1037233\n1043047\n1045882\n1047264\n1047937\n1048265\n1048424\n1048502\n1048540\n1048559\n1048567\n1048572\n1048574\n1020945\n-486111\n-352378\n-1021293\n271357\n-695716\n-1021433\n130781\n98078\n677908\n-842667\n-917952\n-279146\n296432\n654966\n850136\n950305\n1000329\n1024988\n1037067\n1042966\n1045843\n1047245\n1047928\n1048260\n1048422\n1048501\n1048540\n1048558\n1048567\n1048572\n1048574\n752840\n-969470\n150727\n-1042099\n25822\n-556978\n-987318\n214105\n39247\n708771\n-825265\n-927469\n-292599\n287059\n649634\n847346\n948899\n999633\n1024646\n1036900\n1042885\n1045803\n1047225\n1047918\n1048256\n1048420\n1048500\n1048539\n1048558\n1048567\n1048572\n1048574\n-207423\n-999578\n618789\n-943718\n-221158\n-403022\n-940051\n296039\n-19709\n738542\n-807243\n-936647\n-306000\n277662\n644276\n844538\n947483\n998932\n1024302\n1036732\n1042803\n1045763\n1047206\n1047909\n1048251\n1048418\n1048499\n1048538\n1048558\n1048567\n1048572\n1048574\n-976982\n-562359\n942990\n-737404\n-455761\n-238055\n-880263\n376052\n-78602\n767176\n-788616\n-945484\n-319347\n268241\n638890\n841713\n946058\n998226\n1023955\n1036562\n1042720\n1045723\n1047186\n1047899\n1048246\n1048415\n1048498\n1048538\n1048557\n1048567\n1048572\n1048574\n-848308\n137776\n1047957\n-446751\n-664853\n-66584\n-808749\n453624\n-137247\n794628\n-769398\n-953975\n-332637\n258796\n633477\n838871\n944623\n997515\n1023606\n1036392\n1042637\n1045682\n1047167\n1047890\n1048242\n1048413\n1048497\n1048537\n1048557\n1048567\n1048572\n1048574\n60296\n773498\n909287\n-105003\n-836733\n106706\n-726463\n528252\n-195457\n820858\n-749603\n-962118\n-345867\n249329\n628037\n836011\n943179\n996799\n1023255\n1036220\n1042553\n1045641\n1047147\n1047880\n1048237\n1048411\n1048496\n1048537\n1048557\n1048567\n1048571\n1048574\n913464\n1047590\n559218\n248754\n-961780\n277081\n-634500\n599451\n-253050\n845823\n-729246\n-969909\n-359037\n239840\n622571\n833135\n941725\n996079\n1022901\n1036047\n1042468\n1045600\n1047127\n1047870\n1048232\n1048409\n1048494\n1048536\n1048557\n1048567\n1048571\n1048574\n926798\n831910\n79138\n574061\n-1032996\n439886\n-534085\n666760\n-309843\n869486\n-708342\n-977346\n-372142\n230331\n617079\n830241\n940262\n995353\n1022544\n1035872\n1042383\n1045559\n1047106\n1047860\n1048227\n1048406\n1048493\n1048536\n1048556\n1048566\n1048571\n1048574\n88038\n227291\n-419341\n833712\n-1046393\n590672\n-426555\n729741\n-365656\n891810\n-686907\n-984427\n-385181\n220802\n611561\n827330\n938790\n994623\n1022186\n1035697\n1042298\n1045517\n1047086\n1047850\n1048223\n1048404\n1048492\n1048535\n1048556\n1048566\n1048571\n1048574\n-831664\n-483592\n-820328\n998010\n-1001223\n725320\n-313344\n787986\n-420313\n912761\n-664957\n-991148\n-398152\n211253\n606017\n824402\n937308\n993888\n1021824\n1035520\n1042211\n1045475\n1047066\n1047840\n1048218\n1048402\n1048491\n1048535\n1048556\n1048566\n1048571\n1048574\n-986738\n-968384\n-1030599\n1048165\n-900013\n840151\n-195959\n841117\n-473642\n932307\n-642508\n-997507\n-411052\n201687\n600447\n821457\n935817\n993148\n1021461\n1035342\n1042125\n1045433\n1047045\n1047830\n1048213\n1048399\n1048490\n1048534\n1048556\n1048566\n1048571\n1048574\n-234609\n-1000433\n-1001268\n978440\n-748428\n932027\n-75964\n888789\n-525473\n950417\n-619578\n-1003501\n-423879\n192103\n594852\n818496\n934316\n992403\n1021095\n1035163\n1042037\n1045390\n1047024\n1047820\n1048208\n1048397\n1048489\n1048533\n1048555\n1048566\n1048571\n1048574\n733218\n-564756\n-739153\n796809\n-554953\n998439\n45044\n930692\n-575643\n967063\n-596183\n-1009129\n-436631\n182502\n589232\n815517\n932807\n991654\n1020726\n1034983\n1041949\n1045347\n1047003\n1047810\n1048203\n1048394\n1048488\n1048533\n1048555\n1048566\n1048571\n1048574\n1026928\n134958\n-305195\n524047\n-330417\n1037572\n165451\n966555\n-623993\n982221\n-572341\n-1014389\n-449304\n172885\n583587\n812522\n931287\n990899\n1020355\n1034802\n1041861\n1045304\n1046982\n1047800\n1048198\n1048392\n1048486\n1048532\n1048555\n1048566\n1048571\n1048574\n376485\n771576\n199718\n191348\n-87387\n1048356\n283654\n996144\n-670370\n995866\n-548069\n-1019279\n-461898\n163254\n577918\n809510\n929759\n990140\n1019982\n1034619\n1041772\n1045261\n1046961\n1047789\n1048193\n1048389\n1048485\n1048532\n1048554\n1048566\n1048571\n1048574\n-620096\n1047463\n658198\n-163235\n160535\n1030498\n398079\n1019269\n-714628\n1007978\n-523387\n-1023796\n-474410\n153608\n572224\n806481\n928221\n989376\n1019606\n1034435\n1041682\n1045217\n1046940\n1047779\n1048188\n1048387\n1048484\n1048531\n1048554\n1048565\n1048571\n1048573\n-1046564\n833636\n963656\n-499149\n399471\n984484\n507201\n1035777\n-756626\n1018538\n-498312\n-1027939\n-486838\n143949\n566506\n803436\n926674\n988607\n1019228\n1034250\n1041592\n1045173\n1046918\n1047769\n1048183\n1048385\n1048483\n1048531\n1048554\n1048565\n1048571\n1048573\n-510826\n230064\n1045075\n-777974\n616048\n911573\n609568\n1045564\n-796233\n1027530\n-472864\n-1031707\n-499179\n134278\n560764\n800375\n925118\n987833\n1018848\n1034064\n1041501\n1045129\n1046897\n1047758\n1048178\n1048382\n1048482\n1048530\n1048554\n1048565\n1048571\n1048573\n494563\n-481068\n883526\n-967822\n798144\n813756\n703814\n1048564\n-833322\n1034940\n-447061\n-1035098\n-511431\n124594\n554999\n797297\n923552\n987055\n1018465\n1033877\n1041409\n1045084\n1046875\n1047748\n1048173\n1048380\n1048480\n1048529\n1048553\n1048565\n1048571\n1048573\n1045253\n-967290\n516568\n-1046979\n935567\n693705\n788686\n1044759\n-867777\n1040756\n-420923\n-1038111\n-523592\n114901\n549210\n794202\n921977\n986271\n1018079\n1033688\n1041317\n1045039\n1046853\n1047737\n1048167\n1048377\n1048479\n1048529\n1048553\n1048565\n1048571\n1048573\n634942\n-1001280\n29515\n-1006391\n1020626\n554702\n863052\n1034173\n-899489\n1044969\n-394469\n-1040746\n-535660\n105197\n543398\n791092\n920393\n985483\n1017692\n1033498\n1041224\n1044994\n1046831\n1047726\n1048162\n1048374\n1048478\n1048528\n1048553\n1048565\n1048570\n1048573\n-359132\n-567148\n-464401\n-850702\n1048559\n400543\n925921\n1016875\n-928356\n1047574\n-367719\n-1043000\n-547633\n95484\n537563\n787965\n918800\n984690\n1017301\n1033307\n1041131\n1044948\n1046809\n1047715\n1048157\n1048372\n1048477\n1048528\n1048552\n1048565\n1048570\n1048573\n-1023021\n132140\n-850349\n-597716\n1017803\n235440\n976457\n992977\n-954289\n1048565\n-340694\n-1044873\n-559508\n85763\n531705\n784821\n917197\n983893\n1016909\n1033115\n1041037\n1044903\n1046787\n1047705\n1048152\n1048369\n1048475\n1048527\n1048552\n1048564\n1048570\n1048573\n-746350\n769649\n-1038601\n-276369\n930079\n63905\n1013986\n962634\n-977205\n1047943\n-313413\n-1046364\n-571284\n76034\n525825\n781662\n915586\n983090\n1016514\n1032922\n1040943\n1044857\n1046764\n1047694\n1048146\n1048367\n1048474\n1048526\n1048552\n1048564\n1048570\n1048573\n216512\n1047329\n-985390\n76586\n790298\n-109376\n1038009\n926044\n-997031\n1045706\n-285897\n-1047474\n-582958\n66299\n519922\n778487\n913965\n982283\n1016116\n1032727\n1040848\n1044810\n1046742\n1047683\n1048141\n1048364\n1048473\n1048526\n1048552\n1048564\n1048570\n1048573\n980314\n835357\n-703089\n420783\n606283\n-279669\n1048205\n883443\n-1013706\n1041860\n-258167\n-1048201\n-594529\n56558\n513998\n775296\n912335\n981471\n1015716\n1032531\n1040752\n1044764\n1046719\n1047672\n1048136\n1048361\n1048472\n1048525\n1048551\n1048564\n1048570\n1048573\n842820\n232836\n-257327\n716854\n388334\n-442321\n1044438\n835109\n-1027176\n1036409\n-230243\n-1048545\n-605994\n46812\n508052\n772088\n910696\n980654\n1015314\n1032334\n1040656\n1044717\n1046696\n1047660\n1048130\n1048359\n1048470\n1048525\n1048551\n1048564\n1048570\n1048573\n-69559\n-478542\n248260\n930937\n148649\n-592888\n1026759\n781354\n-1037398\n1029362\n-202147\n-1048507\n-617351\n37062\n502085\n768865\n909048\n979832\n1014909\n1032136\n1040560\n1044670\n1046673\n1047649\n1048125\n1048356\n1048469\n1048524\n1048551\n1048564\n1048570\n1048573\n-917986\n-966189\n696130\n1038548\n-99356\n-727256\n995403\n722528\n-1044340\n1020731\n-173899\n-1048085\n-628599\n27309\n496097\n765626\n907390\n979006\n1014502\n1031937\n1040462\n1044622\n1046650\n1047638\n1048119\n1048354\n1048468\n1048523\n1048550\n1048563\n1048570\n1048573\n-922420\n-1002120\n982157\n1027379\n-341799\n-841754\n950788\n659013\n-1047981\n1010528\n-145520\n-1047281\n-639734\n17553\n490087\n762372\n905724\n978174\n1014093\n1031736\n1040364\n1044574\n1046627\n1047627\n1048114\n1048351\n1048466\n1048523\n1048550\n1048563\n1048570\n1048573\n-78786\n-569536\n1039845\n898708\n-565112\n-933254\n893508\n591221\n-1048309\n998769\n-117033\n-1046095\n-650756\n7796\n484057\n759102\n904049\n977339\n1013681\n1031534\n1040266\n1044526\n1046603\n1047615\n1048108\n1048348\n1048465\n1048522\n1048550\n1048563\n1048570\n1048573\n837284\n129320\n855781\n667250\n-756795\n-999256\n824326\n519591\n-1045323\n985472\n-88457\n-1044526\n-661662\n-1961\n478007\n755816\n902364\n976498\n1013267\n1031332\n1040167\n1044478\n1046580\n1047604\n1048103\n1048345\n1048464\n1048521\n1048549\n1048563\n1048570\n1048573\n983559\n767716\n472759\n359478\n-906120\n-1037956\n744164\n444589\n-1039032\n970658\n-59816\n-1042576\n-672451\n-11719\n471936\n752514\n900671\n975652\n1012850\n1031127\n1040067\n1044430\n1046556\n1047592\n1048097\n1048343\n1048462\n1048521\n1048549\n1048563\n1048570\n1048573\n225554\n1047186\n-20175\n10592\n-1004727\n-1048298\n654089\n366701\n-1029456\n954349\n-31129\n-1040245\n-683120\n-21475\n465846\n749198\n898968\n974802\n1012431\n1030922\n1039967\n1044381\n1046532\n1047581\n1048091\n1048340\n1048461\n1048520\n1048549\n1048563\n1048570\n1048573\n-739824\n837071\n-508418\n-339505\n-1047099\n-1029998\n555302\n286434\n-1016626\n936571\n-2419\n-1037535\n-693667\n-31230\n459736\n745865\n897257\n973947\n1012009\n1030716\n1039866\n1044331\n1046508\n1047569\n1048086\n1048337\n1048460\n1048519\n1048548\n1048563\n1048569\n1048573\n-1025011\n235606\n-878460\n-650773\n-1030864\n-983557\n449118\n204307\n-1000581\n917350\n26292\n-1034445\n-704091\n-40982\n453606\n742518\n895537\n973087\n1011585\n1030508\n1039764\n1044282\n1046484\n1047557\n1048080\n1048334\n1048458\n1048519\n1048548\n1048562\n1048569\n1048573\n-367808\n-476011\n-1044270\n-887611\n-956930\n-910243\n336951\n120855\n-981373\n896718\n54984\n-1030978\n-714390\n-50730\n447457\n739155\n893807\n972223\n1011159\n1030299\n1039662\n1044232\n1046460\n1047545\n1048074\n1048332\n1048457\n1048518\n1048548\n1048562\n1048569\n1048573\n627556\n-965082\n-967301\n-1022932\n-829436\n-812060\n220296\n36618\n-959063\n874704\n83635\n-1027134\n-724562\n-60474\n441290\n735777\n892069\n971354\n1010730\n1030089\n1039560\n1044182\n1046436\n1047534\n1048068\n1048329\n1048456\n1048517\n1048547\n1048562\n1048569\n1048573\n1045948\n-1002953\n-665445\n-1041259\n-655517\n-691690\n100706\n-47857\n-933721\n851344\n112223\n-1022916\n-734605\n-70213\n435104\n732384\n890322\n970480\n1010299\n1029878\n1039457\n1044132\n1046411\n1047522\n1048062\n1048326\n1048454\n1048517\n1048547\n1048562\n1048569\n1048573\n502700\n-571920\n-208881\n-940496\n-444908\n-552422\n-20225\n-132021\n-905426\n826673\n140727\n-1018323\n-744517\n-79946\n428899\n728976\n888566\n969601\n1009866\n1029665\n1039353\n1044081\n1046386\n1047510\n1048057\n1048323\n1048453\n1048516\n1048547\n1048562\n1048569\n1048573\n-502728\n126500\n296245\n-732167\n-209398\n-398061\n-140886\n-215328\n-874270\n800729\n169125\n-1013359\n-754297\n-89671\n422676\n725553\n886801\n968718\n1009430\n1029452\n1039249\n1044030\n1046362\n1047497\n1048051\n1048320\n1048451\n1048515\n1048546\n1048562\n1048569\n1048573\n-1045950\n765778\n732498\n-440100\n37833\n-232824\n-259671\n-297237\n-840349\n773552\n197397\n-1008025\n-763943\n-99389\n416436\n722115\n885027\n967830\n1008992\n1029237\n1039144\n1043979\n1046337\n1047485\n1048045\n1048317\n1048450\n1048515\n1048546\n1048561\n1048569\n1048573\n-627531\n1047036\n998453\n-97697\n282947\n-61225\n-374997\n-377218\n-803772\n745184\n225520\n-1002323\n-773453\n-109099\n410178\n718662\n883244\n966937\n1008551\n1029021\n1039038\n1043928\n1046312\n1047473\n1048039\n1048314\n1048449\n1048514\n1048546\n1048561\n1048569\n1048573\n367838\n838780\n1032280\n255879\n512223\n112045\n-485328\n-454750\n-764653\n715669\n253475\n-996255\n-782826\n-118799\n403903\n715195\n881452\n966040\n1008108\n1028804\n1038932\n1043876\n1046286\n1047461\n1048033\n1048312\n1048447\n1048513\n1048545\n1048561\n1048569\n1048572\n1025018\n238374\n826115\n580189\n712830\n282255\n-589194\n-529331\n-723117\n685052\n281239\n-989823\n-792059\n-128488\n397610\n711712\n879652\n965137\n1007662\n1028585\n1038825\n1043824\n1046261\n1047448\n1048027\n1048309\n1048446\n1048513\n1048545\n1048561\n1048569\n1048572\n739801\n-473478\n427887\n838143\n873539\n444753\n-685212\n-600476\n-679296\n653380\n308792\n-983029\n-801151\n-138167\n391301\n708216\n877843\n964230\n1007215\n1028366\n1038718\n1043772\n1046235\n1047436\n1048021\n1048306\n1048444\n1048512\n1048545\n1048561\n1048569\n1048572\n-225585\n-963967\n-69819\n1000238\n985355\n595100\n-772102\n-667724\n-633326\n620701\n336114\n-975877\n-810101\n-147833\n384976\n704704\n876025\n963319\n1006764\n1028145\n1038610\n1043719\n1046210\n1047424\n1048015\n1048303\n1048443\n1048511\n1048544\n1048561\n1048569\n1048572\n-983570\n-1003779\n-551293\n1047934\n1042020\n729187\n-848708\n-730638\n-585354\n587068\n363184\n-968368\n-818907\n-157487\n378634\n701178\n874198\n962403\n1006312\n1027923\n1038502\n1043666\n1046184\n1047411\n1048009\n1048300\n1048441\n1048510\n1048544\n1048560\n1048568\n1048572\n-837265\n-574299\n-904598\n975776\n1040361\n843352\n-914009\n-788810\n-535532\n552530\n389982\n-960505\n-827567\n-167127\n372276\n697638\n872363\n961482\n1005857\n1027700\n1038393\n1043613\n1046158\n1047398\n1048002\n1048297\n1048440\n1048510\n1048544\n1048560\n1048568\n1048572\n78817\n123679\n-1047595\n792018\n980473\n934475\n-967134\n-841863\n-484017\n517142\n416487\n-952292\n-836080\n-176753\n365903\n694084\n870518\n960556\n1005399\n1027476\n1038283\n1043559\n1046132\n1047386\n1047996\n1048294\n1048439\n1048509\n1048543\n1048560\n1048568\n1048572\n922435\n763834\n-947038\n517676\n865706\n1000066\n-1007377\n-889451\n-430971\n480957\n442679\n-943731\n-844445\n-186363\n359514\n690515\n868666\n959626\n1004939\n1027250\n1038173\n1043506\n1046106\n1047373\n1047990\n1048291\n1048437\n1048508\n1048543\n1048560\n1048568\n1048572\n917971\n1046878\n-626307\n184127\n702485\n1038334\n-1034202\n-931267\n-376563\n444032\n468540\n-934826\n-852659\n-195957\n353110\n686932\n866804\n958691\n1004477\n1027023\n1038062\n1043452\n1046080\n1047360\n1047984\n1048288\n1048436\n1048508\n1048543\n1048560\n1048568\n1048572\n69528\n840482\n-159966\n-170482\n499945\n1048232\n-1047250\n-967039\n-320965\n406423\n494050\n-925579\n-860722\n-205535\n346691\n683335\n864934\n957751\n1004013\n1026795\n1037951\n1043397\n1046053\n1047347\n1047978\n1048285\n1048434\n1048507\n1048542\n1048560\n1048568\n1048572\n-842838\n241140\n343565\n-505592\n269422\n1029492\n-1046349\n-996534\n-264351\n368188\n519189\n-915994\n-868631\n-215094\n340257\n679723\n863055\n956807\n1003546\n1026566\n1037839\n1043343\n1046026\n1047334\n1047971\n1048281\n1048433\n1048506\n1048542\n1048559\n1048568\n1048572\n-980303\n-470941\n767221\n-782877\n23820\n982623\n-1031510\n-1019561\n-206902\n329386\n543939\n-906074\n-876386\n-224635\n333809\n676098\n861167\n955858\n1003077\n1026336\n1037726\n1043288\n1046000\n1047321\n1047965\n1048278\n1048431\n1048505\n1048542\n1048559\n1048568\n1048572\n-216481\n-962845\n1012507\n-970623\n-223116\n908908\n-1002930\n-1035971\n-148799\n290078\n568280\n-895824\n-883986\n-234156\n327347\n672459\n859271\n954904\n1002605\n1026105\n1037613\n1043233\n1045973\n1047308\n1047958\n1048275\n1048430\n1048505\n1048541\n1048559\n1048568\n1048572\n746372\n-1004597\n1022398\n-1047358\n-457563\n810359\n-960992\n-1045658\n-90226\n250322\n592196\n-885247\n-891428\n-243657\n320872\n668807\n857366\n953946\n1002131\n1025872\n1037499\n1043177\n1045946\n1047295\n1047952\n1048272\n1048428\n1048504\n1048541\n1048559\n1048568\n1048572\n1023015\n-576675\n794593\n-1004305\n-666401\n689671\n-906252\n-1048557\n-31367\n210181\n615668\n-874346\n-898711\n-253137\n314382\n665140\n855453\n952983\n1001654\n1025638\n1037385\n1043121\n1045919\n1047282\n1047946\n1048269\n1048427\n1048503\n1048541\n1048559\n1048568\n1048572\n359102\n120856\n382055\n-846389\n-837939\n550139\n-839441\n-1044652\n27591\n169717\n638678\n-863126\n-905835\n-262595\n307880\n661460\n853531\n952015\n1001176\n1025403\n1037270\n1043065\n1045891\n1047268\n1047939\n1048266\n1048425\n1048502\n1048540\n1048559\n1048568\n1048572\n-634967\n761884\n-119307\n-591671\n-962577\n395576\n-761448\n-1033966\n86461\n128991\n661209\n-851591\n-912798\n-272031\n301364\n657766\n851601\n951043\n1000694\n1025167\n1037155\n1043009\n1045864\n1047255\n1047933\n1048263\n1048423\n1048502\n1048540\n1048558\n1048567\n1048572\n-1045251\n1046713\n-592931\n-269282\n-1033338\n230206\n-673312\n-1016569\n145059\n88067\n683244\n-839745\n-919598\n-281442\n294836\n654059\n849662\n950066\n1000211\n1024930\n1037039\n1042952\n1045836\n1047242\n1047926\n1048259\n1048422\n1048501\n1048539\n1048558\n1048567\n1048572\n-494535\n842178\n-928705\n83905\n-1046262\n58546\n-576208\n-992575\n203198\n47007\n704767\n-827592\n-926235\n-290830\n288295\n650338\n847715\n949085\n999725\n1024691\n1036922\n1042896\n1045809\n1047228\n1047920\n1048256\n1048420\n1048500\n1048539\n1048558\n1048567\n1048572\n510853\n243905\n-1048567\n427496\n-1000626\n-114714\n-471428\n-962138\n260694\n5874\n725762\n-815137\n-932707\n-300192\n281742\n646604\n845759\n948098\n999237\n1024452\n1036805\n1042838\n1045781\n1047214\n1047913\n1048253\n1048419\n1048499\n1048539\n1048558\n1048567\n1048572\n1046566\n-468400\n-924650\n722193\n-898983\n-284839\n-360369\n-925457\n317366\n-35267\n746212\n-802385\n-939013\n-309528\n275177\n642857\n843794\n947108\n998746\n1024211\n1036687\n1042781\n1045753\n1047201\n1047906\n1048250\n1048417\n1048499\n1048538\n1048558\n1048567\n1048572\n620071\n-961716\n-585762\n934292\n-747024\n-447182\n-244509\n-882769\n373035\n-76354\n766103\n-789339\n-945152\n-318838\n268601\n639096\n841822\n946112\n998253\n1023969\n1036569\n1042723\n1045724\n1047187\n1047900\n1048247\n1048416\n1048498\n1048538\n1048557\n1048567\n1048572\n-376515\n-1005407\n-110692\n1039535\n-553253\n-597308\n-125393\n-834352\n427524\n-117324\n785419\n-776006\n-951123\n-328119\n262013\n635322\n839840\n945113\n997757\n1023725\n1036450\n1042665\n1045696\n1047173\n1047893\n1048243\n1048414\n1048497\n1048538\n1048557\n1048567\n1048572\n-1026934\n-579046\n390113\n1025886\n-328515\n-731114\n-4606\n-780520\n480662\n-158113\n804146\n-762388\n-956926\n-337373\n255415\n631536\n837851\n944108\n997260\n1023481\n1036330\n1042607\n1045668\n1047159\n1047886\n1048240\n1048412\n1048496\n1048537\n1048557\n1048567\n1048572\n-733195\n118033\n800221\n894904\n-85391\n-844944\n116242\n-721622\n532281\n-198658\n822271\n-748493\n-962557\n-346597\n248805\n627736\n835853\n943099\n996760\n1023235\n1036210\n1042548\n1045639\n1047146\n1047879\n1048237\n1048411\n1048496\n1048537\n1048557\n1048567\n1048571\n234640\n759929\n1024287\n661571\n162513\n-935689\n235542\n-658040\n582216\n-238898\n839779\n-734324\n-968018\n-355791\n242185\n623923\n833847\n942085\n996257\n1022988\n1036090\n1042489\n1045610\n1047132\n1047873\n1048233\n1048409\n1048495\n1048536\n1048557\n1048567\n1048571\n986748\n1046540\n1010219\n352574\n401322\n-1000870\n351705\n-590188\n630311\n-278770\n856657\n-719887\n-973307\n-364954\n235555\n620098\n831832\n941067\n995752\n1022740\n1035968\n1042430\n1045582\n1047117\n1047866\n1048230\n1048408\n1048494\n1048536\n1048557\n1048567\n1048571\n831645\n843868\n761287\n3252\n617667\n-1038705\n463182\n-518505\n676413\n-318212\n872892\n-705187\n-978422\n-374086\n228915\n616260\n829809\n940044\n995245\n1022491\n1035846\n1042371\n1045553\n1047103\n1047859\n1048227\n1048406\n1048493\n1048536\n1048556\n1048566\n1048571\n-88069\n246668\n335364\n-346442\n799441\n-1048160\n568490\n-443457\n720377\n-357165\n888474\n-690230\n-983364\n-383186\n222265\n612409\n827778\n939016\n994735\n1022241\n1035724\n1042311\n1045524\n1047089\n1047852\n1048223\n1048404\n1048492\n1048535\n1048556\n1048566\n1048571\n-926813\n-465856\n-168526\n-656512\n936470\n-1028978\n666225\n-365530\n762063\n-395567\n903389\n-675020\n-988130\n-392252\n215606\n608546\n825738\n937984\n994223\n1021989\n1035601\n1042251\n1045494\n1047075\n1047845\n1048220\n1048403\n1048492\n1048535\n1048556\n1048566\n1048571\n-913449\n-960580\n-633236\n-891497\n1021083\n-981683\n755086\n-285231\n801340\n-433361\n917626\n-659564\n-992721\n-401284\n208938\n604670\n823690\n936948\n993709\n1021737\n1035477\n1042191\n1045465\n1047061\n1047838\n1048217\n1048401\n1048491\n1048534\n1048556\n1048566\n1048571\n-60264\n-1006211\n-950727\n-1024520\n1048545\n-907566\n833889\n-203081\n838084\n-470487\n931176\n-643867\n-997136\n-410281\n202261\n600782\n821634\n935907\n993192\n1021483\n1035353\n1042130\n1045435\n1047046\n1047831\n1048213\n1048399\n1048490\n1048534\n1048556\n1048566\n1048571\n848327\n-581413\n-1047184\n-1040368\n1017319\n-808653\n901584\n-119613\n872178\n-506889\n944027\n-627935\n-1001373\n-419243\n195575\n596881\n819570\n934861\n992673\n1021228\n1035228\n1042069\n1045406\n1047032\n1047824\n1048210\n1048398\n1048489\n1048534\n1048555\n1048566\n1048571\n976970\n115209\n-900184\n-937227\n929153\n-687646\n957269\n-35369\n903515\n-542510\n956170\n-611774\n-1005432\n-428169\n188881\n592968\n817497\n933811\n992152\n1020971\n1035103\n1042008\n1045376\n1047017\n1047817\n1048206\n1048396\n1048488\n1048533\n1048555\n1048566\n1048571\n207392\n757968\n-543902\n-726895\n788980\n-547852\n1000204\n49105\n931996\n-577297\n967597\n-595389\n-1009312\n-437057\n182179\n589043\n815417\n932756\n991628\n1020714\n1034977\n1041946\n1045346\n1047003\n1047810\n1048203\n1048394\n1048488\n1048533\n1048555\n1048566\n1048571\n-752862\n1046359\n-61169\n-433426\n604648\n-393089\n1029815\n133260\n957530\n-611194\n978297\n-578787\n-1013013\n-445908\n175470\n585106\n813328\n931696\n991102\n1020455\n1034851\n1041885\n1045316\n1046988\n1047803\n1048199\n1048393\n1048487\n1048533\n1048555\n1048566\n1048571\n-1020938\n845552\n435785\n-90387\n386473\n-227586\n1045708\n216551\n980036\n-644150\n988265\n-561974\n-1016533\n-454720\n168753\n581157\n811231\n930632\n990574\n1020195\n1034723\n1041823\n1045285\n1046973\n1047795\n1048196\n1048391\n1048486\n1048532\n1048555\n1048566\n1048571\n-350368\n249429\n831424\n262991\n146666\n-55866\n1047673\n298436\n999445\n-676114\n997491\n-544955\n-1019873\n-463493\n162029\n577196\n809126\n929564\n990043\n1019934\n1034596\n1041760\n1045255\n1046958\n1047788\n1048192\n1048389\n1048485\n1048532\n1048554\n1048565\n1048571\n642328\n-463308\n1033767\n586289\n-101349\n117382\n1035682\n378384\n1015693\n-707037\n1005969\n-527738\n-1023032\n-472225\n155298\n573223\n807013\n928491\n989510\n1019672\n1034467\n1041698\n1045225\n1046944\n1047781\n1048189\n1048387\n1048484\n1048531\n1048554\n1048565\n1048571\n1044471\n-959437\n995772\n842534\n-343692\n287422\n1009895\n455876\n1028731\n-736872\n1013693\n-510328\n-1026008\n-480917\n148561\n569238\n804892\n927413\n988974\n1019409\n1034339\n1041635\n1045194\n1046929\n1047774\n1048185\n1048386\n1048483\n1048531\n1048554\n1048565\n1048571\n486332\n-1007006\n726271\n1002416\n-566798\n449609\n970656\n530409\n1038517\n-765572\n1020657\n-492731\n-1028802\n-489567\n141817\n565242\n802762\n926331\n988437\n1019145\n1034209\n1041572\n1045163\n1046914\n1047766\n1048182\n1048384\n1048482\n1048530\n1048554\n1048565\n1048571\n-518939\n-583776\n287921\n1047651\n-758180\n599512\n918488\n601500\n1045019\n-793093\n1026855\n-474954\n-1031414\n-498175\n135067\n561234\n800625\n925245\n987896\n1018879\n1034079\n1041508\n1045132\n1046899\n1047759\n1048178\n1048382\n1048482\n1048530\n1048554\n1048565\n1048571\n-1047099\n112384\n-217367\n973065\n-907126\n733035\n854085\n668687\n1048217\n-819394\n1032284\n-457004\n-1033841\n-506739\n128312\n557214\n798480\n924154\n987354\n1018612\n1033949\n1041444\n1045101\n1046883\n1047752\n1048175\n1048380\n1048481\n1048530\n1048553\n1048565\n1048571\n-612562\n756002\n-672120\n787188\n-1005299\n846531\n778305\n731534\n1048102\n-844432\n1036938\n-438887\n-1036085\n-515260\n121551\n553183\n796327\n923058\n986809\n1018344\n1033817\n1041380\n1045070\n1046868\n1047744\n1048171\n1048379\n1048480\n1048529\n1048553\n1048565\n1048571\n385162\n1046170\n-970613\n511280\n-1047204\n936898\n692158\n789633\n1044673\n-868170\n1040815\n-420610\n-1038145\n-523736\n114785\n549141\n794165\n921958\n986262\n1018075\n1033686\n1041316\n1045039\n1046853\n1047737\n1048167\n1048377\n1048479\n1048529\n1048553\n1048565\n1048571\n1028770\n847229\n-1043450\n176896\n-1030496\n1001667\n596791\n842607\n1037941\n-890572\n1043912\n-402179\n-1040020\n-532166\n108015\n545087\n791996\n920854\n985712\n1017804\n1033553\n1041251\n1045007\n1046838\n1047729\n1048164\n1048375\n1048478\n1048528\n1048553\n1048565\n1048571\n726531\n252188\n-873698\n-177720\n-956110\n1039069\n493475\n890113\n1027928\n-911602\n1046225\n-383602\n-1041710\n-540551\n101239\n541022\n789819\n919745\n985161\n1017533\n1033420\n1041187\n1044975\n1046822\n1047722\n1048160\n1048373\n1048477\n1048528\n1048553\n1048565\n1048570\n-243677\n-460757\n-500821\n-512010\n-828209\n1048081\n383585\n931841\n1014665\n-931229\n1047754\n-364884\n-1043215\n-548889\n94460\n536946\n787634\n918631\n984606\n1017260\n1033287\n1041121\n1044944\n1046807\n1047714\n1048156\n1048372\n1048476\n1048528\n1048552\n1048565\n1048570\n-989850\n-958287\n-11509\n-787740\n-653953\n1028459\n268586\n967521\n998195\n-949422\n1048498\n-346033\n-1044534\n-557179\n87676\n532859\n785441\n917513\n984050\n1016986\n1033153\n1041056\n1044912\n1046791\n1047707\n1048153\n1048370\n1048476\n1048527\n1048552\n1048564\n1048570\n-825959\n-1007795\n480479\n-973376\n-443094\n980736\n150009\n996922\n978569\n-966153\n1048455\n-327056\n-1045668\n-565421\n80889\n528762\n783241\n916391\n983491\n1016711\n1033018\n1040990\n1044880\n1046775\n1047699\n1048149\n1048368\n1048475\n1048527\n1048552\n1048564\n1048570\n97314\n-586134\n860760\n-1047686\n-207435\n906219\n29434\n1019852\n955849\n-981396\n1047626\n-307959\n-1046616\n-573614\n74098\n524653\n781032\n915264\n982930\n1016435\n1032883\n1040924\n1044847\n1046760\n1047691\n1048145\n1048366\n1048474\n1048526\n1048552\n1048564\n1048570\n931117\n109558\n1040926\n-1002171\n39835\n806942\n-91534\n1036164\n930107\n-995128\n1046012\n-288750\n-1047377\n-581757\n67305\n520533\n778816\n914133\n982366\n1016157\n1032747\n1040858\n1044815\n1046744\n1047684\n1048141\n1048364\n1048473\n1048526\n1048552\n1048564\n1048570\n908855\n754030\n979088\n-842036\n284875\n685618\n-211281\n1045750\n901424\n-1007328\n1043613\n-269435\n-1047952\n-589850\n60508\n516403\n776591\n912997\n981800\n1015879\n1032611\n1040791\n1044783\n1046728\n1047676\n1048138\n1048363\n1048472\n1048525\n1048551\n1048564\n1048570\n50996\n1045974\n689624\n-585596\n513970\n545561\n-328215\n1048549\n869892\n-1017977\n1040432\n-250022\n-1048341\n-597892\n53709\n512262\n774360\n911857\n981232\n1015599\n1032474\n1040724\n1044750\n1046712\n1047668\n1048134\n1048361\n1048471\n1048525\n1048551\n1048564\n1048570\n-853749\n848900\n239831\n-262181\n714298\n390599\n-440777\n1044543\n835610\n-1027059\n1036470\n-230518\n-1048544\n-605882\n46908\n508111\n772120\n910712\n980662\n1015318\n1032336\n1040657\n1044717\n1046696\n1047661\n1048130\n1048359\n1048470\n1048525\n1048551\n1048564\n1048570\n-973561\n254945\n-265720\n91220\n874645\n224965\n-547467\n1033757\n798686\n-1034559\n1031731\n-210929\n-1048560\n-613820\n40104\n503949\n769872\n909563\n980089\n1015036\n1032198\n1040590\n1044684\n1046680\n1047653\n1048126\n1048357\n1048469\n1048524\n1048551\n1048564\n1048570\n-198286\n-458203\n-709494\n434188\n986038\n53185\n-646865\n1016262\n759237\n-1040467\n1026219\n-191264\n-1048390\n-621704\n33299\n499777\n767617\n908409\n979514\n1014753\n1032059\n1040522\n1044651\n1046664\n1047645\n1048123\n1048355\n1048468\n1048524\n1048550\n1048564\n1048570\n759293\n-957130\n-988320\n727497\n1042242\n-120049\n-737646\n992171\n717388\n-1044772\n1019937\n-171529\n-1048033\n-629535\n26493\n495595\n765355\n907251\n978936\n1014468\n1031920\n1040454\n1044618\n1046648\n1047637\n1048119\n1048353\n1048468\n1048523\n1048550\n1048563\n1048570\n1018781\n-1008576\n-1037373\n937602\n1040109\n-290002\n-818601\n961641\n673270\n-1047469\n1012891\n-151731\n-1047490\n-637311\n19685\n491402\n763084\n906089\n978356\n1014182\n1031780\n1040386\n1044585\n1046632\n1047629\n1048115\n1048351\n1048467\n1048523\n1048550\n1048563\n1048570\n341606\n-588488\n-845249\n1040472\n979761\n-452032\n-888653\n924869\n627025\n-1048552\n1005084\n-131877\n-1046760\n-645032\n12877\n487199\n760806\n904922\n977774\n1013896\n1031640\n1040317\n1044551\n1046616\n1047621\n1048111\n1048350\n1048466\n1048522\n1048550\n1048563\n1048570\n-649639\n106732\n-456615\n1024342\n864575\n-601712\n-946867\n882094\n578797\n-1048022\n996525\n-111976\n-1045845\n-652697\n6068\n482987\n758521\n903751\n977190\n1013608\n1031499\n1040248\n1044518\n1046599\n1047613\n1048107\n1048348\n1048465\n1048522\n1048550\n1048563\n1048570\n-1043610\n752052\n38177\n891057\n700997\n-734952\n-992469\n833595\n528739\n-1045877\n987217\n-92033\n-1044743\n-660305\n-742\n478764\n756227\n902575\n976603\n1013318\n1031357\n1040179\n1044484\n1046583\n1047605\n1048103\n1048346\n1048464\n1048521\n1048549\n1048563\n1048570\n-478090\n1045770\n524093\n655860\n498183\n-848112\n-1024850\n779685\n477009\n-1042123\n977170\n-72057\n-1043456\n-667857\n-7551\n474532\n753926\n901395\n976014\n1013028\n1031215\n1040110\n1044450\n1046566\n1047597\n1048099\n1048344\n1048463\n1048521\n1048549\n1048563\n1048570\n526983\n850565\n888164\n345652\n267486\n-938100\n-1043580\n720715\n423772\n-1036763\n966390\n-52055\n-1041983\n-675350\n-14360\n470290\n751618\n900211\n975423\n1012737\n1031072\n1040040\n1044416\n1046550\n1047589\n1048095\n1048342\n1048462\n1048520\n1048549\n1048563\n1048570\n1047551\n257701\n1045747\n-4088\n21818\n-1002457\n-1048409\n657067\n369194\n-1029808\n954885\n-32033\n-1040325\n-682785\n-21168\n466038\n749302\n899022\n974829\n1012444\n1030929\n1039970\n1044382\n1046533\n1047581\n1048091\n1048340\n1048461\n1048520\n1048549\n1048563\n1048570\n605005\n-455645\n960206\n-353361\n-225072\n-1039426\n-1039273\n589155\n313450\n-1021266\n942665\n-12000\n-1038481\n-690161\n-27975\n461776\n746979\n897829\n974233\n1012150\n1030785\n1039900\n1044348\n1046516\n1047573\n1048087\n1048338\n1048460\n1048520\n1048549\n1048563\n1048569\n-393780\n-955966\n651429\n-662220\n-459365\n-1047995\n-1016294\n517418\n256715\n-1011153\n929737\n8037\n-1036454\n-697477\n-34781\n457506\n744648\n896632\n973635\n1011855\n1030640\n1039829\n1044313\n1046500\n1047565\n1048083\n1048336\n1048459\n1048519\n1048548\n1048563\n1048569\n-1030525\n-1009350\n191203\n-895340\n-667946\n-1027932\n-979777\n442324\n199168\n-999482\n916112\n28072\n-1034241\n-704733\n-41586\n453225\n742310\n895430\n973034\n1011559\n1030495\n1039758\n1044279\n1046483\n1047556\n1048079\n1048334\n1048458\n1048519\n1048548\n1048562\n1048569\n-719810\n-590838\n-313476\n-1026058\n-839141\n-979783\n-930209\n364359\n140991\n-986272\n901801\n48096\n-1031845\n-711928\n-48389\n448936\n739964\n894223\n972431\n1011262\n1030349\n1039687\n1044244\n1046466\n1047548\n1048075\n1048332\n1048457\n1048518\n1048548\n1048562\n1048569\n252695\n103905\n-745276\n-1039426\n-963369\n-904866\n-868250\n284029\n82369\n-971544\n886813\n68103\n-1029266\n-719061\n-55190\n444637\n737611\n893013\n971826\n1010963\n1030203\n1039616\n1044209\n1046449\n1047540\n1048071\n1048330\n1048456\n1048518\n1048548\n1048562\n1048569\n992873\n750069\n-1003807\n-933912\n-1033676\n-805225\n-794726\n201855\n23486\n-955320\n871160\n88084\n-1026503\n-726132\n-61989\n440329\n735251\n891798\n971218\n1010663\n1030056\n1039544\n1044174\n1046432\n1047532\n1048067\n1048328\n1048455\n1048517\n1048547\n1048562\n1048569\n820209\n1045559\n-1028966\n-721587\n-1046127\n-683585\n-710615\n118372\n-35471\n-937625\n854854\n108034\n-1023558\n-733140\n-68785\n436012\n732883\n890579\n970608\n1010363\n1029909\n1039472\n1044139\n1046415\n1047523\n1048063\n1048326\n1048454\n1048517\n1048547\n1048562\n1048569\n-106552\n852224\n-814902\n-426732\n-1000025\n-543267\n-617039\n34120\n-94316\n-918486\n837907\n127944\n-1020431\n-740084\n-75578\n431686\n730508\n889355\n969996\n1010061\n1029761\n1039400\n1044104\n1046397\n1047515\n1048059\n1048324\n1048453\n1048516\n1048547\n1048562\n1048569\n-935349\n260454\n-411383\n-83072\n-897951\n-388107\n-515244\n-50354\n-152863\n-897933\n820331\n147808\n-1017122\n-746965\n-82368\n427352\n728125\n888127\n969382\n1009757\n1029612\n1039327\n1044069\n1046380\n1047507\n1048055\n1048322\n1048452\n1048516\n1048547\n1048562\n1048569\n-904191\n-453084\n87777\n270090\n-745617\n-222343\n-406586\n-134500\n-210926\n-875998\n802141\n167617\n-1013633\n-753780\n-89154\n423008\n725735\n886895\n968765\n1009453\n1029463\n1039254\n1044033\n1046363\n1047498\n1048051\n1048320\n1048452\n1048515\n1048546\n1048562\n1048569\n-41724\n-954795\n566531\n592361\n-551550\n-50504\n-292511\n-217774\n-268323\n-852714\n783349\n187366\n-1009963\n-760531\n-95937\n418656\n723338\n885658\n968146\n1009148\n1029313\n1039181\n1043997\n1046346\n1047490\n1048047\n1048318\n1048451\n1048515\n1048546\n1048562\n1048569\n859104\n-1010116\n913572\n846883\n-326613\n122715\n-174540\n-299634\n-324871\n-828116\n763969\n207046\n-1006114\n-767215\n-102715\n414295\n720934\n884417\n967525\n1008841\n1029163\n1039107\n1043962\n1046328\n1047481\n1048043\n1048316\n1048450\n1048514\n1048546\n1048561\n1048569\n970076\n-593184\n1048219\n1004545\n-83394\n292581\n-54245\n-379549\n-380392\n-802244\n744017\n226650\n-1002085\n-773833\n-109490\n409926\n718523\n883172\n966901\n1008533\n1029012\n1039034\n1043926\n1046311\n1047473\n1048039\n1048314\n1048449\n1048514\n1048546\n1048561\n1048569\n189164\n101076\n939168\n1047317\n164492\n454453\n66773\n-457001\n-434711\n-775136\n723507\n246171\n-997879\n-780385\n-116259\n405548\n716104\n881922\n966275\n1008224\n1028861\n1038960\n1043890\n1046293\n1047464\n1048034\n1048312\n1048448\n1048513\n1048546\n1048561\n1048569\n-765664\n748081\n611771\n970306\n403171\n603908\n186902\n-531487\n-487656\n-746835\n702454\n265603\n-993495\n-786868\n-123024\n401162\n713678\n880669\n965647\n1007914\n1028709\n1038885\n1043853\n1046275\n1047455\n1048030\n1048310\n1048447\n1048513\n1048545\n1048561\n1048569\n-1016544\n1045339\n142145\n782320\n619285\n736864\n304541\n-602523\n-539058\n-717384\n680874\n284938\n-988935\n-793283\n-129784\n396767\n711245\n879410\n965016\n1007603\n1028556\n1038811\n1043817\n1046258\n1047447\n1048026\n1048308\n1048446\n1048512\n1048545\n1048561\n1048569\n-332818\n853877\n-360529\n504859\n800736\n849688\n418124\n-669649\n-588757\n-686829\n658784\n304168\n-984198\n-799630\n-136538\n392365\n708805\n878148\n964384\n1007290\n1028403\n1038736\n1043780\n1046240\n1047438\n1048022\n1048306\n1048445\n1048512\n1048545\n1048561\n1048569\n656899\n263206\n-779383\n169657\n937369\n939296\n526137\n-732429\n-636594\n-655216\n636200\n323288\n-979287\n-805908\n-143286\n387954\n706358\n876881\n963748\n1006976\n1028249\n1038661\n1043744\n1046222\n1047429\n1048018\n1048304\n1048444\n1048512\n1048545\n1048561\n1048569\n1042667\n-450520\n-1017041\n-184950\n1021537\n1003241\n627142\n-790455\n-682418\n-622594\n613139\n342289\n-974201\n-812115\n-150029\n383536\n703904\n875610\n963111\n1006662\n1028095\n1038586\n1043707\n1046204\n1047421\n1048013\n1048302\n1048443\n1048511\n1048544\n1048561\n1048568\n469811\n-953617\n-1018248\n-518403\n1048528\n1039776\n719792\n-843350\n-726086\n-589013\n589619\n361166\n-968942\n-818253\n-156765\n379110\n701443\n874335\n962471\n1006346\n1027940\n1038510\n1043670\n1046186\n1047412\n1048009\n1048300\n1048442\n1048511\n1048544\n1048560\n1048568\n-534987\n-1010875\n-782725\n-792566\n1016832\n1047903\n802855\n-890773\n-767457\n-554526\n565656\n379910\n-963511\n-824319\n-163494\n374675\n698975\n873056\n961829\n1006028\n1027784\n1038434\n1043633\n1046168\n1047403\n1048005\n1048298\n1048441\n1048510\n1048544\n1048560\n1048568\n-1047920\n-595525\n-365228\n-976082\n928223\n1027399\n875224\n-932413\n-806403\n-519184\n541269\n398516\n-957909\n-830314\n-170216\n370234\n696499\n871772\n961185\n1005710\n1027628\n1038358\n1043596\n1046150\n1047394\n1048000\n1048296\n1048440\n1048510\n1048544\n1048560\n1048568\n-597401\n98248\n137181\n-1047963\n787660\n978824\n935934\n-968002\n-842799\n-483044\n516476\n416977\n-952136\n-836237\n-176932\n365784\n694017\n870484\n960539\n1005391\n1027471\n1038281\n1043558\n1046132\n1047385\n1047996\n1048294\n1048438\n1048509\n1048543\n1048560\n1048568\n402366\n746087\n607696\n-999987\n603011\n903506\n984177\n-997309\n-876530\n-446159\n491296\n435285\n-946194\n-842088\n-183640\n361327\n691528\n869192\n959890\n1005070\n1027314\n1038204\n1043521\n1046113\n1047377\n1047992\n1048292\n1048437\n1048509\n1048543\n1048560\n1048568\n1032199\n1045112\n936929\n-837641\n384610\n803503\n1019311\n-1020142\n-907491\n-408588\n465747\n453434\n-940084\n-847866\n-190340\n356863\n689032\n867895\n959239\n1004748\n1027156\n1038127\n1043483\n1046095\n1047368\n1047987\n1048289\n1048436\n1048508\n1048543\n1048560\n1048568\n713033\n855523\n1048338\n-579493\n144683\n681547\n1040867\n-1036355\n-935582\n-370387\n439849\n471417\n-933806\n-853570\n-197032\n352391\n686529\n866595\n958586\n1004425\n1026998\n1038050\n1043446\n1046077\n1047359\n1047983\n1048287\n1048435\n1048508\n1048543\n1048560\n1048568\n-261692\n265956\n916020\n-255068\n-103343\n540970\n1048558\n-1045841\n-960716\n-331616\n413621\n489229\n-927362\n-859201\n-203716\n347912\n684019\n865290\n957930\n1004101\n1026839\n1037972\n1043408\n1046058\n1047350\n1047979\n1048285\n1048434\n1048507\n1048542\n1048560\n1048568\n-995819\n-447952\n570739\n98530\n-345584\n385612\n1042282\n-1048540\n-982812\n-292335\n387084\n506862\n-920754\n-864757\n-210391\n343426\n681503\n863981\n957272\n1003776\n1026679\n1037894\n1043370\n1046040\n1047341\n1047974\n1048283\n1048433\n1048507\n1048542\n1048560\n1048568\n-814394\n-952432\n92768\n440858\n-568482\n219719\n1022122\n-1044433\n-1001801\n-252603\n360255\n524309\n-913982\n-870238\n-217057\n338933\n678979\n862668\n956612\n1003450\n1026519\n1037816\n1043331\n1046021\n1047332\n1047970\n1048281\n1048432\n1048506\n1048542\n1048559\n1048568\n115781\n-1011626\n-406771\n732765\n-759562\n47823\n988348\n-1033547\n-1017624\n-212483\n333157\n541565\n-907047\n-875644\n-223714\n334433\n676449\n861350\n955950\n1003122\n1026358\n1037737\n1043293\n1046002\n1047322\n1047965\n1048279\n1048431\n1048505\n1048542\n1048559\n1048568\n939508\n-597861\n-811740\n940865\n-908129\n-125380\n941408\n-1015953\n-1030229\n-172035\n305809\n558624\n-899951\n-880974\n-230362\n329926\n673912\n860028\n955285\n1002793\n1026197\n1037658\n1043255\n1045984\n1047313\n1047961\n1048277\n1048430\n1048505\n1048541\n1048559\n1048568\n899455\n95418\n-1027990\n1041357\n-1005866\n-295157\n881928\n-991766\n-1039577\n-131323\n278232\n575478\n-892695\n-886228\n-237000\n325412\n671368\n858702\n954618\n1002463\n1026035\n1037579\n1043216\n1045965\n1047304\n1047957\n1048274\n1048429\n1048504\n1048541\n1048559\n1048568\n32448\n744087\n-1005244\n1022748\n-1047304\n-456870\n810700\n-961142\n-1045638\n-90408\n250446\n592123\n-885280\n-891405\n-243628\n320892\n668818\n857372\n953949\n1002132\n1025873\n1037500\n1043177\n1045946\n1047295\n1047952\n1048272\n1048428\n1048504\n1048541\n1048559\n1048568\n-864392\n1044878\n-748791\n887166\n-1030123\n-606101\n728674\n-924279\n-1048394\n-49354\n222472\n608551\n-877708\n-896505\n-250246\n316365\n666261\n856038\n953277\n1001800\n1025710\n1037420\n1043138\n1045927\n1047286\n1047948\n1048270\n1048427\n1048503\n1048541\n1048559\n1048568\n-966514\n857163\n-318252\n650117\n-955286\n-738772\n636941\n-881418\n-1047835\n-8224\n194332\n624757\n-869979\n-901527\n-256853\n311831\n663697\n854700\n952603\n1001467\n1025546\n1037340\n1043099\n1045908\n1047276\n1047943\n1048268\n1048426\n1048503\n1048540\n1048559\n1048568\n-180028\n268704\n186276\n338714\n-826979\n-851258\n536725\n-832836\n-1043964\n32919\n166045\n640734\n-862096\n-906471\n-263449\n307291\n661126\n853357\n951927\n1001132\n1025382\n1037260\n1043060\n1045889\n1047267\n1047939\n1048266\n1048425\n1048502\n1048540\n1048559\n1048567\n771975\n-445381\n647497\n-11428\n-652386\n-940486\n429358\n-778849\n-1036792\n74011\n137634\n656478\n-854060\n-911337\n-270034\n302745\n658549\n852010\n951249\n1000796\n1025217\n1037179\n1043021\n1045870\n1047258\n1047934\n1048263\n1048424\n1048502\n1048540\n1048558\n1048567\n1014228\n-951239\n958183\n-360263\n-441278\n-1004019\n316273\n-719807\n-1026342\n114989\n109120\n671982\n-845872\n-916123\n-276608\n298193\n655966\n850660\n950569\n1000460\n1025052\n1037098\n1042982\n1045850\n1047248\n1047929\n1048261\n1048423\n1048501\n1048540\n1048558\n1048567\n324004\n-1012370\n1046102\n-667895\n-205471\n-1040120\n198974\n-656093\n-1012648\n155790\n80525\n687241\n-837533\n-920831\n-283171\n293634\n653376\n849305\n949886\n1000122\n1024886\n1037017\n1042942\n1045831\n1047239\n1047925\n1048259\n1048422\n1048501\n1048539\n1048558\n1048567\n-664108\n-600194\n890816\n-899138\n41836\n-1047803\n79026\n-588120\n-995752\n196351\n51868\n702249\n-829046\n-925458\n-289721\n289069\n650779\n847945\n949201\n999782\n1024720\n1036936\n1042902\n1045812\n1047230\n1047920\n1048257\n1048420\n1048500\n1048539\n1048558\n1048567\n-1041642\n92588\n528425\n-1027546\n286802\n-1026858\n-41976\n-516331\n-975708\n236609\n23173\n717000\n-820411\n-930006\n-296259\n284499\n648176\n846582\n948514\n999442\n1024552\n1036854\n1042862\n1045792\n1047220\n1047916\n1048254\n1048419\n1048500\n1048539\n1048558\n1048567\n-461495\n742082\n43182\n-1038432\n515715\n-977858\n-162418\n-441190\n-952580\n276504\n-5539\n731489\n-811630\n-934473\n-302784\n279922\n645566\n845215\n947824\n999101\n1024385\n1036772\n1042822\n1045773\n1047211\n1047911\n1048252\n1048418\n1048499\n1048539\n1048558\n1048567\n542948\n1044636\n-452100\n-930552\n715763\n-902141\n-280697\n-363187\n-926440\n315972\n-34248\n745712\n-802705\n-938859\n-309297\n275340\n642950\n843843\n947132\n998758\n1024217\n1036690\n1042782\n1045753\n1047201\n1047906\n1048250\n1048417\n1048499\n1048538\n1048558\n1048567\n1048207\n858796\n-842275\n-716243\n875749\n-801776\n-395237\n-282825\n-897371\n354955\n-62930\n759662\n-793637\n-943163\n-315797\n270753\n640327\n842468\n946438\n998414\n1024048\n1036608\n1042742\n1045734\n1047192\n1047902\n1048248\n1048416\n1048498\n1048538\n1048558\n1048567\n589749\n271449\n-1036631\n-420017\n986718\n-679505\n-504512\n-200629\n-865466\n393390\n-91566\n773334\n-784428\n-947387\n-322283\n266159\n637698\n841088\n945742\n998069\n1023878\n1036525\n1042702\n1045714\n1047182\n1047897\n1048245\n1048415\n1048498\n1048538\n1048557\n1048567\n-410921\n-442807\n-989982\n-75753\n1042459\n-538669\n-607067\n-117130\n-830824\n431220\n-120133\n786725\n-775079\n-951528\n-328756\n261560\n635063\n839704\n945044\n997723\n1023709\n1036442\n1042661\n1045694\n1047172\n1047892\n1048243\n1048414\n1048497\n1048538\n1048557\n1048567\n-1033793\n-950040\n-713174\n277176\n1039853\n-383115\n-701535\n-32871\n-793555\n468386\n-148609\n799828\n-765593\n-955586\n-335215\n256956\n632421\n838316\n944343\n997376\n1023538\n1036358\n1042620\n1045674\n1047163\n1047888\n1048241\n1048413\n1048497\n1048537\n1048557\n1048567\n-706200\n-1013107\n-270562\n598403\n979046\n-217094\n-786659\n51602\n-753778\n504831\n-176975\n812639\n-755970\n-959562\n-341660\n252347\n629773\n836924\n943640\n997028\n1023367\n1036275\n1042580\n1045655\n1047153\n1047883\n1048238\n1048412\n1048496\n1048537\n1048557\n1048567\n270670\n-602522\n234952\n851190\n863440\n-45141\n-861304\n135739\n-711617\n540498\n-205207\n825153\n-746213\n-963455\n-348090\n247732\n627118\n835528\n942935\n996678\n1023195\n1036191\n1042539\n1045635\n1047143\n1047878\n1048236\n1048411\n1048495\n1048537\n1048557\n1048567\n998687\n89757\n685843\n1006626\n699506\n128045\n-924476\n218996\n-667207\n575334\n-233286\n837366\n-736323\n-967264\n-354506\n243112\n624458\n834128\n942227\n996327\n1023023\n1036106\n1042498\n1045615\n1047134\n1047873\n1048234\n1048409\n1048495\n1048536\n1048557\n1048567\n808516\n740072\n977284\n1046932\n496420\n297732\n-975334\n300831\n-620688\n609283\n-261190\n849273\n-726303\n-970990\n-360907\n238488\n621791\n832724\n941518\n995976\n1022850\n1036022\n1042456\n1045594\n1047124\n1047869\n1048232\n1048408\n1048494\n1048536\n1048557\n1048567\n-125001\n1044386\n1041518\n967500\n265549\n459285\n-1013200\n380714\n-572206\n642294\n-288898\n860870\n-716153\n-974631\n-367292\n233858\n619118\n831315\n940806\n995623\n1022677\n1035937\n1042415\n1045574\n1047114\n1047864\n1048229\n1048407\n1048494\n1048536\n1048556\n1048567\n-943593\n860424\n863611\n777413\n19815\n608289\n-1037570\n458126\n-521915\n674316\n-316389\n872152\n-705876\n-978188\n-373662\n229224\n616438\n829903\n940092\n995269\n1022503\n1035852\n1042373\n1045554\n1047104\n1047859\n1048227\n1048406\n1048493\n1048536\n1048556\n1048566\n-894650\n274193\n484925\n498413\n-227028\n740674\n-1048119\n532564\n-469975\n705300\n-343643\n883116\n-695473\n-981661\n-380017\n224585\n613753\n828487\n939375\n994913\n1022328\n1035767\n1042332\n1045534\n1047094\n1047854\n1048225\n1048405\n1048493\n1048535\n1048556\n1048566\n-23170\n-440229\n-6500\n162409\n-461164\n852822\n-1044706\n603546\n-416548\n735199\n-370639\n893758\n-684947\n-985048\n-386355\n219941\n611061\n827066\n938657\n994557\n1022153\n1035681\n1042290\n1045513\n1047084\n1047849\n1048222\n1048404\n1048492\n1048535\n1048556\n1048566\n869612\n-948834\n-496414\n-192170\n-669488\n941670\n-1027378\n670610\n-361805\n763965\n-397358\n904073\n-674299\n-988350\n-392677\n215293\n608364\n825642\n937936\n994199\n1021977\n1035595\n1042248\n1045493\n1047074\n1047845\n1048220\n1048403\n1048492\n1048535\n1048556\n1048566\n962877\n-1013836\n-870918\n-524771\n-840341\n1004790\n-996364\n733323\n-305918\n791554\n-423778\n914058\n-663531\n-991567\n-398982\n210640\n605660\n824214\n937213\n993841\n1021801\n1035509\n1042206\n1045472\n1047064\n1047840\n1048217\n1048401\n1048491\n1048535\n1048556\n1048566\n170877\n-604845\n-1042944\n-797352\n-964158\n1040457\n-952079\n791275\n-249063\n817925\n-449881\n923710\n-652645\n-994697\n-405271\n205983\n602950\n822781\n936487\n993481\n1021624\n1035422\n1042164\n1045452\n1047054\n1047835\n1048215\n1048400\n1048490\n1048534\n1048556\n1048566\n-778226\n86925\n-972497\n-978740\n-1034011\n1047697\n-895111\n844093\n-191422\n843037\n-475647\n933024\n-641643\n-997742\n-411543\n201322\n600234\n821345\n935760\n993120\n1021447\n1035336\n1042121\n1045431\n1047044\n1047830\n1048213\n1048399\n1048490\n1048534\n1048556\n1048566\n-1011832\n738056\n-675957\n-1048188\n-1045988\n1026312\n-826221\n891431\n-133175\n866850\n-501055\n941997\n-630527\n-1000700\n-417797\n196656\n597512\n819904\n935030\n992757\n1021269\n1035249\n1042079\n1045410\n1047034\n1047825\n1048210\n1048398\n1048489\n1048534\n1048555\n1048566\n-315164\n1044128\n-222265\n-997753\n-999421\n976886\n-746324\n932984\n-74507\n889329\n-526088\n950626\n-619298\n-1003571\n-424033\n191987\n594785\n818460\n934298\n992394\n1021090\n1035161\n1042036\n1045390\n1047024\n1047820\n1048208\n1048397\n1048489\n1048533\n1048555\n1048566\n671264\n862045\n283102\n-833205\n-896915\n900770\n-656487\n968482\n-15603\n910439\n-550727\n958908\n-607960\n-1006356\n-430252\n187314\n592051\n817011\n933564\n992030\n1020911\n1035074\n1041993\n1045369\n1047014\n1047815\n1048205\n1048396\n1048488\n1048533\n1048555\n1048566\n1040535\n276935\n722650\n-573361\n-744208\n800044\n-557904\n997694\n43350\n930146\n-574952\n966841\n-596514\n-1009053\n-436453\n182636\n589311\n815559\n932828\n991664\n1020731\n1034986\n1041951\n1045348\n1047004\n1047810\n1048203\n1048394\n1048488\n1048533\n1048555\n1048566\n453143\n-437649\n994191\n-247942\n-549846\n677458\n-451890\n1020431\n102165\n948422\n-598747\n974420\n-584961\n-1011663\n-442635\n177955\n586565\n814103\n932089\n991297\n1020551\n1034897\n1041908\n1045327\n1046993\n1047805\n1048201\n1048393\n1048487\n1048533\n1048555\n1048566\n-550867\n-947621\n1034594\n105835\n-324709\n536364\n-339857\n1036544\n160658\n965237\n-622092\n981643\n-573304\n-1014186\n-448798\n173271\n583814\n812642\n931348\n990929\n1020370\n1034809\n1041864\n1045306\n1046983\n1047800\n1048198\n1048392\n1048486\n1048532\n1048555\n1048566\n-1048412\n-1014558\n834467\n447507\n-81398\n380615\n-223297\n1045931\n218643\n980566\n-644971\n988507\n-561546\n-1016620\n-454943\n168583\n581057\n811178\n930605\n990560\n1020189\n1034720\n1041821\n1045285\n1046973\n1047795\n1048196\n1048391\n1048486\n1048532\n1048555\n1048566\n-582052\n-607164\n440336\n737998\n166470\n214467\n-103762\n1048528\n275937\n994385\n-667367\n995011\n-549687\n-1018967\n-461068\n163891\n578293\n809710\n929860\n990190\n1020007\n1034631\n1041778\n1045264\n1046963\n1047790\n1048193\n1048390\n1048485\n1048532\n1048554\n1048566\n419444\n84093\n-56168\n944083\n405019\n42459\n17155\n1044321\n332358\n1006673\n-689262\n1001152\n-537731\n-1021225\n-467174\n159196\n575525\n808237\n929113\n989819\n1019824\n1034542\n1041734\n1045242\n1046952\n1047785\n1048191\n1048388\n1048485\n1048532\n1048554\n1048565\n1035305\n736035\n-539614\n1042191\n620900\n-130708\n137843\n1033335\n387729\n1017411\n-710640\n1006926\n-525679\n-1023395\n-473260\n154498\n572750\n806761\n928363\n989446\n1019641\n1034452\n1041690\n1045221\n1046942\n1047780\n1048188\n1048387\n1048484\n1048531\n1048554\n1048565\n699312\n1043863\n-897605\n1021104\n802028\n-300305\n256695\n1015643\n441873\n1026583\n-731485\n1012334\n-513534\n-1025476\n-479326\n149796\n569969\n805281\n927611\n989073\n1019457\n1034362\n1041646\n1045200\n1046931\n1047775\n1048186\n1048386\n1048483\n1048531\n1048554\n1048565\n-279626\n863659\n-1046914\n883231\n938265\n-461696\n372128\n991360\n494621\n1034174\n-751782\n1017371\n-501298\n-1027469\n-485372\n145092\n567183\n803797\n926857\n988698\n1019273\n1034272\n1041602\n1045178\n1046921\n1047770\n1048183\n1048385\n1048483\n1048531\n1048554\n1048565\n-1001477\n279675\n-952829\n644342\n1021987\n-610474\n482604\n960641\n545805\n1040172\n-771515\n1022037\n-488972\n-1029372\n-491398\n140385\n564392\n802309\n926101\n988322\n1019088\n1034182\n1041558\n1045157\n1046910\n1047765\n1048181\n1048384\n1048482\n1048530\n1048554\n1048565\n-802574\n-435065\n-637221\n331759\n1048507\n-742572\n586651\n923688\n595264\n1044569\n-790669\n1026330\n-476559\n-1031186\n-497403\n135674\n561594\n800818\n925343\n987945\n1018903\n1034091\n1041514\n1045135\n1046900\n1047760\n1048178\n1048382\n1048482\n1048530\n1048554\n1048565\n134211\n-946401\n-173468\n-18768\n1016341\n-854381\n682885\n880740\n642841\n1047358\n-809231\n1030248\n-464062\n-1032911\n-503387\n130961\n558791\n799322\n924582\n987567\n1018717\n1034000\n1041469\n1045113\n1046889\n1047755\n1048176\n1048381\n1048481\n1048530\n1048554\n1048565\n947603\n-1015272\n330615\n-367148\n927289\n-942848\n770021\n832076\n688385\n1048534\n-827186\n1033789\n-451482\n-1034547\n-509349\n126245\n555983\n797822\n923819\n987188\n1018530\n1033909\n1041425\n1045092\n1046879\n1047749\n1048173\n1048380\n1048481\n1048530\n1048553\n1048565\n889774\n-609479\n757834\n-673537\n786336\n-1005554\n846901\n778011\n731753\n1048095\n-844520\n1036954\n-438822\n-1036093\n-515291\n121527\n553168\n796319\n923054\n986807\n1018343\n1033817\n1041380\n1045070\n1046868\n1047744\n1048171\n1048379\n1048480\n1048529\n1048553\n1048565\n13890\n81260\n1008865\n-902893\n601371\n-1040787\n912500\n718897\n772808\n1046043\n-861222\n1039739\n-426083\n-1037549\n-521210\n116806\n550349\n794811\n922287\n986426\n1018155\n1033725\n1041335\n1045048\n1046857\n1047739\n1048168\n1048377\n1048479\n1048529\n1048553\n1048565\n-874764\n734008\n1025347\n-1028984\n382746\n-1047583\n965944\n655117\n811419\n1042380\n-877277\n1042145\n-413269\n-1038916\n-527107\n112083\n547524\n793300\n921518\n986043\n1017967\n1033633\n1041290\n1045026\n1046847\n1047734\n1048166\n1048376\n1048479\n1048529\n1048553\n1048565\n-959164\n1043590\n803449\n-1037389\n142699\n-1025758\n1006521\n587085\n847465\n1037112\n-892675\n1044171\n-400382\n-1040192\n-532983\n107357\n544693\n791785\n920746\n985659\n1017778\n1033540\n1041245\n1045004\n1046836\n1047729\n1048163\n1048375\n1048478\n1048528\n1048553\n1048565\n-161713\n865268\n394758\n-927146\n-105335\n-975907\n1033691\n515243\n880832\n1030248\n-907403\n1045815\n-387423\n-1041379\n-538835\n102629\n541857\n790266\n919973\n985274\n1017589\n1033448\n1041200\n1044982\n1046825\n1047723\n1048161\n1048374\n1048478\n1048528\n1048553\n1048565\n784416\n282412\n-105710\n-710865\n-347474\n-899393\n1047091\n440056\n911415\n1021796\n-921451\n1047077\n-374395\n-1042475\n-544665\n97899\n539015\n788743\n919197\n984888\n1017399\n1033355\n1041154\n1044960\n1046814\n1047718\n1048158\n1048373\n1048477\n1048528\n1048553\n1048565\n1009357\n-432478\n-581601\n-413281\n-570164\n-798306\n1046544\n362014\n939116\n1011772\n-934808\n1047957\n-361301\n-1043481\n-550472\n93167\n536168\n787217\n918419\n984501\n1017208\n1033261\n1041109\n1044938\n1046804\n1047713\n1048156\n1048371\n1048476\n1048527\n1048552\n1048564\n306300\n-945174\n-922277\n-68430\n-760942\n-675407\n1032057\n281622\n963848\n1000190\n-947464\n1048454\n-348143\n-1044396\n-556256\n88433\n533316\n785686\n917638\n984112\n1017017\n1033168\n1041063\n1044915\n1046793\n1047708\n1048153\n1048370\n1048476\n1048527\n1048552\n1048564\n-678368\n-1015979\n-1048534\n284248\n-909128\n-534056\n1003822\n199402\n985532\n987068\n-959409\n1048568\n-334922\n-1045221\n-562017\n83698\n530458\n784152\n916856\n983723\n1016825\n1033074\n1041017\n1044893\n1046782\n1047702\n1048150\n1048369\n1048475\n1048527\n1048552\n1048564\n-1039347\n-611789\n-931020\n604416\n-1006430\n-378113\n962216\n115888\n1004102\n972426\n-970636\n1048300\n-321642\n-1045956\n-567753\n78960\n527595\n782614\n916071\n983332\n1016633\n1032980\n1040971\n1044871\n1046771\n1047697\n1048148\n1048368\n1048474\n1048527\n1048552\n1048564\n-444755\n78427\n-597055\n855456\n-1047401\n-211839\n907793\n31621\n1019496\n956286\n-981134\n1047648\n-308305\n-1046600\n-573466\n74221\n524727\n781072\n915284\n982940\n1016440\n1032885\n1040925\n1044848\n1046760\n1047692\n1048145\n1048366\n1048474\n1048526\n1048552\n1048564\n558743\n731976\n-124281\n1008656\n-1029748\n-39777\n841277\n-52850\n1031668\n938675\n-990896\n1046614\n-294913\n-1047154\n-579155\n69481\n521854\n779526\n914495\n982547\n1016246\n1032791\n1040879\n1044826\n1046749\n1047686\n1048143\n1048365\n1048473\n1048526\n1048552\n1048564\n1048535\n1043309\n377386\n1046496\n-954458\n133371\n763556\n-136978\n1040578\n919618\n-999916\n1045198\n-281469\n-1047616\n-584819\n64739\n518975\n777977\n913704\n982153\n1016052\n1032696\n1040833\n1044803\n1046738\n1047681\n1048140\n1048364\n1048473\n1048526\n1048551\n1048564\n574309\n866870\n791316\n964646\n-825746\n302875\n675663\n-220218\n1046198\n899145\n-1008186\n1043400\n-267975\n-1047988\n-590458\n59995\n516091\n776423\n912911\n981758\n1015858\n1032600\n1040786\n1044780\n1046727\n1047675\n1048137\n1048362\n1048472\n1048525\n1048551\n1048564\n-427934\n285148\n1021274\n772469\n-650817\n464105\n578771\n-302028\n1048511\n877287\n-1015700\n1041222\n-254433\n-1048270\n-596073\n55251\n513202\n774866\n912116\n981361\n1015662\n1032505\n1040740\n1044757\n1046716\n1047670\n1048135\n1048361\n1048471\n1048525\n1048551\n1048564\n-1036737\n-429887\n1013798\n491943\n-439460\n612654\n474169\n-381878\n1047509\n854079\n-1022452\n1038663\n-240845\n-1048460\n-601662\n50505\n510308\n773305\n911318\n980964\n1015467\n1032409\n1040693\n1044735\n1046705\n1047665\n1048132\n1048360\n1048471\n1048525\n1048551\n1048564\n-692369\n-943940\n770626\n155153\n-203507\n744464\n363251\n-459249\n1043196\n829556\n-1028437\n1035724\n-227215\n-1048560\n-607226\n45759\n507409\n771741\n910518\n980565\n1015270\n1032313\n1040646\n1044712\n1046694\n1047659\n1048130\n1048359\n1048470\n1048524\n1048551\n1048564\n288560\n-1016678\n348293\n-199381\n43837\n855935\n247494\n-533640\n1035584\n803755\n-1033652\n1032408\n-213544\n-1048569\n-612765\n41011\n504504\n770172\n909716\n980165\n1015074\n1032216\n1040599\n1044689\n1046682\n1047654\n1048127\n1048357\n1048469\n1048524\n1048551\n1048564\n1004188\n-614095\n-155014\n-531113\n288728\n944019\n128441\n-604567\n1024698\n776717\n-1038091\n1028714\n-199836\n-1048487\n-618278\n36262\n501595\n768600\n908912\n979764\n1014876\n1032120\n1040552\n1044666\n1046671\n1047648\n1048124\n1048356\n1048469\n1048524\n1048551\n1048564\n796570\n75593\n-622282\n-802100\n517458\n1006312\n7676\n-671571\n1010573\n748483\n-1041752\n1024645\n-186092\n-1048314\n-623764\n31513\n498681\n767024\n908106\n979362\n1014678\n1032023\n1040504\n1044643\n1046660\n1047643\n1048122\n1048355\n1048468\n1048524\n1048550\n1048564\n-143411\n729939\n-944877\n-981350\n717225\n1041110\n-113190\n-734216\n993253\n719096\n-1044631\n1020202\n-172315\n-1048051\n-629224\n26763\n495761\n765445\n907298\n978959\n1014479\n1031926\n1040457\n1044620\n1046649\n1047637\n1048119\n1048353\n1048468\n1048523\n1048550\n1048563\n-951540\n1043021\n-1047800\n-1048362\n876848\n1047463\n-232549\n-792095\n972793\n688603\n-1046728\n1015386\n-158507\n-1047696\n-634658\n22013\n492837\n763861\n906487\n978555\n1014280\n1031828\n1040409\n1044596\n1046637\n1047632\n1048116\n1048352\n1048467\n1048523\n1048550\n1048563\n-884828\n868465\n-907121\n-995472\n987394\n1025198\n-348810\n-844833\n949258\n657049\n-1048039\n1010199\n-144671\n-1047251\n-640065\n17262\n489908\n762274\n905674\n978150\n1014080\n1031730\n1040361\n1044573\n1046626\n1047626\n1048113\n1048351\n1048466\n1048523\n1048550\n1048563\n-4609\n287882\n-555548\n-828728\n1042674\n974922\n-460425\n-892089\n922721\n624483\n-1048565\n1004644\n-130809\n-1046716\n-645445\n12511\n486973\n760684\n904859\n977743\n1013880\n1031632\n1040313\n1044550\n1046615\n1047621\n1048111\n1048349\n1048466\n1048522\n1048550\n1048563\n879847\n-427294\n-74816\n-567202\n1039594\n898010\n-565907\n-933554\n893268\n590956\n-1048304\n998721\n-116924\n-1046089\n-650798\n7759\n484034\n759089\n904042\n977335\n1013679\n1031534\n1040265\n1044526\n1046603\n1047615\n1048108\n1048348\n1048465\n1048522\n1048550\n1048563\n955376\n-942699\n423309\n-240804\n978327\n796563\n-663851\n-968960\n860990\n556518\n-1047257\n992434\n-103018\n-1045373\n-656123\n3008\n481090\n757491\n903223\n976927\n1013478\n1031435\n1040217\n1044503\n1046592\n1047610\n1048105\n1048347\n1048464\n1048522\n1048550\n1048563\n152537\n-1017370\n823021\n113135\n862302\n673352\n-752952\n-998078\n825990\n521224\n-1045425\n985785\n-89094\n-1044565\n-661421\n-1744\n478142\n755889\n902402\n976517\n1013276\n1031336\n1040169\n1044479\n1046580\n1047604\n1048103\n1048345\n1048464\n1048521\n1048549\n1048563\n-790545\n-616396\n1031389\n454135\n698013\n531744\n-832023\n-1020717\n788379\n485128\n-1042810\n978775\n-75154\n-1043667\n-666690\n-6496\n475188\n754283\n901579\n976106\n1013073\n1031237\n1040120\n1044456\n1046569\n1047598\n1048100\n1048344\n1048463\n1048521\n1048549\n1048563\n-1006803\n72758\n999972\n743194\n494655\n375608\n-900012\n-1036732\n748276\n448284\n-1039412\n971408\n-61201\n-1042679\n-671932\n-11247\n472230\n752674\n900753\n975693\n1012870\n1031137\n1040072\n1044432\n1046557\n1047593\n1048097\n1048343\n1048462\n1048521\n1048549\n1048563\n-297411\n727896\n736073\n947254\n263611\n209209\n-956012\n-1046018\n705807\n410751\n-1035234\n963687\n-47237\n-1041601\n-677145\n-15998\n469267\n751061\n899925\n975280\n1012666\n1031038\n1040023\n1044408\n1046546\n1047587\n1048094\n1048341\n1048462\n1048520\n1048549\n1048563\n685419\n1042725\n301046\n1042975\n17813\n37095\n-999277\n-1048516\n661106\n372584\n-1030281\n955613\n-33264\n-1040432\n-682330\n-20749\n466299\n749445\n899095\n974866\n1012462\n1030937\n1039974\n1044384\n1046534\n1047581\n1048092\n1048340\n1048461\n1048520\n1048549\n1048563\n1038078\n870055\n-203970\n1019409\n-228983\n-136033\n-1029232\n-1044208\n614316\n333845\n-1024555\n947191\n-19286\n-1039173\n-687486\n-25500\n463327\n747825\n898263\n974450\n1012257\n1030837\n1039925\n1044360\n1046522\n1047576\n1048089\n1048339\n1048460\n1048520\n1048549\n1048563\n436333\n290613\n-661566\n879253\n-462962\n-305444\n-1045477\n-1033122\n565583\n294591\n-1018061\n938422\n-5304\n-1037824\n-692613\n-30250\n460350\n746201\n897429\n974033\n1012052\n1030736\n1039876\n1044336\n1046511\n1047570\n1048086\n1048337\n1048460\n1048519\n1048548\n1048563\n-566575\n-424697\n-965356\n638535\n-671029\n-466510\n-1047796\n-1015332\n515063\n254883\n-1010803\n929311\n8679\n-1036386\n-697711\n-34999\n457369\n744573\n896593\n973615\n1011846\n1030636\n1039827\n1044312\n1046499\n1047564\n1048083\n1048336\n1048459\n1048519\n1048548\n1048563\n-1048576\n-941451\n-1044712\n324788\n-841537\n-614830\n-1036158\n-990952\n462914\n214783\n-1002787\n919861\n22660\n-1034857\n-702779\n-39748\n454383\n742942\n895755\n973196\n1011639\n1030534\n1039777\n1044288\n1046487\n1047559\n1048081\n1048335\n1048459\n1048519\n1048548\n1048562\n-566521\n-1018054\n-881185\n-26106\n-964944\n-746352\n-1010717\n-960140\n409301\n174353\n-994019\n910075\n36637\n-1033239\n-707818\n-44496\n451392\n741308\n894914\n972776\n1011432\n1030433\n1039728\n1044264\n1046475\n1047553\n1048078\n1048333\n1048458\n1048518\n1048548\n1048562\n436390\n-618692\n-512793\n-374014\n-1034341\n-857482\n-971814\n-923096\n354395\n133653\n-984507\n899956\n50608\n-1031532\n-712827\n-49243\n448397\n739669\n894072\n972355\n1011224\n1030331\n1039678\n1044240\n1046464\n1047547\n1048075\n1048332\n1048457\n1048518\n1048548\n1048562\n1038087\n69923\n-25183\n-679146\n-1045846\n-945185\n-919965\n-880062\n298368\n92748\n-974255\n889509\n64570\n-1029735\n-717806\n-53989\n445397\n738027\n893227\n971933\n1011016\n1030229\n1039628\n1044216\n1046452\n1047541\n1048072\n1048331\n1048457\n1048518\n1048548\n1048562\n685371\n725848\n468282\n-906603\n-998813\n-1007063\n-855863\n-831315\n241398\n51701\n-963274\n878737\n78520\n-1027849\n-722754\n-58734\n442393\n736382\n892380\n971509\n1010807\n1030127\n1039578\n1044191\n1046440\n1047536\n1048069\n1048329\n1048456\n1048517\n1048548\n1048562\n-297472\n1042421\n852877\n-1030371\n-895876\n-1041426\n-780359\n-777173\n183665\n10573\n-951569\n867644\n92457\n-1025874\n-727672\n-63477\n439384\n734733\n891531\n971085\n1010598\n1030024\n1039528\n1044167\n1046428\n1047530\n1048067\n1048328\n1048455\n1048517\n1048547\n1048562\n-1006820\n871638\n1039188\n-1036294\n-742795\n-1047336\n-694462\n-717987\n125351\n-30570\n-939152\n856234\n106377\n-1023810\n-732559\n-68220\n436372\n733080\n890680\n970659\n1010388\n1029921\n1039478\n1044142\n1046416\n1047524\n1048064\n1048327\n1048455\n1048517\n1048547\n1048562\n-790503\n293342\n983900\n-923695\n-548140\n-1024631\n-599313\n-654141\n66641\n-71667\n-926030\n844512\n120278\n-1021657\n-737416\n-72961\n433354\n731424\n889827\n970232\n1010177\n1029818\n1039427\n1044118\n1046404\n1047518\n1048061\n1048325\n1048454\n1048517\n1048547\n1048562\n152599\n-422097\n699868\n-705451\n-322804\n-973931\n-496182\n-586049\n7720\n-112653\n-912214\n832481\n134157\n-1019416\n-742241\n-77700\n430333\n729764\n888972\n969804\n1009966\n1029714\n1039377\n1044093\n1046392\n1047512\n1048058\n1048324\n1048453\n1048516\n1048547\n1048562\n955402\n-940196\n253124\n-406525\n-79401\n-896621\n-386441\n-514154\n-51226\n-153466\n-897714\n820146\n148013\n-1017087\n-747035\n-82438\n427307\n728101\n888114\n969375\n1009754\n1029611\n1039326\n1044068\n1046380\n1047506\n1048055\n1048322\n1048452\n1048516\n1048547\n1048562\n879813\n-1018731\n-252468\n-61103\n168447\n-794815\n-271552\n-438922\n-110009\n-194042\n-882540\n807512\n161842\n-1014670\n-751798\n-87174\n424276\n726434\n887255\n968945\n1009542\n1029507\n1039275\n1044043\n1046368\n1047501\n1048052\n1048321\n1048452\n1048516\n1048547\n1048562\n-4672\n-620984\n-699365\n291306\n406866\n-671292\n-153047\n-360841\n-168445\n-234320\n-866705\n794583\n175643\n-1012165\n-756529\n-91908\n421242\n724763\n886393\n968514\n1009329\n1029402\n1039224\n1044019\n1046356\n1047495\n1048049\n1048320\n1048451\n1048515\n1048546\n1048562\n-884862\n67088\n-983667\n610399\n622512\n-529429\n-32503\n-280418\n-226348\n-274237\n-850220\n781364\n189412\n-1009572\n-761228\n-96641\n418203\n723089\n885529\n968082\n1009116\n1029298\n1039173\n1043994\n1046344\n1047489\n1048047\n1048318\n1048450\n1048515\n1048546\n1048561\n-951514\n723794\n-1039278\n859680\n803316\n-373101\n88474\n-198175\n-283535\n-313731\n-833098\n767859\n203148\n-1006891\n-765895\n-101371\n415161\n721412\n884664\n967648\n1008902\n1029193\n1039122\n1043969\n1046332\n1047483\n1048044\n1048317\n1048450\n1048515\n1048546\n1048561\n-143348\n1042110\n-853270\n1010638\n939158\n-206578\n208273\n-114646\n-339826\n-352743\n-815350\n754074\n216848\n-1004124\n-770529\n-106100\n412114\n719731\n883796\n967213\n1008687\n1029088\n1039071\n1043944\n1046319\n1047477\n1048041\n1048315\n1048449\n1048514\n1048546\n1048561\n796611\n873214\n-468886\n1046008\n1022433\n-34412\n325297\n-30372\n-395043\n-391211\n-796992\n740013\n230509\n-1001270\n-775131\n-110826\n409062\n718046\n882926\n966778\n1008472\n1028982\n1039019\n1043919\n1046307\n1047471\n1048038\n1048314\n1048448\n1048514\n1048546\n1048561\n1004170\n296069\n24507\n961745\n1048482\n138694\n437989\n54098\n-449011\n-429077\n-778035\n725683\n244129\n-998328\n-779701\n-115550\n406007\n716358\n882054\n966341\n1008257\n1028877\n1038968\n1043893\n1046295\n1047465\n1048035\n1048312\n1048448\n1048514\n1048546\n1048561\n288499\n-419495\n512204\n767486\n1015847\n308011\n544846\n138217\n-501559\n-466283\n-758496\n711087\n257705\n-995301\n-784237\n-120271\n402948\n714667\n881179\n965903\n1008040\n1028771\n1038916\n1043868\n1046283\n1047459\n1048032\n1048311\n1048447\n1048513\n1048545\n1048561\n-692416\n-938935\n880819\n485449\n926353\n468913\n644445\n221440\n-552522\n-502770\n-738387\n696232\n271236\n-992187\n-788741\n-124991\n399885\n712972\n880303\n965464\n1007824\n1028664\n1038864\n1043843\n1046270\n1047453\n1048029\n1048310\n1048446\n1048513\n1048545\n1048561\n-1036727\n-1019400\n1044654\n147890\n785010\n617003\n735460\n303225\n-601737\n-538483\n-717725\n681123\n284719\n-988987\n-793211\n-129707\n396817\n711273\n879425\n965024\n1007606\n1028558\n1038812\n1043817\n1046258\n1047447\n1048026\n1048308\n1048446\n1048512\n1048545\n1048561\n-427876\n-623272\n965619\n-206583\n599729\n748235\n816679\n383042\n-649051\n-573368\n-696525\n665764\n298151\n-985702\n-797648\n-134421\n393746\n709571\n878544\n964582\n1007388\n1028451\n1038760\n1043792\n1046245\n1047441\n1048023\n1048307\n1048445\n1048512\n1048545\n1048561\n574362\n64251\n662090\n-537429\n380881\n859024\n887019\n460373\n-694313\n-607369\n-674802\n650163\n311529\n-982331\n-802051\n-139132\n390671\n707866\n877662\n964140\n1007170\n1028344\n1038707\n1043766\n1046233\n1047435\n1048020\n1048305\n1048444\n1048512\n1048545\n1048561\n1048535\n721736\n204633\n-806808\n140715\n946344\n945544\n534716\n-737379\n-640435\n-652574\n634324\n324853\n-978876\n-806420\n-143840\n387592\n706157\n876777\n963696\n1006951\n1028236\n1038655\n1043741\n1046220\n1047429\n1048017\n1048304\n1048443\n1048511\n1048545\n1048561\n558689\n1041791\n-300400\n-983912\n-107328\n1007807\n991473\n605588\n-778114\n-672515\n-629856\n618254\n338119\n-975335\n-810755\n-148546\n384509\n704445\n875890\n963252\n1006731\n1028129\n1038602\n1043715\n1046208\n1047423\n1048014\n1048302\n1048443\n1048511\n1048544\n1048561\n-444813\n874784\n-735592\n-1048485\n-349363\n1041736\n1024196\n672530\n-816390\n-703560\n-606666\n601958\n351324\n-971710\n-815056\n-153248\n381422\n702729\n875002\n962806\n1006511\n1028021\n1038549\n1043689\n1046195\n1047417\n1048011\n1048301\n1048442\n1048511\n1048544\n1048561\n-1039356\n298794\n-999769\n-993141\n-571844\n1047202\n1043276\n735107\n-852084\n-733521\n-583021\n585442\n364468\n-968001\n-819323\n-157947\n378331\n701010\n874111\n962359\n1006290\n1027912\n1038496\n1043664\n1046183\n1047410\n1048008\n1048300\n1048441\n1048510\n1048544\n1048560\n-678320\n-416889\n-1031511\n-824210\n-762318\n1024057\n1048460\n792913\n-885085\n-762353\n-558938\n568712\n377546\n-964208\n-823555\n-162643\n375237\n699287\n873218\n961911\n1006069\n1027804\n1038443\n1043638\n1046170\n1047404\n1048005\n1048298\n1048441\n1048510\n1048544\n1048560\n306360\n-937666\n-823439\n-561014\n-910125\n972933\n1039677\n845573\n-915287\n-790011\n-534437\n551775\n390557\n-960332\n-827753\n-167335\n372139\n697562\n872323\n961462\n1005847\n1027695\n1038390\n1043612\n1046158\n1047398\n1048002\n1048297\n1048440\n1048510\n1048544\n1048560\n1009374\n-1020062\n-423927\n-233654\n-1006990\n895227\n1017045\n892745\n-942596\n-816453\n-509535\n534636\n403499\n-956372\n-831915\n-172024\n369037\n695832\n871426\n961011\n1005624\n1027586\n1038337\n1043586\n1046145\n1047392\n1047999\n1048295\n1048439\n1048509\n1048544\n1048560\n784374\n-625555\n74142\n120430\n-1047494\n793061\n980866\n934123\n-966925\n-841637\n-484250\n517302\n416369\n-952330\n-836043\n-176710\n365931\n694099\n870527\n960560\n1005401\n1027477\n1038284\n1043560\n1046132\n1047386\n1047996\n1048294\n1048439\n1048509\n1048543\n1048560\n-161776\n61415\n554975\n460740\n-1029368\n669228\n931622\n969437\n-988197\n-865526\n-458603\n499779\n429165\n-948205\n-840135\n-181392\n362822\n692363\n869626\n960108\n1005178\n1027367\n1038230\n1043534\n1046119\n1047380\n1047993\n1048292\n1048438\n1048509\n1048543\n1048560\n-959190\n719671\n906782\n748354\n-953627\n527110\n869968\n998460\n-1006345\n-888082\n-432612\n482073\n441884\n-943998\n-844192\n-186070\n359709\n690624\n868722\n959654\n1004954\n1027257\n1038176\n1043507\n1046107\n1047373\n1047990\n1048291\n1048437\n1048508\n1048543\n1048560\n-874729\n1041464\n1047773\n950378\n-824511\n370591\n796725\n1021003\n-1021311\n-909271\n-406296\n464191\n454525\n-939709\n-848213\n-190744\n356593\n688881\n867817\n959199\n1004729\n1027147\n1038122\n1043481\n1046094\n1047367\n1047987\n1048289\n1048436\n1048508\n1048543\n1048560\n13953\n876348\n945170\n1043707\n-649245\n203946\n712870\n1036919\n-1033049\n-929059\n-379676\n446140\n467086\n-935339\n-852198\n-195415\n353473\n687135\n866910\n958744\n1004504\n1027036\n1038068\n1043455\n1046081\n1047361\n1047984\n1048288\n1048436\n1048508\n1048543\n1048560\n889807\n301517\n622826\n1017665\n-437641\n31729\n619519\n1046105\n-1041520\n-947417\n-352771\n427926\n479563\n-930888\n-856148\n-200081\n350349\n685385\n866000\n958287\n1004278\n1026925\n1038014\n1043428\n1046068\n1047355\n1047981\n1048286\n1048435\n1048507\n1048543\n1048560\n947576\n-414280\n155682\n875232\n-201542\n-141354\n517916\n1048502\n-1046699\n-964317\n-325602\n409556\n491955\n-926356\n-860061\n-204744\n347222\n683632\n865089\n957829\n1004051\n1026814\n1037960\n1043402\n1046055\n1047348\n1047978\n1048285\n1048434\n1048507\n1048542\n1048560\n134148\n-936391\n-347656\n632698\n45838\n-310576\n409414\n1044093\n-1048569\n-979731\n-298188\n391036\n504259\n-921744\n-863938\n-209402\n344092\n681876\n864175\n957370\n1003824\n1026703\n1037905\n1043375\n1046042\n1047342\n1047975\n1048283\n1048433\n1048507\n1048542\n1048560\n-802615\n-1020717\n-770168\n317801\n290652\n-471312\n295459\n1032908\n-1047124\n-993637\n-270551\n372374\n516474\n-917052\n-867779\n-214056\n340958\n680117\n863260\n956910\n1003597\n1026591\n1037851\n1043349\n1046029\n1047336\n1047972\n1048282\n1048433\n1048506\n1048542\n1048559\n-1001458\n-627833\n-1013625\n-33443\n519199\n-619171\n177567\n1015019\n-1042368\n-1006013\n-242710\n353575\n528597\n-912281\n-871583\n-218705\n337820\n678354\n862342\n956448\n1003369\n1026479\n1037796\n1043322\n1046016\n1047329\n1047969\n1048280\n1048432\n1048506\n1048542\n1048559\n-279565\n58578\n-1021427\n-380863\n718685\n-750113\n57311\n990542\n-1034317\n-1016840\n-214688\n334648\n540626\n-907431\n-875350\n-223350\n334679\n676588\n861422\n955986\n1003140\n1026367\n1037741\n1043295\n1046003\n1047323\n1047966\n1048279\n1048431\n1048506\n1048542\n1048559\n699359\n717602\n-791759\n-684722\n877945\n-860561\n-63709\n959637\n-1022997\n-1026102\n-186505\n315598\n552559\n-902502\n-879080\n-227990\n331535\n674818\n860501\n955522\n1002911\n1026255\n1037686\n1043268\n1045990\n1047317\n1047963\n1048277\n1048431\n1048505\n1048542\n1048559\n1035295\n1041130\n-378016\n-910269\n988066\n-947497\n-183880\n922503\n-1008441\n-1033783\n-158182\n296433\n564393\n-897495\n-882774\n-232626\n328388\n673046\n859577\n955058\n1002681\n1026142\n1037631\n1043242\n1045977\n1047310\n1047960\n1048276\n1048430\n1048505\n1048541\n1048559\n419386\n877905\n123611\n-1031707\n1042884\n-1008545\n-301602\n879381\n-990698\n-1039873\n-129741\n277159\n576127\n-892411\n-886430\n-237257\n325237\n671270\n858651\n954592\n1002451\n1026029\n1037576\n1043215\n1045964\n1047304\n1047956\n1048274\n1048429\n1048504\n1048541\n1048559\n-582105\n304238\n596500\n-1035148\n1039330\n-1042038\n-415306\n830553\n-969823\n-1044361\n-101202\n257785\n587759\n-887249\n-890049\n-241883\n322083\n669490\n857723\n954125\n1002220\n1025916\n1037521\n1043188\n1045951\n1047297\n1047953\n1048273\n1048428\n1048504\n1048541\n1048559\n-1048411\n-411668\n930709\n-920198\n977604\n-1047061\n-523479\n776333\n-945882\n-1047242\n-72587\n238316\n599286\n-882010\n-893630\n-246504\n318926\n667708\n856793\n953657\n1001988\n1025802\n1037465\n1043160\n1045938\n1047291\n1047950\n1048271\n1048428\n1048504\n1048541\n1048559\n-550813\n-935109\n1048540\n-700003\n861161\n-1023477\n-624678\n717076\n-918950\n-1048510\n-43918\n218760\n610707\n-876695\n-897173\n-251120\n315766\n665922\n855861\n953188\n1001756\n1025688\n1037409\n1043133\n1045924\n1047284\n1047947\n1048270\n1048427\n1048503\n1048541\n1048559\n453200\n-1021364\n922598\n-399749\n696517\n-971929\n-717556\n653164\n-889113\n-1048164\n-15216\n199124\n622019\n-871304\n-900679\n-255731\n312602\n664133\n854927\n952718\n1001523\n1025574\n1037354\n1043106\n1045911\n1047278\n1047944\n1048268\n1048426\n1048503\n1048540\n1048559\n1040543\n-630107\n582163\n-53774\n492888\n-893826\n-800877\n585012\n-856466\n-1046204\n13498\n179416\n633220\n-865837\n-904147\n-260336\n309435\n662341\n853992\n952247\n1001290\n1025460\n1037298\n1043079\n1045898\n1047272\n1047941\n1048267\n1048425\n1048503\n1048540\n1048559\n671216\n55740\n106382\n298350\n261672\n-791303\n-873529\n513064\n-821110\n-1042633\n42201\n159642\n644309\n-860296\n-907576\n-264937\n306266\n660545\n853054\n951775\n1001056\n1025345\n1037242\n1043051\n1045885\n1047265\n1047938\n1048265\n1048425\n1048502\n1048540\n1048559\n-315224\n715527\n-394132\n616352\n15810\n-667159\n-934546\n437786\n-783159\n-1037456\n70873\n139810\n655283\n-854680\n-910967\n-269531\n303093\n658747\n852114\n951301\n1000822\n1025230\n1037185\n1043024\n1045871\n1047259\n1047934\n1048264\n1048424\n1048502\n1048540\n1048558\n-1011849\n1040788\n-803015\n863861\n-230937\n-524788\n-983114\n359667\n-742732\n-1030682\n99491\n119927\n666141\n-848990\n-914320\n-274121\n299917\n656945\n851172\n950827\n1000587\n1025115\n1037129\n1042997\n1045858\n1047252\n1047931\n1048262\n1048423\n1048502\n1048540\n1048558\n-778184\n879456\n-1025206\n1012570\n-464758\n-368079\n-1018586\n279213\n-699956\n-1022322\n128035\n100000\n676880\n-843226\n-917635\n-278704\n296738\n655140\n850228\n950351\n1000352\n1024999\n1037073\n1042969\n1045844\n1047245\n1047928\n1048260\n1048422\n1048501\n1048540\n1048558\n170939\n306956\n-1009049\n1045469\n-672566\n-201313\n-1040491\n196947\n-654968\n-1012387\n156483\n80036\n687499\n-837390\n-920910\n-283282\n293556\n653331\n849281\n949874\n1000116\n1024883\n1037016\n1042941\n1045831\n1047239\n1047925\n1048259\n1048422\n1048501\n1048539\n1048558\n962902\n-409053\n-758300\n958797\n-842730\n-29046\n-1048536\n113403\n-607909\n-1000893\n184814\n60044\n697996\n-831481\n-924147\n-287854\n290371\n651520\n848333\n949396\n999879\n1024767\n1036959\n1042914\n1045817\n1047232\n1047922\n1048257\n1048421\n1048500\n1048539\n1048558\n869577\n-933819\n-331256\n762466\n-965726\n144013\n-1042613\n29123\n-558929\n-987858\n213006\n40029\n708368\n-825500\n-927345\n-292421\n287184\n649705\n847383\n948918\n999642\n1024651\n1036902\n1042886\n1045804\n1047226\n1047918\n1048256\n1048420\n1048500\n1048539\n1048558\n-23233\n-1022003\n172802\n478931\n-1034668\n313138\n-1022803\n-55346\n-508181\n-973302\n241038\n20000\n718615\n-819447\n-930503\n-296981\n283993\n647887\n846431\n948438\n999404\n1024534\n1036845\n1042858\n1045790\n1047219\n1047915\n1048254\n1048419\n1048500\n1048539\n1048558\n-894683\n-632376\n636685\n140620\n-1045700\n473708\n-989369\n-139456\n-455827\n-957248\n268890\n-37\n728733\n-813324\n-933623\n-301535\n280800\n646066\n845477\n947956\n999166\n1024417\n1036788\n1042830\n1045777\n1047212\n1047912\n1048253\n1048419\n1048499\n1048539\n1048558\n-943565\n52902\n952546\n-213774\n-998202\n621335\n-942756\n-222661\n-402031\n-939719\n296540\n-20073\n738722\n-807130\n-936703\n-306083\n277604\n644242\n844521\n947474\n998927\n1024300\n1036731\n1042802\n1045763\n1047206\n1047909\n1048251\n1048418\n1048499\n1048538\n1048558\n-124938\n713448\n1046952\n-543718\n-894833\n751986\n-883585\n-304421\n-346965\n-920744\n323968\n-40103\n748580\n-800866\n-939744\n-310624\n274405\n642415\n843563\n946991\n998688\n1024182\n1036673\n1042774\n1045749\n1047199\n1047905\n1048249\n1048417\n1048499\n1048538\n1048558\n808556\n1040439\n897954\n-811477\n-741380\n862092\n-812644\n-384205\n-290802\n-900351\n351152\n-60117\n758305\n-794533\n-942745\n-315160\n271203\n640585\n842603\n946507\n998448\n1024064\n1036616\n1042746\n1045736\n1047193\n1047902\n1048248\n1048416\n1048498\n1048538\n1048558\n998668\n881000\n540192\n-986426\n-546431\n948643\n-730879\n-461495\n-233719\n-878572\n378074\n-80110\n767895\n-788131\n-945706\n-319688\n267999\n638751\n841641\n946021\n998208\n1023946\n1036558\n1042718\n1045722\n1047186\n1047899\n1048246\n1048415\n1048498\n1048538\n1048557\n270609\n309672\n56842\n-1048556\n-320898\n1009276\n-639378\n-535790\n-175897\n-855440\n404711\n-100073\n777348\n-781661\n-948627\n-324210\n264792\n636915\n840677\n945535\n997967\n1023828\n1036500\n1042690\n1045708\n1047179\n1047896\n1048245\n1048415\n1048497\n1048538\n1048557\n-706247\n-406435\n-439723\n-990762\n-77404\n1042334\n-539360\n-606608\n-117520\n-830991\n431046\n-120000\n786663\n-775123\n-951508\n-328726\n261582\n635075\n839710\n945047\n997725\n1023709\n1036442\n1042661\n1045694\n1047172\n1047892\n1048243\n1048414\n1048497\n1048538\n1048557\n-1033782\n-932523\n-834058\n-819653\n170423\n1046914\n-432158\n-673489\n-58771\n-805262\n457057\n-139883\n795838\n-768518\n-954350\n-333235\n258369\n633232\n838742\n944558\n997483\n1023590\n1036384\n1042633\n1045681\n1047166\n1047889\n1048241\n1048413\n1048497\n1048537\n1048557\n-410863\n-1022635\n-1034484\n-554799\n408711\n1022890\n-319199\n-735998\n164\n-778293\n482726\n-159715\n804872\n-761846\n-957151\n-337737\n255154\n631386\n837772\n944068\n997240\n1023471\n1036326\n1042604\n1045667\n1047159\n1047886\n1048240\n1048412\n1048496\n1048537\n1048557\n589802\n-634640\n-994405\n-226492\n624123\n970918\n-201988\n-793731\n59099\n-750126\n508032\n-179489\n813762\n-755109\n-959911\n-342232\n251937\n629537\n836800\n943577\n996997\n1023352\n1036267\n1042576\n1045653\n1047152\n1047883\n1048238\n1048412\n1048496\n1048537\n1048557\n1048205\n50064\n-723139\n127718\n804602\n892420\n-82087\n-846312\n117846\n-720805\n532958\n-199197\n822508\n-748306\n-962631\n-346720\n248717\n627685\n835826\n943085\n996753\n1023232\n1036209\n1042547\n1045639\n1047145\n1047879\n1048237\n1048411\n1048496\n1048537\n1048557\n542894\n711362\n-283752\n467322\n940047\n789539\n38908\n-893400\n176221\n-690373\n557483\n-218832\n831108\n-741438\n-965311\n-351201\n245494\n625830\n834850\n942592\n996508\n1023112\n1036150\n1042519\n1045625\n1047139\n1047876\n1048235\n1048410\n1048495\n1048537\n1048557\n-461552\n1040082\n221605\n753477\n1022876\n665086\n159384\n-934690\n234039\n-658878\n581591\n-238388\n839560\n-734506\n-967950\n-355674\n242270\n623972\n833872\n942098\n996263\n1022992\n1036091\n1042490\n1045611\n1047132\n1047873\n1048233\n1048409\n1048495\n1048536\n1048557\n-1041649\n882538\n675441\n953456\n1048454\n522462\n277737\n-969913\n291117\n-626369\n605263\n-257856\n847862\n-727511\n-970548\n-360141\n239042\n622111\n832892\n941603\n996018\n1022871\n1036032\n1042461\n1045597\n1047125\n1047869\n1048232\n1048408\n1048494\n1048536\n1048557\n-664059\n312386\n972245\n1044388\n1015348\n365564\n392391\n-998841\n347275\n-592895\n628481\n-277230\n856014\n-720452\n-973105\n-364600\n235812\n620247\n831910\n941106\n995772\n1022750\n1035973\n1042432\n1045583\n1047118\n1047866\n1048230\n1048408\n1048494\n1048536\n1048557\n324064\n-403814\n1043013\n1015871\n925413\n198678\n501818\n-1021287\n402335\n-558509\n651228\n-296503\n864013\n-713331\n-975621\n-369051\n232580\n618379\n830926\n940609\n995525\n1022629\n1035914\n1042403\n1045569\n1047111\n1047863\n1048229\n1048407\n1048494\n1048536\n1048556\n1014244\n-931221\n871294\n871168\n783681\n26363\n604561\n-1037104\n456122\n-523262\n673486\n-315668\n871859\n-706148\n-978096\n-373495\n229346\n616509\n829940\n940110\n995278\n1022507\n1035854\n1042374\n1045555\n1047104\n1047859\n1048227\n1048406\n1048493\n1048536\n1048556\n771933\n-1023260\n497009\n626829\n598085\n-146672\n699250\n-1046190\n508468\n-487210\n695239\n-334717\n879550\n-698904\n-980529\n-377931\n226109\n614636\n828953\n939611\n995030\n1022386\n1035795\n1042345\n1045540\n1047097\n1047856\n1048225\n1048405\n1048493\n1048536\n1048556\n-180090\n-636900\n7176\n310798\n379014\n-315699\n784625\n-1048486\n559207\n-450407\n716471\n-353644\n887084\n-691600\n-982922\n-382359\n222870\n612760\n827963\n939110\n994782\n1022264\n1035735\n1042316\n1045526\n1047090\n1047853\n1048224\n1048404\n1048492\n1048535\n1048556\n-966539\n47226\n-484326\n-40779\n138730\n-476101\n859549\n-1043977\n608177\n-412912\n737166\n-372443\n894461\n-684236\n-985272\n-386780\n219629\n610880\n826971\n938608\n994533\n1022141\n1035675\n1042287\n1045512\n1047084\n1047849\n1048222\n1048404\n1048492\n1048535\n1048556\n-864356\n709272\n-863228\n-387692\n-109320\n-623495\n923023\n-1032692\n655225\n-374780\n757308\n-391105\n901678\n-676812\n-987582\n-391193\n216385\n608998\n825977\n938105\n994283\n1022019\n1035615\n1042258\n1045498\n1047077\n1047846\n1048220\n1048403\n1048492\n1048535\n1048556\n32511\n1039717\n-1041439\n-690264\n-351251\n-753854\n974202\n-1014705\n700201\n-336071\n776882\n-409624\n908736\n-669330\n-989849\n-395597\n213140\n607113\n824981\n937601\n994033\n1021896\n1035555\n1042229\n1045483\n1047070\n1047842\n1048219\n1048402\n1048491\n1048535\n1048556\n899488\n884069\n-977529\n-913890\n-573522\n-863617\n1012405\n-990132\n742963\n-296845\n795874\n-427994\n915631\n-661790\n-992075\n-399994\n209892\n605225\n823984\n937096\n993783\n1021773\n1035495\n1042199\n1045469\n1047063\n1047839\n1048217\n1048401\n1048491\n1048535\n1048556\n939480\n315097\n-686354\n-1032993\n-763692\n-949784\n1037122\n-959132\n783377\n-257162\n814269\n-446207\n922364\n-654193\n-994259\n-404382\n206642\n603334\n822984\n936590\n993532\n1021649\n1035435\n1042170\n1045455\n1047056\n1047836\n1048215\n1048400\n1048490\n1048534\n1048556\n115718\n-401190\n-235610\n-1033952\n-911118\n-1010001\n1048023\n-921908\n821314\n-217083\n832053\n-464257\n928933\n-646539\n-996402\n-408762\n203390\n601440\n821983\n936083\n993280\n1021526\n1035374\n1042140\n1045440\n1047049\n1047832\n1048214\n1048400\n1048490\n1048534\n1048556\n-814434\n-929911\n269910\n-916656\n-1007547\n-1042623\n1044965\n-878700\n856655\n-176669\n849213\n-482138\n935336\n-638829\n-998502\n-413134\n200136\n599543\n820979\n935575\n993028\n1021402\n1035314\n1042111\n1045426\n1047042\n1047829\n1048212\n1048399\n1048490\n1048534\n1048556\n-995799\n-1023876\n712679\n-694521\n-1047583\n-1046759\n1027988\n-829789\n889287\n-135984\n865737\n-499843\n941574\n-631063\n-1000560\n-417497\n196881\n597643\n819973\n935065\n992775\n1021277\n1035253\n1042081\n1045411\n1047035\n1047825\n1048210\n1048398\n1048489\n1048534\n1048555\n-261631\n-639155\n989759\n-392953\n-1028984\n-1022296\n997317\n-775493\n919108\n-95089\n881611\n-517366\n947644\n-623243\n-1002576\n-421851\n193623\n595740\n818966\n934555\n992522\n1021153\n1035192\n1042051\n1045397\n1047027\n1047822\n1048209\n1048397\n1048489\n1048534\n1048555\n713079\n44387\n1036732\n-46442\n-952792\n-969901\n953361\n-716163\n946023\n-54047\n896824\n-534699\n953545\n-615369\n-1004549\n-426197\n190363\n593835\n817957\n934043\n992268\n1021028\n1035131\n1042021\n1045382\n1047020\n1047818\n1048207\n1048396\n1048489\n1048533\n1048555\n1032188\n707177\n842677\n305380\n-823272\n-891008\n896707\n-652185\n969948\n-12923\n911365\n-551837\n959277\n-607442\n-1006480\n-430534\n187101\n591926\n816945\n933531\n992013\n1020903\n1035070\n1041992\n1045368\n1047013\n1047815\n1048205\n1048396\n1048488\n1048533\n1048555\n402308\n1039344\n452710\n622276\n-647671\n-787770\n828108\n-583975\n990806\n28221\n925223\n-568774\n964838\n-599462\n-1008369\n-434863\n183837\n590015\n815932\n933017\n991758\n1020778\n1035008\n1041962\n1045353\n1047006\n1047811\n1048204\n1048395\n1048488\n1048533\n1048555\n-597453\n885594\n-42507\n868001\n-435820\n-663009\n748478\n-511974\n1008531\n69322\n938386\n-585503\n970227\n-591430\n-1010215\n-439182\n180572\n588101\n814917\n932502\n991502\n1020652\n1034947\n1041932\n1045339\n1046999\n1047808\n1048202\n1048394\n1048487\n1048533\n1048555\n-1047918\n317807\n-527842\n1014452\n-199576\n-520134\n658878\n-436650\n1023069\n110316\n950846\n-602018\n975445\n-583347\n-1012019\n-443492\n177305\n586183\n813900\n931986\n991246\n1020526\n1034885\n1041902\n1045324\n1046992\n1047805\n1048200\n1048393\n1048487\n1048533\n1048555\n-534932\n-398563\n-890460\n1044879\n47839\n-363047\n560502\n-358493\n1034371\n151141\n962593\n-618313\n980488\n-575214\n-1013780\n-447793\n174036\n584263\n812881\n931469\n990990\n1020400\n1034823\n1041871\n1045309\n1046985\n1047801\n1048199\n1048392\n1048486\n1048532\n1048555\n469867\n-928594\n-1046056\n955801\n292576\n-196042\n454659\n-278008\n1042404\n191732\n973619\n-634383\n985357\n-567031\n-1015498\n-452085\n170765\n582341\n811860\n930951\n990732\n1020273\n1034762\n1041841\n1045295\n1046978\n1047798\n1048197\n1048391\n1048486\n1048532\n1048555\n1042673\n-1024486\n-958457\n757408\n520938\n-23680\n342760\n-195720\n1047141\n232029\n983914\n-650221\n990051\n-558798\n-1017173\n-456368\n167493\n580415\n810837\n930432\n990474\n1020147\n1034700\n1041811\n1045280\n1046970\n1047794\n1048195\n1048391\n1048486\n1048532\n1048555\n656850\n-641406\n-648028\n472389\n720142\n149329\n226296\n-112161\n1048568\n271968\n993471\n-665821\n994569\n-550517\n-1018806\n-460642\n164218\n578486\n809812\n929912\n990216\n1020020\n1034637\n1041781\n1045265\n1046963\n1047791\n1048193\n1048390\n1048485\n1048532\n1048554\n-332878\n41547\n-186940\n133342\n879039\n318258\n106817\n-27874\n1046680\n311488\n1002283\n-681178\n998911\n-542189\n-1020395\n-464905\n160943\n576555\n808785\n929391\n989957\n1019892\n1034575\n1041750\n1045250\n1046956\n1047787\n1048192\n1048389\n1048485\n1048532\n1048554\n-1016560\n705076\n317609\n-220955\n988735\n478491\n-14084\n56594\n1041482\n350529\n1010344\n-696287\n1003074\n-533814\n-1021942\n-469160\n157665\n574621\n807757\n928869\n989698\n1019765\n1034513\n1041720\n1045235\n1046949\n1047783\n1048190\n1048388\n1048484\n1048531\n1048554\n-765621\n1038964\n748318\n-549981\n1043090\n625651\n-134798\n140695\n1032992\n389030\n1017648\n-711141\n1007059\n-525392\n-1023445\n-473405\n154386\n572684\n806726\n928345\n989438\n1019637\n1034450\n1041689\n1045220\n1046942\n1047780\n1048188\n1048387\n1048484\n1048531\n1048554\n189226\n887112\n1005051\n-816105\n1039063\n755718\n-253717\n223882\n1021237\n426932\n1024188\n-725736\n1010865\n-516925\n-1024906\n-477640\n151106\n570744\n805694\n927821\n989177\n1019509\n1034387\n1041659\n1045206\n1046934\n1047776\n1048187\n1048386\n1048484\n1048531\n1048554\n970100\n320514\n1028123\n-988891\n976878\n865136\n-369256\n305617\n1006253\n464177\n1029960\n-740066\n1014491\n-508413\n-1026323\n-481865\n147824\n568802\n804659\n927295\n988916\n1019380\n1034325\n1041628\n1045191\n1046927\n1047773\n1048185\n1048386\n1048483\n1048531\n1048554\n859068\n-395933\n812168\n-1048576\n860016\n950918\n-479876\n385367\n988087\n500707\n1034960\n-754125\n1017937\n-499857\n-1027696\n-486080\n144540\n566856\n803623\n926769\n988654\n1019251\n1034262\n1041597\n1045176\n1046920\n1047769\n1048183\n1048385\n1048483\n1048531\n1048554\n-41787\n-927271\n407394\n-988334\n695018\n1010719\n-584104\n462617\n966798\n536466\n1039184\n-767909\n1021202\n-491258\n-1029027\n-490285\n141255\n564908\n802585\n926241\n988392\n1019122\n1034198\n1041566\n1045161\n1046912\n1047766\n1048181\n1048384\n1048482\n1048530\n1048554\n-904223\n-1025088\n-92095\n-815055\n491120\n1042905\n-680552\n536864\n942452\n571399\n1042628\n-781413\n1024286\n-482616\n-1030314\n-494481\n137969\n562958\n801545\n925712\n988129\n1018993\n1034135\n1041535\n1045146\n1046905\n1047762\n1048180\n1048383\n1048482\n1048530\n1048554\n-935321\n-643651\n-570172\n-548557\n259732\n1046598\n-767934\n607627\n915127\n605452\n1045291\n-794631\n1027187\n-473932\n-1031557\n-498666\n134681\n561004\n800503\n925183\n987866\n1018864\n1034072\n1041505\n1045130\n1046898\n1047759\n1048178\n1048382\n1048482\n1048530\n1048554\n-106489\n38708\n-915692\n-219320\n13807\n1021695\n-845087\n674446\n884909\n638573\n1047170\n-807559\n1029905\n-465208\n-1032757\n-502841\n131392\n559048\n799459\n924652\n987602\n1018734\n1034008\n1041473\n1045115\n1046890\n1047755\n1048176\n1048381\n1048481\n1048530\n1048554\n820248\n702970\n-1048323\n135001\n-232890\n968878\n-910983\n736888\n851893\n670711\n1048264\n-820192\n1032440\n-456443\n-1033914\n-507005\n128102\n557089\n798413\n924120\n987337\n1018604\n1033945\n1041442\n1045100\n1046883\n1047751\n1048174\n1048380\n1048481\n1048530\n1048553\n992853\n1038577\n-937232\n473881\n-466553\n889590\n-964745\n794547\n816184\n701816\n1048571\n-832526\n1034792\n-447638\n-1035027\n-511159\n124810\n555127\n797365\n923587\n987072\n1018473\n1033881\n1041411\n1045085\n1046875\n1047748\n1048173\n1048380\n1048480\n1048529\n1048553\n252633\n888624\n-608246\n758563\n-674102\n785996\n-1005656\n847049\n777894\n731840\n1048093\n-844555\n1036960\n-438795\n-1036096\n-515303\n121517\n553163\n796316\n923053\n986806\n1018343\n1033817\n1041380\n1045070\n1046868\n1047744\n1048171\n1048379\n1048480\n1048529\n1048553\n-719856\n323218\n-137850\n956488\n-843921\n660927\n-1033171\n894053\n737146\n760738\n1046828\n-856277\n1038943\n-429914\n-1037122\n-519436\n118223\n551195\n795264\n922518\n986540\n1018212\n1033753\n1041349\n1045054\n1046861\n1047741\n1048169\n1048378\n1048480\n1048529\n1048553\n-1030513\n-393300\n364595\n1045018\n-966504\n517801\n-1046924\n935255\n694066\n788464\n1044779\n-867685\n1040742\n-420996\n-1038104\n-523558\n114928\n549226\n794211\n921981\n986273\n1018080\n1033688\n1041317\n1045039\n1046853\n1047737\n1048167\n1048377\n1048479\n1048529\n1048553\n-393721\n-925941\n782276\n1014027\n-1034992\n360528\n-1046731\n970387\n648793\n814976\n1041946\n-878777\n1042355\n-412041\n-1039042\n-527670\n111631\n547253\n793156\n921444\n986006\n1017949\n1033624\n1041286\n1045024\n1046846\n1047733\n1048166\n1048376\n1048479\n1048529\n1048553\n605056\n-1025682\n1018087\n867062\n-1045549\n193404\n-1032595\n999221\n601468\n840234\n1038332\n-889548\n1043783\n-403050\n-1039936\n-531771\n108333\n545278\n792098\n920906\n985738\n1017817\n1033560\n1041255\n1045009\n1046838\n1047730\n1048164\n1048375\n1048478\n1048528\n1048553\n1047548\n-645892\n1017205\n620929\n-997587\n20996\n-1004705\n1021569\n552242\n864198\n1033939\n-899994\n1045026\n-394025\n-1040786\n-535860\n105035\n543300\n791039\n920366\n985470\n1017685\n1033495\n1041223\n1044993\n1046831\n1047726\n1048162\n1048374\n1048478\n1048528\n1048553\n526929\n35868\n779835\n303780\n-893788\n-151985\n-963432\n1037288\n501270\n886831\n1028771\n-910111\n1046083\n-384965\n-1041593\n-539939\n101735\n541320\n789979\n919826\n985201\n1017553\n1033430\n1041191\n1044978\n1046823\n1047722\n1048160\n1048374\n1048477\n1048528\n1048553\n-478146\n700859\n361163\n-48113\n-739963\n-320814\n-909326\n1046274\n448714\n908098\n1022832\n-919896\n1046953\n-375872\n-1042355\n-544007\n98434\n539337\n788916\n919285\n984932\n1017420\n1033365\n1041160\n1044962\n1046816\n1047719\n1048158\n1048373\n1048477\n1048528\n1048553\n-1043616\n1038181\n-141476\n-394503\n-544721\n-480878\n-843107\n1048469\n394738\n927968\n1016125\n-929345\n1047638\n-366747\n-1043074\n-548064\n95133\n537351\n787851\n918742\n984661\n1017287\n1033300\n1041128\n1044947\n1046808\n1047715\n1048157\n1048372\n1048477\n1048528\n1048552\n-649590\n890129\n-611222\n-695773\n-318991\n-627803\n-765657\n1043859\n339515\n946408\n1008657\n-938455\n1048136\n-357590\n-1043749\n-552109\n91830\n535363\n786785\n918198\n984391\n1017154\n1033235\n1041096\n1044931\n1046801\n1047711\n1048155\n1048371\n1048476\n1048527\n1048552\n341666\n325920\n-938867\n-917467\n-75406\n-757576\n-678009\n1032475\n283219\n963391\n1000432\n-947222\n1048448\n-348402\n-1044379\n-556143\n88526\n533372\n785716\n917654\n984120\n1017020\n1033170\n1041064\n1044916\n1046793\n1047708\n1048153\n1048370\n1048476\n1048527\n1048552\n1018796\n-390664\n-1048237\n-1034229\n172399\n-866650\n-581329\n1014389\n226027\n978891\n991457\n-955643\n1048573\n-339183\n-1044966\n-560166\n85222\n531378\n784646\n917108\n983848\n1016887\n1033104\n1041032\n1044900\n1046785\n1047704\n1048151\n1048369\n1048475\n1048527\n1048552\n759249\n-924604\n-913904\n-1032705\n410555\n-952046\n-476906\n989719\n168120\n992884\n981739\n-963716\n1048512\n-329935\n-1045508\n-564177\n81916\n529382\n783574\n916561\n983576\n1016753\n1033039\n1041000\n1044885\n1046778\n1047700\n1048149\n1048368\n1048475\n1048527\n1048552\n-198348\n-1026269\n-567099\n-913069\n625731\n-1011431\n-366130\n958627\n109682\n1005348\n971284\n-971436\n1048265\n-320659\n-1046007\n-568176\n78610\n527384\n782500\n916013\n983303\n1016618\n1032973\n1040968\n1044869\n1046770\n1047697\n1048148\n1048367\n1048474\n1048527\n1048552\n-973584\n-648128\n-88450\n-689005\n805885\n-1043181\n-250477\n921312\n50898\n1016264\n960102\n-978802\n1047831\n-311355\n-1046461\n-572164\n75303\n525382\n781424\n915464\n983030\n1016484\n1032907\n1040936\n1044853\n1046763\n1047693\n1048146\n1048367\n1048474\n1048526\n1048552\n-853712\n33028\n410762\n-386138\n940932\n-1046430\n-131487\n878018\n-8048\n1025615\n948199\n-985810\n1047210\n-302024\n-1046871\n-576140\n71996\n523379\n780347\n914914\n982756\n1016349\n1032841\n1040904\n1044837\n1046755\n1047689\n1048144\n1048366\n1048474\n1048526\n1048552\n51059\n698743\n814477\n-39108\n1023315\n-1021088\n-10747\n829025\n-66968\n1033387\n935585\n-992458\n1046404\n-292667\n-1047237\n-580104\n68687\n521372\n779267\n914363\n982481\n1016214\n1032775\n1040871\n1044822\n1046747\n1047685\n1048142\n1048365\n1048473\n1048526\n1048552\n908887\n1037778\n1028836\n312394\n1048421\n-967848\n110137\n774651\n-125677\n1039568\n922270\n-998744\n1045411\n-283284\n-1047559\n-584057\n65378\n519363\n778186\n913811\n982206\n1016078\n1032709\n1040839\n1044806\n1046740\n1047682\n1048140\n1048364\n1048473\n1048526\n1048552\n931088\n891628\n1004002\n628168\n1014846\n-888166\n229554\n715250\n-183988\n1044148\n908263\n-1004665\n1044233\n-273877\n-1047837\n-587997\n62068\n517352\n777103\n913258\n981931\n1015943\n1032642\n1040807\n1044790\n1046732\n1047678\n1048139\n1048363\n1048472\n1048525\n1048551\n97251\n328620\n745750\n872098\n924469\n-784217\n345913\n651206\n-241718\n1047121\n893575\n-1010219\n1042869\n-264447\n-1048070\n-591925\n58758\n515338\n776018\n912704\n981654\n1015807\n1032576\n1040774\n1044774\n1046724\n1047674\n1048137\n1048362\n1048472\n1048525\n1048551\n-825998\n-388026\n314121\n1016284\n782349\n-658841\n457665\n582936\n-298683\n1048481\n878217\n-1015404\n1041319\n-254993\n-1048260\n-595841\n55447\n513322\n774931\n912149\n981378\n1015671\n1032509\n1040742\n1044758\n1046716\n1047670\n1048135\n1048361\n1048471\n1048525\n1048551\n-989829\n-923260\n-190539\n1044237\n596439\n-515465\n563320\n510883\n-354704\n1048227\n862201\n-1020219\n1039584\n-245517\n-1048405\n-599745\n52135\n511303\n773842\n911592\n981100\n1015534\n1032442\n1040709\n1044743\n1046709\n1047667\n1048133\n1048360\n1048471\n1048525\n1048551\n-243615\n-1026848\n-650900\n952759\n377146\n-358006\n661471\n435514\n-409604\n1046359\n845538\n-1024661\n1037665\n-236020\n-1048506\n-603637\n48823\n509281\n772751\n911035\n980823\n1015397\n1032375\n1040676\n1044727\n1046701\n1047663\n1048131\n1048359\n1048471\n1048525\n1048551\n726577\n-650360\n-959935\n752313\n136744\n-190765\n750812\n357318\n-463209\n1042880\n828241\n-1028728\n1035561\n-226503\n-1048562\n-607516\n45511\n507257\n771659\n910476\n980544\n1015260\n1032308\n1040643\n1044711\n1046693\n1047659\n1048129\n1048359\n1048470\n1048524\n1048551\n1028758\n30187\n-1045796\n465824\n-111312\n-18312\n830151\n276803\n-515349\n1037795\n810323\n-1032421\n1033272\n-216966\n-1048575\n-611383\n42198\n505231\n770565\n909917\n980265\n1015123\n1032241\n1040611\n1044695\n1046685\n1047655\n1048128\n1048358\n1048470\n1048524\n1048551\n385103\n696621\n-888522\n126059\n-353137\n154641\n898433\n194492\n-565860\n1031112\n791797\n-1035736\n1030800\n-207410\n-1048543\n-615237\n38885\n503202\n769469\n909356\n979986\n1014985\n1032173\n1040578\n1044679\n1046677\n1047651\n1048126\n1048357\n1048469\n1048524\n1048551\n-612613\n1037368\n-524678\n-228125\n-575197\n323369\n954746\n110918\n-614582\n1022841\n772678\n-1038673\n1028145\n-197836\n-1048467\n-619078\n35571\n501171\n768371\n908795\n979706\n1014847\n1032106\n1040545\n1044662\n1046670\n1047648\n1048124\n1048356\n1048469\n1048524\n1048551\n-1047096\n893120\n-38852\n-556217\n-765063\n483262\n998343\n26624\n-661361\n1012996\n752979\n-1041230\n1025307\n-188245\n-1048347\n-622907\n32257\n499137\n767271\n908232\n979425\n1014709\n1032038\n1040512\n1044646\n1046662\n1047644\n1048122\n1048355\n1048468\n1048524\n1048550\n-518884\n331317\n456007\n-820694\n-912107\n629951\n1028641\n-57842\n-706050\n1001591\n732715\n-1043408\n1022286\n-178638\n-1048183\n-626723\n28942\n497101\n766170\n907669\n979144\n1014571\n1031970\n1040479\n1044630\n1046654\n1047640\n1048120\n1048354\n1048468\n1048523\n1048550\n486388\n-385384\n844849\n-991308\n-1008100\n759429\n1045237\n-141933\n-748506\n988644\n711902\n-1045204\n1019084\n-169015\n-1047974\n-630526\n25627\n495062\n765066\n907104\n978863\n1014432\n1031902\n1040445\n1044614\n1046646\n1047636\n1048118\n1048353\n1048467\n1048523\n1048550\n1044477\n-921910\n1037274\n-1048544\n-1047668\n868158\n1047910\n-225103\n-788596\n974174\n690556\n-1046619\n1015700\n-159378\n-1047721\n-634316\n22312\n493021\n763961\n906538\n978581\n1014293\n1031834\n1040412\n1044598\n1046638\n1047632\n1048116\n1048352\n1048467\n1048523\n1048550\n642278\n-1027420\n988545\n-985857\n-1028597\n953168\n1036624\n-306812\n-826193\n958204\n668691\n-1047651\n1012136\n-149727\n-1047424\n-638094\n18997\n490978\n762854\n905971\n978298\n1014154\n1031766\n1040379\n1044582\n1046630\n1047628\n1048114\n1048351\n1048467\n1048523\n1048550\n-350428\n-652587\n709992\n-810417\n-951954\n1012135\n1011531\n-386529\n-861177\n940760\n646325\n-1048301\n1008392\n-140063\n-1047083\n-641858\n15682\n488932\n761746\n905403\n978014\n1014014\n1031698\n1040345\n1044565\n1046622\n1047624\n1048113\n1048350\n1048466\n1048522\n1048550\n-1020952\n27347\n266373\n-542288\n-822030\n1043449\n972963\n-463738\n-893440\n921866\n623475\n-1048569\n1004469\n-130386\n-1046698\n-645609\n12366\n486884\n760635\n904835\n977731\n1013874\n1031629\n1040312\n1044549\n1046614\n1047621\n1048111\n1048349\n1048466\n1048522\n1048550\n-752818\n694495\n-239174\n-212137\n-646095\n1046254\n921435\n-537937\n-922877\n901553\n600157\n-1048453\n1000366\n-120699\n-1046269\n-649346\n9050\n484833\n759523\n904265\n977446\n1013734\n1031560\n1040278\n1044533\n1046606\n1047617\n1048109\n1048348\n1048465\n1048522\n1048550\n207454\n1036950\n-689116\n142277\n-433998\n1020474\n857633\n-608645\n-949397\n879852\n576389\n-1047954\n996086\n-111001\n-1045795\n-653070\n5735\n482780\n758408\n903694\n977161\n1013594\n1031492\n1040245\n1044516\n1046598\n1047613\n1048107\n1048348\n1048465\n1048522\n1048550\n976993\n894606\n-978846\n480418\n-197609\n966812\n782407\n-675402\n-972916\n856797\n552189\n-1047073\n991629\n-101293\n-1045277\n-656781\n2419\n480725\n757293\n903122\n976876\n1013453\n1031423\n1040211\n1044500\n1046590\n1047609\n1048105\n1048347\n1048464\n1048522\n1048550\n848290\n334012\n-1041007\n763613\n49840\n886736\n696759\n-737776\n-993359\n832422\n527574\n-1045809\n986996\n-91577\n-1044716\n-660479\n-897\n478668\n756175\n902548\n976590\n1013312\n1031354\n1040177\n1044483\n1046582\n1047605\n1048103\n1048346\n1048464\n1048521\n1048549\n-60327\n-382740\n-861146\n959472\n294499\n782432\n601830\n-795362\n-1010661\n806765\n502564\n-1044164\n982187\n-81853\n-1044110\n-664163\n-4213\n476608\n755055\n901974\n976303\n1013171\n1031285\n1040144\n1044467\n1046574\n1047601\n1048101\n1048345\n1048463\n1048521\n1048549\n-913480\n-920552\n-481079\n1045596\n522675\n656751\n498885\n-847785\n-1024768\n779866\n477177\n-1042137\n977203\n-72121\n-1043460\n-667833\n-7529\n474545\n753934\n901399\n976016\n1013029\n1031215\n1040110\n1044450\n1046566\n1047597\n1048099\n1048344\n1048463\n1048521\n1048549\n-926783\n-1027984\n10834\n1012134\n721596\n513126\n389294\n-894706\n-1035635\n751766\n451433\n-1039730\n972046\n-62384\n-1042766\n-671489\n-10845\n472481\n752811\n900823\n975728\n1012887\n1031146\n1040076\n1044434\n1046558\n1047593\n1048097\n1048343\n1048462\n1048521\n1048549\n-88006\n-654808\n500227\n862913\n880129\n355482\n274517\n-935820\n-1043229\n722509\n425350\n-1036943\n966716\n-52641\n-1042029\n-675132\n-14160\n470414\n751686\n900246\n975440\n1012745\n1031076\n1040042\n1044417\n1046550\n1047589\n1048095\n1048342\n1048462\n1048521\n1048549\n831683\n24506\n873324\n614999\n989400\n188125\n156084\n-970860\n-1047524\n692140\n398948\n-1033777\n961214\n-42893\n-1041247\n-678761\n-17476\n468345\n750559\n899667\n975151\n1012603\n1031006\n1040008\n1044401\n1046542\n1047585\n1048094\n1048341\n1048462\n1048520\n1048549\n986727\n692363\n1043383\n296747\n1043293\n15629\n35572\n-999599\n-1048507\n660704\n372247\n-1030234\n955541\n-33142\n-1040421\n-682376\n-20791\n466273\n749431\n899088\n974862\n1012460\n1030937\n1039974\n1044384\n1046534\n1047581\n1048092\n1048340\n1048461\n1048520\n1048549\n234579\n1036524\n970868\n-55444\n1038792\n-157295\n-85414\n-1021851\n-1046176\n628252\n345266\n-1026315\n949698\n-23387\n-1039552\n-685976\n-24106\n464200\n748300\n898508\n974572\n1012317\n1030867\n1039940\n1044367\n1046526\n1047577\n1048090\n1048339\n1048461\n1048520\n1048549\n-733240\n896085\n672638\n-401294\n976148\n-325921\n-205263\n-1037470\n-1040537\n594832\n318027\n-1022021\n943686\n-13631\n-1038638\n-689563\n-27421\n462124\n747168\n897926\n974282\n1012174\n1030796\n1039905\n1044351\n1046518\n1047574\n1048088\n1048338\n1048460\n1048520\n1048549\n-1026921\n336705\n218028\n-701247\n858869\n-485642\n-322377\n-1046356\n-1031609\n560496\n290549\n-1017353\n937506\n-3874\n-1037681\n-693136\n-30736\n460046\n746035\n897344\n973991\n1012031\n1030726\n1039871\n1044334\n1046510\n1047570\n1048086\n1048337\n1048460\n1048519\n1048548\n-376456\n-380093\n-287272\n-920998\n693518\n-632095\n-435197\n-1048450\n-1019419\n525297\n262854\n-1012314\n931160\n5884\n-1036680\n-696694\n-34050\n457965\n744899\n896760\n973699\n1011887\n1030656\n1039837\n1044317\n1046501\n1047566\n1048084\n1048336\n1048459\n1048519\n1048548\n620122\n-919188\n-725784\n-1035414\n489349\n-761278\n-542220\n-1043740\n-1004007\n489289\n234961\n-1006906\n924648\n15641\n-1035636\n-700238\n-37364\n455882\n743762\n896176\n973407\n1011743\n1030585\n1039802\n1044300\n1046493\n1047562\n1048082\n1048335\n1048459\n1048519\n1048548\n1046562\n-1028541\n-995560\n-1031407\n257791\n-869661\n-642020\n-1032256\n-985420\n452528\n206892\n-1001130\n917972\n25397\n-1034547\n-703768\n-40678\n453797\n742622\n895590\n973114\n1011599\n1030514\n1039768\n1044284\n1046485\n1047558\n1048080\n1048334\n1048458\n1048519\n1048548\n510798\n-657026\n-1033880\n-909438\n11805\n-954284\n-733269\n-1014072\n-963718\n415070\n178668\n-994988\n911132\n35151\n-1033415\n-707283\n-43991\n451710\n741482\n895004\n972821\n1011454\n1030444\n1039733\n1044267\n1046477\n1047554\n1048078\n1048334\n1048458\n1048518\n1048548\n-494591\n21665\n-831836\n-683455\n-234843\n-1012833\n-814750\n-989306\n-938970\n376973\n150310\n-988483\n904130\n44901\n-1032240\n-710784\n-47304\n449621\n740339\n894416\n972527\n1011309\n1030373\n1039698\n1044250\n1046469\n1047550\n1048076\n1048333\n1048457\n1048518\n1048548\n')
FILES['tables/sin.txt'] = ('cf807ea8b90744db265a17bd68906d40c0b089bfb8933655aa190086ca6dfee3', '# sin.txt\n# S[p, i] = round(2^20 sin(p 100000^(-i/32))), p = 0 .. 511, i = 0 .. 31, row p then i\n# 16384 values, one per line, generated by make_tables.py\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n0\n882346\n673770\n490679\n349509\n246333\n172729\n120819\n84407\n58935\n41138\n28711\n20037\n13983\n9758\n6809\n4752\n3316\n2314\n1615\n1127\n786\n549\n383\n267\n186\n130\n91\n63\n44\n31\n22\n15\n953467\n1032536\n867281\n659045\n478878\n340739\n240028\n168267\n117683\n82212\n57400\n40066\n27963\n19514\n13618\n9503\n6632\n4628\n3229\n2254\n1573\n1097\n766\n534\n373\n260\n182\n127\n88\n62\n43\n30\n147975\n908567\n1042251\n893205\n684619\n499440\n356040\n251034\n176059\n123159\n86046\n60080\n41938\n29269\n20426\n14255\n9948\n6942\n4844\n3380\n2359\n1646\n1149\n802\n559\n390\n272\n190\n133\n93\n65\n45\n-793565\n359821\n974909\n1025208\n852042\n644495\n467310\n332172\n233879\n163917\n114628\n80073\n55905\n39022\n27234\n19006\n13263\n9256\n6459\n4507\n3145\n2195\n1532\n1069\n746\n520\n363\n253\n177\n123\n86\n60\n-1005505\n-357150\n680913\n1039957\n971775\n771941\n572354\n411154\n290959\n204423\n143124\n100037\n69863\n48771\n34040\n23757\n16579\n11569\n8074\n5634\n3932\n2744\n1915\n1336\n932\n651\n454\n317\n221\n154\n108\n75\n-292988\n-907145\n228613\n935764\n1037117\n878296\n669775\n487468\n347120\n244614\n171512\n119964\n83808\n58516\n40845\n28507\n19894\n13883\n9688\n6761\n4718\n3292\n2297\n1603\n1119\n781\n545\n380\n265\n185\n129\n90\n688900\n-1033027\n-276837\n724547\n1044410\n960655\n758274\n560618\n402183\n284428\n199772\n139847\n97739\n68256\n47648\n33256\n23209\n16197\n11303\n7888\n5504\n3841\n2680\n1870\n1305\n911\n636\n444\n310\n216\n151\n105\n1037417\n-675945\n-717926\n430463\n993247\n1016767\n836673\n630129\n455975\n323804\n227882\n159679\n111652\n77990\n54450\n38005\n26524\n18510\n12918\n9014\n6291\n4390\n3063\n2138\n1492\n1041\n726\n507\n354\n247\n172\n120\n432138\n-2842\n-992106\n87146\n886490\n1045099\n903927\n695550\n508325\n362682\n255821\n179452\n125545\n87717\n61248\n42754\n29839\n20824\n14532\n10141\n7077\n4938\n3446\n2405\n1678\n1171\n817\n570\n398\n278\n194\n135\n-570447\n671590\n-1035632\n-266138\n730116\n1044877\n959140\n756458\n559068\n401001\n283569\n199161\n139416\n97437\n68045\n47501\n33153\n23137\n16147\n11268\n7863\n5487\n3829\n2672\n1865\n1301\n908\n634\n442\n309\n215\n150\n-1048566\n1032037\n-838387\n-588983\n532876\n1016107\n1001577\n812455\n608043\n438703\n311103\n218796\n153262\n107148\n74838\n52247\n36467\n25451\n17761\n12395\n8649\n6036\n4212\n2939\n2051\n1431\n999\n697\n486\n339\n237\n165\n-562637\n909982\n-446227\n-844466\n305810\n959576\n1030672\n863180\n655096\n475729\n338405\n238352\n167081\n116850\n81628\n56992\n39781\n27764\n19376\n13521\n9436\n6585\n4595\n3206\n2238\n1561\n1090\n760\n531\n370\n258\n180\n440577\n362488\n49676\n-1003366\n61628\n876827\n1046039\n908302\n700079\n512023\n365453\n257820\n180870\n126542\n88415\n61737\n43094\n30077\n20990\n14648\n10222\n7133\n4978\n3474\n2424\n1692\n1180\n824\n575\n401\n280\n195\n1038727\n-354477\n534030\n-1047510\n-186004\n770122\n1047472\n947529\n742848\n547529\n392226\n277195\n194627\n136222\n95198\n66479\n46407\n32390\n22604\n15775\n11008\n7682\n5361\n3741\n2611\n1822\n1271\n887\n619\n432\n301\n210\n681876\n-905716\n894228\n-971849\n-423224\n642375\n1034952\n980606\n783268\n582191\n418706\n296468\n208349\n145891\n101977\n71221\n49720\n34703\n24219\n16901\n11795\n8231\n5744\n4008\n2797\n1952\n1362\n950\n663\n463\n323\n225\n-301889\n-1033511\n1046529\n-785036\n-636757\n497078\n1008646\n1007319\n821212\n615957\n444871\n315633\n222034\n155548\n108752\n75961\n53032\n37015\n25833\n18028\n12581\n8779\n6127\n4275\n2983\n2082\n1453\n1014\n707\n494\n345\n240\n-1008098\n-678115\n955524\n-508438\n-814649\n338200\n968905\n1027494\n856560\n648775\n470703\n334682\n235680\n165191\n115522\n80699\n56343\n39328\n27447\n19155\n13367\n9328\n6509\n4543\n3170\n2212\n1544\n1077\n752\n525\n366\n255\n-787467\n-5683\n642371\n-173690\n-946944\n170081\n916257\n1041000\n889200\n680594\n496182\n353610\n249284\n174819\n122288\n85436\n59654\n41640\n29061\n20281\n14153\n9877\n6892\n4810\n3356\n2342\n1634\n1141\n796\n555\n388\n270\n157158\n669405\n179875\n180924\n-1026239\n-2684\n851405\n1047750\n919029\n711364\n521289\n372408\n262843\n184433\n129048\n90171\n62964\n43952\n30675\n21408\n14940\n10425\n7275\n5077\n3543\n2472\n1725\n1204\n840\n586\n409\n285\n957292\n1031530\n-324440\n514845\n-1048093\n-175376\n775211\n1047700\n945953\n741040\n546005\n391071\n276356\n194030\n135803\n94904\n66274\n46264\n32289\n22534\n15726\n10974\n7658\n5344\n3729\n2602\n1816\n1267\n884\n617\n431\n301\n877297\n911391\n-753327\n789883\n-1011284\n-343277\n688691\n1040849\n969885\n769574\n570312\n409590\n289820\n203611\n142552\n99635\n69582\n48575\n33903\n23661\n16512\n11523\n8041\n5611\n3916\n2733\n1907\n1331\n929\n648\n452\n316\n-9281\n365154\n-1007074\n974581\n-917873\n-501798\n592998\n1027244\n990752\n796924\n594191\n427960\n303232\n213174\n149295\n104365\n72891\n50886\n35517\n24787\n17298\n12072\n8424\n5879\n4102\n2863\n1998\n1394\n973\n679\n474\n331\n-887326\n-351801\n-1026688\n1047815\n-773087\n-646610\n489406\n1006971\n1008486\n823046\n617625\n446174\n316590\n222719\n156031\n109092\n76198\n53198\n37131\n25914\n18084\n12620\n8807\n6146\n4289\n2993\n2088\n1457\n1017\n710\n495\n346\n-949568\n-904281\n-807610\n1001209\n-585031\n-773755\n379295\n980163\n1023033\n847901\n640595\n464224\n329892\n232244\n162761\n113816\n79505\n55508\n38745\n27040\n18871\n13169\n9190\n6413\n4475\n3123\n2179\n1521\n1061\n741\n517\n361\n-138781\n-1033987\n-400772\n840094\n-364230\n-879759\n264131\n946993\n1034344\n871451\n663085\n482106\n343135\n241749\n169485\n118539\n82811\n57819\n40358\n28167\n19657\n13718\n9573\n6680\n4662\n3253\n2270\n1584\n1105\n771\n538\n376\n799600\n-680280\n99241\n582896\n-123043\n-961728\n145449\n907677\n1042386\n893659\n685078\n499811\n356317\n251234\n176201\n123259\n86116\n60129\n41972\n29293\n20443\n14266\n9956\n6947\n4848\n3383\n2361\n1647\n1150\n802\n560\n391\n1002833\n-8525\n576181\n259031\n125031\n-1017419\n24829\n862469\n1047133\n914491\n706557\n517334\n369436\n260696\n182909\n127976\n89420\n62439\n43585\n30420\n21229\n14815\n10338\n7215\n5035\n3513\n2452\n1711\n1194\n833\n581\n406\n284065\n667215\n919166\n-94459\n366108\n-1045314\n-96121\n811664\n1048569\n933914\n727506\n534667\n382489\n270136\n189610\n132691\n92724\n64749\n45198\n31546\n22015\n15364\n10721\n7482\n5221\n3643\n2542\n1774\n1238\n864\n603\n421\n-695871\n1031016\n1048456\n-437146\n586692\n-1044648\n-215791\n755592\n1046690\n951900\n747910\n551806\n395474\n279553\n196303\n137403\n96026\n67058\n46812\n32672\n22802\n15912\n11104\n7749\n5407\n3774\n2633\n1838\n1282\n895\n624\n436\n-1036026\n912792\n933993\n-729836\n774439\n-1015441\n-332586\n694615\n1041501\n968420\n767753\n568743\n408389\n288945\n202988\n142113\n99328\n69367\n48425\n33798\n23588\n16461\n11487\n8016\n5594\n3904\n2724\n1901\n1327\n926\n646\n451\n-423664\n367816\n602387\n-939053\n918840\n-958490\n-444951\n629129\n1033021\n983449\n787020\n585472\n421231\n298312\n209664\n146819\n102628\n71676\n50038\n34925\n24374\n17010\n11870\n8283\n5780\n4034\n2815\n1964\n1371\n957\n668\n466\n578213\n-349123\n130733\n-1040870\n1011812\n-875352\n-551390\n559561\n1021274\n996964\n805697\n601988\n433999\n307654\n216332\n151522\n105927\n73984\n51650\n36051\n25160\n17558\n12253\n8551\n5967\n4164\n2906\n2028\n1415\n987\n689\n481\n1048484\n-902839\n-371314\n-1023642\n1048152\n-768297\n-650483\n486361\n1006299\n1008944\n823770\n618284\n446689\n316969\n222990\n156223\n109226\n76292\n53263\n37177\n25946\n18107\n12636\n8818\n6153\n4294\n2996\n2091\n1459\n1018\n711\n496\n554783\n-1034456\n-787035\n-889338\n1025825\n-640252\n-740912\n410004\n988142\n1019370\n841225\n634354\n459299\n326256\n229638\n160920\n112523\n78600\n54876\n38303\n26732\n18655\n13019\n9085\n6340\n4424\n3087\n2154\n1503\n1049\n732\n511\n-448982\n-682440\n-1019781\n-653320\n946083\n-494713\n-821472\n330986\n966862\n1028227\n858050\n650192\n471828\n335515\n236278\n165613\n115819\n80907\n56488\n39429\n27518\n19204\n13402\n9352\n6526\n4554\n3178\n2218\n1548\n1080\n754\n526\n-1039956\n-11367\n-1015439\n-342580\n813386\n-335658\n-891089\n249820\n942524\n1035500\n874231\n665793\n484274\n344745\n242907\n170304\n119114\n83214\n58101\n40555\n28304\n19753\n13784\n9619\n6713\n4684\n3269\n2281\n1592\n1111\n775\n541\n-674799\n665021\n-775020\n7340\n635164\n-167432\n-948837\n167033\n915207\n1041179\n889756\n681151\n496633\n353946\n249525\n174991\n122408\n85520\n59713\n41681\n29090\n20301\n14167\n9887\n6899\n4814\n3360\n2345\n1636\n1142\n797\n556\n310765\n1030495\n-354418\n356421\n421391\n5368\n-993946\n83161\n884997\n1045255\n904614\n696260\n508903\n363115\n256134\n179674\n125701\n87826\n61325\n42807\n29876\n20850\n14550\n10154\n7086\n4945\n3451\n2408\n1680\n1173\n818\n571\n1010613\n914188\n148582\n664738\n184032\n178022\n-1025815\n-1250\n851989\n1047722\n918794\n711114\n521083\n372253\n262731\n184353\n128992\n90132\n62937\n43933\n30662\n21399\n14933\n10421\n7272\n5075\n3541\n2471\n1725\n1203\n840\n586\n781308\n370476\n617038\n897028\n-63627\n345812\n-1044019\n-85653\n816287\n1048575\n932285\n725709\n533171\n381359\n269318\n189029\n132282\n92437\n64548\n45058\n31448\n21947\n15316\n10688\n7459\n5205\n3632\n2535\n1769\n1234\n861\n601\n-166328\n-346442\n942041\n1026724\n-307725\n504153\n-1048317\n-169500\n778004\n1047814\n945077\n740040\n545164\n390432\n275893\n193701\n135571\n94742\n66160\n46184\n32234\n22496\n15699\n10955\n7645\n5335\n3723\n2598\n1813\n1265\n883\n616\n-961042\n-901390\n1048030\n1038992\n-534600\n648721\n-1038651\n-252247\n737262\n1045440\n957161\n754099\n557059\n399471\n282456\n198369\n138858\n97046\n67771\n47310\n33020\n23044\n16082\n11223\n7832\n5465\n3814\n2661\n1857\n1296\n904\n631\n-872179\n-1034917\n910364\n932429\n-731552\n775564\n-1015150\n-333357\n694189\n1041455\n968526\n767884\n568856\n408476\n289008\n203033\n142144\n99350\n69383\n48436\n33806\n23593\n16465\n11490\n8018\n5595\n3905\n2725\n1901\n1327\n926\n646\n18562\n-684595\n561050\n719224\n-887558\n881217\n-978126\n-412304\n648922\n1035868\n979166\n781388\n580552\n417445\n295547\n207693\n145429\n101653\n70994\n49561\n34592\n24141\n16847\n11757\n8204\n5725\n3995\n2788\n1946\n1358\n947\n661\n892237\n-14208\n81298\n423759\n-993887\n962794\n-928074\n-488574\n601603\n1028685\n989071\n794607\n592144\n426378\n302074\n212348\n148712\n103956\n72605\n50687\n35378\n24690\n17230\n12024\n8391\n5855\n4086\n2851\n1990\n1389\n969\n676\n945594\n662821\n-417354\n79829\n-1044587\n1018066\n-865659\n-561673\n552382\n1019918\n998234\n807535\n603631\n435274\n308588\n216999\n151994\n106258\n74215\n51812\n36164\n25239\n17613\n12291\n8577\n5986\n4177\n2915\n2034\n1419\n991\n691\n129576\n1029966\n-818977\n-273231\n-1036820\n1045522\n-791713\n-631127\n501415\n1009581\n1006649\n820169\n615010\n444132\n315090\n221646\n155274\n108560\n75826\n52937\n36949\n25787\n17996\n12558\n8764\n6116\n4268\n2978\n2078\n1450\n1012\n706\n-805573\n915576\n-1030197\n-595041\n-971021\n1044413\n-707222\n-696485\n448862\n997690\n1014309\n832503\n626281\n452952\n321577\n226288\n158552\n110861\n77436\n54063\n37735\n26336\n18379\n12826\n8950\n6246\n4359\n3042\n2122\n1481\n1034\n721\n-1000082\n373133\n-1001909\n-848796\n-850873\n1014768\n-613309\n-757322\n394891\n984262\n1021209\n844534\n637440\n461733\n328052\n230925\n161829\n113162\n79047\n55188\n38521\n26884\n18762\n13093\n9137\n6376\n4449\n3105\n2167\n1512\n1055\n736\n-275120\n-343759\n-740689\n-1005473\n-683101\n957399\n-511228\n-813245\n339671\n969319\n1027342\n856255\n648485\n470473\n334512\n235558\n165104\n115462\n80657\n56313\n39307\n27433\n19145\n13360\n9323\n6506\n4540\n3168\n2211\n1543\n1077\n751\n702786\n-899935\n-307267\n-1047153\n-477095\n873871\n-402336\n-863888\n283377\n952883\n1032706\n867665\n659415\n479173\n340959\n240186\n168378\n117761\n82267\n57438\n40093\n27981\n19527\n13627\n9510\n6636\n4631\n3232\n2255\n1574\n1098\n766\n1034554\n-1035370\n197590\n-969069\n-244385\n766468\n-288085\n-908925\n226187\n934980\n1037295\n878757\n670228\n487832\n347391\n244809\n171650\n120060\n83876\n58564\n40878\n28530\n19910\n13894\n9696\n6766\n4722\n3295\n2299\n1605\n1120\n781\n415158\n-686745\n656510\n-780151\n2003\n638124\n-169997\n-948063\n168282\n915638\n1041106\n889528\n680922\n496448\n353808\n249426\n174920\n122359\n85486\n59689\n41664\n29078\n20293\n14162\n9883\n6896\n4813\n3358\n2344\n1635\n1141\n796\n-585933\n-17049\n962800\n-502006\n248279\n492345\n-49645\n-981048\n109846\n894886\n1044137\n899975\n691495\n505021\n360210\n254039\n178189\n124657\n87095\n60813\n42450\n29627\n20676\n14429\n10069\n7027\n4903\n3422\n2388\n1666\n1163\n811\n-1048319\n660617\n1045249\n-166447\n480659\n333114\n71369\n-1007665\n51062\n872756\n1046384\n910093\n701945\n513551\n366598\n258647\n181456\n126954\n88704\n61938\n43235\n30175\n21059\n14696\n10255\n7157\n4994\n3485\n2432\n1697\n1184\n826\n-546886\n1029429\n884691\n188150\n686135\n164782\n191432\n-1027742\n-7884\n849282\n1047847\n919879\n712270\n522036\n372969\n263249\n184721\n129250\n90313\n63063\n44021\n30724\n21442\n14963\n10442\n7287\n5085\n3548\n2476\n1728\n1206\n841\n457352\n916958\n518453\n521227\n853208\n-8052\n308946\n-1041149\n-66804\n824500\n1048525\n929328\n722468\n530476\n379325\n267846\n187984\n131546\n91921\n64188\n44807\n31272\n21824\n15230\n10628\n7417\n5176\n3612\n2520\n1759\n1227\n856\n1041102\n375787\n31680\n794691\n972526\n-180667\n422344\n-1047799\n-125514\n798449\n1048416\n938439\n732538\n538869\n385666\n272437\n191245\n133842\n93530\n65313\n45592\n31821\n22207\n15498\n10815\n7547\n5267\n3675\n2565\n1790\n1249\n872\n667668\n-341073\n-462457\n977266\n1037411\n-348345\n530116\n-1047648\n-183826\n771168\n1047521\n947207\n742477\n547217\n391989\n277023\n194504\n136136\n95138\n66437\n46378\n32369\n22590\n15765\n11001\n7677\n5357\n3739\n2609\n1821\n1270\n887\n-319617\n-898473\n-849079\n1048069\n1044230\n-506505\n630827\n-1040697\n-241558\n742700\n1045840\n955628\n752285\n555516\n398297\n281603\n197762\n138430\n96746\n67562\n47163\n32918\n22973\n16032\n11188\n7807\n5448\n3802\n2653\n1851\n1292\n902\n-1013048\n-1035816\n-1038300\n999003\n992603\n-650827\n723135\n-1026992\n-298526\n713089\n1043375\n963701\n761959\n563768\n404587\n286177\n201017\n140724\n98354\n68686\n47949\n33466\n23356\n16299\n11374\n7937\n5539\n3865\n2697\n1882\n1313\n917\n-775087\n-688889\n-986129\n835680\n885419\n-777368\n805810\n-1006622\n-354549\n682379\n1040128\n971422\n771497\n571971\n410860\n290745\n204270\n143016\n99961\n69810\n48734\n34015\n23739\n16566\n11561\n8067\n5630\n3929\n2742\n1913\n1335\n932\n175485\n-19891\n-704695\n576780\n728677\n-882669\n877752\n-979718\n-409453\n650619\n1036101\n978788\n780898\n580124\n417117\n295308\n207522\n145308\n101568\n70935\n49520\n34563\n24121\n16833\n11747\n8198\n5721\n3992\n2786\n1944\n1357\n947\n964717\n658408\n-259427\n251912\n531150\n-963854\n938002\n-946455\n-463061\n617857\n1031297\n985797\n790160\n588228\n423355\n299864\n210771\n147600\n103176\n72059\n50305\n35111\n24504\n17101\n11934\n8328\n5811\n4055\n2830\n1975\n1378\n962\n866993\n1028884\n246154\n-101767\n303894\n-1018705\n985757\n-907050\n-515206\n584144\n1025719\n992446\n799282\n596280\n429576\n304414\n214018\n149890\n104782\n73183\n51091\n35660\n24887\n17368\n12120\n8458\n5902\n4119\n2874\n2006\n1400\n977\n-27841\n918333\n694508\n-443807\n59628\n-1045723\n1020382\n-861758\n-565722\n549531\n1019373\n998732\n808261\n604280\n435778\n308958\n217263\n152180\n106389\n74307\n51876\n36208\n25270\n17635\n12306\n8588\n5993\n4182\n2918\n2037\n1421\n992\n-897078\n378439\n981396\n-735088\n-187974\n-1044170\n1041414\n-810873\n-614449\n514073\n1012262\n1004654\n817097\n612228\n441962\n313496\n220506\n154469\n107995\n75431\n52661\n36757\n25653\n17902\n12493\n8718\n6084\n4245\n2963\n2067\n1443\n1007\n-941545\n-338385\n1040122\n-942296\n-425056\n-1014089\n1048575\n-754724\n-661234\n477822\n1004392\n1010209\n825788\n620124\n448128\n318027\n223747\n156757\n109601\n76555\n53447\n37305\n26035\n18169\n12679\n8848\n6175\n4309\n3007\n2098\n1464\n1022\n-120361\n-897005\n857032\n-1041733\n-638347\n-956301\n1041768\n-693678\n-705928\n440836\n995769\n1015395\n834331\n627965\n454274\n322552\n226985\n159045\n111207\n77679\n54232\n37853\n26418\n18436\n12866\n8978\n6265\n4372\n3051\n2129\n1486\n1037\n811483\n-1036254\n474692\n-1022025\n-815908\n-872385\n1021084\n-628129\n-748391\n403171\n986399\n1020210\n842727\n635752\n460402\n327070\n230221\n161332\n112812\n78802\n55017\n38402\n26801\n18704\n13052\n9108\n6356\n4436\n3095\n2160\n1507\n1052\n997253\n-691029\n-18009\n-885428\n-947803\n-764634\n986800\n-558504\n-788488\n364886\n976290\n1024653\n850972\n643484\n466510\n331581\n233455\n163618\n114418\n79926\n55802\n38950\n27184\n18971\n13239\n9239\n6447\n4499\n3139\n2191\n1529\n1067\n266153\n-22732\n-506522\n-647562\n-1026648\n-635992\n939370\n-485253\n-826091\n326038\n965449\n1028721\n859066\n651161\n472598\n336085\n236687\n165903\n116023\n81049\n56588\n39498\n27567\n19238\n13425\n9369\n6538\n4562\n3184\n2222\n1550\n1082\n-709647\n656194\n-877275\n-335635\n-1048030\n-489973\n879428\n-408854\n-861084\n286689\n953883\n1032414\n867007\n658781\n478667\n340583\n239916\n168187\n117627\n82173\n57373\n40047\n27949\n19505\n13612\n9499\n6629\n4626\n3228\n2253\n1572\n1097\n-1033001\n1028332\n-1044072\n14680\n-1010753\n-330568\n807771\n-329800\n-893354\n246898\n941603\n1035730\n874795\n666344\n484715\n345073\n243142\n170471\n119232\n83296\n58158\n40595\n28332\n19772\n13798\n9629\n6719\n4689\n3272\n2283\n1593\n1112\n-406619\n919701\n-968135\n363316\n-916903\n-162130\n725355\n-248606\n-922799\n206727\n928616\n1038668\n882426\n673849\n490743\n349557\n246367\n172753\n120836\n84419\n58943\n41143\n28715\n20039\n13985\n9759\n6810\n4752\n3316\n2314\n1615\n1127\n593607\n381088\n-667118\n670399\n-771733\n10737\n633277\n-165799\n-949327\n166238\n914933\n1041226\n889901\n681296\n496750\n354033\n249589\n175035\n122440\n85542\n59728\n41692\n29098\n20307\n14171\n9889\n6901\n4816\n3361\n2345\n1636\n1142\n1048073\n-335694\n-211004\n900807\n-583368\n183310\n532763\n-81916\n-972855\n125493\n900564\n1043404\n897217\n688684\n502736\n358502\n252808\n177316\n124043\n86665\n60513\n42240\n29480\n20574\n14357\n10019\n6992\n4879\n3405\n2376\n1658\n1157\n538945\n-895530\n294166\n1028189\n-362351\n350875\n425152\n2500\n-993306\n84554\n885519\n1045201\n904374\n696012\n508701\n362964\n256025\n179597\n125646\n87788\n61298\n42788\n29863\n20841\n14544\n10149\n7083\n4942\n3449\n2407\n1680\n1172\n-465686\n-1036685\n730946\n1037976\n-121054\n508854\n311879\n86898\n-1010617\n43485\n869811\n1046617\n911370\n703280\n514645\n367418\n259239\n181876\n127249\n88911\n62083\n43336\n30246\n21108\n14730\n10279\n7173\n5006\n3493\n2438\n1701\n1187\n-1042168\n-693164\n997789\n929049\n127020\n652930\n194450\n170733\n-1024733\n2350\n853450\n1047650\n918204\n710487\n520566\n371865\n262451\n184154\n128852\n90034\n62868\n43885\n30629\n21375\n14917\n10410\n7264\n5069\n3537\n2469\n1723\n1202\n-660485\n-25573\n1032659\n713865\n367984\n779167\n74432\n253460\n-1035610\n-38790\n836450\n1048300\n924875\n717632\n526466\n376304\n265660\n186432\n130454\n91156\n63653\n44433\n31011\n21642\n15103\n10540\n7355\n5133\n3582\n2499\n1744\n1217\n328444\n653975\n827447\n417035\n588351\n884115\n-46577\n334542\n-1043212\n-79869\n818822\n1048568\n931381\n724716\n532344\n380735\n268866\n188708\n132056\n92279\n64438\n44981\n31394\n21909\n15290\n10670\n7446\n5196\n3626\n2530\n1766\n1232\n1015404\n1027773\n429864\n72509\n775788\n964908\n-166967\n413452\n-1047517\n-120826\n800580\n1048453\n937722\n731736\n538199\n385159\n272070\n190984\n133658\n93401\n65223\n45529\n31777\n22177\n15476\n10800\n7537\n5259\n3670\n2561\n1787\n1247\n768806\n921063\n-67657\n-280311\n919803\n1019338\n-285132\n489679\n-1048509\n-161596\n781738\n1047955\n943896\n738694\n544032\n389574\n275271\n193259\n135259\n94523\n66007\n46078\n32160\n22444\n15663\n10930\n7627\n5323\n3714\n2592\n1809\n1262\n-184629\n383734\n-549449\n-601071\n1012336\n1045918\n-399499\n562728\n-1046187\n-202118\n762310\n1047075\n949902\n745587\n549841\n393982\n278469\n195533\n136860\n95646\n66792\n46626\n32542\n22711\n15849\n11060\n7718\n5386\n3759\n2623\n1830\n1277\n-968316\n-333000\n-903501\n-853085\n1048207\n1043921\n-508545\n632125\n-1040558\n-242328\n742310\n1045812\n955739\n752416\n555628\n398381\n281664\n197805\n138461\n96768\n67577\n47174\n32925\n22978\n16035\n11190\n7809\n5449\n3803\n2654\n1852\n1292\n-861738\n-894049\n-1047499\n-1007531\n1025409\n1013403\n-610816\n697419\n-1031639\n-282166\n721754\n1044167\n961406\n759179\n561391\n402772\n284857\n200077\n140062\n97890\n68361\n47722\n33308\n23245\n16222\n11320\n7900\n5513\n3847\n2685\n1873\n1307\n37118\n-1037108\n-947967\n-1046745\n945217\n955197\n-704952\n758186\n-1019458\n-321569\n700656\n1042141\n966902\n765877\n567130\n407156\n288047\n202348\n141662\n99011\n69146\n48270\n33691\n23512\n16408\n11450\n7991\n5576\n3891\n2715\n1895\n1322\n901848\n-695294\n-628043\n-966241\n812121\n870893\n-789697\n814033\n-1004054\n-360476\n679033\n1039735\n972227\n772509\n572845\n411530\n291233\n204618\n143261\n100133\n69931\n48818\n34073\n23779\n16595\n11581\n8081\n5639\n3935\n2746\n1916\n1337\n937423\n-28413\n-162107\n-775227\n633569\n762795\n-863923\n864596\n-985476\n-398829\n656901\n1036948\n977378\n779073\n578536\n415896\n294417\n206887\n144861\n101255\n70715\n49366\n34456\n24047\n16781\n11711\n8172\n5703\n3980\n2777\n1938\n1352\n111136\n651751\n341517\n-495550\n419556\n633856\n-926641\n909548\n-963783\n-436568\n634276\n1033783\n982356\n785570\n584203\n420254\n297598\n209155\n146460\n102376\n71500\n49915\n34839\n24314\n16968\n11841\n8263\n5766\n4024\n2808\n1959\n1367\n-817329\n1027206\n765742\n-159195\n182060\n487598\n-977016\n948597\n-939043\n-473634\n611175\n1030241\n987159\n791999\n589845\n424603\n300777\n211422\n148059\n103498\n72284\n50463\n35221\n24581\n17154\n11971\n8354\n5830\n4068\n2839\n1981\n1382\n-994346\n922418\n1011942\n195366\n-65626\n328019\n-1014376\n981488\n-911334\n-509972\n587617\n1026322\n991786\n798360\n595463\n428943\n303952\n213688\n149657\n104619\n73069\n51011\n35604\n24848\n17341\n12101\n8445\n5893\n4112\n2870\n2003\n1397\n-257165\n386377\n1022877\n527584\n-309639\n159478\n-1038225\n1008010\n-880743\n-545524\n563617\n1022029\n996237\n804651\n601055\n433275\n307124\n215953\n151255\n105740\n73853\n51559\n35987\n25115\n17527\n12231\n8535\n5956\n4156\n2901\n2024\n1412\n716452\n-330305\n796005\n799461\n-536321\n-13421\n-1048245\n1027990\n-847369\n-580236\n539195\n1017362\n1000511\n810873\n606622\n437598\n310292\n218217\n152853\n106861\n74637\n52107\n36369\n25382\n17713\n12361\n8626\n6020\n4201\n2931\n2046\n1427\n1031367\n-892561\n384072\n979902\n-732985\n-185952\n-1044301\n1041297\n-811315\n-614054\n514369\n1012324\n1004607\n817025\n612163\n441911\n313458\n220479\n154450\n107982\n75422\n52655\n36752\n25649\n17900\n12491\n8717\n6083\n4245\n2962\n2067\n1443\n398048\n-1037523\n-117154\n1048271\n-888623\n-353404\n-1026446\n1047846\n-772697\n-646927\n489157\n1006916\n1008524\n823105\n617678\n446216\n316621\n222741\n156047\n109103\n76206\n53203\n37135\n25917\n18086\n12622\n8808\n6146\n4289\n2993\n2089\n1458\n-601234\n-697418\n-591142\n996749\n-994524\n-511199\n-994919\n1047594\n-731635\n-678804\n463578\n1001141\n1012262\n829115\n623168\n450511\n319781\n225002\n157644\n110223\n76990\n53751\n37517\n26184\n18273\n12752\n8899\n6210\n4333\n3024\n2110\n1473\n-1047744\n-31254\n-927697\n831226\n-1044760\n-655028\n-950140\n1040544\n-688261\n-709636\n437652\n995000\n1015820\n835053\n628631\n454797\n322937\n227261\n159240\n111344\n77774\n54299\n37900\n26451\n18459\n12882\n8989\n6273\n4378\n3055\n2132\n1488\n-530963\n649522\n-1048574\n570636\n-1036519\n-780960\n-892704\n1026739\n-642711\n-739375\n411397\n988495\n1019197\n840918\n634068\n459074\n326090\n229519\n160836\n112464\n78558\n54847\n38283\n26718\n18646\n13012\n9080\n6336\n4422\n3086\n2153\n1503\n473983\n1026631\n-925669\n244781\n-970264\n-885555\n-823377\n1006271\n-595129\n-767976\n384834\n981630\n1022393\n846710\n639478\n463342\n329240\n231777\n162431\n113584\n79343\n55395\n38665\n26985\n18832\n13142\n9171\n6400\n4466\n3117\n2175\n1518\n1043151\n923766\n-587558\n-109070\n-849701\n-965955\n-743082\n979272\n-545665\n-795395\n357983\n974406\n1025407\n852430\n644860\n467599\n332386\n234033\n164026\n114704\n80127\n55943\n39048\n27252\n19019\n13272\n9262\n6463\n4510\n3147\n2196\n1533\n653251\n389017\n-112846\n-450446\n-681580\n-1019964\n-652889\n945917\n-494476\n-821588\n330862\n966826\n1028239\n858075\n650216\n471848\n335530\n236288\n165621\n115824\n80911\n56491\n39431\n27519\n19205\n13402\n9353\n6527\n4554\n3178\n2218\n1548\n-337246\n-327606\n388101\n-740305\n-475311\n-1046105\n-553999\n906423\n-441724\n-846517\n303494\n958894\n1030888\n863646\n655544\n476086\n338669\n238542\n167215\n116944\n81695\n57039\n39813\n27786\n19391\n13532\n9443\n6590\n4599\n3209\n2239\n1563\n-1017680\n-891066\n798819\n-945493\n-242437\n-1043665\n-447730\n861045\n-387576\n-870142\n275899\n950611\n1033354\n869142\n660845\n480315\n341806\n240794\n168809\n118064\n82478\n57586\n40196\n28054\n19578\n13662\n9534\n6653\n4643\n3240\n2261\n1578\n-762464\n-1037931\n1023821\n-1042545\n4006\n-1012710\n-335497\n810080\n-332202\n-892427\n248096\n941981\n1035636\n874564\n666118\n484534\n344939\n243046\n170402\n119184\n83262\n58134\n40579\n28321\n19764\n13793\n9625\n6717\n4687\n3271\n2282\n1593\n193758\n-699538\n1010798\n-1020359\n250224\n-954086\n-218795\n753856\n-275778\n-913339\n220107\n933007\n1037734\n879909\n671363\n488743\n348069\n245296\n171995\n120303\n84046\n58682\n40961\n28588\n19951\n13923\n9716\n6780\n4731\n3302\n2304\n1608\n971840\n-34094\n762775\n-881474\n482438\n-869395\n-99178\n692740\n-218482\n-932844\n191954\n923692\n1039647\n885178\n676579\n492942\n351195\n247545\n173588\n121422\n84830\n59230\n41344\n28855\n20137\n14053\n9807\n6843\n4776\n3333\n2326\n1623\n856417\n647289\n337417\n-641773\n687648\n-760950\n21760\n627128\n-160496\n-950912\n163656\n914040\n1041376\n890370\n681767\n497131\n354317\n249793\n175180\n122541\n85614\n59778\n41726\n29122\n20324\n14183\n9897\n6907\n4820\n3363\n2347\n1638\n-46392\n1026049\n-166387\n-328672\n854371\n-631715\n142407\n557446\n-102002\n-967517\n135236\n904054\n1042919\n895486\n686926\n501309\n357436\n252040\n176772\n123660\n86397\n60326\n42109\n29389\n20510\n14313\n9988\n6970\n4864\n3394\n2369\n1653\n-906548\n925107\n-631508\n22019\n973273\n-485220\n261158\n484145\n-43186\n-982632\n106714\n893739\n1044277\n900524\n692056\n505478\n360552\n254285\n178363\n124779\n87181\n60874\n42492\n29656\n20696\n14443\n10079\n7033\n4908\n3425\n2390\n1668\n-933228\n391654\n-949811\n370192\n1037700\n-325469\n376430\n407702\n15767\n-996234\n78112\n883096\n1045449\n905483\n697157\n509636\n363664\n256530\n179954\n125898\n87964\n61421\n42874\n29923\n20883\n14573\n10170\n7097\n4952\n3456\n2412\n1683\n-101903\n-324906\n-1047294\n676026\n1044046\n-156824\n486688\n328614\n74671\n-1008302\n49452\n872132\n1046435\n910365\n702229\n513783\n366772\n258773\n181545\n127017\n88748\n61969\n43257\n30190\n21069\n14703\n10261\n7160\n4997\n3487\n2433\n1698\n823112\n-889565\n-901294\n904542\n991956\n16104\n590463\n247392\n133338\n-1018817\n20754\n860849\n1047235\n915168\n707271\n517920\n369877\n261014\n183135\n128135\n89531\n62517\n43639\n30457\n21256\n14833\n10351\n7224\n5041\n3518\n2455\n1713\n991361\n-1038331\n-545753\n1029604\n884344\n188593\n686373\n164565\n191583\n-1027764\n-7959\n849251\n1047849\n919891\n712283\n522046\n372977\n263255\n184725\n129253\n90315\n63065\n44022\n30725\n21442\n14963\n10442\n7287\n5085\n3549\n2476\n1728\n248157\n-701652\n-63332\n1036910\n727235\n355930\n773140\n80670\n249223\n-1035128\n-36666\n837343\n1048276\n924535\n717265\n526162\n376075\n265494\n186314\n130371\n91098\n63612\n44405\n30992\n21629\n15094\n10533\n7350\n5129\n3579\n2498\n1743\n-723201\n-36934\n433813\n925622\n529422\n513541\n849609\n-3749\n306075\n-1040898\n-65345\n825130\n1048517\n929098\n722217\n530267\n379168\n267732\n187903\n131489\n91882\n64160\n44787\n31259\n21815\n15224\n10624\n7414\n5173\n3610\n2519\n1758\n-1029651\n645051\n830102\n708471\n301976\n657122\n914760\n-88144\n361959\n-1045066\n-93976\n812615\n1048572\n933581\n727138\n534361\n382258\n269968\n189491\n132607\n92665\n64708\n45170\n31526\n22001\n15354\n10715\n7477\n5218\n3641\n2541\n1773\n-389445\n1025460\n1033402\n410290\n57629\n782749\n967727\n-171966\n416699\n-1047625\n-122536\n799804\n1048440\n937984\n732029\n538444\n385343\n272204\n191079\n133725\n93448\n65255\n45552\n31793\n22188\n15484\n10805\n7540\n5262\n3672\n2562\n1788\n608815\n926441\n996448\n65184\n-189944\n886990\n1007802\n-254673\n470122\n-1048570\n-151004\n786700\n1048122\n942305\n736889\n542515\n388425\n274438\n192667\n134843\n94231\n65803\n45935\n32060\n22374\n15614\n10896\n7604\n5306\n3703\n2584\n1803\n1047334\n394289\n727832\n-287377\n-426886\n966997\n1034454\n-335726\n522058\n-1047901\n-179359\n773310\n1047617\n946544\n741717\n546576\n391503\n276670\n194254\n135960\n95014\n66351\n46317\n32327\n22561\n15744\n10987\n7667\n5350\n3734\n2605\n1818\n522939\n-322203\n290004\n-607070\n-639934\n1020583\n1047326\n-414601\n572344\n-1045619\n-207579\n759636\n1046926\n950702\n746515\n550626\n394578\n278901\n195840\n137077\n95797\n66898\n46700\n32594\n22747\n15874\n11078\n7730\n5395\n3765\n2627\n1833\n-482244\n-888057\n-215247\n-857332\n-817165\n1046286\n1046248\n-490784\n620820\n-1041726\n-235644\n745686\n1046049\n954777\n751281\n554664\n397648\n281131\n197426\n138194\n96580\n67446\n47082\n32861\n22934\n16004\n11169\n7794\n5439\n3795\n2649\n1848\n-1044053\n-1038723\n-670456\n-1009540\n-948658\n1043402\n1031233\n-563782\n667334\n-1036230\n-263532\n731463\n1044986\n958770\n756015\n558691\n400714\n283360\n199012\n139311\n97363\n67993\n47465\n33128\n23120\n16134\n11259\n7857\n5483\n3826\n2670\n1863\n-645965\n-703761\n-969791\n-1046286\n-1027053\n1012011\n1002481\n-633122\n711738\n-1029138\n-291223\n716973\n1043737\n962679\n760718\n562706\n403776\n285587\n200597\n140428\n98146\n68541\n47847\n33395\n23306\n16265\n11350\n7921\n5527\n3857\n2692\n1878\n346020\n-39774\n-1043662\n-963366\n-1047964\n952970\n960376\n-698352\n753892\n-1020461\n-318695\n702221\n1042303\n966505\n765388\n566710\n406834\n287813\n202182\n141544\n98929\n69088\n48230\n33662\n23493\n16395\n11441\n7984\n5571\n3888\n2713\n1893\n1019877\n642808\n-874894\n-770266\n-1010218\n867892\n905479\n-759049\n793662\n-1010214\n-345928\n687213\n1040683\n970248\n770026\n570702\n409888\n290037\n203766\n142661\n99712\n69636\n48612\n33930\n23679\n16525\n11532\n8047\n5616\n3919\n2735\n1908\n756063\n1024863\n-502723\n-489069\n-915930\n759101\n838520\n-814820\n830924\n-998410\n-372902\n671954\n1038878\n973906\n774632\n574682\n412938\n292260\n205350\n143777\n100495\n70184\n48995\n34197\n23866\n16655\n11623\n8111\n5660\n3950\n2756\n1923\n-202872\n927769\n-13675\n-151936\n-770376\n629571\n760392\n-865303\n865558\n-985070\n-399596\n656450\n1036888\n977480\n779205\n578651\n415984\n294482\n206933\n144893\n101277\n70731\n49377\n34464\n24052\n16785\n11713\n8174\n5704\n3981\n2778\n1938\n-975287\n396921\n478552\n202573\n-581703\n482839\n672135\n-910169\n897456\n-970213\n-425990\n640705\n1034714\n980970\n783745\n582608\n419026\n296702\n208516\n146009\n102060\n71278\n49760\n34731\n24238\n16915\n11804\n8237\n5748\n4011\n2799\n1953\n-851028\n-319497\n859521\n533914\n-360471\n322916\n574925\n-949128\n926517\n-953862\n-452065\n624727\n1032356\n984374\n788252\n586552\n422064\n298920\n210098\n147125\n102842\n71826\n50142\n34998\n24425\n17045\n11895\n8301\n5793\n4042\n2821\n1968\n55662\n-886543\n1040662\n804191\n-119064\n154170\n470056\n-981928\n952649\n-936042\n-477801\n608521\n1029814\n987693\n792726\n590485\n425097\n301137\n211680\n148240\n103625\n72373\n50525\n35265\n24611\n17175\n11986\n8364\n5837\n4073\n2842\n1983\n911177\n-1039108\n979862\n982491\n129008\n-18788\n358927\n-1008354\n975769\n-916781\n-503179\n592092\n1027090\n990927\n797166\n594406\n428126\n303353\n213261\n149356\n104407\n72921\n50907\n35532\n24798\n17305\n12077\n8427\n5881\n4104\n2864\n1998\n928960\n-705865\n691255\n1048423\n369858\n-191233\n243016\n-1028235\n995804\n-896108\n-528180\n575448\n1024182\n994075\n801573\n598314\n431151\n305567\n214842\n150471\n105190\n73468\n51290\n35799\n24984\n17436\n12167\n8491\n5925\n4135\n2885\n2014\n92661\n-42614\n241940\n994445\n590008\n-358453\n123868\n-1041443\n1012691\n-874056\n-552784\n558593\n1021093\n997137\n805946\n602210\n434171\n307780\n216422\n151586\n105972\n74015\n51672\n36066\n25171\n17566\n12258\n8554\n5969\n4166\n2907\n2029\n-828829\n640561\n-263623\n826731\n777134\n-515880\n3071\n-1047892\n1026376\n-850658\n-576974\n541534\n1017822\n1000112\n810285\n606094\n437187\n309991\n218001\n152701\n106754\n74563\n52055\n36333\n25357\n17696\n12349\n8618\n6014\n4197\n2928\n2044\n-988298\n1024258\n-707898\n564464\n920763\n-659211\n-117768\n-1047540\n1036816\n-825950\n-600731\n524277\n1014369\n1003001\n814590\n609965\n440199\n312201\n219581\n153816\n107537\n75110\n52437\n36600\n25543\n17826\n12440\n8681\n6058\n4227\n2950\n2059\n-239130\n929090\n-987594\n237637\n1012856\n-784532\n-237038\n-1040388\n1043979\n-799970\n-624038\n506829\n1010737\n1005803\n818860\n613824\n443206\n314409\n221159\n154930\n108319\n75657\n52820\n36867\n25730\n17956\n12531\n8744\n6102\n4258\n2972\n2074\n729893\n399549\n-1037686\n-116368\n1048258\n-888418\n-353151\n-1026485\n1047841\n-772758\n-646877\n489196\n1006925\n1008518\n823096\n617670\n446209\n316616\n222737\n156045\n109101\n76205\n53202\n37134\n25916\n18086\n12621\n8808\n6146\n4289\n2993\n2089\n1027856\n-316790\n-846529\n-457064\n1024988\n-968031\n-464559\n-1005919\n1048391\n-744357\n-669231\n471385\n1002934\n1011146\n827297\n621503\n449208\n318821\n224315\n157159\n109883\n76752\n53584\n37401\n26103\n18216\n12712\n8871\n6190\n4320\n3015\n2104\n380813\n-885022\n-458564\n-745485\n944348\n-1021196\n-569779\n-978824\n1045626\n-714810\n-691083\n453401\n998764\n1013686\n831464\n625324\n452202\n321025\n225892\n158273\n110665\n77299\n53967\n37668\n26289\n18346\n12803\n8934\n6235\n4351\n3036\n2119\n-616348\n-1039485\n36012\n-948644\n810853\n-1046460\n-667410\n-945377\n1039555\n-684162\n-712417\n435251\n994417\n1016138\n835595\n629132\n455191\n323227\n227469\n159386\n111447\n77846\n54349\n37935\n26475\n18476\n12894\n8998\n6279\n4382\n3058\n2134\n-1046841\n-707964\n522215\n-1043305\n631972\n-1043132\n-756150\n-905794\n1030198\n-652460\n-733216\n416943\n989893\n1018502\n839691\n632927\n458176\n325427\n229045\n160500\n112229\n78394\n54732\n38202\n26662\n18606\n12984\n9061\n6323\n4412\n3079\n2149\n-514873\n-45453\n887010\n-1018643\n417720\n-1011305\n-834818\n-860331\n1017584\n-619754\n-753466\n398482\n985192\n1020778\n843751\n636709\n461156\n327626\n230620\n161614\n113010\n78941\n55114\n38469\n26848\n18737\n13075\n9124\n6367\n4443\n3101\n2164\n490466\n638309\n1045585\n-877477\n180087\n-951847\n-902367\n-809285\n1001753\n-586094\n-773151\n379876\n980317\n1022966\n847777\n640478\n464132\n329823\n232195\n162727\n113792\n79488\n55496\n38736\n27035\n18867\n13166\n9188\n6412\n4474\n3122\n2179\n1044874\n1023646\n961075\n-635953\n-67625\n-866382\n-957895\n-752987\n982755\n-551531\n-792256\n361131\n975268\n1025065\n851766\n644233\n467103\n332019\n233769\n163840\n114574\n80035\n55879\n39003\n27221\n18997\n13257\n9251\n6456\n4505\n3144\n2194\n638629\n930404\n653126\n-321694\n-311552\n-757247\n-1000663\n-691801\n960650\n-516119\n-810767\n342255\n970045\n1027075\n855719\n647976\n470069\n334213\n235343\n164953\n115355\n80582\n56261\n39270\n27407\n19127\n13348\n9315\n6500\n4536\n3165\n2209\n-354768\n402175\n193333\n29357\n-538042\n-627422\n-1030103\n-626126\n935508\n-479913\n-828670\n323253\n964649\n1028997\n859636\n651705\n473031\n336406\n236917\n166065\n116137\n81129\n56644\n39537\n27594\n19257\n13438\n9378\n6544\n4567\n3187\n2224\n-1021993\n-314080\n-311408\n377051\n-734416\n-480455\n-1045821\n-556387\n907408\n-442967\n-845952\n304133\n959082\n1030829\n863518\n655421\n475988\n338596\n238489\n167178\n116918\n81676\n57026\n39804\n27780\n19387\n13529\n9441\n6588\n4598\n3208\n2239\n-749602\n-883495\n-743750\n681621\n-889685\n-320361\n-1047608\n-483036\n876440\n-405339\n-862599\n284902\n953344\n1032572\n867362\n659123\n478940\n340786\n240061\n168290\n117700\n82223\n57408\n40071\n27967\n19517\n13620\n9505\n6633\n4628\n3230\n2254\n211970\n-1039855\n-1003179\n908233\n-995156\n-151514\n-1035441\n-406551\n842701\n-367088\n-878600\n265567\n947437\n1034225\n871170\n662812\n481888\n342973\n241633\n169402\n118481\n82770\n57791\n40338\n28153\n19647\n13711\n9568\n6677\n4659\n3251\n2269\n978658\n-710058\n-1029381\n1030969\n-1044928\n21472\n-1009481\n-327427\n806298\n-328271\n-893942\n246136\n941362\n1035789\n874942\n666487\n484830\n345159\n243204\n170514\n119262\n83317\n58173\n40606\n28339\n19777\n13802\n9631\n6721\n4690\n3273\n2284\n845572\n-48292\n-816264\n1035792\n-1036214\n193872\n-970075\n-246177\n767345\n-288948\n-908613\n226614\n935118\n1037264\n878676\n670149\n487768\n347343\n244774\n171626\n120043\n83864\n58555\n40873\n28526\n19907\n13892\n9695\n6765\n4721\n3295\n2299\n-64928\n636052\n-413375\n922151\n-969502\n360975\n-917746\n-163330\n725967\n-249181\n-922603\n207010\n928709\n1038648\n882373\n673797\n490701\n349525\n246344\n172737\n120824\n84411\n58937\n41140\n28712\n20038\n13983\n9758\n6810\n4752\n3316\n2314\n-915734\n1023026\n85618\n703042\n-848526\n518215\n-853193\n-79423\n682294\n-209030\n-935901\n187329\n922135\n1039943\n886033\n677430\n493629\n351706\n247913\n173849\n121606\n84958\n59320\n41407\n28899\n20168\n14074\n9821\n6854\n4783\n3338\n2329\n-924618\n931711\n564706\n403525\n-680057\n661297\n-777275\n4999\n636463\n-168557\n-948498\n167581\n915396\n1041147\n889656\n681050\n496552\n353885\n249482\n174960\n122386\n85505\n59702\n41674\n29085\n20298\n14165\n9885\n6898\n4814\n3359\n2344\n-83413\n404798\n912507\n57857\n-473525\n786311\n-691004\n89389\n588621\n-127825\n-960383\n147771\n908495\n1042262\n893242\n684656\n499470\n356063\n251050\n176071\n123167\n86052\n60084\n41941\n29271\n20428\n14256\n9948\n6942\n4844\n3381\n2359\n834482\n-311367\n1048161\n-294429\n-240488\n889841\n-595528\n173199\n538917\n-86896\n-971548\n127908\n901432\n1043286\n896789\n688248\n502383\n358238\n252618\n177181\n123948\n86599\n60467\n42208\n29458\n20558\n14346\n10012\n6986\n4875\n3402\n2374\n985158\n-881961\n940129\n-613040\n6008\n969060\n-492119\n255885\n487510\n-45833\n-981985\n107998\n894209\n1044220\n900299\n691826\n505291\n360412\n254184\n178292\n124729\n87146\n60849\n42474\n29644\n20688\n14437\n10075\n7031\n4906\n3424\n2389\n230084\n-1040217\n613529\n-861538\n252169\n1021802\n-382156\n336910\n434562\n-4699\n-991685\n88048\n886827\n1045063\n903771\n695390\n508194\n362584\n255751\n179402\n125510\n87692\n61231\n42741\n29831\n20818\n14528\n10138\n7075\n4937\n3445\n2404\n-736528\n-712146\n144291\n-1011500\n484215\n1046627\n-267101\n415748\n380239\n36441\n-1000642\n68066\n879287\n1045816\n907204\n698939\n511092\n364754\n257316\n180512\n126290\n88239\n61613\n43008\n30017\n20948\n14619\n10202\n7119\n4968\n3467\n2419\n-1025980\n-51130\n-358493\n-1045775\n689159\n1042856\n-148489\n491888\n324715\n77526\n-1008848\n48059\n871591\n1046478\n910600\n702474\n513984\n366923\n258881\n181622\n127071\n88786\n61996\n43275\n30203\n21078\n14710\n10265\n7163\n4999\n3488\n2434\n-372150\n633790\n-777932\n-960444\n855530\n1010592\n-27899\n564836\n268164\n118491\n-1016298\n28035\n863740\n1047050\n913957\n705995\n516872\n369090\n260446\n182732\n127851\n89333\n62378\n43542\n30390\n21208\n14800\n10328\n7207\n5030\n3510\n2449\n623832\n1022399\n-1016511\n-765266\n974017\n950718\n93063\n634117\n210765\n159274\n-1022986\n8000\n855735\n1047531\n917275\n709501\n519754\n371255\n262010\n183841\n128632\n89879\n62760\n43809\n30576\n21338\n14891\n10392\n7252\n5060\n3531\n2464\n1046266\n933011\n-1018763\n-482564\n1037986\n864867\n212785\n699283\n152700\n199812\n-1028906\n-12037\n847578\n1047921\n920555\n712992\n522632\n373418\n263573\n184951\n129412\n90426\n63142\n44076\n30762\n21469\n14982\n10455\n7296\n5091\n3553\n2479\n506768\n407418\n-784165\n-144670\n1043858\n755388\n329673\n759911\n94153\n240042\n-1034056\n-32070\n839271\n1048220\n923796\n716469\n525504\n375579\n265135\n186060\n130192\n90973\n63525\n44343\n30949\n21599\n15073\n10518\n7340\n5122\n3574\n2494\n-498651\n-308653\n-367258\n209770\n991305\n625269\n442169\n815606\n35307\n279902\n-1038429\n-52092\n830814\n1048429\n926998\n719931\n528371\n377738\n266697\n187168\n130973\n91519\n63907\n44610\n31135\n21729\n15164\n10582\n7384\n5153\n3596\n2509\n-1045612\n-880421\n135032\n540218\n883266\n478068\n548776\n866008\n-23650\n319331\n-1042024\n-72094\n822209\n1048547\n930161\n723379\n531232\n379896\n268259\n188277\n131753\n92066\n64289\n44877\n31322\n21859\n15254\n10645\n7429\n5184\n3617\n2524\n-631243\n-1040571\n605929\n808881\n725791\n317804\n648072\n910789\n-82532\n358269\n-1044838\n-92070\n813458\n1048574\n933285\n726811\n534088\n382052\n269819\n189385\n132533\n92613\n64671\n45144\n31508\n21989\n15345\n10709\n7473\n5215\n3639\n2539\n363488\n-714229\n935955\n985032\n527692\n148858\n738736\n949659\n-141154\n396655\n-1046869\n-112012\n804563\n1048511\n936370\n730229\n536939\n384206\n271379\n190494\n133313\n93159\n65053\n45411\n31694\n22119\n15436\n10772\n7517\n5246\n3661\n2554\n1024030\n-53968\n1048382\n1048523\n300058\n-24156\n819560\n982365\n-199329\n434430\n-1048114\n-131914\n795524\n1048356\n939414\n733631\n539785\n386358\n272939\n191602\n134093\n93706\n65436\n45678\n31881\n22249\n15527\n10835\n7561\n5276\n3682\n2569\n743083\n631524\n917073\n992093\n55629\n-196509\n889467\n1008696\n-256874\n471537\n-1048573\n-151767\n786344\n1048111\n942420\n737019\n542625\n388508\n274498\n192709\n134873\n94252\n65818\n45945\n32067\n22379\n15618\n10899\n7605\n5307\n3704\n2585\n-221051\n1021764\n572555\n822196\n-191914\n-363493\n947526\n1028480\n-313607\n507917\n-1048246\n-171565\n777025\n1047775\n945385\n740391\n545459\n390656\n276056\n193817\n135652\n94799\n66200\n46212\n32254\n22509\n15708\n10962\n7650\n5338\n3725\n2600\n-981952\n934304\n94926\n558264\n-428715\n-520547\n992963\n1041588\n-369348\n543515\n-1047133\n-191300\n767567\n1047348\n948311\n743748\n548289\n392803\n277613\n194924\n136432\n95345\n66582\n46479\n32440\n22639\n15799\n11025\n7694\n5369\n3747\n2615\n-840051\n410035\n-404773\n230482\n-641520\n-663378\n1025174\n1047936\n-423922\n578277\n-1045235\n-210966\n757972\n1046831\n951197\n747090\n551112\n394947\n279170\n196031\n137212\n95892\n66964\n46746\n32626\n22770\n15890\n11089\n7738\n5400\n3768\n2630\n74189\n-305936\n-810367\n-123660\n-818418\n-788084\n1043729\n1047483\n-477155\n612148\n-1042553\n-230554\n748243\n1046223\n954042\n750417\n553930\n397090\n280726\n197138\n137991\n96438\n67346\n47013\n32813\n22900\n15981\n11152\n7782\n5431\n3790\n2645\n920220\n-878874\n-1027560\n-463659\n-949509\n-891258\n1048381\n1040232\n-528880\n645076\n-1039090\n-250058\n738380\n1045524\n956848\n753728\n556743\n399230\n282281\n198245\n138771\n96984\n67728\n47280\n32999\n23030\n16071\n11215\n7827\n5462\n3811\n2660\n920204\n-1040918\n-1005858\n-750628\n-1027455\n-970082\n1039068\n1026229\n-578934\n677012\n-1034847\n-269471\n728387\n1044735\n959613\n757023\n559550\n401369\n283836\n199351\n139550\n97531\n68110\n47547\n33185\n23160\n16162\n11279\n7871\n5492\n3833\n2675\n74158\n-716307\n-750306\n-951748\n-1047893\n-1022401\n1015915\n1005565\n-627156\n707905\n-1029828\n-288785\n718264\n1043855\n962337\n760304\n562352\n403506\n285390\n200457\n140329\n98077\n68493\n47814\n33372\n23290\n16253\n11342\n7915\n5523\n3854\n2690\n-840069\n-56806\n-320316\n-1044015\n-1009680\n-1046787\n979229\n978375\n-673396\n737707\n-1024037\n-307994\n708013\n1042885\n965021\n763568\n565148\n405641\n286944\n201563\n141108\n98623\n68875\n48081\n33558\n23420\n16344\n11405\n7959\n5554\n3876\n2705\n-981940\n629253\n184143\n-1016876\n-914953\n-1042573\n929499\n944835\n-717507\n766374\n-1017478\n-327091\n697636\n1041825\n967664\n766817\n567938\n407773\n288497\n202669\n141888\n99170\n69257\n48348\n33745\n23550\n16435\n11469\n8003\n5585\n3897\n2720\n-221020\n1021122\n645791\n-873437\n-769015\n-1009873\n867388\n905163\n-759350\n793861\n-1010157\n-346068\n687135\n1040674\n970267\n770050\n570722\n409904\n290049\n203774\n142667\n99716\n69639\n48614\n33931\n23680\n16525\n11532\n8048\n5616\n3919\n2735\n743105\n935591\n957300\n-630101\n-580035\n-949582\n793723\n859616\n-798792\n820126\n-1002077\n-364919\n676512\n1039433\n972828\n773267\n573501\n412033\n291600\n204879\n143446\n100262\n70021\n48881\n34117\n23810\n16616\n11596\n8092\n5647\n3940\n2750\n1024023\n412649\n1046249\n-314700\n-358590\n-863347\n709486\n808490\n-835709\n845128\n-993247\n-383636\n665769\n1038103\n975349\n776469\n576274\n414160\n293151\n205984\n144224\n100808\n70403\n49148\n34304\n23940\n16707\n11659\n8136\n5678\n3962\n2765\n363459\n-303216\n891957\n36694\n-117074\n-753524\n615797\n752117\n-869984\n868829\n-983671\n-402213\n654907\n1036682\n977828\n779654\n579042\n416285\n294701\n207089\n145003\n101355\n70785\n49415\n34490\n24070\n16798\n11722\n8180\n5708\n3984\n2780\n-631268\n-877321\n530296\n383891\n130995\n-623113\n513906\n690862\n-901508\n891191\n-973358\n-420644\n643929\n1035172\n980266\n782824\n581803\n418407\n296250\n208194\n145782\n101901\n71167\n49682\n34676\n24200\n16889\n11786\n8224\n5739\n4005\n2795\n-1045610\n-1041257\n45347\n687182\n371732\n-475677\n405170\n625123\n-930183\n912182\n-962315\n-438921\n632836\n1033571\n982663\n785977\n584559\n420528\n297799\n209298\n146561\n102447\n71549\n49949\n34863\n24331\n16979\n11849\n8269\n5770\n4027\n2810\n-498623\n-718380\n-450144\n911879\n591662\n-315245\n291037\n555327\n-955916\n931768\n-950551\n-457037\n621631\n1031882\n985018\n789114\n587309\n422647\n299346\n210402\n147339\n102993\n71931\n50216\n35049\n24461\n17070\n11912\n8313\n5801\n4048\n2825\n506796\n-59643\n-840983\n1032283\n778477\n-146200\n173026\n481927\n-978628\n949920\n-938073\n-474987\n610315\n1030103\n987332\n792235\n590053\n424763\n300894\n211506\n148118\n103539\n72313\n50483\n35235\n24591\n17161\n11976\n8357\n5832\n4070\n2840\n1046268\n626978\n-1036302\n1034624\n921719\n26839\n52712\n405398\n-998245\n966609\n-924893\n-492763\n598891\n1028235\n989605\n795340\n592791\n426878\n302440\n212609\n148896\n104085\n72695\n50750\n35422\n24721\n17252\n12039\n8401\n5863\n4091\n2855\n623807\n1020472\n-990694\n918634\n1013372\n199145\n-68305\n326239\n-1014707\n981809\n-911018\n-510360\n587360\n1026278\n991835\n798428\n595523\n428990\n303986\n213712\n149674\n104631\n73077\n51017\n35608\n24851\n17343\n12102\n8446\n5894\n4113\n2870\n-372180\n936871\n-714761\n697579\n1048305\n366010\n-188413\n244962\n-1027961\n995498\n-896461\n-527770\n575724\n1024232\n994024\n801500\n598249\n431101\n305531\n214815\n150453\n105177\n73459\n51284\n35795\n24981\n17433\n12166\n8490\n5924\n4134\n2885\n-1025986\n415260\n-272655\n396740\n1024564\n522875\n-306010\n162096\n-1037965\n1007654\n-881232\n-544987\n563987\n1022097\n996170\n804556\n600970\n433209\n307075\n215918\n151231\n105723\n73841\n51550\n35981\n25111\n17524\n12229\n8534\n5955\n4156\n2900\n-736505\n-300495\n232840\n50526\n943476\n665454\n-419531\n78177\n-1044687\n1018259\n-865341\n-562005\n552149\n1019874\n998275\n807595\n603684\n435315\n308619\n217021\n152009\n106269\n74223\n51817\n36167\n25241\n17615\n12293\n8578\n5986\n4177\n2915\n230115\n-875761\n684203\n-301467\n809581\n789852\n-527464\n-6249\n-1048107\n1027295\n-848802\n-578818\n540213\n1017562\n1000338\n810617\n606392\n437419\n310162\n218123\n152787\n106815\n74605\n52084\n36354\n25371\n17706\n12356\n8622\n6017\n4199\n2930\n985169\n-1041588\n976497\n-618980\n630373\n892670\n-628371\n-90634\n-1048213\n1034750\n-831626\n-595420\n528181\n1015163\n1002358\n813623\n609094\n439521\n311704\n219225\n153565\n107360\n74987\n52351\n36540\n25501\n17797\n12419\n8667\n6048\n4220\n2945\n834463\n-720447\n1041767\n-865701\n415882\n971098\n-720908\n-174432\n-1045005\n1040612\n-813827\n-611804\n516054\n1012675\n1004336\n816612\n611790\n441621\n313245\n220327\n154342\n107906\n75369\n52618\n36726\n25631\n17887\n12483\n8711\n6079\n4242\n2960\n-83444\n-62480\n864838\n-1013410\n178114\n1022994\n-803842\n-257097\n-1038494\n1044871\n-795417\n-627965\n503837\n1010100\n1006272\n819585\n614480\n443718\n314786\n221428\n155120\n108452\n75751\n52885\n36913\n25762\n17978\n12546\n8755\n6110\n4263\n2975\n-924633\n624698\n486845\n-1045214\n-69623\n1046940\n-876068\n-338093\n-1028699\n1047522\n-776411\n-643896\n491529\n1007437\n1008165\n822540\n617164\n445814\n316326\n222530\n155898\n108998\n76133\n53152\n37099\n25892\n18069\n12609\n8799\n6140\n4285\n2990\n-915719\n1019815\n-4334\n-957475\n-313464\n1042282\n-936625\n-416895\n-1015653\n1048560\n-756823\n-659593\n479134\n1004687\n1010016\n825479\n619842\n447907\n317865\n223631\n156675\n109544\n76515\n53419\n37285\n26022\n18160\n12673\n8844\n6171\n4307\n3005\n-64897\n938143\n-494505\n-760230\n-539760\n1009147\n-984706\n-492992\n-999395\n1047983\n-736668\n-675048\n466654\n1001850\n1011825\n828400\n622513\n449998\n319403\n224731\n157453\n110089\n76896\n53685\n37472\n26152\n18250\n12736\n8888\n6202\n4328\n3020\n845591\n417868\n-869709\n-476035\n-735844\n948441\n-1019670\n-565888\n-979978\n1045792\n-715960\n-690257\n454091\n998927\n1013590\n831305\n625178\n452087\n320941\n225832\n158230\n110635\n77278\n53952\n37658\n26282\n18341\n12799\n8932\n6233\n4350\n3035\n978646\n-297771\n-1042717\n-137396\n-890743\n861821\n-1041052\n-635112\n-957463\n1041992\n-694715\n-705214\n441447\n995916\n1015313\n834193\n627837\n454174\n322477\n226932\n159007\n111181\n77660\n54219\n37844\n26412\n18432\n12863\n8976\n6264\n4371\n3050\n211939\n-874195\n-973305\n216957\n-995786\n751655\n-1048566\n-700214\n-931920\n1036587\n-672949\n-719914\n428725\n992820\n1016993\n837063\n630490\n456258\n324014\n228032\n159785\n111726\n78042\n54486\n38031\n26542\n18523\n12926\n9020\n6295\n4393\n3065\n-749624\n-1041912\n-677612\n546496\n-1045093\n620952\n-1042113\n-760771\n-903432\n1029586\n-650678\n-734350\n415926\n989637\n1018630\n839916\n633136\n458341\n325549\n229132\n160562\n112272\n78424\n54753\n38217\n26672\n18614\n12990\n9065\n6326\n4414\n3080\n-1021986\n-722509\n-224382\n813532\n-1035906\n473283\n-1021779\n-816391\n-872087\n1020999\n-627920\n-748519\n403054\n986369\n1020224\n842752\n635776\n460421\n327083\n230231\n161339\n112817\n78806\n55020\n38403\n26802\n18704\n13053\n9109\n6356\n4436\n3095\n-354739\n-65316\n281015\n987524\n-968738\n312684\n-987834\n-866712\n-837985\n1010841\n-604691\n-762414\n390110\n983016\n1021776\n845571\n638410\n462499\n328617\n231330\n162116\n113363\n79187\n55286\n38590\n26932\n18795\n13116\n9153\n6387\n4457\n3110\n638654\n622413\n721078\n1048571\n-847348\n143542\n-940731\n-911408\n-801235\n999126\n-581008\n-776030\n377096\n979577\n1023284\n848372\n641037\n464574\n330150\n232429\n162892\n113908\n79569\n55553\n38776\n27062\n18886\n13180\n9197\n6418\n4479\n3125\n1044871\n1019150\n993500\n989692\n-678532\n-29522\n-881096\n-950188\n-761951\n985873\n-556890\n-789364\n364016\n976054\n1024749\n851156\n643658\n466647\n331682\n233528\n163669\n114454\n79951\n55820\n38962\n27192\n18977\n13243\n9241\n6449\n4500\n3140\n490438\n939409\n1034945\n817620\n-471737\n-201780\n-809726\n-982802\n-720258\n971102\n-532354\n-802409\n350870\n972445\n1026170\n853923\n646273\n468718\n333214\n234626\n164446\n114999\n80333\n56087\n39149\n27322\n19068\n13306\n9286\n6480\n4522\n3156\n-514901\n420472\n835777\n552037\n-238538\n-368524\n-727569\n-1009036\n-676288\n954835\n-507419\n-815161\n337663\n968753\n1027549\n856672\n648881\n470787\n334745\n235724\n165222\n115545\n80715\n56354\n39335\n27453\n19158\n13370\n9330\n6511\n4543\n3171\n-1046843\n-295046\n442301\n223316\n8011\n-525200\n-635721\n-1028722\n-630180\n937098\n-482103\n-827615\n324395\n964977\n1028884\n859403\n651482\n472854\n336274\n236822\n165999\n116090\n81096\n56621\n39521\n27583\n19249\n13433\n9374\n6542\n4565\n3186\n-616322\n-872623\n-54004\n-130946\n254112\n-667526\n-535405\n-1041731\n-582080\n917919\n-456426\n-839767\n311069\n961117\n1030176\n862116\n654077\n474918\n337803\n237920\n166775\n116635\n81478\n56887\n39708\n27713\n19340\n13496\n9418\n6572\n4586\n3201\n380842\n-1042228\n-537755\n-470231\n485991\n-791615\n-427957\n-1047979\n-532139\n897326\n-430407\n-851613\n297688\n957175\n1031424\n864812\n656665\n476980\n339332\n239017\n167551\n117181\n81860\n57154\n39894\n27843\n19431\n13560\n9463\n6603\n4608\n3216\n1027862\n-724566\n-896483\n-755735\n690667\n-894075\n-314809\n-1047426\n-480516\n875352\n-404065\n-863147\n284255\n953149\n1032629\n867491\n659247\n479039\n340859\n240114\n168328\n117726\n82242\n57421\n40080\n27973\n19522\n13623\n9507\n6634\n4630\n3231\n729870\n-68152\n-1046790\n-954805\n856687\n-972107\n-197467\n-1040074\n-427374\n852029\n-377420\n-874366\n270770\n949040\n1033790\n870151\n661823\n481096\n342386\n241211\n169104\n118271\n82623\n57688\n40267\n28103\n19612\n13686\n9551\n6665\n4651\n3246\n-239161\n620124\n-953731\n-1044673\n974757\n-1023580\n-77495\n-1025971\n-372881\n827395\n-350492\n-885267\n257238\n944850\n1034908\n872794\n664391\n483151\n343912\n242307\n169880\n118816\n83005\n57955\n40453\n28233\n19703\n13750\n9595\n6696\n4673\n3261\n-988309\n1018478\n-638941\n-1015060\n1038268\n-1047087\n43510\n-1005210\n-317210\n801487\n-323301\n-895843\n243660\n940577\n1035982\n875418\n666953\n485204\n345437\n243403\n170655\n119361\n83387\n58221\n40639\n28363\n19794\n13813\n9639\n6727\n4694\n3276\n-828810\n940668\n-175604\n-869354\n1043667\n-1041985\n163935\n-977925\n-260535\n774345\n-295868\n-906093\n230038\n936224\n1037012\n878025\n669509\n487254\n346961\n244499\n171431\n119907\n83769\n58488\n40826\n28493\n19885\n13877\n9684\n6758\n4716\n3291\n92693\n423074\n328558\n-624219\n990650\n-1008415\n282176\n-944293\n-203036\n746010\n-268213\n-916012\n216376\n931789\n1037999\n880613\n672057\n489301\n348484\n245595\n172207\n120452\n84150\n58755\n41012\n28623\n19975\n13940\n9728\n6788\n4737\n3306\n928974\n-292318\n756335\n-307690\n882185\n-947293\n396658\n-904532\n-144896\n716527\n-240356\n-925596\n202675\n927273\n1038942\n883184\n674599\n491347\n350007\n246690\n172983\n120997\n84532\n59022\n41198\n28753\n20066\n14003\n9772\n6819\n4759\n3321\n911161\n-871044\n1008272\n44029\n724344\n-860289\n505857\n-858900\n-86298\n685941\n-212320\n-934842\n188938\n922677\n1039841\n885736\n677134\n493390\n351528\n247785\n173758\n121542\n84914\n59289\n41385\n28883\n20157\n14067\n9816\n6850\n4780\n3336\n55631\n-1042536\n1025799\n390712\n525960\n-749781\n608318\n-807694\n-27427\n654298\n-184124\n-943747\n175168\n918002\n1040696\n888270\n679663\n495430\n353049\n248880\n174533\n122087\n85295\n59555\n41571\n29013\n20248\n14130\n9860\n6881\n4802\n3351\n-851046\n-726618\n804839\n692709\n298138\n-618787\n702675\n-751245\n31531\n621648\n-155790\n-952307\n161366\n913246\n1041507\n890786\n682184\n497469\n354569\n249975\n175309\n122632\n85677\n59822\n41757\n29144\n20339\n14193\n9905\n6912\n4823\n3366\n-975275\n-70988\n396764\n915480\n53629\n-470887\n787673\n-689921\n90389\n588041\n-127340\n-960520\n147536\n908412\n1042274\n893284\n684699\n499504\n356088\n251069\n176084\n123177\n86058\n60089\n41944\n29274\n20429\n14257\n9949\n6943\n4845\n3381\n-202841\n617830\n-103554\n1033547\n-193882\n-310121\n862179\n-624119\n148962\n553528\n-98794\n-968382\n133679\n903499\n1042998\n895763\n687207\n501538\n357607\n252163\n176859\n123722\n86440\n60356\n42130\n29404\n20520\n14320\n9993\n6974\n4866\n3396\n756085\n1017798\n-579797\n1033406\n-430541\n-140883\n925200\n-554266\n207063\n518163\n-70174\n-975890\n119799\n898508\n1043677\n898224\n689708\n503568\n359124\n253256\n177634\n124266\n86822\n60622\n42316\n29534\n20611\n14383\n10037\n7004\n4888\n3411\n1019869\n941921\n-921244\n915072\n-643103\n32205\n975897\n-480816\n264510\n482001\n-41501\n-983042\n105897\n893439\n1044313\n900667\n692202\n505597\n360641\n254350\n178409\n124811\n87203\n60889\n42503\n29664\n20702\n14447\n10082\n7035\n4909\n3426\n345991\n425673\n-1048513\n692081\n-819669\n204413\n1013594\n-404246\n321121\n445096\n-12797\n-989835\n91977\n888292\n1044904\n903091\n694690\n507623\n362157\n255443\n179184\n125356\n87585\n61156\n42689\n29794\n20793\n14510\n10126\n7066\n4931\n3441\n-645990\n-289588\n-932015\n389936\n-950357\n371036\n1037790\n-325051\n376716\n407505\n15917\n-996266\n78040\n883069\n1045451\n905496\n697170\n509646\n363672\n256535\n179959\n125901\n87966\n61423\n42875\n29924\n20883\n14573\n10170\n7097\n4953\n3456\n-1044051\n-869459\n-598835\n43193\n-1027853\n527521\n1048163\n-243747\n431121\n369288\n44618\n-1002334\n64089\n877769\n1045955\n907883\n699643\n511667\n365186\n257628\n180733\n126446\n88348\n61689\n43062\n30054\n20974\n14637\n10214\n7128\n4974\n3471\n-482216\n-1042837\n-126433\n-308490\n-1047819\n669594\n1044573\n-160861\n484162\n330502\n73286\n-1008035\n50127\n872393\n1046414\n910251\n702110\n513686\n366699\n258720\n181508\n126990\n88730\n61956\n43248\n30184\n21065\n14700\n10258\n7159\n4996\n3486\n522966\n-728664\n375364\n-624890\n-1009137\n793372\n1027070\n-76931\n535673\n291206\n101899\n-1013369\n36156\n866942\n1046829\n912600\n704569\n515702\n368211\n259812\n182282\n127535\n89111\n62223\n43434\n30314\n21156\n14764\n10303\n7190\n5017\n3501\n1047335\n-73823\n789892\n-869821\n-913973\n895474\n995885\n7499\n585491\n251463\n130436\n-1018332\n22178\n861416\n1047200\n914931\n707021\n517715\n369723\n260903\n183056\n128080\n89493\n62490\n43620\n30444\n21246\n14827\n10347\n7220\n5039\n3516\n608789\n615532\n1020780\n-1015270\n-767652\n973110\n951435\n91879\n633457\n211332\n158875\n-1022924\n8197\n855815\n1047526\n917243\n709466\n519726\n371233\n261994\n183831\n128624\n89874\n62756\n43807\n30574\n21337\n14890\n10391\n7251\n5060\n3531\n-389475\n1017111\n1014349\n-1044601\n-578366\n1024159\n894311\n175664\n679421\n170876\n187195\n-1027142\n-5786\n850139\n1047809\n919536\n711904\n521735\n372743\n263085\n184605\n129169\n90256\n63023\n43993\n30704\n21428\n14954\n10435\n7282\n5082\n3546\n-1029657\n943166\n772094\n-954459\n-356707\n1047226\n825274\n258308\n723236\n130157\n215374\n-1030985\n-19768\n844391\n1048047\n921810\n714335\n523741\n374252\n264176\n185379\n129713\n90637\n63290\n44179\n30834\n21519\n15017\n10480\n7313\n5103\n3561\n-723178\n428268\n350336\n-755155\n-115083\n1041681\n745245\n339276\n764766\n89237\n243392\n-1034451\n-33746\n838569\n1048241\n924066\n716759\n525744\n375760\n265266\n186152\n130258\n91019\n63557\n44366\n30964\n21610\n15080\n10524\n7344\n5125\n3576\n248188\n-286855\n-152871\n-469484\n132982\n1007676\n655289\n418042\n803877\n48180\n271228\n-1037540\n-47718\n832674\n1048391\n926302\n719176\n527745\n377267\n266356\n186926\n130802\n91400\n63823\n44552\n31095\n21700\n15144\n10568\n7375\n5146\n3591\n991371\n-867867\n-620537\n-130116\n373604\n946139\n556604\n494094\n840447\n7049\n298860\n-1040250\n-61682\n826708\n1048497\n928519\n721585\n529743\n378773\n267446\n187700\n131347\n91781\n64090\n44738\n31225\n21791\n15207\n10612\n7406\n5168\n3606\n823092\n-1043130\n-943936\n224133\n593315\n858751\n450504\n566940\n874360\n-34093\n326268\n-1042580\n-75635\n820669\n1048558\n930718\n723988\n531738\n380278\n268535\n188473\n131891\n92163\n64357\n44925\n31355\n21882\n15270\n10656\n7436\n5189\n3621\n-101934\n-730704\n-1047881\n552747\n779817\n747902\n338404\n636106\n905509\n-75183\n353431\n-1044529\n-89575\n814560\n1048576\n932897\n726383\n533732\n381783\n269624\n189247\n132435\n92544\n64623\n45111\n31485\n21973\n15334\n10701\n7467\n5211\n3636\n-933242\n-76657\n-908206\n818144\n922673\n616618\n221797\n701144\n933795\n-116156\n380329\n-1046097\n-103498\n808380\n1048549\n935057\n728771\n535722\n383286\n270713\n190020\n132980\n92926\n64890\n45297\n31615\n22063\n15397\n10745\n7498\n5232\n3651\n-906532\n613229\n-557384\n989968\n1013885\n468487\n102234\n761631\n959129\n-156951\n406942\n-1047283\n-117403\n802131\n1048478\n937198\n731151\n537710\n384789\n271802\n190794\n133524\n93307\n65157\n45483\n31745\n22154\n15461\n10789\n7529\n5254\n3666\n-46360\n1016416\n-76977\n1048568\n1048349\n307556\n-18690\n817175\n981431\n-197505\n433250\n-1048087\n-131287\n795811\n1048362\n939319\n733524\n539695\n386290\n272890\n191567\n134068\n93689\n65424\n45670\n31875\n22245\n15524\n10833\n7560\n5276\n3681\n856435\n944404\n421326\n987242\n1024136\n138222\n-139365\n867415\n1000630\n-237754\n459233\n-1048507\n-145148\n789423\n1048203\n941421\n735890\n541678\n387791\n273978\n192340\n134612\n94070\n65690\n45856\n32005\n22336\n15587\n10877\n7591\n5297\n3696\n971828\n430860\n821676\n813005\n942601\n-34888\n-258183\n912025\n1016666\n-277637\n484872\n-1048545\n-158983\n782967\n1047999\n943504\n738248\n543658\n389291\n275065\n193113\n135156\n94451\n65957\n46042\n32135\n22427\n15651\n10922\n7622\n5319\n3711\n193727\n-284121\n1030996\n545783\n808307\n-207045\n-373563\n950716\n1029487\n-317092\n510147\n-1048200\n-172790\n776442\n1047751\n945568\n740600\n545635\n390790\n276152\n193885\n135701\n94833\n66224\n46229\n32265\n22517\n15714\n10966\n7652\n5340\n3726\n-762486\n-866269\n1000622\n216139\n628771\n-373545\n-483966\n983237\n1039054\n-356060\n535040\n-1047472\n-186566\n769851\n1047459\n947612\n742943\n547609\n392287\n277239\n194658\n136245\n95214\n66490\n46415\n32395\n22608\n15777\n11010\n7683\n5362\n3742\n-1017672\n-1043415\n737615\n-138225\n414043\n-529840\n-587923\n1009376\n1045336\n-394479\n559531\n-1046362\n-200309\n763192\n1047123\n949637\n745280\n549581\n393784\n278326\n195431\n136789\n95595\n66757\n46601\n32525\n22699\n15841\n11054\n7714\n5383\n3757\n-337216\n-732740\n303121\n-476780\n176140\n-671658\n-684049\n1028964\n1048313\n-432291\n583603\n-1044870\n-214016\n756468\n1046742\n951642\n747608\n551551\n395281\n279412\n196203\n137333\n95977\n67024\n46787\n32655\n22790\n15904\n11099\n7745\n5405\n3772\n653276\n-79491\n-201844\n-760805\n-71622\n-795125\n-771063\n1041873\n1047976\n-469437\n607237\n-1042996\n-227685\n749678\n1046317\n953627\n749930\n553517\n396776\n280498\n196976\n137877\n96358\n67290\n46974\n32785\n22881\n15967\n11143\n7776\n5426\n3787\n1043148\n610922\n-659884\n-957816\n-315375\n-896868\n-847806\n1048021\n1044326\n-505860\n630416\n-1040741\n-241314\n742824\n1045849\n955593\n752243\n555481\n398270\n281583\n197748\n138421\n96739\n67557\n47160\n32915\n22971\n16031\n11187\n7807\n5448\n3802\n473955\n1015714\n-964508\n-1045280\n-541476\n-974107\n-913255\n1047366\n1037374\n-541505\n653122\n-1038106\n-254900\n735904\n1045336\n957540\n754550\n557443\n399763\n282669\n198520\n138965\n97120\n67824\n47346\n33045\n23062\n16094\n11231\n7838\n5469\n3817\n-530990\n945635\n-1044896\n-1013195\n-737270\n-1024732\n-966540\n1039914\n1027143\n-576315\n675339\n-1035092\n-268440\n728922\n1044779\n959466\n756849\n559401\n401255\n283754\n199292\n139508\n97502\n68090\n47533\n33176\n23153\n16157\n11275\n7868\n5491\n3832\n-1047746\n433449\n-882357\n-865229\n-891798\n-1047359\n-1006950\n1025713\n1013664\n-610239\n697049\n-1031700\n-281933\n721876\n1044178\n961373\n759140\n561357\n402747\n284838\n200064\n140052\n97883\n68357\n47719\n33306\n23244\n16221\n11320\n7899\n5512\n3847\n-601209\n-281385\n-514682\n-618305\n-996411\n-1041371\n-1033947\n1004854\n996981\n-643222\n718236\n-1027932\n-295376\n714767\n1043533\n963261\n761423\n563310\n404237\n285922\n200836\n140596\n98264\n68623\n47905\n33436\n23334\n16284\n11364\n7930\n5534\n3862\n398077\n-864665\n-27349\n-300666\n-1045255\n-1006930\n-1047172\n977473\n977145\n-675216\n738885\n-1023788\n-308766\n707597\n1042844\n965128\n763699\n565260\n405727\n287006\n201608\n141140\n98645\n68890\n48091\n33566\n23425\n16348\n11408\n7961\n5555\n3877\n1031372\n-1043693\n466343\n51361\n-1035594\n-944978\n-1046448\n943749\n954221\n-706169\n758979\n-1019270\n-322101\n700365\n1042111\n966976\n765968\n567208\n407215\n288090\n202379\n141683\n99027\n69157\n48278\n33696\n23516\n16411\n11452\n7992\n5577\n3892\n716429\n-734770\n851615\n397514\n-967969\n-857208\n-1031784\n903899\n928280\n-736036\n778505\n-1014380\n-335379\n693073\n1041334\n968804\n768228\n569152\n408703\n289173\n203151\n142227\n99408\n69423\n48464\n33826\n23607\n16474\n11496\n8023\n5598\n3907\n-257196\n-82324\n1038897\n698203\n-846167\n-746018\n-1003377\n858183\n899404\n-764769\n797447\n-1009119\n-348597\n685721\n1040513\n970612\n770481\n571094\n410189\n290256\n203922\n142771\n99789\n69690\n48650\n33956\n23698\n16538\n11541\n8054\n5620\n3922\n-994356\n608610\n984648\n919037\n-677003\n-614445\n-961605\n806896\n867685\n-792324\n815791\n-1003491\n-361754\n678309\n1039649\n972400\n772727\n573034\n411675\n291339\n204693\n143314\n100170\n69957\n48836\n34086\n23788\n16601\n11585\n8084\n5642\n3937\n-817310\n1015005\n701480\n1034760\n-469947\n-466084\n-907023\n750373\n833223\n-818660\n833523\n-997495\n-374845\n670839\n1038740\n974168\n774964\n574970\n413159\n292421\n205464\n143858\n100551\n70223\n49023\n34216\n23879\n16664\n11629\n8115\n5663\n3952\n111167\n946860\n255226\n1032136\n-236588\n-304989\n-840360\n688980\n796126\n-843735\n850630\n-991136\n-387871\n663310\n1037788\n975916\n777194\n576904\n414643\n293503\n206235\n144401\n100932\n70490\n49209\n34346\n23970\n16728\n11673\n8146\n5685\n3967\n937438\n436035\n-250365\n911466\n10014\n-135561\n-762503\n623114\n756513\n-867511\n867099\n-984414\n-400827\n655724\n1036791\n977644\n779416\n578835\n416126\n294585\n207006\n144945\n101314\n70756\n49395\n34476\n24061\n16791\n11718\n8177\n5706\n3982\n901832\n-278646\n-697749\n686550\n256055\n37571\n-674489\n553205\n714508\n-889951\n882919\n-977333\n-413712\n648082\n1035752\n979352\n781630\n580763\n417607\n295666\n207777\n145488\n101695\n71023\n49581\n34606\n24151\n16854\n11762\n8208\n5728\n3997\n37086\n-863054\n-982914\n383113\n487764\n209676\n-577490\n479705\n670244\n-911021\n898076\n-969895\n-426524\n640383\n1034668\n981040\n783837\n582688\n419088\n296747\n208548\n146032\n102076\n71290\n49768\n34736\n24242\n16918\n11806\n8239\n5749\n4012\n-861756\n-1043963\n-1039564\n35858\n692173\n376052\n-472799\n403092\n623861\n-930688\n912559\n-962103\n-439259\n632629\n1033541\n982707\n786035\n584610\n420567\n297827\n209318\n146575\n102457\n71556\n49954\n34866\n24333\n16981\n11850\n8270\n5771\n4027\n-968304\n-736795\n-854527\n-315497\n857840\n532154\n-361810\n323863\n575505\n-948922\n926359\n-953960\n-451917\n624819\n1032370\n984355\n788226\n586530\n422046\n298908\n210089\n147118\n102838\n71823\n50140\n34996\n24424\n17044\n11894\n8300\n5792\n4042\n-184598\n-85157\n-470824\n-630769\n975493\n673717\n-246002\n242531\n525331\n-965695\n939463\n-945468\n-464494\n616956\n1031156\n985982\n790409\n588446\n423524\n299987\n210859\n147662\n103219\n72089\n50327\n35126\n24515\n17108\n11939\n8331\n5814\n4057\n768827\n606294\n22341\n-873899\n1038546\n796872\n-126917\n159626\n473495\n-980982\n951864\n-936631\n-476988\n609040\n1029898\n987589\n792584\n590360\n425000\n301067\n211629\n148205\n103600\n72356\n50513\n35256\n24605\n17171\n11983\n8362\n5835\n4072\n1015396\n1014288\n510312\n-1017080\n1043471\n898256\n-6141\n75684\n420163\n-994757\n963550\n-927452\n-489398\n601070\n1028597\n989176\n794751\n592271\n426476\n302146\n212399\n148748\n103981\n72622\n50699\n35386\n24696\n17235\n12027\n8393\n5857\n4087\n328414\n948077\n879641\n-1043937\n989992\n975097\n114717\n-8748\n365502\n-1007001\n974514\n-917934\n-501721\n593049\n1027252\n990742\n796910\n594179\n427951\n303225\n213169\n149291\n104362\n72889\n50885\n35516\n24787\n17298\n12071\n8424\n5878\n4102\n-660510\n438618\n1044464\n-951397\n881101\n1025298\n234046\n-93124\n309686\n-1017695\n984747\n-908081\n-513954\n584976\n1025864\n992288\n799061\n596084\n429424\n304303\n213939\n149834\n104743\n73156\n51072\n35646\n24878\n17361\n12115\n8455\n5900\n4117\n-1042164\n-275906\n966462\n-750044\n722895\n1047485\n350258\n-176896\n252891\n-1026822\n994242\n-897897\n-526096\n576853\n1024433\n993814\n801204\n597986\n430897\n305382\n214709\n150377\n105124\n73422\n51258\n35777\n24968\n17425\n12160\n8486\n5921\n4132\n-465658\n-861437\n663769\n-462909\n524227\n1041053\n461804\n-259519\n195296\n-1034367\n1002992\n-887384\n-538145\n568679\n1022959\n995319\n803339\n599886\n432369\n306459\n215478\n150920\n105505\n73689\n51444\n35907\n25059\n17488\n12204\n8516\n5943\n4147\n538973\n-1044226\n206757\n-122829\n296217\n1006178\n567199\n-340458\n137084\n-1040320\n1010989\n-876548\n-550098\n560457\n1021441\n996804\n805466\n601782\n433839\n307537\n216248\n151463\n105886\n73955\n51630\n36037\n25150\n17551\n12248\n8547\n5965\n4162\n1048074\n-738814\n-298323\n231298\n51629\n943812\n665039\n-419188\n78438\n-1044671\n1018228\n-865391\n-561953\n552186\n1019881\n998269\n807585\n603676\n435309\n308614\n217017\n152006\n106267\n74222\n51816\n36167\n25241\n17615\n12292\n8578\n5986\n4177\n593581\n-87989\n-734047\n558971\n-195850\n855660\n754020\n-495196\n19545\n-1047414\n1024704\n-853919\n-573708\n543867\n1018277\n999712\n809696\n605566\n436777\n309691\n217787\n152549\n106648\n74488\n52003\n36297\n25332\n17678\n12337\n8609\n6008\n4192\n-406648\n603973\n-999113\n822715\n-432367\n744129\n832957\n-567991\n-39411\n-1048544\n1030411\n-842135\n-585361\n535501\n1016631\n1001136\n811799\n607454\n438245\n310767\n218556\n153092\n107029\n74755\n52189\n36427\n25422\n17741\n12381\n8640\n6029\n4207\n-1033006\n1013563\n-1031898\n992363\n-644684\n612268\n900799\n-637099\n-98241\n-1048059\n1035346\n-830043\n-596910\n527088\n1014941\n1002539\n813894\n609338\n439711\n311843\n219325\n153635\n107410\n75021\n52375\n36557\n25513\n17805\n12425\n8671\n6051\n4222\n-709624\n949288\n-824779\n1048514\n-820917\n463678\n956642\n-702073\n-156762\n-1045961\n1039504\n-817648\n-608353\n518630\n1013209\n1003921\n815981\n611220\n441176\n312919\n220094\n154178\n107791\n75288\n52561\n36687\n25604\n17868\n12469\n8701\n6072\n4237\n266184\n441198\n-425908\n984745\n-951202\n302420\n999742\n-762489\n-214786\n-1042252\n1042883\n-804954\n-619687\n510127\n1011434\n1005282\n818059\n613099\n442641\n313994\n220862\n154720\n108172\n75554\n52748\n36817\n25695\n17931\n12513\n8732\n6094\n4252\n997263\n-273163\n71981\n808349\n-1028248\n132899\n1029525\n-817957\n-272132\n-1036938\n1045480\n-791967\n-630912\n501580\n1009617\n1006623\n820129\n614974\n444104\n315069\n221631\n155263\n108552\n75821\n52934\n36947\n25785\n17995\n12558\n8763\n6115\n4267\n811463\n-859813\n553135\n539502\n-1047741\n-40253\n1045594\n-868116\n-328617\n-1030028\n1047292\n-778690\n-642024\n492989\n1007757\n1007944\n822191\n616847\n445566\n316143\n222399\n155806\n108933\n76087\n53120\n37077\n25876\n18058\n12602\n8794\n6137\n4282\n-120392\n-1044480\n905692\n208951\n-1008591\n-212305\n1047735\n-912641\n-384063\n-1021532\n1048320\n-765129\n-653022\n484356\n1005854\n1009243\n824245\n618717\n447027\n317218\n223168\n156348\n109314\n76354\n53306\n37207\n25967\n18122\n12646\n8825\n6158\n4297\n-941559\n-740828\n1047686\n-145498\n-912990\n-378556\n1035921\n-951243\n-438295\n-1011463\n1048561\n-751289\n-663904\n475681\n1003909\n1010522\n826291\n620583\n448487\n318291\n223936\n156891\n109695\n76620\n53493\n37337\n26058\n18185\n12690\n8856\n6180\n4313\n-897061\n-90820\n946106\n-483306\n-766287\n-534465\n1010307\n-983670\n-491142\n-999837\n1048017\n-737174\n-674668\n466965\n1001922\n1011780\n828328\n622447\n449947\n319365\n224704\n157433\n110076\n76887\n53679\n37467\n26149\n18248\n12734\n8887\n6201\n4328\n-27809\n601648\n624567\n-765838\n-576694\n-675771\n971236\n-1009714\n-542436\n-986671\n1046686\n-722790\n-685312\n458208\n999892\n1013018\n830357\n624308\n451404\n320438\n225472\n157976\n110457\n77153\n53865\n37597\n26239\n18312\n12779\n8917\n6223\n4343\n867010\n1012831\n157824\n-960780\n-354823\n-798614\n919228\n-1029203\n-592014\n-971985\n1044570\n-708142\n-695834\n449411\n997820\n1014234\n832378\n626165\n452861\n321511\n226240\n158518\n110837\n77420\n54051\n37727\n26330\n18375\n12823\n8948\n6244\n4358\n964705\n950491\n-345611\n-1045836\n-113092\n-899638\n854975\n-1042014\n-639722\n-955804\n1041672\n-693236\n-706232\n440576\n995706\n1015430\n834390\n628020\n454317\n322583\n227008\n159061\n111218\n77686\n54237\n37857\n26421\n18438\n12867\n8979\n6266\n4373\n175454\n443774\n-768696\n-1011279\n134968\n-976081\n779333\n-1048061\n-685406\n-938150\n1037992\n-678076\n-716505\n431702\n993550\n1016605\n836394\n629871\n455772\n323655\n227775\n159603\n111599\n77953\n54424\n37987\n26512\n18502\n12911\n9010\n6288\n4388\n-775108\n-270418\n-1013068\n-861061\n375475\n-1025857\n693310\n-1047306\n-728924\n-919052\n1033534\n-662669\n-726650\n422791\n991353\n1017759\n838390\n631720\n457226\n324727\n228543\n160145\n111980\n78219\n54610\n38117\n26602\n18565\n12956\n9041\n6309\n4403\n-1013040\n-858184\n-1021914\n-612362\n594965\n-1047604\n598053\n-1039753\n-770138\n-898539\n1028300\n-647020\n-736667\n413844\n989113\n1018892\n840377\n633565\n458678\n325798\n229310\n160688\n112360\n78486\n54796\n38247\n26693\n18628\n13000\n9072\n6331\n4418\n-319587\n-1044727\n-793177\n-293627\n781155\n-1040729\n494829\n-1025452\n-808917\n-876643\n1022296\n-631134\n-746552\n404860\n986832\n1020004\n842356\n635407\n460130\n326869\n230077\n161230\n112741\n78752\n54982\n38377\n26784\n18692\n13044\n9103\n6352\n4433\n667693\n-742836\n-380036\n58691\n923623\n-1005419\n385013\n-1004496\n-845138\n-853397\n1015525\n-615018\n-756304\n395842\n984509\n1021095\n844327\n637246\n461580\n327939\n230844\n161772\n113122\n79019\n55168\n38507\n26875\n18755\n13088\n9133\n6374\n4448\n1041099\n-93651\n121459\n404296\n1014394\n-942639\n270069\n-977020\n-878688\n-828836\n1007993\n-598678\n-765922\n386789\n982145\n1022165\n846289\n639082\n463030\n329009\n231611\n162314\n113502\n79285\n55355\n38637\n26966\n18818\n13132\n9164\n6395\n4463\n457323\n599318\n594716\n703662\n1048389\n-854105\n151528\n-943203\n-909459\n-803000\n999705\n-582119\n-775404\n377703\n979739\n1023215\n848242\n640915\n464478\n330079\n232378\n162856\n113883\n79552\n55541\n38767\n27056\n18882\n13177\n9195\n6417\n4478\n-546913\n1012092\n929709\n922549\n1023704\n-742236\n30968\n-903265\n-937355\n-775927\n990667\n-565347\n-784747\n368584\n977292\n1024243\n850188\n642745\n465925\n331148\n233145\n163398\n114264\n79818\n55727\n38897\n27147\n18945\n13221\n9226\n6438\n4493\n-1048320\n951688\n1048556\n1035922\n941721\n-610087\n-90004\n-857464\n-962288\n-747660\n980886\n-548369\n-793952\n359433\n974803\n1025250\n852124\n644572\n467371\n332217\n233912\n163940\n114644\n80084\n55913\n39028\n27238\n19008\n13265\n9257\n6460\n4508\n-585907\n446347\n923626\n1030816\n807030\n-461269\n-209777\n-806098\n-984179\n-718241\n970370\n-531191\n-803015\n350251\n972274\n1026236\n854052\n646395\n468816\n333286\n234678\n164482\n115025\n80351\n56100\n39158\n27329\n19072\n13309\n9288\n6481\n4523\n415187\n-267672\n583964\n907815\n627168\n-299849\n-326757\n-749499\n-1002959\n-687716\n959126\n-513818\n-811935\n341038\n969703\n1027201\n855972\n648216\n470260\n334354\n235444\n165024\n115405\n80617\n56286\n39288\n27419\n19135\n13354\n9319\n6503\n4538\n1034559\n-856548\n108537\n680985\n412202\n-130236\n-439383\n-688037\n-1018567\n-656133\n947163\n-496258\n-820711\n331797\n967092\n1028145\n857883\n650033\n471702\n335422\n236210\n165566\n115786\n80884\n56472\n39418\n27510\n19199\n13398\n9349\n6524\n4553\n702763\n-1044967\n-392123\n376271\n174165\n42935\n-546157\n-622109\n-1030956\n-623539\n934490\n-478517\n-829341\n322526\n964440\n1029068\n859785\n651847\n473144\n336489\n236977\n166108\n116167\n81150\n56658\n39548\n27601\n19262\n13442\n9380\n6546\n4568\n-275150\n-744839\n-801620\n28521\n-73620\n214933\n-645656\n-552143\n-1040085\n-589985\n921116\n-460601\n-837823\n313227\n961747\n1029970\n861679\n653658\n474584\n337556\n237742\n166650\n116547\n81416\n56844\n39678\n27692\n19325\n13486\n9411\n6567\n4583\n-1000092\n-96481\n-1024749\n-322490\n-317284\n381058\n-736554\n-478594\n-1045926\n-555522\n907052\n-442517\n-846157\n303902\n959014\n1030850\n863564\n655466\n476024\n338623\n238508\n167191\n116928\n81683\n57030\n39808\n27782\n19389\n13530\n9442\n6589\n4598\n-805553\n596984\n-1009636\n-636617\n-543190\n536773\n-817641\n-401938\n-1048460\n-520205\n892307\n-424271\n-854340\n294550\n956240\n1031710\n865441\n657270\n477462\n339689\n239274\n167733\n117308\n81949\n57217\n39938\n27873\n19452\n13575\n9473\n6611\n4613\n129607\n1011345\n-759795\n-877934\n-738693\n677822\n-887837\n-322674\n-1047680\n-484086\n876893\n-405871\n-862371\n285172\n953426\n1032548\n867309\n659072\n478899\n340755\n240039\n168275\n117689\n82216\n57403\n40068\n27964\n19515\n13619\n9504\n6632\n4628\n945607\n952877\n-333311\n-1018841\n-892850\n800351\n-946206\n-241315\n-1043587\n-447222\n860822\n-387322\n-870248\n275770\n950571\n1033365\n869168\n660870\n480335\n341820\n240805\n168816\n118069\n82482\n57589\n40198\n28055\n19579\n13663\n9535\n6654\n4643\n892220\n448917\n170664\n-1043221\n-997033\n901014\n-991972\n-158391\n-1036195\n-409669\n844105\n-368632\n-877971\n266344\n947677\n1034161\n871018\n662665\n481770\n342885\n241570\n169358\n118450\n82748\n57775\n40328\n28146\n19642\n13707\n9565\n6675\n4658\n18530\n-264923\n634962\n-948287\n-1045412\n977059\n-1024524\n-74438\n-1025528\n-371486\n826755\n-349807\n-885538\n256895\n944743\n1034936\n872860\n664456\n483203\n343950\n242335\n169899\n118830\n83015\n57961\n40458\n28236\n19705\n13751\n9596\n6697\n4673\n-872197\n-854905\n951639\n-744897\n-1035278\n1026409\n-1043429\n9998\n-1011618\n-332731\n808785\n-330854\n-892948\n247424\n941768\n1035689\n874693\n666245\n484636\n345014\n243100\n170441\n119210\n83281\n58148\n40588\n28327\n19769\n13796\n9627\n6718\n4688\n-961030\n-1045199\n1047071\n-456311\n-967198\n1047716\n-1048436\n94369\n-994510\n-293463\n790209\n-311781\n-900198\n237931\n938755\n1036421\n876518\n668030\n486067\n346078\n243865\n170982\n119591\n83548\n58334\n40718\n28418\n19832\n13840\n9658\n6740\n4703\n-166297\n-746836\n899071\n-115537\n-844982\n1040398\n-1039476\n178128\n-974258\n-253743\n771040\n-292593\n-907289\n228417\n935701\n1037132\n878333\n669812\n487497\n347142\n244630\n171523\n119971\n83814\n58520\n40848\n28509\n19895\n13884\n9689\n6761\n4718\n781329\n-99310\n542048\n238452\n-675473\n1004654\n-1016671\n260730\n-950926\n-213633\n751293\n-273299\n-914218\n218884\n932608\n1037821\n880140\n671590\n488926\n348205\n245394\n172065\n120352\n84080\n58706\n40978\n28599\n19959\n13928\n9720\n6783\n4733\n1010604\n594646\n59006\n565168\n-468156\n941461\n-980323\n341640\n-924587\n-173194\n730983\n-253905\n-920984\n209332\n929476\n1038489\n881938\n673366\n490354\n349267\n246159\n172606\n120732\n84347\n58892\n41108\n28690\n20022\n13973\n9751\n6804\n4748\n310735\n1010591\n-437755\n827245\n-234636\n852545\n-930916\n420333\n-895326\n-132488\n710124\n-234419\n-927587\n199762\n926304\n1039136\n883727\n675138\n491781\n350330\n246923\n173147\n121112\n84613\n59078\n41238\n28781\n20085\n14017\n9781\n6826\n4763\n-674823\n954060\n-832743\n994710\n12017\n740337\n-869110\n496298\n-863234\n-91578\n688733\n-214846\n-934025\n190174\n923094\n1039762\n885508\n676907\n493206\n351392\n247687\n173688\n121493\n84879\n59265\n41368\n28872\n20149\n14061\n9812\n6847\n4778\n-1039951\n451483\n-1034128\n1048408\n257997\n607902\n-795726\n569041\n-828413\n-50527\n666826\n-195196\n-940297\n180570\n919844\n1040366\n887279\n678672\n494631\n352453\n248451\n174229\n121873\n85146\n59451\n41498\n28962\n20212\n14105\n9843\n6869\n4793\n-448954\n-262173\n-995091\n982199\n489536\n458857\n-711743\n638092\n-790973\n-9399\n644419\n-175474\n-946402\n170951\n916556\n1040949\n889042\n680435\n496054\n353514\n249215\n174771\n122253\n85412\n59637\n41628\n29053\n20276\n14149\n9874\n6890\n4808\n554810\n-853257\n-724707\n803654\n693676\n297276\n-618280\n703001\n-751032\n31744\n621528\n-155688\n-952338\n161316\n913229\n1041510\n890795\n682194\n497476\n354575\n249979\n175312\n122634\n85678\n59823\n41758\n29144\n20339\n14194\n9905\n6912\n4823\n1048484\n-1045423\n-285837\n533194\n858990\n127572\n-516581\n763347\n-708717\n72839\n598171\n-135845\n-958105\n151668\n909864\n1042050\n892540\n683949\n498897\n355635\n250742\n175853\n123014\n85945\n60009\n41888\n29235\n20402\n14238\n9936\n6934\n4838\n578186\n-748828\n219486\n201753\n976226\n-45617\n-408000\n818739\n-664162\n113821\n574366\n-115952\n-963701\n142006\n906460\n1042569\n894276\n685701\n500316\n356695\n251506\n176393\n123394\n86211\n60195\n42018\n29326\n20466\n14282\n9967\n6955\n4853\n-423693\n-102139\n673782\n-152764\n1038821\n-217559\n-293985\n868817\n-617506\n154628\n550130\n-96017\n-969126\n132332\n903018\n1043066\n896003\n687450\n501735\n357754\n252269\n176934\n123775\n86477\n60382\n42148\n29416\n20529\n14326\n9997\n6977\n4868\n-1036031\n592303\n971431\n-489808\n1043272\n-383558\n-176054\n913256\n-568899\n195196\n525482\n-76047\n-974379\n122647\n899538\n1043542\n897721\n689196\n503152\n358813\n253032\n177475\n124155\n86743\n60568\n42278\n29507\n20592\n14370\n10028\n6998\n4884\n-695847\n1009829\n1043234\n-770833\n989330\n-539077\n-55778\n951768\n-518493\n235465\n500440\n-56050\n-979459\n112951\n896020\n1043996\n899430\n690938\n504568\n359872\n253795\n178016\n124535\n87010\n60754\n42408\n29598\n20656\n14415\n10059\n7020\n4899\n284096\n955235\n872498\n-963696\n880014\n-679867\n65241\n984102\n-466448\n275370\n475022\n-36031\n-984364\n103245\n892464\n1044429\n901130\n692677\n505983\n360930\n254558\n178557\n124915\n87276\n60940\n42538\n29689\n20719\n14459\n10090\n7041\n4914\n1002842\n454046\n498916\n-1046341\n721443\n-802083\n185391\n1010050\n-412928\n314852\n449248\n-16000\n-989095\n93531\n888870\n1044840\n902821\n694412\n507397\n361988\n255321\n179097\n125295\n87542\n61126\n42668\n29779\n20782\n14503\n10121\n7063\n4929\n799580\n-259421\n9342\n-1009314\n522491\n-902384\n303072\n1029442\n-358102\n353849\n423137\n4037\n-993649\n83808\n885239\n1045230\n904503\n696145\n508809\n363045\n256083\n179638\n125675\n87809\n61312\n42798\n29870\n20846\n14547\n10152\n7084\n4944\n-138812\n-851602\n-482403\n-856851\n294296\n-978030\n416716\n1042152\n-302145\n392301\n396709\n24073\n-998027\n74078\n881571\n1045599\n906176\n697873\n510221\n364102\n256846\n180179\n126056\n88075\n61499\n42928\n29961\n20909\n14591\n10183\n7106\n4959\n-949581\n-1045639\n-861996\n-606388\n49628\n-1026955\n524808\n1048099\n-245232\n430149\n369984\n44100\n-1002227\n64341\n877866\n1045946\n907840\n699599\n511631\n365158\n257608\n180719\n126436\n88341\n61685\n43058\n30052\n20972\n14636\n10213\n7127\n4974\n-887310\n-750815\n-1041185\n-286573\n-197817\n-1047821\n625910\n1047243\n-187544\n467335\n342981\n64110\n-1006249\n54599\n874123\n1046272\n909495\n701321\n513040\n366214\n258370\n181260\n126816\n88607\n61871\n43188\n30142\n21036\n14680\n10244\n7149\n4989\n-9250\n-104966\n-978311\n66019\n-434191\n-1040060\n718675\n1039591\n-129264\n503801\n315721\n84098\n-1010092\n44853\n870344\n1046576\n911140\n703039\n514447\n367270\n259132\n181800\n127196\n88874\n62057\n43318\n30233\n21099\n14724\n10275\n7170\n5004\n877314\n589956\n-687991\n411059\n-646262\n-1003882\n801867\n1025191\n-70574\n539491\n288224\n104054\n-1013756\n35102\n866528\n1046858\n912777\n704754\n515854\n368325\n259894\n182340\n127576\n89140\n62243\n43448\n30324\n21162\n14768\n10306\n7192\n5019\n957280\n1009060\n-237721\n709087\n-822161\n-940276\n874377\n1004137\n-11662\n574351\n260511\n123973\n-1017239\n25348\n862675\n1047119\n914404\n706466\n517259\n369380\n260656\n182881\n127956\n89406\n62429\n43578\n30415\n21226\n14813\n10337\n7213\n5034\n157126\n956404\n267816\n926015\n-952043\n-850980\n935241\n976566\n47288\n608326\n232603\n143847\n-1020542\n15592\n858786\n1047359\n916023\n708174\n518663\n370434\n261417\n183421\n128336\n89672\n62615\n43708\n30505\n21289\n14857\n10368\n7235\n5049\n-787488\n456606\n711088\n1037034\n-1028638\n-738434\n983647\n942657\n106088\n641365\n204520\n163667\n-1023663\n5835\n854861\n1047577\n917632\n709879\n520066\n371488\n262179\n183961\n128716\n89939\n62801\n43838\n30596\n21353\n14901\n10399\n7256\n5064\n-1008090\n-256666\n989042\n1029446\n-1047659\n-605712\n1018950\n902629\n164553\n673416\n176284\n183429\n-1026601\n-3923\n850900\n1047774\n919232\n711580\n521467\n372542\n262940\n184501\n129096\n90205\n62988\n43968\n30687\n21416\n14945\n10429\n7278\n5079\n-301858\n-849941\n1037055\n904119\n-1008042\n-456442\n1040680\n856744\n222497\n704431\n147916\n203123\n-1029358\n-13680\n846903\n1047949\n920823\n713278\n522867\n373595\n263701\n185042\n129476\n90471\n63174\n44098\n30778\n21479\n14989\n10460\n7300\n5094\n681900\n-1045848\n843965\n675387\n-912003\n-294701\n1048548\n805298\n279738\n734360\n119436\n222743\n-1031931\n-23436\n842870\n1048102\n922404\n714972\n524266\n374648\n264462\n185582\n129856\n90737\n63360\n44228\n30869\n21543\n15034\n10491\n7321\n5109\n1038723\n-752796\n454663\n369410\n-764918\n-124908\n1042449\n748625\n336094\n763159\n90868\n242281\n-1034321\n-33190\n838802\n1048234\n923976\n716663\n525664\n375700\n265223\n186122\n130236\n91003\n63546\n44358\n30959\n21606\n15078\n10522\n7343\n5124\n440548\n-107793\n-40342\n21183\n-575020\n48298\n1022464\n687093\n391388\n790783\n62231\n261731\n-1036526\n-42942\n834698\n1048345\n925540\n718350\n527061\n376752\n265983\n186662\n130616\n91270\n63732\n44488\n31050\n21669\n15122\n10553\n7364\n5139\n-562664\n587605\n-525969\n-329466\n-352938\n220184\n988859\n621102\n445445\n817190\n33547\n281086\n-1038548\n-52689\n830559\n1048434\n927093\n720034\n528456\n377803\n266744\n187202\n130996\n91536\n63918\n44618\n31141\n21733\n15166\n10584\n7386\n5154\n-1048566\n1008284\n-889313\n-642434\n-111101\n386055\n942083\n551080\n498094\n842338\n4839\n300338\n-1040385\n-62432\n826385\n1048501\n928638\n721715\n529850\n378854\n267504\n187741\n131376\n91802\n64104\n44748\n31232\n21796\n15210\n10615\n7407\n5169\n-570421\n957565\n-1045904\n-881926\n136954\n541377\n882757\n477481\n549167\n866189\n-23874\n319480\n-1042037\n-72170\n822176\n1048547\n930173\n723392\n531243\n379904\n268265\n188281\n131756\n92068\n64290\n44878\n31322\n21859\n15255\n10645\n7429\n5184\n432166\n459162\n-959333\n-1020551\n377344\n681909\n811673\n400783\n598505\n888706\n-52568\n338506\n-1043503\n-81901\n817933\n1048572\n931699\n725065\n532634\n380954\n269025\n188821\n132136\n92334\n64477\n45008\n31413\n21923\n15299\n10676\n7450\n5199\n1037422\n-253910\n-649730\n-1042455\n596613\n803809\n729777\n321484\n645950\n909855\n-81223\n357408\n-1044784\n-91625\n813655\n1048575\n933216\n726735\n534025\n382004\n269785\n189361\n132515\n92600\n64663\n45138\n31504\n21986\n15343\n10707\n7472\n5214\n688877\n-848274\n-189072\n-945131\n782490\n903748\n638160\n240099\n691354\n929603\n-109817\n376179\n-1045879\n-101342\n809343\n1048556\n934723\n728401\n535414\n383053\n270544\n189900\n132895\n92867\n64849\n45268\n31595\n22049\n15387\n10738\n7493\n5229\n-293019\n-1046049\n315543\n-739712\n924569\n978995\n538043\n157155\n734571\n947920\n-138329\n394813\n-1046788\n-111049\n804996\n1048516\n936221\n730064\n536801\n384102\n271304\n190440\n133275\n93133\n65035\n45398\n31685\n22113\n15432\n10769\n7515\n5244\n-1005514\n-754771\n746798\n-449691\n1014899\n1027494\n430759\n73191\n775467\n964777\n-166737\n413303\n-1047511\n-120747\n800616\n1048454\n937710\n731723\n538188\n385150\n272063\n190980\n133655\n93399\n65221\n45528\n31776\n22176\n15476\n10800\n7536\n5259\n-793544\n-110620\n1004432\n-108239\n1048425\n1047920\n317736\n-11248\n813910\n980149\n-195020\n431642\n-1048048\n-130435\n796202\n1048371\n939189\n733379\n539573\n386198\n272823\n191519\n134035\n93665\n65407\n45658\n31867\n22239\n15520\n10831\n7558\n5274\n148006\n585249\n1028546\n245594\n1023269\n1039715\n200482\n-95614\n849781\n994012\n-223157\n449824\n-1048399\n-140111\n791754\n1048266\n940659\n735031\n540957\n387245\n273582\n192059\n134414\n93931\n65593\n45788\n31958\n22303\n15564\n10861\n7579\n5289\n953481\n1007500\n813537\n571337\n940839\n1003103\n80557\n-179359\n882965\n1006344\n-251126\n467841\n-1048563\n-149775\n787273\n1048140\n942120\n736679\n542340\n388292\n274341\n192598\n134794\n94197\n65779\n45918\n32048\n22366\n15608\n10892\n7601\n5304\n882329\n958720\n409389\n831736\n805749\n939085\n-40442\n-261940\n913358\n1017126\n-278907\n485687\n-1048540\n-159426\n782759\n1047992\n943571\n738324\n543721\n389339\n275100\n193137\n135174\n94463\n65965\n46048\n32139\n22430\n15653\n10923\n7623\n5319\n-32\n461716\n-89936\n997008\n625561\n849409\n-160901\n-342821\n940863\n1026343\n-306479\n503356\n-1048331\n-169063\n778211\n1047822\n945012\n739965\n545101\n390385\n275859\n193677\n135554\n94730\n66152\n46178\n32230\n22493\n15697\n10954\n7644\n5334\n-882363\n-251152\n-568353\n1048251\n410360\n736526\n-279217\n-421478\n965393\n1033979\n-333822\n520841\n-1047936\n-178686\n773631\n1047631\n946444\n741603\n546480\n391430\n276617\n194216\n135933\n94996\n66338\n46308\n32321\n22556\n15741\n10985\n7666\n5349\n-953454\n-846600\n-914634\n979604\n172190\n603519\n-393814\n-497398\n986872\n1040023\n-360914\n538136\n-1047354\n-188293\n769019\n1047419\n947867\n743237\n547857\n392475\n277375\n194755\n136313\n95262\n66524\n46438\n32411\n22620\n15785\n11016\n7687\n5364\n-147944\n-1046242\n-1048274\n798919\n-75617\n454024\n-503165\n-570091\n1005230\n1044466\n-387735\n555235\n-1046586\n-197884\n764373\n1047185\n949280\n744867\n549233\n393520\n278134\n195294\n136693\n95528\n66710\n46568\n32502\n22683\n15829\n11047\n7709\n5379\n793586\n-756741\n-938202\n526861\n-319192\n292124\n-605814\n-639083\n1020411\n1047301\n-414265\n572130\n-1045632\n-207458\n759696\n1046930\n950684\n746494\n550608\n394564\n278892\n195833\n137072\n95794\n66896\n46698\n32593\n22746\n15874\n11077\n7730\n5394\n1005496\n-113445\n-610010\n194545\n-544902\n122242\n-700393\n-703927\n1032366\n1048523\n-440485\n588817\n-1044492\n-217013\n754987\n1046653\n952078\n748117\n551982\n395608\n279650\n196372\n137452\n96060\n67082\n46828\n32684\n22810\n15918\n11108\n7752\n5409\n292958\n582889\n-139998\n-160022\n-740113\n-50979\n-785643\n-764203\n1041057\n1048130\n-466375\n605289\n-1043166\n-226550\n750245\n1046354\n953463\n749737\n553354\n396651\n280407\n196911\n137831\n96326\n67268\n46958\n32775\n22873\n15962\n11139\n7773\n5424\n-688924\n1006708\n362562\n-496286\n-893898\n-222808\n-860427\n-819519\n1046456\n1046124\n-491915\n621540\n-1041655\n-236068\n745473\n1046035\n954838\n751353\n554725\n397694\n281165\n197450\n138211\n96592\n67454\n47088\n32865\n22936\n16006\n11170\n7795\n5439\n-1037413\n959867\n780831\n-775790\n-997652\n-388549\n-923751\n-869516\n1048548\n1042507\n-517085\n637563\n-1039958\n-245565\n740668\n1045693\n956204\n752965\n556094\n398737\n281923\n197989\n138591\n96858\n67640\n47218\n32956\n23000\n16051\n11201\n7816\n5454\n-432109\n464265\n1017566\n-966566\n-1045565\n-543674\n-974769\n-913869\n1047324\n1037285\n-541869\n653354\n-1038077\n-255040\n735833\n1045330\n957560\n754574\n557463\n399779\n282680\n198528\n138970\n97124\n67826\n47348\n33047\n23063\n16095\n11232\n7838\n5470\n570474\n-248392\n1017729\n-1046794\n-1034958\n-683945\n-1012803\n-952292\n1042789\n1030466\n-566246\n668907\n-1036011\n-264494\n730966\n1044946\n958906\n756178\n558830\n400820\n283437\n199067\n139350\n97390\n68012\n47478\n33138\n23126\n16139\n11262\n7859\n5485\n1048566\n-844920\n781282\n-1007299\n-966422\n-805530\n-1037347\n-984533\n1034958\n1022060\n-590198\n684215\n-1033761\n-273924\n726069\n1044540\n960243\n757780\n560195\n401861\n284194\n199606\n139729\n97657\n68198\n47608\n33228\n23190\n16183\n11293\n7881\n5500\n562611\n-1046428\n363196\n-852599\n-843795\n-905106\n-1048072\n-1010385\n1023855\n1012080\n-613708\n699273\n-1031327\n-283331\n721141\n1044113\n961570\n759377\n561560\n402902\n284951\n200144\n140109\n97923\n68384\n47738\n33319\n23253\n16227\n11324\n7902\n5515\n-440606\n-758705\n-139328\n-600385\n-673940\n-979953\n-1044837\n-1029679\n1009515\n1000542\n-636757\n714076\n-1028709\n-292714\n716182\n1043664\n962888\n760971\n562923\n403942\n285707\n200683\n140488\n98189\n68571\n47868\n33410\n23316\n16272\n11355\n7924\n5530\n-1038731\n-116270\n-609461\n-279505\n-466363\n-1028026\n-1027684\n-1042290\n991983\n987464\n-659330\n728618\n-1025909\n-302071\n711194\n1043194\n964196\n762561\n564284\n404981\n286464\n201221\n140868\n98455\n68757\n47998\n33501\n23380\n16316\n11386\n7946\n5545\n-681852\n580525\n-937900\n73343\n-232684\n-1048011\n-996842\n-1048136\n971315\n972865\n-681407\n742894\n-1022926\n-311402\n706175\n1042703\n965494\n764147\n565645\n406020\n287220\n201760\n141247\n98721\n68943\n48128\n33591\n23443\n16360\n11417\n7967\n5560\n301919\n1005910\n-1048290\n417802\n14019\n-1039363\n-952721\n-1047180\n947576\n956768\n-702974\n756899\n-1019761\n-320706\n701126\n1042190\n966783\n765730\n567004\n407059\n287976\n202298\n141626\n98987\n69129\n48258\n33682\n23506\n16404\n11448\n7989\n5575\n1008107\n961008\n-914964\n714477\n259937\n-1002318\n-895910\n-1039427\n920842\n939198\n-724014\n770627\n-1016415\n-329982\n696048\n1041656\n968062\n767309\n568361\n408097\n288732\n202837\n142006\n99253\n69315\n48388\n33773\n23570\n16448\n11478\n8010\n5590\n787446\n466811\n-568920\n929436\n491307\n-937887\n-827165\n-1024927\n891197\n920181\n-744510\n784074\n-1012888\n-339229\n690941\n1041100\n969332\n768884\n569718\n409135\n289488\n203375\n142385\n99519\n69501\n48518\n33864\n23633\n16493\n11509\n8032\n5605\n-157189\n-245630\n-90609\n1038094\n695177\n-847832\n-747402\n-1003776\n858734\n899748\n-764449\n797235\n-1009181\n-348447\n685805\n1040523\n970591\n770456\n571072\n410172\n290244\n203913\n142765\n99785\n69687\n48648\n33954\n23697\n16537\n11540\n8053\n5620\n-957305\n-843234\n408767\n1028025\n860137\n-734613\n-657683\n-976110\n823556\n877930\n-783814\n810105\n-1005295\n-357635\n680639\n1039924\n971841\n772023\n572426\n411209\n290999\n204451\n143144\n100051\n69873\n48778\n34045\n23760\n16581\n11571\n8075\n5635\n-877280\n-1046606\n813110\n900379\n976955\n-601322\n-559204\n-942109\n785775\n854760\n-802592\n822678\n-1001230\n-366792\n675445\n1039305\n973082\n773587\n573778\n412245\n291755\n204990\n143523\n100317\n70059\n48908\n34136\n23823\n16625\n11602\n8096\n5650\n9313\n-760664\n1028415\n669756\n1039091\n-451603\n-453275\n-901993\n745509\n830274\n-820767\n834952\n-996987\n-375918\n670223\n1038663\n974312\n775148\n575129\n413281\n292510\n205528\n143903\n100583\n70245\n49038\n34227\n23887\n16670\n11633\n8118\n5665\n887343\n-119093\n1004625\n362531\n1043069\n-289545\n-341309\n-856023\n702887\n804509\n-838328\n846920\n-992566\n-385011\n664972\n1038001\n975533\n776704\n576479\n414317\n293265\n206066\n144282\n100849\n70431\n49168\n34317\n23950\n16714\n11664\n8139\n5680\n949554\n578156\n747272\n13844\n988664\n-119576\n-224797\n-804497\n658042\n777506\n-855259\n858579\n-987969\n-394070\n659693\n1037317\n976744\n778257\n577827\n415351\n294020\n206604\n144661\n101115\n70617\n49298\n34408\n24013\n16758\n11694\n8161\n5695\n138750\n1005104\n316187\n-336427\n878923\n53660\n-105290\n-747749\n611117\n749305\n-871550\n869925\n-983196\n-403095\n654386\n1036612\n977945\n779806\n579173\n416386\n294775\n207142\n145040\n101381\n70803\n49428\n34499\n24077\n16802\n11725\n8182\n5710\n-799621\n962141\n-188407\n-648220\n719988\n225430\n15619\n-686149\n562260\n719951\n-887186\n880953\n-978249\n-412086\n649052\n1035886\n979137\n781351\n580519\n417420\n295529\n207680\n145420\n101646\n70989\n49558\n34590\n24140\n16846\n11756\n8204\n5725\n-1002824\n469354\n-649200\n-885875\n520754\n391041\n136321\n-620095\n511625\n689488\n-902158\n891659\n-973127\n-421040\n643690\n1035138\n980318\n782892\n581863\n418453\n296283\n208217\n145799\n101912\n71175\n49688\n34680\n24203\n16891\n11787\n8225\n5740\n-284035\n-242867\n-959061\n-1022212\n292373\n545968\n255206\n-550016\n459373\n657964\n-916453\n902040\n-967833\n-429958\n638301\n1034369\n981490\n784429\n583205\n419486\n297038\n208755\n146178\n102178\n71361\n49818\n34771\n24267\n16935\n11818\n8247\n5755\n695894\n-841542\n-1045951\n-1041637\n47627\n685978\n370692\n-476368\n405669\n625426\n-930061\n912091\n-962366\n-438839\n632886\n1033579\n982652\n785963\n584547\n420519\n297792\n209293\n146557\n102444\n71547\n49948\n34862\n24330\n16979\n11849\n8268\n5770\n1036021\n-1046776\n-889671\n-941929\n-199784\n807246\n481240\n-399628\n350682\n591926\n-942971\n921810\n-956728\n-447682\n627443\n1032767\n983805\n787493\n585887\n421551\n298545\n209831\n146936\n102710\n71733\n50078\n34953\n24393\n17023\n11880\n8290\n5785\n423635\n-762617\n-526553\n-734492\n-436013\n906458\n585378\n-320295\n294587\n557514\n-955175\n931191\n-950921\n-456487\n621974\n1031935\n984947\n789019\n587225\n422582\n299299\n210368\n147315\n102976\n71919\n50208\n35043\n24457\n17067\n11910\n8312\n5800\n-578239\n-121916\n-41017\n-443049\n-647838\n980905\n681719\n-238882\n237560\n522244\n-966662\n940233\n-944944\n-465251\n616479\n1031081\n986080\n790541\n588562\n423613\n300053\n210906\n147694\n103242\n72105\n50338\n35134\n24520\n17112\n11941\n8333\n5815\n-1048483\n575783\n454054\n-100935\n-823403\n1028551\n768978\n-155919\n179782\n486169\n-977424\n948931\n-938798\n-473976\n610958\n1030206\n987202\n792059\n589898\n424644\n300806\n211443\n148074\n103508\n72291\n50468\n35225\n24583\n17156\n11972\n8355\n5830\n-554757\n1004290\n843564\n252724\n-952881\n1048096\n845995\n-71944\n121436\n449346\n-987453\n957283\n-932487\n-482659\n605411\n1029310\n988315\n793573\n591232\n425674\n301559\n211981\n148453\n103774\n72477\n50598\n35316\n24647\n17200\n12003\n8376\n5845\n449011\n963267\n1036955\n577478\n-1029025\n1039005\n911743\n12497\n62706\n411831\n-996742\n965285\n-926009\n-491301\n599839\n1028393\n989418\n795084\n592565\n426703\n302312\n212518\n148832\n104040\n72664\n50728\n35406\n24710\n17244\n12034\n8398\n5860\n1039960\n471894\n989266\n836185\n-1047574\n1001526\n965345\n96858\n3778\n373682\n-1005284\n972935\n-919366\n-499900\n594241\n1027454\n990511\n796590\n593896\n427732\n303065\n213055\n149211\n104306\n72850\n50858\n35497\n24773\n17288\n12065\n8419\n5875\n674774\n-240102\n711585\n999257\n-1007488\n936684\n1006089\n180590\n-55163\n334958\n-1013072\n980229\n-912560\n-508455\n588619\n1026495\n991594\n798093\n595226\n428761\n303818\n213592\n149590\n104572\n73036\n50988\n35588\n24837\n17333\n12096\n8441\n5890\n-310795\n-839844\n268468\n1048043\n-911013\n846250\n1033432\n263150\n-113928\n295718\n-1020100\n987166\n-905592\n-516967\n582971\n1025514\n992668\n799592\n596555\n429789\n304570\n214130\n149969\n104837\n73222\n51118\n35679\n24900\n17377\n12126\n8462\n5905\n-1010621\n-1046939\n-237064\n976962\n-763547\n732695\n1047009\n344002\n-172334\n256023\n-1026363\n993742\n-898463\n-525434\n577299\n1024513\n993731\n801087\n597882\n430816\n305323\n214667\n150348\n105103\n73408\n51248\n35769\n24963\n17421\n12157\n8484\n5920\n-781287\n-764564\n-687481\n794146\n-573344\n599121\n1046639\n422622\n-230195\n215933\n-1031856\n999955\n-891174\n-533855\n571602\n1023490\n994784\n802578\n599208\n431843\n306075\n215204\n150727\n105369\n73594\n51378\n35860\n25027\n17465\n12188\n8505\n5935\n166359\n-124738\n-978067\n520502\n-351051\n449179\n1032327\n498498\n-287328\n175511\n-1036576\n1005803\n-883726\n-542231\n565882\n1022446\n995828\n804065\n600533\n432870\n306827\n215741\n151106\n105635\n73780\n51507\n35951\n25090\n17510\n12219\n8527\n5950\n961055\n573406\n-1041265\n187327\n-109109\n286964\n1004265\n571139\n-343553\n134819\n-1040519\n1011284\n-876122\n-550559\n560137\n1021382\n996861\n805549\n601856\n433896\n307579\n216278\n151484\n105901\n73966\n51637\n36042\n25154\n17554\n12250\n8548\n5965\n872161\n1003469\n-862380\n-167272\n138940\n116909\n962825\n640073\n-398692\n93919\n-1043681\n1016395\n-868361\n-558839\n554369\n1020296\n997885\n807028\n603177\n434922\n308330\n216815\n151863\n106167\n74152\n51767\n36132\n25217\n17598\n12281\n8570\n5980\n-18593\n964387\n-483003\n-502740\n379212\n-56341\n908560\n704853\n-452570\n52874\n-1046061\n1021135\n-860446\n-567072\n548578\n1019190\n998898\n808503\n604497\n435947\n309082\n217351\n152242\n106432\n74338\n51897\n36223\n25280\n17642\n12312\n8591\n5995\n-892254\n474430\n8667\n-780709\n598259\n-228051\n842193\n765058\n-505017\n11748\n-1047656\n1025503\n-852378\n-575255\n542763\n1018062\n999902\n809975\n605816\n436971\n309833\n217888\n152621\n106698\n74524\n52027\n36314\n25344\n17686\n12342\n8613\n6010\n-945580\n-237335\n498322\n-969388\n783821\n-393530\n764607\n820298\n-555868\n-29396\n-1048466\n1029496\n-844159\n-583388\n536926\n1016914\n1000895\n811443\n607133\n437995\n310584\n218425\n153000\n106964\n74710\n52157\n36405\n25407\n17731\n12373\n8635\n6025\n-129544\n-838139\n872123\n-1047196\n925512\n-548258\n676837\n870214\n-604961\n-70495\n-1048489\n1033113\n-835789\n-591471\n531066\n1015744\n1001879\n812906\n608449\n439019\n311335\n218962\n153379\n107230\n74896\n52287\n36495\n25470\n17775\n12404\n8656\n6041\n805594\n-1047093\n1043166\n-1005236\n1015401\n-688006\n580051\n914482\n-652142\n-111485\n-1047726\n1036352\n-827271\n-599502\n525183\n1014554\n1002852\n814366\n609763\n440042\n312086\n219498\n153757\n107496\n75082\n52417\n36586\n25534\n17819\n12435\n8678\n6056\n1000073\n-766506\n971685\n-848305\n1048457\n-808956\n475538\n952814\n-697262\n-152303\n-1046178\n1039214\n-818606\n-607482\n519278\n1013343\n1003816\n815821\n611076\n441064\n312836\n220035\n154136\n107762\n75268\n52547\n36677\n25597\n17863\n12466\n8699\n6071\n275089\n-127559\n674299\n-594353\n1022829\n-907805\n364691\n984963\n-740177\n-192887\n-1043845\n1041695\n-809795\n-615409\n513352\n1012111\n1004769\n817273\n612388\n442086\n313587\n220571\n154515\n108027\n75454\n52677\n36768\n25660\n17907\n12497\n8721\n6086\n-702810\n571025\n220147\n-272424\n939953\n-981850\n248986\n1010718\n-780751\n-233174\n-1040730\n1043797\n-800840\n-623282\n507404\n1010859\n1005713\n818721\n613698\n443108\n314337\n221108\n154894\n108293\n75639\n52807\n36858\n25724\n17952\n12528\n8742\n6101\n-1034549\n1002641\n-285187\n80663\n804466\n-1029070\n129964\n1029914\n-818858\n-273102\n-1036834\n1045517\n-791743\n-631102\n501434\n1009585\n1006646\n820165\n615006\n444129\n315087\n221644\n155272\n108559\n75825\n52937\n36949\n25787\n17996\n12558\n8764\n6116\n-415129\n965499\n-724218\n424524\n623953\n-1048174\n9211\n1042426\n-854376\n-312610\n-1032161\n1046855\n-782505\n-638867\n495443\n1008291\n1007569\n821604\n616313\n445149\n315837\n222180\n155651\n108825\n76011\n53067\n37040\n25850\n18040\n12589\n8785\n6131\n585959\n476962\n-994878\n719832\n408516\n-1038639\n-111664\n1048171\n-887192\n-351636\n-1026713\n1047812\n-773128\n-646577\n489432\n1006977\n1008483\n823040\n617619\n446169\n316587\n222717\n156030\n109090\n76197\n53197\n37131\n25914\n18084\n12620\n8807\n6146\n1048319\n-234566\n-1034239\n932811\n170214\n-1000728\n-231052\n1047114\n-917204\n-390121\n-1020496\n1048385\n-763613\n-654231\n483399\n1005641\n1009386\n824472\n618923\n447189\n317336\n223253\n156408\n109356\n76383\n53327\n37221\n25977\n18129\n12651\n8828\n6161\n546859\n-836428\n-833153\n1039104\n-77615\n-935474\n-347362\n1039261\n-944317\n-428005\n-1013514\n1048576\n-753962\n-661828\n477347\n1004285\n1010279\n825899\n620226\n448208\n318086\n223789\n156787\n109622\n76569\n53457\n37312\n26040\n18173\n12682\n8850\n6176\n-457380\n-1047241\n-438368\n1026554\n-321100\n-844662\n-459046\n1024663\n-968443\n-465230\n-1005771\n1048384\n-744178\n-669368\n471274\n1002908\n1011162\n827323\n621527\n449226\n318835\n224325\n157166\n109888\n76755\n53587\n37403\n26104\n18217\n12713\n8871\n6191\n-1041106\n-768443\n58332\n896595\n-546612\n-730772\n-564614\n1003414\n-989508\n-501739\n-997275\n1047809\n-734261\n-676849\n465181\n1001511\n1012034\n828743\n622827\n450244\n319584\n224861\n157544\n110153\n76941\n53717\n37494\n26167\n18261\n12744\n8893\n6206\n-667644\n-130379\n541470\n664091\n-741530\n-596917\n-662662\n975653\n-1007445\n-537475\n-988030\n1046851\n-724214\n-684273\n459069\n1000093\n1012897\n830158\n624125\n451261\n320333\n225397\n157923\n110419\n77127\n53847\n37584\n26230\n18305\n12774\n8914\n6221\n319647\n568640\n898723\n355635\n-894944\n-446752\n-751883\n941559\n-1022197\n-572384\n-978045\n1045511\n-714037\n-691637\n452937\n998655\n1013750\n831570\n625422\n452278\n321081\n225933\n158301\n110685\n77313\n53977\n37675\n26294\n18350\n12805\n8936\n6236\n1013056\n1001806\n1047034\n6504\n-998267\n-284381\n-831088\n901355\n-1033717\n-606411\n-967326\n1043790\n-703734\n-698941\n446786\n997196\n1014592\n832977\n626717\n453295\n321830\n226468\n158680\n110951\n77499\n54107\n37766\n26357\n18394\n12836\n8958\n6251\n775066\n966604\n951922\n-343370\n-1045715\n-114241\n-899223\n855300\n-1041970\n-639505\n-955882\n1041687\n-693306\n-706184\n440617\n995716\n1015424\n834381\n628011\n454310\n322578\n227004\n159058\n111216\n77685\n54237\n37857\n26420\n18438\n12867\n8979\n6266\n-175516\n479491\n635499\n-653973\n-1034634\n59021\n-955380\n803695\n-1046928\n-671614\n-943722\n1039204\n-682754\n-713367\n434429\n994216\n1016246\n835780\n629303\n455326\n323326\n227540\n159437\n111482\n77871\n54367\n37947\n26484\n18482\n12898\n9001\n6281\n-964730\n-231795\n171331\n-889781\n-965643\n230670\n-998811\n746873\n-1048576\n-702688\n-930853\n1036341\n-672081\n-720487\n428222\n992696\n1017058\n837176\n630594\n456341\n324074\n228075\n159815\n111748\n78057\n54496\n38038\n26547\n18526\n12929\n9022\n6296\n-866975\n-834712\n-332670\n-1023823\n-842604\n396017\n-1028937\n685203\n-1046909\n-732681\n-917287\n1033100\n-661288\n-727546\n421998\n991155\n1017860\n838567\n631884\n457355\n324822\n228611\n160194\n112013\n78243\n54626\n38129\n26610\n18571\n12959\n9044\n6311\n27873\n-1047380\n-759330\n-1040769\n-672404\n550544\n-1045358\n619087\n-1041933\n-761546\n-903033\n1029482\n-650378\n-734541\n415755\n989594\n1018652\n839954\n633172\n458368\n325569\n229146\n160572\n112279\n78429\n54756\n38220\n26674\n18615\n12990\n9065\n6326\n897094\n-770373\n-1009454\n-938681\n-464568\n690029\n-1047854\n548952\n-1033662\n-789238\n-888101\n1025487\n-639352\n-741472\n409495\n988013\n1019433\n841337\n634458\n459382\n326317\n229682\n160950\n112545\n78615\n54886\n38310\n26737\n18659\n13021\n9087\n6341\n941532\n-133198\n-1024892\n-729235\n-230730\n810661\n-1036392\n475255\n-1022124\n-815715\n-872504\n1021119\n-628213\n-748340\n403218\n986411\n1020204\n842716\n635743\n460394\n327064\n230217\n161329\n112810\n78801\n55016\n38401\n26801\n18703\n13052\n9108\n6356\n120329\n566250\n-802055\n-436386\n16022\n909145\n-1011125\n398473\n-1007354\n-840936\n-856252\n1016377\n-616962\n-755142\n396924\n984790\n1020965\n844091\n637026\n461406\n327811\n230753\n161707\n113076\n78987\n55146\n38492\n26864\n18747\n13083\n9130\n6371\n-811503\n1000963\n-392750\n-93626\n261877\n982789\n-972389\n319104\n-989400\n-864862\n-839359\n1011264\n-605601\n-761880\n390613\n983147\n1021716\n845462\n638308\n462418\n328558\n231288\n162085\n113342\n79173\n55276\n38583\n26927\n18792\n13114\n9151\n6386\n-997243\n967702\n107865\n259841\n493075\n1029582\n-920701\n237665\n-968317\n-887457\n-821836\n1005782\n-594132\n-768551\n384286\n981485\n1022457\n846829\n639588\n463429\n329304\n231823\n162464\n113607\n79359\n55406\n38673\n26991\n18836\n13145\n9173\n6401\n-266123\n482016\n583403\n583590\n696675\n1048245\n-856749\n154683\n-944174\n-908685\n-803697\n999933\n-582558\n-775156\n377942\n979803\n1023187\n848191\n640867\n464440\n330051\n232358\n162842\n113873\n79545\n55536\n38764\n27054\n18880\n13175\n9194\n6416\n709670\n-229023\n923306\n840594\n861281\n1038267\n-781384\n70697\n-917045\n-928514\n-784955\n993718\n-570880\n-781693\n371583\n978100\n1023907\n849549\n642144\n465450\n330797\n232893\n163220\n114139\n79730\n55666\n38855\n27117\n18924\n13206\n9216\n6431\n1032995\n-832989\n1048551\n1001457\n977681\n999923\n-695611\n-13747\n-887017\n-946913\n-765624\n987141\n-559100\n-788163\n365208\n976377\n1024617\n850904\n643420\n466459\n331543\n233428\n163599\n114404\n79916\n55796\n38946\n27181\n18969\n13237\n9237\n6446\n406589\n-1047512\n930021\n1047783\n1039358\n934259\n-600572\n-98103\n-854185\n-963855\n-745720\n980203\n-547222\n-794565\n358817\n974635\n1025317\n852254\n644694\n467468\n332289\n233963\n163977\n114670\n80102\n55926\n39036\n27244\n19013\n13268\n9259\n6461\n-593633\n-772298\n595272\n974272\n1042862\n843069\n-497534\n-181821\n-818653\n-979312\n-725256\n972907\n-535246\n-800897\n352411\n972872\n1026006\n853600\n645967\n468476\n333035\n234498\n164355\n114935\n80288\n56056\n39127\n27307\n19057\n13299\n9281\n6476\n-1048072\n-136017\n122130\n789333\n987995\n728845\n-387868\n-264360\n-780532\n-993261\n-704248\n965256\n-523174\n-807161\n345991\n971089\n1026685\n854942\n647238\n469484\n333780\n235033\n164733\n115201\n80474\n56186\n39218\n27371\n19101\n13330\n9302\n6491\n-538918\n563856\n-379407\n514117\n877829\n594708\n-273035\n-345183\n-739944\n-1005681\n-682713\n957253\n-511010\n-813354\n339556\n969287\n1027354\n856279\n648508\n470491\n334526\n235567\n165111\n115467\n80660\n56316\n39308\n27434\n19145\n13361\n9324\n6506\n465714\n1000113\n-792736\n180101\n718530\n444322\n-154566\n-423765\n-697016\n-1016553\n-660665\n948900\n-498755\n-819478\n333106\n967464\n1028013\n857613\n649776\n471498\n335271\n236102\n165489\n115732\n80846\n56446\n39399\n27497\n19190\n13391\n9345\n6521\n1042171\n968792\n-1021763\n-174514\n519015\n281797\n-34037\n-499597\n-651885\n-1025859\n-638122\n940200\n-486411\n-825530\n326643\n965622\n1028661\n858942\n651043\n472504\n336016\n236637\n165867\n115998\n81032\n56575\n39490\n27561\n19234\n13422\n9367\n6536\n660461\n484538\n-1013242\n-509169\n290449\n111572\n86945\n-572187\n-604693\n-1033586\n-615100\n931157\n-473980\n-831510\n320165\n963759\n1029299\n860267\n652308\n473510\n336760\n237171\n166246\n116263\n81218\n56705\n39581\n27624\n19278\n13453\n9388\n6551\n-328474\n-226249\n-769155\n-785590\n45627\n-61701\n206768\n-641063\n-555589\n-1039721\n-591618\n921775\n-461466\n-837419\n313675\n961877\n1029927\n861588\n653571\n474515\n337505\n237706\n166624\n116529\n81404\n56835\n39671\n27687\n19322\n13484\n9410\n6566\n-1015412\n-831259\n-346249\n-972162\n-201750\n-233288\n323838\n-705778\n-504729\n-1044256\n-567691\n912055\n-448869\n-843255\n307171\n959976\n1030544\n862905\n654833\n475520\n338249\n238240\n167002\n116794\n81590\n56965\n39762\n27751\n19366\n13515\n9431\n6581\n-768784\n-1047636\n157157\n-1047547\n-437834\n-398501\n436593\n-765912\n-452273\n-1047182\n-543339\n902003\n-436192\n-849018\n300654\n958054\n1031152\n864217\n656093\n476524\n338994\n238774\n167380\n117060\n81776\n57095\n39853\n27814\n19411\n13546\n9453\n6596\n184660\n-774218\n624025\n-1003122\n-649412\n-552826\n543533\n-821076\n-398388\n-1048496\n-518580\n891621\n-423438\n-854708\n294124\n956113\n1031748\n865526\n657352\n477527\n339738\n239309\n167758\n117325\n81961\n57225\n39944\n27877\n19455\n13577\n9474\n6611\n968328\n-138834\n945815\n-843970\n-824641\n-692048\n643233\n-870910\n-343243\n-1048196\n-493432\n880913\n-410609\n-860324\n287582\n954152\n1032335\n866830\n658609\n478530\n340481\n239843\n168136\n117591\n82147\n57355\n40034\n27941\n19499\n13607\n9496\n6627\n861720\n561458\n1047714\n-588291\n-953715\n-812361\n734365\n-915092\n-287012\n-1046282\n-467913\n869884\n-397706\n-865865\n281028\n952172\n1032911\n868130\n659865\n479532\n341225\n240377\n168514\n117856\n82333\n57485\n40125\n28004\n19543\n13638\n9517\n6642\n-37149\n999255\n906032\n-265329\n-1029408\n-910479\n815715\n-953335\n-229875\n-1042757\n-442044\n858537\n-384733\n-871331\n274462\n950172\n1033477\n869425\n661119\n480534\n341968\n240911\n168891\n118122\n82519\n57615\n40216\n28067\n19587\n13669\n9539\n6657\n-901864\n969876\n553709\n87979\n-1047484\n-983722\n886199\n-985391\n-172010\n-1037626\n-415843\n846877\n-371692\n-876721\n267884\n948152\n1034032\n870717\n662372\n481535\n342712\n241445\n169269\n118388\n82705\n57745\n40307\n28131\n19632\n13700\n9560\n6672\n-937409\n487057\n72655\n431225\n-1006931\n-1030087\n944879\n-1011050\n-113602\n-1030898\n-389331\n834907\n-358584\n-882036\n261295\n946113\n1034578\n872004\n663623\n482536\n343455\n241979\n169647\n118653\n82891\n57875\n40397\n28194\n19676\n13731\n9582\n6687\n-111105\n-223473\n-425291\n725151\n-910019\n-1048309\n990973\n-1030148\n-54835\n-1022582\n-362527\n822633\n-345412\n-887275\n254695\n944055\n1035112\n873287\n664872\n483536\n344198\n242513\n170025\n118918\n83077\n58005\n40488\n28257\n19720\n13762\n9603\n6702\n817349\n-829524\n-824361\n936141\n-762173\n-1037888\n1023866\n-1042560\n4106\n-1012692\n-335450\n810058\n-332180\n-892436\n248085\n941977\n1035637\n874566\n666120\n484536\n344940\n243047\n170403\n119184\n83263\n58135\n40579\n28321\n19764\n13793\n9625\n6717\n994336\n-1047753\n-1031778\n1040063\n-571667\n-999111\n1043121\n-1048205\n63034\n-1001243\n-308123\n797187\n-318888\n-897520\n241464\n939880\n1036151\n875840\n667366\n485535\n345683\n243580\n170781\n119449\n83448\n58265\n40670\n28384\n19809\n13823\n9647\n6732\n257135\n-776131\n-999318\n1025032\n-349163\n-933037\n1048481\n-1047048\n121762\n-988251\n-280564\n784025\n-305539\n-902527\n234832\n937764\n1036655\n877110\n668611\n486533\n346425\n244114\n171159\n119715\n83634\n58394\n40760\n28447\n19853\n13854\n9668\n6747\n-716475\n-141650\n-734529\n892767\n-107117\n-841470\n1039875\n-1039094\n180106\n-973739\n-252795\n770577\n-292136\n-907455\n228191\n935628\n1037148\n878376\n669854\n487531\n347167\n244648\n171536\n119980\n83820\n58524\n40851\n28511\n19897\n13885\n9690\n6762\n-1031361\n559056\n-298970\n658395\n140924\n-726913\n1017418\n-1024397\n237880\n-957727\n-224836\n756848\n-278681\n-912305\n221540\n933473\n1037631\n879638\n671096\n488528\n347909\n245181\n171914\n120246\n84006\n58654\n40942\n28574\n19941\n13916\n9711\n6777\n-398018\n998390\n206095\n348721\n381078\n-592495\n981408\n-1003050\n294902\n-940240\n-196708\n742842\n-265177\n-917076\n214880\n931299\n1038104\n880895\n672335\n489525\n348651\n245715\n172292\n120511\n84192\n58784\n41032\n28637\n19985\n13947\n9733\n6792\n601260\n970953\n663246\n-836\n599903\n-441889\n932325\n-975194\n350992\n-921306\n-168434\n728565\n-251625\n-921767\n208211\n929106\n1038566\n882148\n673574\n490521\n349392\n246248\n172669\n120777\n84378\n58914\n41123\n28701\n20030\n13978\n9754\n6807\n1047743\n489571\n966200\n-350298\n785150\n-279210\n870824\n-941009\n405972\n-900953\n-140032\n714022\n-238029\n-926379\n201533\n926894\n1039018\n883397\n674811\n491517\n350133\n246781\n173047\n121042\n84564\n59044\n41214\n28764\n20074\n14009\n9776\n6822\n530936\n-220696\n1044524\n-659695\n926452\n-108903\n797723\n-900716\n459669\n-879213\n-111526\n699218\n-224390\n-930910\n194846\n924662\n1039460\n884641\n676046\n492512\n350874\n247315\n173425\n121308\n84750\n59174\n41305\n28827\n20118\n14039\n9797\n6837\n-474012\n-827783\n880009\n-893643\n1015899\n64380\n713995\n-854577\n511912\n-856119\n-82936\n684159\n-210711\n-935361\n188152\n922412\n1039891\n885882\n677279\n493507\n351615\n247848\n173802\n121573\n84935\n59304\n41395\n28891\n20162\n14070\n9819\n6852\n-1043155\n-1047861\n510902\n-1025383\n1048485\n235904\n620757\n-802891\n562537\n-831707\n-54284\n668850\n-196995\n-939731\n181449\n920143\n1040311\n887117\n678511\n494501\n352356\n248381\n174180\n121838\n85121\n59434\n41486\n28954\n20206\n14101\n9840\n6867\n-653226\n-778039\n23016\n-1039849\n1022386\n400982\n519250\n-745995\n611384\n-806014\n-25592\n653297\n-183244\n-944019\n174739\n917855\n1040722\n888349\n679741\n495494\n353096\n248914\n174558\n122104\n85307\n59564\n41577\n29017\n20251\n14132\n9862\n6882\n337275\n-144465\n-470220\n-935387\n939063\n555105\n410827\n-684257\n658298\n-779080\n3120\n637505\n-169460\n-948225\n168021\n915548\n1041122\n889576\n680970\n496487\n353837\n249447\n174935\n122369\n85493\n59694\n41668\n29081\n20295\n14163\n9883\n6897\n1017688\n556650\n-854136\n-723943\n803180\n694062\n296931\n-618078\n703130\n-750947\n31830\n621480\n-155647\n-952350\n161296\n913222\n1041511\n890799\n682197\n497479\n354577\n249980\n175313\n122635\n85679\n59824\n41758\n29144\n20339\n14194\n9905\n6912\n762442\n997518\n-1039475\n-429701\n622342\n814056\n179080\n-547887\n745740\n-721657\n60515\n605229\n-141805\n-956392\n154565\n910878\n1041890\n892017\n683423\n498470\n355317\n250513\n175690\n122900\n85865\n59953\n41849\n29208\n20383\n14225\n9926\n6927\n-193789\n972022\n-983149\n-86313\n406671\n911808\n58844\n-474140\n785992\n-691257\n89156\n588756\n-127938\n-960351\n147826\n908514\n1042259\n893232\n684646\n499462\n356056\n251046\n176068\n123165\n86050\n60083\n41940\n29271\n20427\n14255\n9948\n6942\n-971852\n492082\n-698253\n266946\n168238\n984648\n-62176\n-397316\n823759\n-659792\n117729\n572069\n-114049\n-964227\n141082\n906133\n1042617\n894441\n685869\n500452\n356796\n251578\n176445\n123431\n86236\n60213\n42030\n29334\n20472\n14286\n9970\n6957\n-856398\n-217917\n-251020\n589674\n-79612\n1030586\n-182368\n-317914\n858922\n-627311\n146214\n555172\n-100139\n-968020\n134332\n903732\n1042965\n895647\n687089\n501442\n357535\n252111\n176823\n123696\n86422\n60343\n42121\n29398\n20516\n14317\n9991\n6972\n46423\n-826036\n254571\n844961\n-323006\n1048366\n-300131\n-236448\n891370\n-593864\n174590\n538073\n-86212\n-971728\n127576\n901313\n1043302\n896848\n688308\n502431\n358274\n252644\n177200\n123961\n86608\n60473\n42212\n29461\n20560\n14348\n10013\n6987\n906564\n-1047962\n700977\n1003608\n-548320\n1037503\n-413896\n-153447\n920999\n-559503\n202834\n520777\n-72269\n-975353\n120815\n898875\n1043629\n898045\n689526\n503420\n359013\n253176\n177577\n124227\n86794\n60603\n42303\n29524\n20604\n14379\n10034\n7002\n933214\n-779941\n984415\n1047472\n-742945\n998293\n-522148\n-69450\n947717\n-524280\n230927\n503292\n-58313\n-978893\n114048\n896419\n1043946\n899237\n690741\n504408\n359752\n253709\n177955\n124492\n86980\n60733\n42393\n29588\n20649\n14410\n10056\n7017\n101871\n-147279\n1038988\n971534\n-895986\n931809\n-623444\n14997\n971439\n-488250\n258846\n485622\n-44347\n-982348\n107277\n893945\n1044252\n900425\n691955\n505396\n360490\n254241\n178332\n124757\n87165\n60863\n42484\n29651\n20693\n14440\n10077\n7032\n-823131\n554240\n852009\n784482\n-998878\n839866\n-716436\n99347\n992089\n-451468\n286571\n467775\n-30373\n-985719\n100501\n891452\n1044547\n901609\n693168\n506383\n361229\n254773\n178709\n125023\n87351\n60993\n42575\n29714\n20737\n14471\n10099\n7047\n-991351\n996638\n466947\n507707\n-1045861\n724976\n-799885\n183052\n1009603\n-413991\n314082\n449757\n-16393\n-989004\n93721\n888941\n1044832\n902788\n694378\n507369\n361967\n255306\n179087\n125288\n87537\n61123\n42666\n29778\n20781\n14502\n10120\n7062\n-248127\n973084\n-26674\n172865\n-1034307\n590279\n-872679\n265569\n1023926\n-375877\n341356\n431575\n-2411\n-992203\n86937\n886412\n1045107\n903963\n695588\n508355\n362705\n255838\n179464\n125553\n87723\n61252\n42756\n29841\n20825\n14533\n10142\n7077\n723224\n494590\n-514093\n-181748\n-964861\n439454\n-933848\n346362\n1035011\n-337184\n368375\n413236\n11572\n-995316\n80150\n883864\n1045371\n905133\n696795\n509340\n363442\n256370\n179841\n125818\n87909\n61382\n42847\n29904\n20870\n14564\n10163\n7092\n1029645\n-215137\n-881992\n-515574\n-841411\n276622\n-982579\n424908\n1042824\n-297972\n395118\n394745\n25552\n-998343\n73359\n881299\n1045625\n906299\n698001\n510325\n364180\n256902\n180219\n126084\n88095\n61512\n42938\n29968\n20914\n14595\n10185\n7107\n389416\n-824282\n-1044839\n-790433\n-670866\n106233\n-1018221\n500696\n1047340\n-258301\n421564\n376111\n39529\n-1001284\n66565\n878715\n1045869\n907461\n699205\n511309\n364917\n257434\n180596\n126349\n88280\n61642\n43029\n30031\n20958\n14626\n10206\n7122\n-608841\n-1048056\n-964773\n-974889\n-462772\n-67059\n-1040299\n573234\n1048545\n-218232\n447695\n357339\n53498\n-1004138\n59768\n876113\n1046101\n908618\n700408\n512292\n365654\n257966\n180973\n126614\n88466\n61772\n43119\n30094\n21002\n14656\n10228\n7137\n-1047332\n-781838\n-660408\n-1047846\n-228776\n-238518\n-1048521\n642051\n1046435\n-177827\n473489\n338436\n67457\n-1006905\n52968\n873493\n1046324\n909771\n701608\n513275\n366391\n258498\n181350\n126879\n88652\n61902\n43210\n30158\n21046\n14687\n10249\n7152\n-522911\n-150092\n-202507\n-1000960\n18024\n-403461\n-1042776\n706702\n1041017\n-137148\n498929\n319410\n81405\n-1009585\n46167\n870856\n1046536\n910919\n702808\n514258\n367128\n259029\n181727\n127145\n88838\n62032\n43301\n30221\n21091\n14718\n10271\n7167\n482272\n551825\n302475\n-839593\n263816\n-557381\n-1023141\n766765\n1032308\n-96259\n523994\n300267\n95338\n-1012177\n39363\n868200\n1046737\n912063\n704005\n515239\n367864\n259561\n182104\n127410\n89024\n62162\n43391\n30284\n21135\n14749\n10292\n7182\n1044056\n995751\n737135\n-582200\n494842\n-696072\n-989877\n821853\n1020335\n-55221\n548667\n281015\n109254\n-1014682\n32558\n865526\n1046928\n913202\n705201\n516220\n368600\n260093\n182481\n127675\n89209\n62292\n43482\n30348\n21179\n14780\n10314\n7198\n645940\n974139\n1000420\n-258221\n698171\n-815745\n-943427\n871606\n1005137\n-14098\n572928\n261660\n123151\n-1017099\n25751\n862835\n1047109\n914337\n706395\n517201\n369336\n260624\n182858\n127940\n89395\n62422\n43573\n30411\n21223\n14811\n10336\n7213\n-346050\n497094\n1031119\n95292\n862422\n-913130\n-884411\n915702\n986761\n27047\n596759\n242210\n137026\n-1019428\n18944\n860126\n1047279\n915468\n707588\n518181\n370072\n261156\n183235\n128206\n89581\n62551\n43664\n30474\n21267\n14842\n10357\n7228\n-1019884\n-212355\n822096\n437905\n978403\n-985568\n-813614\n953855\n965265\n68150\n620143\n222671\n150876\n-1021668\n12135\n857400\n1047439\n916594\n708778\n519160\n370808\n261687\n183612\n128471\n89767\n62681\n43754\n30538\n21312\n14872\n10379\n7243\n-756041\n-822523\n421944\n730436\n1039621\n-1031077\n-731979\n985817\n940718\n109148\n643062\n203050\n164700\n-1023820\n5326\n854655\n1047588\n917715\n709968\n520139\n371543\n262218\n183989\n128736\n89952\n62811\n43845\n30601\n21356\n14903\n10400\n7258\n202903\n-1048141\n-76304\n939425\n1042651\n-1048416\n-640594\n1011381\n913196\n149978\n665499\n183356\n178494\n-1025884\n-1483\n851894\n1047727\n918833\n711155\n521117\n372279\n262750\n184366\n129001\n90138\n62941\n43936\n30664\n21400\n14934\n10422\n7273\n975298\n-783729\n-556812\n1040971\n987322\n-1037110\n-540676\n1030381\n882788\n190577\n687437\n163595\n192257\n-1027858\n-8292\n849114\n1047855\n919945\n712341\n522094\n373014\n263281\n184743\n129266\n90324\n63071\n44026\n30728\n21444\n14965\n10443\n7288\n851009\n-152904\n-907868\n1023460\n876732\n-997469\n-433556\n1042693\n849588\n230883\n708859\n143774\n205985\n-1029744\n-15101\n846318\n1047972\n921053\n713525\n523071\n373748\n263812\n185120\n129532\n90510\n63201\n44117\n30791\n21488\n14996\n10465\n7303\n-55694\n549407\n-1047856\n888895\n717070\n-930575\n-320661\n1048238\n813703\n270833\n729750\n123900\n219677\n-1031540\n-21909\n843504\n1048080\n922157\n714707\n524048\n374483\n264343\n185497\n129797\n90696\n63331\n44208\n30854\n21533\n15027\n10486\n7318\n-911192\n994857\n-944230\n652666\n517273\n-838256\n-203495\n1046979\n775245\n310366\n750093\n103981\n233330\n-1033247\n-28717\n840672\n1048177\n923256\n715888\n525023\n375217\n264874\n185874\n130062\n90881\n63461\n44299\n30918\n21577\n15058\n10508\n7333\n-928945\n975187\n-621081\n341790\n288524\n-723034\n-83618\n1038926\n734337\n349422\n769874\n84024\n246941\n-1034865\n-35523\n837823\n1048263\n924351\n717067\n525998\n375951\n265405\n186251\n130327\n91067\n63590\n44389\n30981\n21621\n15088\n10529\n7348\n-92630\n499594\n-153539\n-8176\n43626\n-588058\n37373\n1024129\n691107\n387939\n789078\n64037\n260509\n-1036393\n-42327\n834958\n1048339\n925441\n718244\n526973\n376685\n265936\n186628\n130592\n91253\n63720\n44480\n31044\n21665\n15119\n10551\n7363\n828849\n-209571\n349700\n-357207\n-203715\n-437015\n157866\n1002685\n645692\n425859\n807690\n44026\n274030\n-1037831\n-49130\n832075\n1048404\n926527\n719420\n527947\n377419\n266466\n187004\n130857\n91439\n63850\n44571\n31108\n21710\n15150\n10572\n7378\n988287\n-820757\n771637\n-665384\n-439653\n-274032\n276257\n974734\n598235\n463123\n825697\n23999\n287502\n-1039180\n-55931\n829174\n1048459\n927608\n720594\n528920\n378153\n266997\n187381\n131122\n91624\n63980\n44662\n31171\n21754\n15181\n10594\n7393\n239099\n-1048219\n1014178\n-897461\n-650983\n-103562\n390967\n940457\n548887\n499674\n843084\n3963\n300923\n-1040438\n-62729\n826257\n1048503\n928685\n721766\n529893\n378886\n267528\n187758\n131387\n91810\n64110\n44752\n31234\n21798\n15212\n10615\n7408\n-729916\n-785614\n1020935\n-1026893\n-825877\n69737\n500469\n900075\n497804\n535456\n859839\n-16074\n314291\n-1041606\n-69525\n823323\n1048537\n929757\n722936\n530865\n379619\n268058\n188135\n131653\n91996\n64240\n44843\n31298\n21842\n15243\n10637\n7423\n-1027850\n-155715\n790336\n-1038879\n-954546\n241131\n603305\n853852\n445148\n570413\n875950\n-36105\n327602\n-1042684\n-76317\n820372\n1048560\n930825\n724105\n531836\n380352\n268589\n188511\n131918\n92182\n64370\n44934\n31361\n21886\n15274\n10659\n7438\n-380783\n546985\n375994\n-932046\n-1029787\n405937\n698105\n802087\n391084\n604493\n891403\n-56123\n340856\n-1043672\n-83107\n817404\n1048573\n931888\n725272\n532807\n381084\n269119\n188888\n132183\n92367\n64500\n45024\n31424\n21931\n15304\n10680\n7453\n616373\n993956\n-125762\n-718615\n-1047391\n559653\n783606\n745116\n335783\n637641\n906188\n-76121\n354049\n-1044569\n-89893\n814420\n1048576\n932946\n726437\n533777\n381817\n269649\n189265\n132448\n92553\n64630\n45115\n31488\n21975\n15335\n10702\n7468\n1046839\n976228\n-598280\n-422995\n-1006371\n698077\n858669\n683309\n279421\n669807\n920294\n-96091\n367179\n-1045376\n-96675\n811418\n1048568\n934000\n727601\n534747\n382549\n270179\n189641\n132713\n92739\n64759\n45206\n31551\n22019\n15366\n10723\n7483\n514846\n502091\n-931705\n-78996\n-909023\n817429\n922294\n617068\n222176\n700943\n933709\n-116025\n380243\n-1046093\n-103453\n808400\n1048549\n935050\n728763\n535716\n383281\n270710\n190018\n132978\n92925\n64889\n45297\n31614\n22063\n15397\n10745\n7498\n-490494\n-206786\n-1048520\n274038\n-760796\n914447\n973633\n546821\n164228\n730999\n946425\n-135918\n393240\n-1046719\n-110227\n805366\n1048520\n936095\n729923\n536684\n384013\n271240\n190394\n133243\n93110\n65019\n45387\n31678\n22107\n15428\n10766\n7513\n-1044876\n-818986\n-921566\n595729\n-569986\n986481\n1012004\n473025\n105761\n759929\n958431\n-155760\n406167\n-1047254\n-116996\n802315\n1048480\n937135\n731081\n537652\n384745\n271770\n190771\n133508\n93296\n65149\n45478\n31741\n22152\n15459\n10788\n7528\n-638604\n-1048290\n-580359\n849287\n-347274\n1031562\n1036894\n396159\n46960\n787689\n969718\n-175546\n419022\n-1047698\n-123761\n799247\n1048430\n938171\n732238\n538619\n385476\n272300\n191147\n133773\n93482\n65279\n45569\n31804\n22196\n15490\n10809\n7543\n354798\n-787493\n-104226\n1005710\n-105125\n1048460\n1047972\n316722\n-11990\n814237\n980278\n-195268\n431803\n-1048052\n-130520\n796163\n1048370\n939202\n733393\n539585\n386207\n272829\n191524\n134038\n93667\n65409\n45659\n31868\n22240\n15520\n10831\n7558\n1022000\n-158524\n396139\n1047109\n142909\n1036711\n1045091\n235230\n-70902\n839531\n990102\n-214918\n444506\n-1048315\n-137273\n793062\n1048299\n940229\n734546\n540551\n386938\n273359\n191900\n134303\n93853\n65539\n45750\n31931\n22284\n15551\n10852\n7573\n749580\n544558\n804406\n968749\n382944\n996638\n1028289\n152211\n-129590\n863532\n999185\n-234490\n457131\n-1048487\n-144021\n789946\n1048217\n941251\n735698\n541516\n387669\n273889\n192277\n134568\n94039\n65668\n45841\n31994\n22328\n15582\n10874\n7588\n-212000\n993047\n1025659\n779592\n601545\n929335\n997790\n68203\n-187868\n886203\n1007518\n-253977\n469674\n-1048569\n-150763\n786813\n1048125\n942268\n736847\n542481\n388399\n274419\n192653\n134833\n94225\n65798\n45932\n32058\n22373\n15613\n10895\n7603\n-978669\n977262\n1008458\n501272\n786476\n836641\n954000\n-16246\n-245552\n907510\n1015096\n-273370\n482134\n-1048559\n-157498\n783664\n1048023\n943281\n737995\n543445\n389130\n274948\n193030\n135098\n94410\n65928\n46022\n32121\n22417\n15644\n10917\n7618\n-845554\n504584\n756802\n165621\n927388\n721088\n897502\n-100591\n-302460\n927420\n1021912\n-292664\n494508\n-1048459\n-164226\n780499\n1047910\n944290\n739142\n544408\n389860\n275478\n193406\n135363\n94596\n66058\n46113\n32184\n22461\n15675\n10938\n7633\n64960\n-204000\n329199\n-188972\n1016393\n585834\n829049\n-184282\n-358411\n945902\n1027963\n-311851\n506794\n-1048268\n-170948\n777317\n1047787\n945293\n740286\n545371\n390589\n276007\n193782\n135628\n94782\n66188\n46204\n32248\n22505\n15705\n10960\n7648\n915749\n-817208\n-174939\n-521953\n1048510\n434574\n749552\n-266778\n-413230\n962927\n1033242\n-330924\n518990\n-1047987\n-177663\n774120\n1047653\n946292\n741429\n546333\n391319\n276536\n194159\n135893\n94967\n66318\n46294\n32311\n22549\n15736\n10982\n7663\n924603\n-1048352\n-638405\n-795237\n1021940\n271441\n660072\n-347542\n-466742\n978470\n1037747\n-349876\n531093\n-1047614\n-184370\n770907\n1047508\n947287\n742570\n547295\n392048\n277066\n194535\n136158\n95153\n66448\n46385\n32374\n22594\n15767\n11003\n7678\n')
for name, (digest, body) in FILES.items():
    os.makedirs(os.path.dirname(name) or '.', exist_ok=True)
    with open(name, 'w', encoding='utf-8') as f:
        f.write(body)
    assert hashlib.sha256(open(name, 'rb').read()).hexdigest() == digest, name
print('wrote and checked', ', '.join(FILES))

In [ ]:
import time
import numpy as np
import exact_llm as X

t0 = time.time()
ex = X.Exact()
hl, hc, cont_e, lg_e = X.prompt_run(ex, 8, keep=True)
print("exact    logits %s  continuations %s  (%.0f s, %d values clamped)" % (hl, hc, time.time() - t0, sum(ex.clamped.values())))
del ex

In [ ]:
t0 = time.time()
f = X.Float32(X.read_safetensors(X.model_file("model.safetensors")))
fl, fc, cont_f, lg_f = X.prompt_run(f, 8, keep=True)
print("float32  logits %s  continuations %s  (%.0f s)" % (fl, fc, time.time() - t0))
del f

In [ ]:
e = np.concatenate(lg_e).astype(np.float64) / 2.0 ** 16
g = np.concatenate(lg_f).astype(np.float64)
print("over %d prompt positions: top-1 agreement %d; |logit difference| largest %.5f, mean %.6f"
      % (e.shape[0], int((e.argmax(-1) == g.argmax(-1)).sum()), np.abs(e - g).max(), np.abs(e - g).mean()))
print("greedy continuations of 48 tokens identical: %d of %d" % (sum(a == b for a, b in zip(cont_e, cont_f)), len(cont_e)))
tok = X.tokenizer()
dec = {i: t for t, i in tok.vocab.items()}
byte = {c: b for b, c in X._byte_encoder().items()}
text = lambda ids: bytes(byte[c] for i in ids for c in dec[i]).decode("utf-8", "replace")
print("prompt 1: %r -> %r" % (X.prompts()[0], text(cont_e[0])))

In [ ]:
el, ec = X.expected()
print("exact    logits %s  continuations %s  %s" % (hl, hc, "MATCH" if (hl, hc) == (el, ec) else "DIFFERENT"))
print("float32  logits %s  continuations %s  (this machine)" % (fl, fc))